In [2]:
#!/usr/bin/env python3
"""
Clean 'University_comparison_subset_Carla.csv' into a 2-column CSV:
- name (institution_name)
- 2013_program_name

Specifics:
- Drop blank/empty rows
- Split the first column on the FIRST comma only
- Strip whitespace
- Preserve special characters and commas in the program-name column
- Add Unicode normalization for robust matching (hyphens/quotes/spaces)
"""

from pathlib import Path
import unicodedata
import pandas as pd

INPUT_PATH = Path("University_comparison_subset_Carla.csv")
OUTPUT_PATH = Path("University_comparison_subset_Carla_clean.csv")


def normalize_name(s: str) -> str:
    """
    Normalize institution names for matching:
    - Unicode normalize (NFKC) to reduce compatibility variants
    - Standardize hyphen/dash variants to ASCII '-'
    - Standardize curly quotes/apostrophes to ASCII equivalents
    - Convert non-breaking spaces to regular spaces
    - Collapse repeated whitespace
    """
    if s is None:
        return ""

    s = str(s)

    # Unicode normalization
    s = unicodedata.normalize("NFKC", s)

    # Standardize dash/hyphen variants
    dash_chars = {
        "\u2010",  # hyphen
        "\u2011",  # non-breaking hyphen
        "\u2012",  # figure dash
        "\u2013",  # en dash
        "\u2014",  # em dash
        "\u2212",  # minus sign
        "\u00AD",  # soft hyphen
    }
    for ch in dash_chars:
        s = s.replace(ch, "-")

    # Standardize quotes/apostrophes
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    # Standardize spaces
    s = s.replace("\u00A0", " ")  # non-breaking space
    s = " ".join(s.split())       # collapse whitespace

    return s.strip()


def main() -> None:
    df = pd.read_csv(
        INPUT_PATH,
        header=0,              # expects first row header like: "name"
        dtype=str,
        keep_default_na=False,  # keep empty strings as empty strings
    )

    if df.shape[1] < 1:
        raise ValueError(f"{INPUT_PATH} has no columns.")

    raw_col = df.columns[0]

    # Normalize: trim, convert empty strings to NA, drop blank rows
    df[raw_col] = df[raw_col].astype(str).str.strip()
    df.loc[df[raw_col].eq(""), raw_col] = pd.NA
    df = df.dropna(subset=[raw_col]).copy()

    # Split on FIRST comma only (preserve commas in the program name portion)
    parts = df[raw_col].str.split(",", n=1, expand=True)

    # Preserve original name fidelity in 'name' but normalize unicode variants
    df["name"] = parts[0].fillna("").astype(str).map(normalize_name)

    df["2013_program_name"] = ""
    if parts.shape[1] > 1:
        df["2013_program_name"] = parts[1].fillna("").astype(str).str.strip()

    # Normalize whitespace only (do NOT strip commas or special characters)
    df["2013_program_name"] = (
        df["2013_program_name"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Drop any rows that ended up with no institution name after cleaning
    df.loc[df["name"].eq(""), "name"] = pd.NA
    df = df.dropna(subset=["name"]).copy()

    out = df[["name", "2013_program_name"]].reset_index(drop=True)

    out.to_csv(OUTPUT_PATH, index=False)
    print(f"Wrote {len(out):,} rows -> {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

print(pd.read_csv(OUTPUT_PATH).head())
print(pd.read_csv(OUTPUT_PATH).shape)

ModuleNotFoundError: No module named 'pandas'

In [26]:
#!/usr/bin/env python3
"""
Inner-merge with the Carnegie institution list, and also output the Carla rows
that do NOT match any Carnegie 'name' (inverse inner merge).

Inputs:
- ace-institutional-classifications.csv               (Carnegie/ACE institutional metadata; must include column: 'name')
- University_comparison_subset_Carla_clean.csv        (cleaned subset; must include column: 'name')

Outputs:
- Carnegie_Carla_2013_subset.csv                      (inner join on 'name')
- 2013_exclusive_set.csv                              (Carla-only rows not matched in Carnegie)
"""

from pathlib import Path
import pandas as pd

# ----------------------------
# Definitions
# ----------------------------
CARNEGIE_PATH = Path("ace-institutional-classifications.csv")
CARLA_CLEAN_PATH = Path("University_comparison_subset_Carla_clean.csv")

OUTPUT_MERGED_PATH = Path("Carnegie_Carla_2013_subset.csv")
OUTPUT_EXCLUSIVE_PATH = Path("2013_exclusive_set.csv")

JOIN_COL = "name"


def main() -> None:
    carnegie = pd.read_csv(CARNEGIE_PATH, dtype=str, keep_default_na=False)
    carla = pd.read_csv(CARLA_CLEAN_PATH, dtype=str, keep_default_na=False)

    # Basic sanity checks
    for pth, df in [(CARNEGIE_PATH, carnegie), (CARLA_CLEAN_PATH, carla)]:
        if JOIN_COL not in df.columns:
            raise ValueError(f"Missing join column '{JOIN_COL}' in {pth}")

    # Inner merge (matched rows)
    merged = carnegie.merge(carla, on=JOIN_COL, how="inner")
    merged.to_csv(OUTPUT_MERGED_PATH, index=False)
    print(f"Wrote: {OUTPUT_MERGED_PATH} | rows,cols = {merged.shape}")

    # Inverse inner merge: Carla rows that do NOT match Carnegie on JOIN_COL
    carnegie_names = set(carnegie[JOIN_COL].astype(str).str.strip())
    carla_only = carla[~carla[JOIN_COL].astype(str).str.strip().isin(carnegie_names)].copy()
    carla_only.to_csv(OUTPUT_EXCLUSIVE_PATH, index=False)
    print(f"Wrote: {OUTPUT_EXCLUSIVE_PATH} | rows,cols = {carla_only.shape}")


if __name__ == "__main__":
    main()

print(pd.read_csv(OUTPUT_MERGED_PATH).head())
print(pd.read_csv(OUTPUT_MERGED_PATH).shape)
print(pd.read_csv(OUTPUT_EXCLUSIVE_PATH).head())
print(pd.read_csv(OUTPUT_EXCLUSIVE_PATH).shape)

Wrote: Carnegie_Carla_2013_subset.csv | rows,cols = (310, 17)
Wrote: 2013_exclusive_set.csv | rows,cols = (51, 2)
   unitid                 name         city          state  \
0  188429   Adelphi University  Garden City       New York   
1  138600  Agnes Scott College      Decatur        Georgia   
2  168546       Albion College       Albion       Michigan   
3  164465      Amherst College      Amherst  Massachusetts   
4  143084    Augustana College  Rock Island       Illinois   

                  control                       Institutional Classification  \
0  Private not-for-profit  Professions-focused Undergraduate/Graduate-Doc...   
1  Private not-for-profit                   Special Focus: Arts and Sciences   
2  Private not-for-profit                   Special Focus: Arts and Sciences   
3  Private not-for-profit                   Special Focus: Arts and Sciences   
4  Private not-for-profit                                Mixed Baccalaureate   

          Student Access and Ear

In [ ]:
# we have 51 non-matches between these sets. This is due to actual name mismatch (eg Auburn University Main Campus vs Augusta University) - these instances could be matched with partial matches, but would need to be checked anyway due to instances like: Arizona State University vs Arizona State University Campus Immersion AND Arizona State University Digital Immersion. There, there are 2 entries in Carnegie for the 1 in the 2013 set. Would be settled with merging on the uniqueID

In [30]:
#!/usr/bin/env python3
"""
Add institution Web Address to a merged Carnegie subset using NCES IPEDS institution profiles.

For each unitid, fetch:
  https://nces.ed.gov/ipeds/institution-profile/<unitid>

Then extract the "Web Address" value (e.g., www.aamu.edu/) and write a new CSV with a
new column 'Web_address' inserted immediately after 'name'.

NOTE: This version intentionally limits to the FIRST ROW ONLY for testing.
Remove the df = df.head(1) line to run the full dataset.
"""

from pathlib import Path
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ----------------------------
# Definitions
# ----------------------------
INPUT_PATH = Path("Carnegie_Carla_2013_subset.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
NEW_COL = "Web_address"

NCES_PROFILE_URL = "https://nces.ed.gov/ipeds/institution-profile/{}"

REQUEST_TIMEOUT_SEC = 30
SLEEP_BETWEEN_REQUESTS_SEC = 2.0  # be polite; increase if needed

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; IPEDS-scraper/1.0; +https://nces.ed.gov/)",
}


def extract_web_address_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # Normalize helper for label matching
    def norm(txt: str) -> str:
        return re.sub(r"\s+", " ", (txt or "")).strip().lower()

    # Look for any element whose text normalizes to "web address"
    label_el = None
    for el in soup.find_all(string=True):
        if norm(el) == "web address":
            label_el = el.parent
            break

    if label_el:
        # Try: link within the same "row"/container
        container = label_el
        for _ in range(4):  # climb a few levels to a reasonable container
            if container.parent is None:
                break
            container = container.parent

            a = container.find("a", href=True)
            if a:
                val = a.get_text(" ", strip=True)
                if val:
                    return val

        # Try: first link after the label in document order
        a = label_el.find_next("a", href=True)
        if a:
            val = a.get_text(" ", strip=True)
            if val:
                return val

    # Regex fallback: "Web Address" ... <a ...>VALUE</a>
    m = re.search(r"Web\s*Address.*?<a[^>]*>([^<]+)</a>", html, flags=re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip()

    return ""


def get_web_address(unitid: str) -> str:
    url = NCES_PROFILE_URL.format(unitid)
    resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC)
    resp.raise_for_status()
    return extract_web_address_from_html(resp.text)


def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

    if UNITID_COL not in df.columns:
        raise ValueError(f"Missing required column '{UNITID_COL}' in {INPUT_PATH}")
    if NAME_COL not in df.columns:
        raise ValueError(f"Missing required column '{NAME_COL}' in {INPUT_PATH}")

    # TEST MODE: only process the first row (REMOVE THIS LINE for full run)
    df = df.head(1).copy()

    web_addresses = []
    for _, row in df.iterrows():
        unitid = str(row[UNITID_COL]).strip()
        if not unitid:
            web_addresses.append("")
            continue

        try:
            web = get_web_address(unitid)
        except Exception as e:
            print(f"[WARN] unitid={unitid}: failed to fetch/parse web address ({e})")
            web = ""

        web_addresses.append(web)

        # Polite delay between requests
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # Insert new column immediately after 'name'
    df.insert(df.columns.get_loc(NAME_COL) + 1, NEW_COL, web_addresses)

    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Wrote: {OUTPUT_PATH} | rows,cols = {df.shape}")


if __name__ == "__main__":
    main()

print(pd.read_csv(OUTPUT_MERGED_PATH).head())

Wrote: Carnegie_Carla_2013_subset_with_web.csv | rows,cols = (1, 18)
   unitid                 name         city          state  \
0  188429   Adelphi University  Garden City       New York   
1  138600  Agnes Scott College      Decatur        Georgia   
2  168546       Albion College       Albion       Michigan   
3  164465      Amherst College      Amherst  Massachusetts   
4  143084    Augustana College  Rock Island       Illinois   

                  control                       Institutional Classification  \
0  Private not-for-profit  Professions-focused Undergraduate/Graduate-Doc...   
1  Private not-for-profit                   Special Focus: Arts and Sciences   
2  Private not-for-profit                   Special Focus: Arts and Sciences   
3  Private not-for-profit                   Special Focus: Arts and Sciences   
4  Private not-for-profit                                Mixed Baccalaureate   

          Student Access and Earnings Classification  \
0                    

In [32]:
# Debug: find exact format around URLs
import re
import requests

url = "https://nces.ed.gov/ipeds/institution-profile/188429"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text

# Find first occurrence of 'www.' (case-insensitive)
m_www = re.search(r"www\.[^\s<\"']+", html, flags=re.IGNORECASE)
print("FIRST www match:", m_www.group(0) if m_www else None)

# Print raw surrounding context to reveal hidden characters
if m_www:
    i = m_www.start()
    print("\n--- CONTEXT AROUND www (repr) ---")
    print(repr(html[max(0, i-250): i+250]))

# Find "Web Address" and print up to the first www after it
m_wa = re.search(r"Web\s*Address", html, flags=re.IGNORECASE)
print("\nWeb Address found at index:", m_wa.start() if m_wa else None)

if m_wa and m_www and m_www.start() > m_wa.start():
    print("\n--- FROM 'Web Address' TO 'www' (repr) ---")
    print(repr(html[m_wa.start(): m_www.start() + 80]))

# Even simpler: capture minimal text between Web Address and the URL
m = re.search(r"(Web\s*Address.{0,800}?)(www\.[^\s<\"']+)", html, flags=re.IGNORECASE | re.DOTALL)
print("\nRegex capture url:", m.group(2) if m else None)
print("\nBetween Web Address and url (repr):", repr(m.group(1)) if m else None)

FIRST www match: www.googletagmanager.com/gtm.js?id=

--- CONTEXT AROUND www (repr) ---
'ags and analytics) -->\n\t<script>(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({\'gtm.start\':\n\t\tnew Date().getTime(),event:\'gtm.js\'});var f=d.getElementsByTagName(s)[0],\n\t\tj=d.createElement(s),dl=l!=\'dataLayer\'?\'&l=\'+l:\'\';j.async=true;j.src=\n\t\t\'https://www.googletagmanager.com/gtm.js?id=\'+i+dl;f.parentNode.insertBefore(j,f);\n\t\t})(window,document,\'script\',\'dataLayer\',\'GTM-KLQ8LQNJ\');\n\t</script>\n</div>\n\n<!-- Google Tag Manager (noscript) -->\n<noscript><iframe src="https://www.googletagmanager.com/ns.'

Web Address found at index: None

Regex capture url: None

Between Web Address and url (repr): None


In [10]:
# Use selenium to get through js:
#!/usr/bin/env python3
"""
Test extraction of institution website ("Web Address") from an NCES IPEDS profile
page using Selenium-rendered body text and a simple regex for 'www.' domains.

This script is intended as a debugging/validation step before scaling.
"""

from pathlib import Path
import re
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

# ----------------------------
# Definitions
# ----------------------------
UNITID = "188429"  # change as needed for testing
NCES_PROFILE_URL = "https://nces.ed.gov/ipeds/institution-profile/{}"

SLEEP_AFTER_LOAD_SEC = 10
WAIT_TIMEOUT_SEC = 30


def get_web_address_from_body_text(driver) -> str:
    txt = driver.find_element(By.TAG_NAME, "body").text
    m = re.search(r"\bwww\.[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?:/[^\s]*)?", txt)
    return m.group(0) if m else ""


def main() -> None:
    url = NCES_PROFILE_URL.format(UNITID)

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)

        # Give JS time to load/render (requested)
        time.sleep(SLEEP_AFTER_LOAD_SEC)

        # Optional: wait until body has some text
        WebDriverWait(driver, WAIT_TIMEOUT_SEC).until(
            lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
        )

        web = get_web_address_from_body_text(driver)
        print("BODY www match:", web if web else None)

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

BODY www match: www.adelphi.edu/


In [ ]:
# we hit with the simple search after the right js match. Integrate into original below

In [12]:
#!/usr/bin/env python3
"""
Add institution Web Address to a merged Carnegie subset using NCES IPEDS institution profiles.

For each unitid, load (JS-rendered) page via Selenium:
  https://nces.ed.gov/ipeds/institution-profile/<unitid>

Then extract the institution website from the rendered BODY text using a simple regex
for the first 'www.'-style domain (works for cases like: www.adelphi.edu/).

Writes a new CSV with a new column 'Web_address' inserted immediately after 'name'.

NOTE: This version intentionally limits to the FIRST ROW ONLY for testing.
Remove the df = df.head(1) line to run the full dataset.
"""

from pathlib import Path
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

# ----------------------------
# Definitions
# ----------------------------
INPUT_PATH = Path("Carnegie_Carla_2013_subset.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
NEW_COL = "Web_address"

NCES_PROFILE_URL = "https://nces.ed.gov/ipeds/institution-profile/{}"

SLEEP_AFTER_PAGE_LOAD_SEC = 8.0         # allow JS to render
WAIT_TIMEOUT_SEC = 30                   # max wait for body text to appear
SLEEP_BETWEEN_REQUESTS_SEC = 2.0        # polite pacing between institutions

WWW_REGEX = re.compile(r"\bwww\.[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?:/[^\s]*)?", re.IGNORECASE)


def extract_web_address_from_body_text(driver) -> str:
    txt = driver.find_element(By.TAG_NAME, "body").text
    m = WWW_REGEX.search(txt)
    return m.group(0).strip() if m else ""


def make_driver() -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    return webdriver.Chrome(options=options)


def get_web_address(driver, unitid: str) -> str:
    url = NCES_PROFILE_URL.format(unitid)
    driver.get(url)

    # Requested sleep to allow JS content to render
    time.sleep(SLEEP_AFTER_PAGE_LOAD_SEC)

    # Ensure we have some rendered body text before regex extraction
    WebDriverWait(driver, WAIT_TIMEOUT_SEC).until(
        lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
    )

    return extract_web_address_from_body_text(driver)


def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

    if UNITID_COL not in df.columns:
        raise ValueError(f"Missing required column '{UNITID_COL}' in {INPUT_PATH}")
    if NAME_COL not in df.columns:
        raise ValueError(f"Missing required column '{NAME_COL}' in {INPUT_PATH}")

    # TEST MODE: only process the first row (REMOVE THIS LINE for full run)
    #df = df.head(1).copy()

    web_addresses = []
    driver = make_driver()
    try:
        for _, row in df.iterrows():
            unitid = str(row[UNITID_COL]).strip()
            if not unitid:
                web_addresses.append("")
                continue

            try:
                web = get_web_address(driver, unitid)
            except Exception as e:
                print(f"[WARN] unitid={unitid}: failed to fetch/parse web address ({e})")
                web = ""

            web_addresses.append(web)

            # Polite delay between requests
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    finally:
        driver.quit()

    # Insert new column immediately after 'name'
    df.insert(df.columns.get_loc(NAME_COL) + 1, NEW_COL, web_addresses)

    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Wrote: {OUTPUT_PATH} | rows,cols = {df.shape}")


if __name__ == "__main__":
    main()

print(pd.read_csv(OUTPUT_PATH).head())
print(pd.read_csv(OUTPUT_PATH).shape)

Wrote: Carnegie_Carla_2013_subset_with_web.csv | rows,cols = (310, 18)
   unitid                 name          Web_address         city  \
0  188429   Adelphi University     www.adelphi.edu/  Garden City   
1  138600  Agnes Scott College  www.agnesscott.edu/      Decatur   
2  168546       Albion College      www.albion.edu/       Albion   
3  164465      Amherst College     www.amherst.edu/      Amherst   
4  143084    Augustana College   www.augustana.edu/  Rock Island   

           state                 control  \
0       New York  Private not-for-profit   
1        Georgia  Private not-for-profit   
2       Michigan  Private not-for-profit   
3  Massachusetts  Private not-for-profit   
4       Illinois  Private not-for-profit   

                        Institutional Classification  \
0  Professions-focused Undergraduate/Graduate-Doc...   
1                   Special Focus: Arts and Sciences   
2                   Special Focus: Arts and Sciences   
3                   Special Foc

In [ ]:
#missed some where "www" is missing and only a .edu
#need fallback search for .edu
#first eg is California Polytechnic State University-San Luis Obispo -> 	calpoly.edu/

In [16]:
#!/usr/bin/env python3
"""
Add institution Web Address to a merged Carnegie subset using NCES IPEDS institution profiles.

For each unitid, load (JS-rendered) page via Selenium:
  https://nces.ed.gov/ipeds/institution-profile/<unitid>

Extraction strategy (fast + resilient):
1) Regex for first 'www.'-style domain in rendered BODY text
2) Fallback regex for first '.edu' domain even if 'www' is missing

Runtime improvements:
- Reuse a single Selenium driver for all rows
- Replace fixed long sleeps with an explicit wait on BODY text length (shorter when page loads quickly)
- Keep only a small, polite inter-request sleep

NOTE: This version intentionally limits to the FIRST ROW ONLY for testing.
Remove the df = df.head(1) line to run the full dataset.
"""

from pathlib import Path
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

# ----------------------------
# Definitions
# ----------------------------
INPUT_PATH = Path("Carnegie_Carla_2013_subset.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
NEW_COL = "Web_address"

NCES_PROFILE_URL = "https://nces.ed.gov/ipeds/institution-profile/{}"

WAIT_TIMEOUT_SEC = 20                    # explicit wait; reduces need for long sleeps
MIN_SLEEP_AFTER_GET_SEC = 0.5            # tiny pause to let JS kick off
SLEEP_BETWEEN_REQUESTS_SEC = 1.0         # polite pacing

# Primary: explicit www.
WWW_REGEX = re.compile(r"\bwww\.[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?:/[^\s]*)?", re.IGNORECASE)

# Fallback: .edu (allow optional scheme/www; keep minimal trailing path)
# Matches examples like: adelphi.edu/ , www.adelphi.edu/ , https://adelphi.edu
EDU_REGEX = re.compile(
    r"\b(?:https?://)?(?:www\.)?[A-Za-z0-9.-]+\.edu(?:/[^\s]*)?",
    re.IGNORECASE,
)


def make_driver() -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")

    # Small perf wins (generally safe)
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--blink-settings=imagesEnabled=false")  # don't load images
    return webdriver.Chrome(options=options)


def extract_web_address_from_body_text(body_text: str) -> str:
    """
    Extract an institution web address from rendered body text.
    Preference:
      1) first www.<domain>
      2) first <domain>.edu (even without www)
    """
    m = WWW_REGEX.search(body_text)
    if m:
        return m.group(0).strip()

    m = EDU_REGEX.search(body_text)
    if m:
        # Normalize: drop scheme if present to keep consistent style with 'www.adelphi.edu/'
        val = m.group(0).strip()
        val = re.sub(r"^https?://", "", val, flags=re.IGNORECASE)
        return val

    return ""


def get_web_address(driver, unitid: str) -> str:
    url = NCES_PROFILE_URL.format(unitid)
    driver.get(url)

    # Tiny pause to allow initial JS execution (keeps us from polling too aggressively)
    time.sleep(MIN_SLEEP_AFTER_GET_SEC)

    # Wait until body text seems populated; faster than fixed long sleeps
    wait = WebDriverWait(driver, WAIT_TIMEOUT_SEC)
    wait.until(lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 200)

    body_text = driver.find_element(By.TAG_NAME, "body").text
    return extract_web_address_from_body_text(body_text)


def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

    if UNITID_COL not in df.columns:
        raise ValueError(f"Missing required column '{UNITID_COL}' in {INPUT_PATH}")
    if NAME_COL not in df.columns:
        raise ValueError(f"Missing required column '{NAME_COL}' in {INPUT_PATH}")

    # TEST MODE: only process the first row (REMOVE THIS LINE for full run)
    #df = df.head(21).copy()

    web_addresses = []
    driver = make_driver()
    try:
        for _, row in df.iterrows():
            unitid = str(row[UNITID_COL]).strip()
            if not unitid:
                web_addresses.append("")
                continue

            try:
                web = get_web_address(driver, unitid)
            except Exception as e:
                print(f"[WARN] unitid={unitid}: failed to fetch/parse web address ({e})")
                web = ""

            web_addresses.append(web)

            # Polite delay between requests
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    finally:
        driver.quit()

    # Insert new column immediately after 'name'
    df.insert(df.columns.get_loc(NAME_COL) + 1, NEW_COL, web_addresses)

    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Wrote: {OUTPUT_PATH} | rows,cols = {df.shape}")


if __name__ == "__main__":
    main()

print(pd.read_csv(OUTPUT_PATH).head())
print(pd.read_csv(OUTPUT_PATH).shape)

Wrote: Carnegie_Carla_2013_subset_with_web.csv | rows,cols = (310, 18)
   unitid                 name          Web_address         city  \
0  188429   Adelphi University     www.adelphi.edu/  Garden City   
1  138600  Agnes Scott College  www.agnesscott.edu/      Decatur   
2  168546       Albion College      www.albion.edu/       Albion   
3  164465      Amherst College     www.amherst.edu/      Amherst   
4  143084    Augustana College   www.augustana.edu/  Rock Island   

           state                 control  \
0       New York  Private not-for-profit   
1        Georgia  Private not-for-profit   
2       Michigan  Private not-for-profit   
3  Massachusetts  Private not-for-profit   
4       Illinois  Private not-for-profit   

                        Institutional Classification  \
0  Professions-focused Undergraduate/Graduate-Doc...   
1                   Special Focus: Arts and Sciences   
2                   Special Focus: Arts and Sciences   
3                   Special Foc

In [17]:
# only have a couple that don't fill, but they look like they should. Add a fallback with longer js wait when URL not found:

In [18]:
#!/usr/bin/env python3
"""
Add institution Web Address to a merged Carnegie subset using NCES IPEDS institution profiles.

For each unitid, load (JS-rendered) page via Selenium:
  https://nces.ed.gov/ipeds/institution-profile/<unitid>

Extraction strategy:
1) Regex for first 'www.'-style domain in rendered BODY text
2) Fallback regex for first '.edu' domain even if 'www' is missing

Runtime improvements:
- Reuse a single Selenium driver for all rows
- Prefer explicit waits over long sleeps
- Polite inter-request delay

Retry fallback:
- If extraction fails (empty result), retry once with a longer post-load sleep (10s).

NOTE: This version intentionally limits to the FIRST ROW ONLY for testing.
Remove the df = df.head(1) line to run the full dataset.
"""

from pathlib import Path
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

# ----------------------------
# Definitions
# ----------------------------
INPUT_PATH = Path("Carnegie_Carla_2013_subset.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
NEW_COL = "Web_address"

NCES_PROFILE_URL = "https://nces.ed.gov/ipeds/institution-profile/{}"

WAIT_TIMEOUT_SEC = 20
MIN_SLEEP_AFTER_GET_SEC = 0.5            # default fast path
RETRY_SLEEP_AFTER_GET_SEC = 10.0         # retry slow path if empty extraction
SLEEP_BETWEEN_REQUESTS_SEC = 1.0

WWW_REGEX = re.compile(r"\bwww\.[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?:/[^\s]*)?", re.IGNORECASE)
EDU_REGEX = re.compile(
    r"\b(?:https?://)?(?:www\.)?[A-Za-z0-9.-]+\.edu(?:/[^\s]*)?",
    re.IGNORECASE,
)


def make_driver() -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--blink-settings=imagesEnabled=false")
    return webdriver.Chrome(options=options)


def extract_web_address_from_body_text(body_text: str) -> str:
    m = WWW_REGEX.search(body_text)
    if m:
        return m.group(0).strip()

    m = EDU_REGEX.search(body_text)
    if m:
        val = m.group(0).strip()
        val = re.sub(r"^https?://", "", val, flags=re.IGNORECASE)
        return val

    return ""


def _load_and_extract(driver, unitid: str, post_get_sleep_sec: float) -> str:
    url = NCES_PROFILE_URL.format(unitid)
    driver.get(url)
    time.sleep(post_get_sleep_sec)

    wait = WebDriverWait(driver, WAIT_TIMEOUT_SEC)
    wait.until(lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 200)

    body_text = driver.find_element(By.TAG_NAME, "body").text
    return extract_web_address_from_body_text(body_text)


def get_web_address_with_retry(driver, unitid: str) -> str:
    """
    Fast attempt with short sleep; if extraction is empty, retry once with longer sleep.
    """
    web = _load_and_extract(driver, unitid, MIN_SLEEP_AFTER_GET_SEC)
    if web:
        return web

    # Retry slow path (requested)
    return _load_and_extract(driver, unitid, RETRY_SLEEP_AFTER_GET_SEC)


def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

    if UNITID_COL not in df.columns:
        raise ValueError(f"Missing required column '{UNITID_COL}' in {INPUT_PATH}")
    if NAME_COL not in df.columns:
        raise ValueError(f"Missing required column '{NAME_COL}' in {INPUT_PATH}")

    # TEST MODE: only process the first row (REMOVE THIS LINE for full run)
    # df = df.head(1).copy()

    web_addresses = []
    driver = make_driver()
    try:
        for _, row in df.iterrows():
            unitid = str(row[UNITID_COL]).strip()
            if not unitid:
                web_addresses.append("")
                continue

            try:
                web = get_web_address_with_retry(driver, unitid)
            except Exception as e:
                print(f"[WARN] unitid={unitid}: failed to fetch/parse web address ({e})")
                web = ""

            web_addresses.append(web)
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    finally:
        driver.quit()

    df.insert(df.columns.get_loc(NAME_COL) + 1, NEW_COL, web_addresses)

    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Wrote: {OUTPUT_PATH} | rows,cols = {df.shape}")


if __name__ == "__main__":
    main()

print(pd.read_csv(OUTPUT_PATH).head())
print(pd.read_csv(OUTPUT_PATH).shape)

Wrote: Carnegie_Carla_2013_subset_with_web.csv | rows,cols = (310, 18)
   unitid                 name          Web_address         city  \
0  188429   Adelphi University     www.adelphi.edu/  Garden City   
1  138600  Agnes Scott College  www.agnesscott.edu/      Decatur   
2  168546       Albion College      www.albion.edu/       Albion   
3  164465      Amherst College     www.amherst.edu/      Amherst   
4  143084    Augustana College   www.augustana.edu/  Rock Island   

           state                 control  \
0       New York  Private not-for-profit   
1        Georgia  Private not-for-profit   
2       Michigan  Private not-for-profit   
3  Massachusetts  Private not-for-profit   
4       Illinois  Private not-for-profit   

                        Institutional Classification  \
0  Professions-focused Undergraduate/Graduate-Doc...   
1                   Special Focus: Arts and Sciences   
2                   Special Focus: Arts and Sciences   
3                   Special Foc

In [19]:
# great, that catches them all

In [20]:
#!/usr/bin/env python3
"""
Find (best-effort) a 2025–2026 "list of majors/programs" page for each institution,
restricted to that institution's base domain, and then scan that page for:

1) "majors proxy" terms (approximate prefix matches, case-insensitive):
   - art*
   - anthropology*
   - math*
   - bio*

2) If >0 proxy terms found, also scan for Afri* (case-insensitive) and record matches.

Outputs a CSV with:
- unitid, name, Web_address
- majors_url (or 'N/A')
- >0_major_found (1/0)
- majors_found (pipe-separated list of proxy terms found)
- per-term indicator columns (1/0) for each proxy term and afri_found (1/0)
- afri_matches (pipe-separated unique Afri* tokens found)

NOTE: This is a heuristic web-crawl (not a search engine query). It:
- Tries common majors/programs URL paths first (fast)
- Falls back to a shallow, domain-restricted crawl to find pages likely about majors/programs

TEST MODE: df = df.head(N) near the top. Remove/adjust for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ----------------------------
# Definitions / Config
# ----------------------------
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

# Politeness / runtime controls
REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.8  # increase if you see rate limiting
MAX_BYTES_PER_PAGE = 2_000_000    # safety cap

# Crawl fallback controls (domain-restricted)
CRAWL_MAX_PAGES = 25             # per institution
CRAWL_MAX_DEPTH = 2              # from the homepage
CRAWL_LINK_KEYWORDS = ("major", "majors", "program", "programs", "degree", "degrees", "catalog", "academics")

# Term definitions (approximate/prefix style). Case-insensitive.
# We treat these as "proxy" evidence that we're on a majors/programs listing.
MAJOR_PROXY_TERMS = {
    "art": r"\bart[a-z]*\b",
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
}

# Afri* scan (only after majors page is identified)
AFRI_REGEX = r"\bafri[a-z\-]*\b"

# Heuristic: try these paths first (fast). We will join them to the base URL.
COMMON_MAJORS_PATHS = [
    "/academics/majors",
    "/academics/undergraduate/majors",
    "/academics/programs",
    "/academics/programs-and-degrees",
    "/programs",
    "/programs/undergraduate",
    "/degrees",
    "/majors",
    "/undergraduate/majors",
    "/catalog",
    "/catalog/undergraduate",
    "/academic-programs",
    "/academics",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/1.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# TEST MODE: subset near the top
TEST_HEAD_N = 5  # change or set to None for full run


# ----------------------------
# Helpers
# ----------------------------
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    # If it's like "www.adelphi.edu/" -> add https://
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def fetch_html(url: str, session: requests.Session) -> str:
    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()

    # Safety cap on payload size
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    # Best-effort decode
    return content.decode(r.encoding or "utf-8", errors="replace")


def text_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # Remove obvious non-content
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    txt = soup.get_text(separator=" ", strip=True)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


PROXY_PATTERNS = compile_patterns(MAJOR_PROXY_TERMS)
AFRI_PATTERN = re.compile(AFRI_REGEX, flags=re.IGNORECASE)


@dataclass
class PageScanResult:
    url: str
    proxy_hits: Dict[str, int]          # per proxy term
    proxy_terms_found: List[str]        # list of proxy term keys that hit
    afri_matches: List[str]             # unique Afri* tokens found (only for majors page)


def scan_page_for_terms(url: str, session: requests.Session) -> PageScanResult:
    html = fetch_html(url, session=session)
    txt = text_from_html(html)

    proxy_hits: Dict[str, int] = {}
    proxy_terms_found: List[str] = []
    for key, pat in PROXY_PATTERNS.items():
        hit = 1 if pat.search(txt) else 0
        proxy_hits[key] = hit
        if hit:
            proxy_terms_found.append(key)

    # Only compute Afri* matches later when we decide this is a majors page.
    return PageScanResult(
        url=url,
        proxy_hits=proxy_hits,
        proxy_terms_found=proxy_terms_found,
        afri_matches=[],
    )


def extract_links(html: str, base_url: str) -> List[str]:
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", "").strip()
        if not href:
            continue
        abs_url = urljoin(base_url, href)
        abs_url, _frag = urldefrag(abs_url)  # drop #fragment
        links.append(abs_url)
    return links


def shallow_crawl_candidates(base_url: str, session: requests.Session) -> List[str]:
    """
    Domain-restricted shallow crawl from homepage to find likely majors/program pages.
    Returns candidate URLs in discovery order.
    """
    base_netloc = urlparse(base_url).netloc
    seen: Set[str] = set()
    queue: List[Tuple[str, int]] = [(base_url, 0)]
    candidates: List[str] = []

    while queue and len(seen) < CRAWL_MAX_PAGES:
        url, depth = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)

        try:
            html = fetch_html(url, session=session)
        except Exception:
            continue

        # Candidate heuristic: URL contains relevant keywords
        url_l = url.lower()
        if any(k in url_l for k in CRAWL_LINK_KEYWORDS):
            candidates.append(url)

        if depth >= CRAWL_MAX_DEPTH:
            continue

        # Expand links
        for link in extract_links(html, url):
            if link in seen:
                continue
            if not same_domain(link, base_netloc):
                continue

            # Queue links that look relevant earlier, but still allow some exploration
            link_l = link.lower()
            if any(k in link_l for k in CRAWL_LINK_KEYWORDS):
                queue.insert(0, (link, depth + 1))
            else:
                queue.append((link, depth + 1))

        # Politeness between fetches
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # De-dup preserve order
    out = []
    s2 = set()
    for u in candidates:
        if u not in s2:
            s2.add(u)
            out.append(u)
    return out


def choose_majors_page(base_url: str, session: requests.Session) -> PageScanResult | None:
    """
    Find a plausible majors/programs page:
    1) Try common paths first; accept the first page with any proxy hit.
    2) If none, do a shallow crawl and test candidate pages; accept first with any proxy hit.
    Returns PageScanResult or None.
    """
    # Fast path: common URL patterns
    tried: Set[str] = set()
    for path in COMMON_MAJORS_PATHS:
        url = urljoin(base_url, path)
        if url in tried:
            continue
        tried.add(url)
        try:
            res = scan_page_for_terms(url, session=session)
        except Exception:
            continue
        if len(res.proxy_terms_found) > 0:
            return res
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # Slow path: shallow crawl candidates
    for cand in shallow_crawl_candidates(base_url, session=session):
        if cand in tried:
            continue
        tried.add(cand)
        try:
            res = scan_page_for_terms(cand, session=session)
        except Exception:
            continue
        if len(res.proxy_terms_found) > 0:
            return res
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    return None


def add_afri_matches(majors_res: PageScanResult, session: requests.Session) -> PageScanResult:
    """
    Given a majors page already chosen, fetch it again and extract Afri* tokens from body text.
    (We refetch to keep the scan_page_for_terms step small and predictable.)
    """
    try:
        html = fetch_html(majors_res.url, session=session)
        txt = text_from_html(html)
        matches = sorted(set(m.group(0) for m in AFRI_PATTERN.finditer(txt)))
        majors_res.afri_matches = matches
    except Exception:
        majors_res.afri_matches = []
    return majors_res


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()

    # Prepare output rows
    out_rows: List[Dict[str, str]] = []

    for idx, row in df.iterrows():
        unitid = str(row[UNITID_COL]).strip()
        name = str(row[NAME_COL]).strip()
        base = ensure_scheme(str(row[WEB_COL]).strip())

        print(f"[{idx+1}/{len(df)}] unitid={unitid} | {name}")

        majors_url = "N/A"
        proxy_hits = {k: 0 for k in MAJOR_PROXY_TERMS.keys()}
        proxy_terms_found: List[str] = []
        afri_matches: List[str] = []
        afri_found = 0

        if base:
            # Find majors page
            majors_res = None
            try:
                majors_res = choose_majors_page(base, session=session)
            except Exception as e:
                print(f"  [WARN] majors page search failed: {e}")

            if majors_res is not None:
                majors_url = majors_res.url
                proxy_hits = {**proxy_hits, **majors_res.proxy_hits}
                proxy_terms_found = majors_res.proxy_terms_found

                # If majors proxy hit, scan for Afri*
                majors_res = add_afri_matches(majors_res, session=session)
                afri_matches = majors_res.afri_matches
                afri_found = 1 if len(afri_matches) > 0 else 0

        gt0_major_found = 1 if len(proxy_terms_found) > 0 else 0

        # Build output record
        rec: Dict[str, str] = dict(row)  # keep all original columns
        rec["majors_url"] = majors_url
        rec[">0_major_found"] = str(gt0_major_found)
        rec["majors_found"] = "|".join(proxy_terms_found) if proxy_terms_found else ""

        # Add per-term indicator columns
        for term_key in MAJOR_PROXY_TERMS.keys():
            rec[f"found_{term_key}"] = str(proxy_hits.get(term_key, 0))

        rec["afri_found"] = str(afri_found)
        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""

        out_rows.append(rec)

        # Politeness (between institutions)
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    out_df = pd.DataFrame(out_rows)

    # Put majors_url and key scan cols near the front (after name/unitid/web)
    preferred_front = [UNITID_COL, NAME_COL, WEB_COL, "majors_url", ">0_major_found", "majors_found"]
    front_cols = [c for c in preferred_front if c in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in front_cols]
    out_df = out_df[front_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)


if __name__ == "__main__":
    main()

[1/5] unitid=188429 | Adelphi University
[2/5] unitid=138600 | Agnes Scott College
[3/5] unitid=168546 | Albion College
[4/5] unitid=164465 | Amherst College
[5/5] unitid=143084 | Augustana College

Wrote: Carnegie_Carla_2013_subset_majors_scan.csv | rows,cols = (5, 27)
   unitid                 name          Web_address  \
0  188429   Adelphi University     www.adelphi.edu/   
1  138600  Agnes Scott College  www.agnesscott.edu/   
2  168546       Albion College      www.albion.edu/   
3  164465      Amherst College     www.amherst.edu/   
4  143084    Augustana College   www.augustana.edu/   

                                          majors_url >0_major_found  \
0                   https://www.adelphi.edu/programs              1   
1                  https://www.agnesscott.edu/majors              1   
2          https://www.albion.edu/academics/programs              1   
3  https://www.amherst.edu/academiclife/college-c...              1   
4                   https://www.augustana.e

In [ ]:
# good start, but we are missing black/ethinc potential matches. 
# Add:

In [22]:
#!/usr/bin/env python3
"""
Find (best-effort) a 2025–2026 "list of majors/programs" page for each institution,
restricted to that institution's base domain, and then scan that page for:

CONTROL (proxy) terms (approximate prefix matches, case-insensitive):
- art*
- anthropology*
- math*
- bio*

If ANY control term is found, set:
- any_control_found = 1 else 0

If a majors-like page is found (via any_control_found==1), also scan that same page for
case-insensitive approximate matches:
- Afri*
- *ethnic*
- *black*

Then compute:
- any_potential_AfrAmr_found = 1 if any of (afri OR ethnic OR black) matches exist else 0

Outputs a CSV with:
- unitid, name, Web_address
- majors_url (or 'N/A')
- any_control_found
- afri_matches
- ethnic_matches
- black_matches
- any_potential_AfrAmr_found
- plus per-control-term indicator columns (found_art, found_anthropology, found_math, found_bio)

NOTE: This is heuristic web-crawl (not a search engine query).
- Tries common majors/programs URL paths first (fast)
- Falls back to a shallow, domain-restricted crawl to find likely majors/program pages

TEST MODE: df = df.head(N) near the top. Remove/adjust for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import re
import time
import unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ----------------------------
# Definitions / Config
# ----------------------------
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

# Politeness / runtime controls
REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.8  # increase if you see rate limiting
MAX_BYTES_PER_PAGE = 2_000_000    # safety cap

# Crawl fallback controls (domain-restricted)
CRAWL_MAX_PAGES = 25             # per institution
CRAWL_MAX_DEPTH = 2              # from the homepage
CRAWL_LINK_KEYWORDS = ("major", "majors", "program", "programs", "degree", "degrees", "catalog", "academics")

# Term definitions (approximate/prefix style). Case-insensitive.
# These serve as "proxy" evidence we found a majors/program listing page.
CONTROL_TERMS = {
    "art": r"\bart[a-z]*\b",
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
}

# Additional searches on the majors page (case-insensitive)
AFRI_REGEX = r"\bafri[a-z\-]*\b"
ETHNIC_REGEX = r"\bethnic[a-z\-]*\b"
BLACK_REGEX = r"\bblack[a-z\-]*\b"

# Heuristic: try these paths first (fast). We will join them to the base URL.
COMMON_MAJORS_PATHS = [
    "/academics/majors",
    "/academics/undergraduate/majors",
    "/academics/programs",
    "/academics/programs-and-degrees",
    "/programs",
    "/programs/undergraduate",
    "/degrees",
    "/majors",
    "/undergraduate/majors",
    "/catalog",
    "/catalog/undergraduate",
    "/academic-programs",
    "/academics",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/1.1; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# TEST MODE: subset near the top
TEST_HEAD_N = 5  # change or set to None for full run


# ----------------------------
# Unicode/Text normalization helpers
# ----------------------------
def normalize_unicode_text(s: str) -> str:
    """
    Normalize unicode text for more stable matching while preserving meaning.
    - NFKC normalization (compatibility normalization)
    - Standardize hyphen variants to '-'
    - Standardize curly quotes to straight quotes
    - Convert NBSP to normal space
    - Collapse whitespace
    """
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    # Dashes/hyphens
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    # Quotes
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    # Spaces
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ----------------------------
# Helpers
# ----------------------------
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def fetch_html(url: str, session: requests.Session) -> str:
    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()

    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    html = content.decode(r.encoding or "utf-8", errors="replace")
    return html


def text_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    txt = soup.get_text(separator=" ", strip=True)
    txt = normalize_unicode_text(txt)
    return txt


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATTERNS = compile_patterns(CONTROL_TERMS)
AFRI_PATTERN = re.compile(AFRI_REGEX, flags=re.IGNORECASE)
ETHNIC_PATTERN = re.compile(ETHNIC_REGEX, flags=re.IGNORECASE)
BLACK_PATTERN = re.compile(BLACK_REGEX, flags=re.IGNORECASE)


@dataclass
class PageScanResult:
    url: str
    control_hits: Dict[str, int]
    any_control_found: int
    afri_matches: List[str]
    ethnic_matches: List[str]
    black_matches: List[str]
    any_potential_AfrAmr_found: int


def scan_for_controls(url: str, session: requests.Session) -> Tuple[Dict[str, int], int]:
    html = fetch_html(url, session=session)
    txt = text_from_html(html)

    hits: Dict[str, int] = {}
    for key, pat in CONTROL_PATTERNS.items():
        hits[key] = 1 if pat.search(txt) else 0

    any_control = 1 if sum(hits.values()) > 0 else 0
    return hits, any_control


def scan_for_targets(url: str, session: requests.Session) -> Tuple[List[str], List[str], List[str], int]:
    html = fetch_html(url, session=session)
    txt = text_from_html(html)

    afri = sorted(set(m.group(0) for m in AFRI_PATTERN.finditer(txt)))
    ethnic = sorted(set(m.group(0) for m in ETHNIC_PATTERN.finditer(txt)))
    black = sorted(set(m.group(0) for m in BLACK_PATTERN.finditer(txt)))

    any_target = 1 if (len(afri) + len(ethnic) + len(black)) > 0 else 0
    return afri, ethnic, black, any_target


def extract_links(html: str, base_url: str) -> List[str]:
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", "").strip()
        if not href:
            continue
        abs_url = urljoin(base_url, href)
        abs_url, _frag = urldefrag(abs_url)
        links.append(abs_url)
    return links


def shallow_crawl_candidates(base_url: str, session: requests.Session) -> List[str]:
    base_netloc = urlparse(base_url).netloc
    seen: Set[str] = set()
    queue: List[Tuple[str, int]] = [(base_url, 0)]
    candidates: List[str] = []

    while queue and len(seen) < CRAWL_MAX_PAGES:
        url, depth = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)

        try:
            html = fetch_html(url, session=session)
        except Exception:
            continue

        url_l = url.lower()
        if any(k in url_l for k in CRAWL_LINK_KEYWORDS):
            candidates.append(url)

        if depth >= CRAWL_MAX_DEPTH:
            continue

        for link in extract_links(html, url):
            if link in seen:
                continue
            if not same_domain(link, base_netloc):
                continue

            link_l = link.lower()
            if any(k in link_l for k in CRAWL_LINK_KEYWORDS):
                queue.insert(0, (link, depth + 1))
            else:
                queue.append((link, depth + 1))

        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    out = []
    s2 = set()
    for u in candidates:
        if u not in s2:
            s2.add(u)
            out.append(u)
    return out


def choose_majors_page(base_url: str, session: requests.Session) -> Tuple[str, Dict[str, int], int]:
    """
    Returns: (majors_url or "", control_hits, any_control_found)
    Strategy:
      - Try common paths first; accept first with any_control_found==1
      - Else shallow crawl; accept first with any_control_found==1
    """
    tried: Set[str] = set()

    for path in COMMON_MAJORS_PATHS:
        url = urljoin(base_url, path)
        if url in tried:
            continue
        tried.add(url)
        try:
            hits, any_control = scan_for_controls(url, session=session)
        except Exception:
            continue
        if any_control == 1:
            return url, hits, any_control
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    for cand in shallow_crawl_candidates(base_url, session=session):
        if cand in tried:
            continue
        tried.add(cand)
        try:
            hits, any_control = scan_for_controls(cand, session=session)
        except Exception:
            continue
        if any_control == 1:
            return cand, hits, any_control
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    return "", {k: 0 for k in CONTROL_TERMS.keys()}, 0


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for idx, row in df.iterrows():
        unitid = str(row[UNITID_COL]).strip()
        name = str(row[NAME_COL]).strip()
        base = ensure_scheme(str(row[WEB_COL]).strip())

        print(f"[{idx+1}/{len(df)}] unitid={unitid} | {name}")

        majors_url = "N/A"
        control_hits = {k: 0 for k in CONTROL_TERMS.keys()}
        any_control_found = 0

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []
        any_potential = 0

        if base:
            try:
                chosen_url, control_hits, any_control_found = choose_majors_page(base, session=session)
                if chosen_url:
                    majors_url = chosen_url

                if any_control_found == 1 and majors_url != "N/A":
                    afri_matches, ethnic_matches, black_matches, any_potential = scan_for_targets(
                        majors_url, session=session
                    )
            except Exception as e:
                print(f"  [WARN] majors search/scan failed: {e}")

        rec: Dict[str, str] = dict(row)  # keep all original columns
        rec["majors_url"] = majors_url

        # Organization requested: final columns
        rec["any_control_found"] = str(any_control_found)
        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential)

        # Per-control-term indicator columns (each its own column)
        for term_key in CONTROL_TERMS.keys():
            rec[f"found_{term_key}"] = str(control_hits.get(term_key, 0))

        out_rows.append(rec)
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    out_df = pd.DataFrame(out_rows)

    # Put requested "final columns" together near the front after key identifiers
    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL, "majors_url",
        "any_control_found", "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found"
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]

    # Keep control indicator columns next
    control_cols = [f"found_{k}" for k in CONTROL_TERMS.keys() if f"found_{k}" in out_df.columns]

    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)


if __name__ == "__main__":
    main()

[1/5] unitid=188429 | Adelphi University
[2/5] unitid=138600 | Agnes Scott College
[3/5] unitid=168546 | Albion College
[4/5] unitid=164465 | Amherst College
[5/5] unitid=143084 | Augustana College

Wrote: Carnegie_Carla_2013_subset_majors_scan.csv | rows,cols = (5, 28)
   unitid                 name          Web_address  \
0  188429   Adelphi University     www.adelphi.edu/   
1  138600  Agnes Scott College  www.agnesscott.edu/   
2  168546       Albion College      www.albion.edu/   
3  164465      Amherst College     www.amherst.edu/   
4  143084    Augustana College   www.augustana.edu/   

                                          majors_url any_control_found  \
0                   https://www.adelphi.edu/programs                 1   
1                  https://www.agnesscott.edu/majors                 1   
2          https://www.albion.edu/academics/programs                 1   
3  https://www.amherst.edu/academiclife/college-c...                 1   
4                   https://

In [23]:
# "art" is maybe not the best search control (lots of words have "art" in them)

In [ ]:
# not capturing "year" urls: https://www.amherst.edu/academiclife/college-catalog/2526
# not capturing majors as links (need JS awareness): https://www.agnesscott.edu/academics/majors-minors/index.html
# compensate:

In [26]:
#!/usr/bin/env python3
"""
Majors/Catalog discovery + term scanning (2025–26 prioritized)

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv
  Required cols: unitid, name, Web_address

GOAL:
For each institution (domain-restricted), find the best candidate "majors/programs/catalog" page
for the 2025–2026 school year, then scan that page (and optionally 1-hop child pages) for:

A) CONTROL majors/program proxies (case-insensitive):
   - anthropology*
   - math*
   - bio*
   - linguistics*
   - chem* (chemistry/chemical)
   - architect*  (architecture/architectural)

B) STRUCTURAL catalog language (case-insensitive):
   - major(s)
   - minor(s)
   - degree(s)
   - program(s)
   - bachelor / BA / BS / B.S. / B.A. etc.

C) TARGET terms (case-insensitive):
   - Afri*
   - Ethnic*
   - Black*

OUTPUT (per institution):
- majors_url (best candidate or N/A)
- any_control_found (1/0)
- struct_hits (pipe-separated: which structural categories matched, e.g. majors|programs|BA|BS)
- any_struct_found (1/0)
- afri_matches, ethnic_matches, black_matches (pipe-separated unique tokens)
- any_potential_AfrAmr_found (1/0) based on (afri OR ethnic OR black) > 0
- found_<controlterm> columns (1/0) for each control term

Key improvements vs prior version:
1) Year-aware candidate generation + prioritization for 2025–26 (URL + anchor text + page text).
2) Remove ambiguous control term 'art*'; add more precise controls.
3) Anchor-text scanning is first-class (we scan both page text + anchor text).
4) 1-hop link expansion from the majors page (domain-restricted, prioritized) to catch cases where
   the majors list is mostly links to detail pages (e.g., Africana Studies link).

TEST MODE:
- Set TEST_HEAD_N to small N (e.g., 5). Set to None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import re
import time
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ----------------------------
# Config / Definitions
# ----------------------------
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v2.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

# Politeness / runtime
REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.8
MAX_BYTES_PER_PAGE = 2_000_000

# Crawl fallback controls (domain-restricted)
CRAWL_MAX_PAGES = 35         # per institution
CRAWL_MAX_DEPTH = 2
ONE_HOP_MAX_PAGES = 25       # pages to fetch from majors_url outbound links

# URL keywords we care about
PAGE_KEYWORDS = ("major", "majors", "minor", "minors", "program", "programs", "degree", "degrees",
                 "catalog", "catalogue", "bulletin", "academics", "undergraduate")

# Control terms (removed art; added linguistics/chem/architect)
CONTROL_TERMS = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    # safer than chem* alone (avoids "chem" in random contexts a bit)
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
}

# Structural terms treated as their own hit types
STRUCT_PATTERNS = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

# Targets
AFRI_REGEX = r"\bafri[a-z\-]*\b"
ETHNIC_REGEX = r"\bethnic[a-z\-]*\b"
BLACK_REGEX = r"\bblack[a-z\-]*\b"

# 2025–2026 year indicators (URL/text). Prefer these to avoid older catalogs.
YEAR_TOKENS = [
    "2526",
    "20252026",
    "2025-2026",
    "2025–2026",
    "2025—2026",
    "2025_2026",
    "202526",
    "25-26",
    "25–26",
    "25_26",
]
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE
)

# Candidate path templates (fast path)
COMMON_PATHS = [
    "/academics/majors",
    "/academics/undergraduate/majors",
    "/academics/programs",
    "/academics/programs-and-degrees",
    "/programs",
    "/programs/undergraduate",
    "/degrees",
    "/majors",
    "/undergraduate/majors",
    "/academic-programs",
    "/academics",
    "/catalog",
    "/catalog/undergraduate",
    "/catalogue",
    "/bulletin",
    "/academics/catalog",
    "/college-catalog",
    "/course-catalog",
]

# Year-aware templates
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",  # Amherst-like pattern
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/2.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# TEST MODE
TEST_HEAD_N = 5  # set None for full run


# ----------------------------
# Unicode normalization (preserve meaning, stabilize matching)
# ----------------------------
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    # Standardize hyphens/dashes
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    # Quotes
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    # NBSP + collapse whitespace
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ----------------------------
# Fetch + parse helpers
# ----------------------------
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def fetch_html(url: str, session: requests.Session) -> str:
    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]
    return content.decode(r.encoding or "utf-8", errors="replace")


def parse_page(html: str) -> Tuple[str, List[Tuple[str, str]]]:
    """
    Returns:
      - full_text (normalized), including anchor text
      - links: list of (anchor_text_norm, abs_url) where anchor_text_norm is normalized text
    """
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Extract links with anchor text
    links: List[Tuple[str, str]] = []
    # base_url handled outside; here hrefs may still be relative, so caller should urljoin
    for a in soup.find_all("a", href=True):
        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        href = (a.get("href") or "").strip()
        if href:
            links.append((a_text, href))

    text = soup.get_text(separator=" ", strip=True)
    text = normalize_unicode_text(text)
    return text, links


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_PATTERNS)

AFRI_PAT = re.compile(AFRI_REGEX, flags=re.IGNORECASE)
ETHNIC_PAT = re.compile(ETHNIC_REGEX, flags=re.IGNORECASE)
BLACK_PAT = re.compile(BLACK_REGEX, flags=re.IGNORECASE)


@dataclass
class ScanSummary:
    control_hits: Dict[str, int]
    any_control_found: int
    struct_hits: List[str]            # categories matched (majors|minors|...)
    any_struct_found: int
    afri_matches: List[str]
    ethnic_matches: List[str]
    black_matches: List[str]
    any_potential: int


def scan_text(text: str) -> ScanSummary:
    # controls
    control_hits = {k: 1 if pat.search(text) else 0 for k, pat in CONTROL_PATS.items()}
    any_control = 1 if sum(control_hits.values()) > 0 else 0

    # structural
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(text)]
    any_struct = 1 if len(struct_hits) > 0 else 0

    # targets
    afri = sorted(set(m.group(0) for m in AFRI_PAT.finditer(text)))
    ethnic = sorted(set(m.group(0) for m in ETHNIC_PAT.finditer(text)))
    black = sorted(set(m.group(0) for m in BLACK_PAT.finditer(text)))
    any_potential = 1 if (len(afri) + len(ethnic) + len(black)) > 0 else 0

    return ScanSummary(
        control_hits=control_hits,
        any_control_found=any_control,
        struct_hits=struct_hits,
        any_struct_found=any_struct,
        afri_matches=afri,
        ethnic_matches=ethnic,
        black_matches=black,
        any_potential=any_potential,
    )


def score_candidate(url: str, anchor_text_norm: str, page_text_norm: str, summary: ScanSummary) -> int:
    """
    Score candidate pages to prefer:
      - 2025–26 year indicators (URL, anchor text, page text) HIGH priority
      - majors/catalog/program keywords in URL
      - structural hits (majors/minors/programs/degrees/BA/BS)
      - control hits (more likely a majors list)
    """
    score = 0
    u = url.lower()

    # Year priority (avoid older catalogs)
    if YEAR_REGEX.search(u):
        score += 50
    if YEAR_REGEX.search(anchor_text_norm):
        score += 40
    if YEAR_REGEX.search(page_text_norm):
        score += 35

    # Keyword hints
    if any(k in u for k in ("catalog", "catalogue", "bulletin")):
        score += 20
    if any(k in u for k in ("majors", "major", "program", "degree", "academics")):
        score += 10

    # Structural and controls
    score += 5 * len(summary.struct_hits)
    score += 3 * sum(summary.control_hits.values())

    # Slight bump if targets appear (useful)
    if summary.any_potential:
        score += 3

    return score


def build_year_candidates(base_url: str) -> List[str]:
    candidates: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            candidates.append(urljoin(base_url, tpl.format(y=y)))
    # de-dup preserve order
    out, seen = [], set()
    for u in candidates:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def extract_links_abs(html: str, page_url: str) -> List[Tuple[str, str]]:
    """
    Returns list of (anchor_text_norm, abs_url) domain-unfiltered.
    """
    text, links = parse_page(html)
    out = []
    for a_text, href in links:
        abs_url = urljoin(page_url, href)
        abs_url, _frag = urldefrag(abs_url)
        out.append((a_text, abs_url))
    return out


def prioritize_links(links: List[Tuple[str, str]]) -> List[Tuple[str, str]]:
    """
    Sort links by:
      - year indicators in anchor or url first
      - then keyword presence in url
    """
    def link_score(item: Tuple[str, str]) -> int:
        a_text, u = item
        s = 0
        ul = u.lower()
        if YEAR_REGEX.search(ul):
            s += 50
        if YEAR_REGEX.search(a_text):
            s += 40
        if any(k in ul for k in ("catalog", "catalogue", "bulletin")):
            s += 20
        if any(k in ul for k in ("majors", "major", "program", "degree")):
            s += 10
        return -s  # negative for ascending sort

    return sorted(links, key=link_score)


def find_best_majors_page(base_url: str, session: requests.Session) -> Tuple[str, ScanSummary]:
    """
    Try (in order):
      1) Year-aware templates
      2) Common paths
      3) Shallow crawl from homepage (year-aware prioritization on link text + url)
    Choose the BEST scoring page rather than first hit.
    """
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()

    best_url = ""
    best_score = -1
    best_summary = ScanSummary(
        control_hits={k: 0 for k in CONTROL_TERMS},
        any_control_found=0,
        struct_hits=[],
        any_struct_found=0,
        afri_matches=[],
        ethnic_matches=[],
        black_matches=[],
        any_potential=0,
    )

    def consider(url: str, anchor_text_norm: str = "") -> None:
        nonlocal best_url, best_score, best_summary
        if url in tried:
            return
        tried.add(url)
        try:
            html = fetch_html(url, session=session)
            page_text, _links = parse_page(html)
            summary = scan_text(page_text)
            score = score_candidate(url, anchor_text_norm, page_text, summary)
            if score > best_score:
                best_score = score
                best_url = url
                best_summary = summary
        except Exception:
            return
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # 1) Year candidates first
    for url in build_year_candidates(base_url):
        consider(url, anchor_text_norm="")

    # 2) Common path candidates
    for path in COMMON_PATHS:
        consider(urljoin(base_url, path), anchor_text_norm="")

    # 3) Shallow crawl (domain restricted)
    queue: List[Tuple[str, int]] = [(base_url, 0)]
    seen: Set[str] = set()
    pages_fetched = 0

    while queue and pages_fetched < CRAWL_MAX_PAGES:
        url, depth = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)
        if not same_domain(url, base_netloc):
            continue

        try:
            html = fetch_html(url, session=session)
        except Exception:
            continue
        pages_fetched += 1

        # Consider this page itself as a candidate if it looks relevant
        ul = url.lower()
        if any(k in ul for k in PAGE_KEYWORDS) or YEAR_REGEX.search(ul):
            # anchor_text unknown at this stage
            page_text, _links = parse_page(html)
            summary = scan_text(page_text)
            score = score_candidate(url, "", page_text, summary)
            if score > best_score:
                best_score = score
                best_url = url
                best_summary = summary

        if depth >= CRAWL_MAX_DEPTH:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)
            continue

        # Extract and prioritize next links, but only keep domain links
        raw_links = extract_links_abs(html, url)
        dom_links = [(a_text, u) for (a_text, u) in raw_links if same_domain(u, base_netloc)]
        dom_links = prioritize_links(dom_links)

        # Enqueue; push higher priority first
        for a_text, u in dom_links[:50]:
            if u not in seen:
                queue.append((u, depth + 1))

        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # Acceptance: if best_score never improved meaningfully, return N/A
    # We’ll accept if either control OR structural hits exist (broader than controls alone).
    if best_url and (best_summary.any_control_found == 1 or best_summary.any_struct_found == 1):
        return best_url, best_summary

    return "", best_summary


def one_hop_expand(majors_url: str, base_netloc: str, session: requests.Session) -> ScanSummary:
    """
    Fetch majors_url, then follow up to ONE_HOP_MAX_PAGES outgoing links (domain-restricted),
    prioritized by year indicators and majors/program keywords.
    Aggregate target matches across all pages; controls/struct are aggregated via OR.
    """
    agg_control = {k: 0 for k in CONTROL_TERMS}
    agg_struct: Set[str] = set()
    agg_afri: Set[str] = set()
    agg_ethnic: Set[str] = set()
    agg_black: Set[str] = set()

    try:
        html = fetch_html(majors_url, session=session)
    except Exception:
        return scan_text("")

    # Include majors_url itself
    page_text, links = parse_page(html)
    base_summary = scan_text(page_text)

    for k, v in base_summary.control_hits.items():
        agg_control[k] = max(agg_control[k], v)
    for s in base_summary.struct_hits:
        agg_struct.add(s)
    agg_afri.update(base_summary.afri_matches)
    agg_ethnic.update(base_summary.ethnic_matches)
    agg_black.update(base_summary.black_matches)

    # Build absolute, domain-restricted link list
    abs_links = []
    for a_text, href in links:
        u = urljoin(majors_url, href)
        u, _ = urldefrag(u)
        if same_domain(u, base_netloc):
            abs_links.append((a_text, u))

    abs_links = prioritize_links(abs_links)

    visited: Set[str] = {majors_url}
    fetched = 0

    for a_text, u in abs_links:
        if fetched >= ONE_HOP_MAX_PAGES:
            break
        if u in visited:
            continue
        visited.add(u)

        # Only follow links likely to be academic/program related OR that include our target hints
        ul = u.lower()
        a_text_l = a_text.lower()
        if not (
            any(k in ul for k in ("major", "program", "degree", "catalog", "catalogue", "academics"))
            or YEAR_REGEX.search(ul)
            or YEAR_REGEX.search(a_text)
            or re.search(r"\b(afri|black|ethnic)\b", a_text_l)
        ):
            continue

        try:
            html2 = fetch_html(u, session=session)
            txt2, _ = parse_page(html2)
            summ2 = scan_text(txt2)

            for k, v in summ2.control_hits.items():
                agg_control[k] = max(agg_control[k], v)
            for s in summ2.struct_hits:
                agg_struct.add(s)
            agg_afri.update(summ2.afri_matches)
            agg_ethnic.update(summ2.ethnic_matches)
            agg_black.update(summ2.black_matches)

            fetched += 1
        except Exception:
            pass
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    any_control = 1 if sum(agg_control.values()) > 0 else 0
    any_struct = 1 if len(agg_struct) > 0 else 0
    any_potential = 1 if (len(agg_afri) + len(agg_ethnic) + len(agg_black)) > 0 else 0

    return ScanSummary(
        control_hits=agg_control,
        any_control_found=any_control,
        struct_hits=sorted(agg_struct),
        any_struct_found=any_struct,
        afri_matches=sorted(agg_afri),
        ethnic_matches=sorted(agg_ethnic),
        black_matches=sorted(agg_black),
        any_potential=any_potential,
    )


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row[UNITID_COL]).strip()
        name = str(row[NAME_COL]).strip()
        base_url = ensure_scheme(str(row[WEB_COL]).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        majors_url = "N/A"
        summary = ScanSummary(
            control_hits={k: 0 for k in CONTROL_TERMS},
            any_control_found=0,
            struct_hits=[],
            any_struct_found=0,
            afri_matches=[],
            ethnic_matches=[],
            black_matches=[],
            any_potential=0,
        )

        if base_url:
            try:
                best_url, best_summary = find_best_majors_page(base_url, session=session)
                if best_url:
                    majors_url = best_url

                    # 1-hop expansion from majors page (anchor-text driven sites)
                    base_netloc = urlparse(base_url).netloc
                    summary = one_hop_expand(best_url, base_netloc, session=session)
                else:
                    # no acceptable page found
                    summary = best_summary
            except Exception as e:
                print(f"  [WARN] failed on institution: {e}")

        # Build record
        rec: Dict[str, str] = dict(row)
        rec["majors_url"] = majors_url

        # Requested organization columns
        rec["any_control_found"] = str(summary.any_control_found)
        rec["struct_hits"] = "|".join(summary.struct_hits) if summary.struct_hits else ""
        rec[">0_struct_hits_found"] = str(summary.any_struct_found)
        rec["afri_matches"] = "|".join(summary.afri_matches) if summary.afri_matches else ""
        rec["ethnic_matches"] = "|".join(summary.ethnic_matches) if summary.ethnic_matches else ""
        rec["black_matches"] = "|".join(summary.black_matches) if summary.black_matches else ""
        rec["any_potential_AfrAmr_found"] = str(summary.any_potential)

        # Per-control indicator columns
        for k in CONTROL_TERMS:
            rec[f"found_{k}"] = str(summary.control_hits.get(k, 0))

        out_rows.append(rec)

        # Polite delay between institutions
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    out_df = pd.DataFrame(out_rows)

    # Column order: identifiers + majors_url + requested summary columns + controls + rest
    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL, "majors_url",
        "any_control_found", "struct_hits", ">0_struct_hits_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)


if __name__ == "__main__":
    main()

[1/5] unitid=188429 | Adelphi University
[2/5] unitid=138600 | Agnes Scott College
[3/5] unitid=168546 | Albion College
[4/5] unitid=164465 | Amherst College
[5/5] unitid=143084 | Augustana College

Wrote: Carnegie_Carla_2013_subset_majors_scan_v2.csv | rows,cols = (5, 32)
   unitid                 name          Web_address  \
0  188429   Adelphi University     www.adelphi.edu/   
1  138600  Agnes Scott College  www.agnesscott.edu/   
2  168546       Albion College      www.albion.edu/   
3  164465      Amherst College     www.amherst.edu/   
4  143084    Augustana College   www.augustana.edu/   

                                          majors_url any_control_found  \
0              https://www.adelphi.edu/bulletin/2526                 1   
1  https://www.agnesscott.edu/graduate-studies/gr...                 1   
2          https://www.albion.edu/academics/programs                 1   
3  https://www.amherst.edu/academiclife/college-c...                 1   
4  https://www.augustana.

In [ ]:
# run time has increased, doesn't look like I used selenium as a fallback and instead used it for everytime -> fix that and make sure we only make 1 request per page and store text to process.
# grabbed incorrect agnes scott page. Grabbed graduate page, likely due to lots of key words present. Need to prioritize undergrad and adjust scoring metrics.

In [29]:
	# 1.	Speed: two-phase search + early stopping + caching + optional concurrency; Selenium only as fallback when JS required.
	# 2.	Relevance: penalize “graduate” strongly, boost “undergraduate”, and make year tokens dominate; optionally treat catalog discovery as step 1 then majors list as step 2.
	# 3.	Program titles: shift from token matches → structured extraction of title candidates (anchors/headings/list items), confirm via 1-hop to program pages, dedupe + stoplist for non-program entities.

In [30]:
#!/usr/bin/env python3
"""
v3: 2025–26 UNDERGRAD majors/catalog discovery + program-title extraction (fast + precise)

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv
  Required cols: unitid, name, Web_address

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v3.csv

Key upgrades vs v2:
1) Speed: two-phase search with early stopping + URL/HTML disk cache.
2) Relevance: hard preference for 2025–26 pages; penalize graduate pages, boost undergrad pages.
3) Evidence: STRUCTURAL hits (major/minor/degree/program + BA/BS/etc) as first-class signals.
4) Recall: anchor-text scanning + limited 1-hop expansion.
5) Descriptive: extract de-duplicated PROGRAM TITLES (not just tokens) for afri/ethnic/black.

Notes:
- Uses requests/BeautifulSoup by default.
- Optional Selenium fallback ONLY when a page appears JS-rendered/content-thin.
  Set USE_SELENIUM_FALLBACK=True if you have selenium+chromedriver working.

TEST MODE:
- Set TEST_HEAD_N (e.g., 5). Set None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import hashlib
import re
import time
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ============================
# Config
# ============================
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v3.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

# Test mode
TEST_HEAD_N = 5  # set None for full run

# Politeness / runtime
REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.4
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.4
MAX_BYTES_PER_PAGE = 2_000_000

# Cache (big speed win)
CACHE_DIR = Path(".cache_v3_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600  # 14 days

# Two-phase budgets (per institution)
PHASE1_MAX_CANDIDATES = 18
PHASE2_CRAWL_MAX_PAGES = 25
PHASE2_CRAWL_MAX_DEPTH = 2

# 1-hop expansion budgets
ONE_HOP_MAX_LINKS_TOTAL = 25
ONE_HOP_MAX_FETCHES = 12
ONE_HOP_MIN_ANCHOR_LEN = 4

# Optional Selenium fallback
USE_SELENIUM_FALLBACK = False
SELENIUM_MIN_SLEEP_SEC = 0.7
SELENIUM_WAIT_TIMEOUT_SEC = 15

# Heuristic: if requests-extracted text is too short, treat as “maybe JS-rendered”
MIN_TEXT_LEN_FOR_TRUST = 900

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/3.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# ============================
# Term sets / scoring signals
# ============================
YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

# Penalize grad-only pages
GRAD_PENALTY_REGEX = re.compile(
    r"\b(graduate|grad\s+studies|master'?s|ph\.?d|doctoral|mba|m\.?s\.?|m\.?a\.?|mfa|postbac|certificate)\b",
    flags=re.IGNORECASE,
)

# Boost undergrad pages
UNDERGRAD_BOOST_REGEX = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COMMON_PATHS = [
    "/academics/majors-minors",
    "/academics/majors-minors/",
    "/academics/majors",
    "/academics/undergraduate/majors",
    "/academics/programs",
    "/academic-programs",
    "/programs",
    "/undergraduate/majors",
    "/majors",
    "/catalog",
    "/catalogue",
    "/college-catalog",
    "/course-catalog",
    "/bulletin",
]

YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]

PAGE_KEYWORDS = (
    "major", "majors", "minor", "minors", "program", "programs", "degree", "degrees",
    "catalog", "catalogue", "bulletin", "academics", "undergraduate"
)

# ============================
# Unicode normalization
# ============================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ============================
# Pattern compilation
# ============================
def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}

CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)

# ============================
# Cache + fetch
# ============================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.html", CACHE_DIR / f"{key}.meta"

def fetch_html_cached(url: str, session: requests.Session) -> str:
    html_path, meta_path = _cache_paths(url)

    if html_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return html_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]
    html = content.decode(r.encoding or "utf-8", errors="replace")

    try:
        html_path.write_text(html, encoding="utf-8", errors="replace")
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass

    return html

# ============================
# Optional Selenium fallback
# ============================
def selenium_fetch_rendered(url: str) -> Optional[Tuple[str, str]]:
    if not USE_SELENIUM_FALLBACK:
        return None
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
    except Exception:
        return None

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--blink-settings=imagesEnabled=false")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_MIN_SLEEP_SEC)
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT_SEC).until(
            lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
        )
        body_text = driver.find_element(By.TAG_NAME, "body").text
        html = driver.page_source
        return normalize_unicode_text(body_text), html
    except Exception:
        return None
    finally:
        driver.quit()

# ============================
# HTML parse helpers
# ============================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url

def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False

def parse_html_text_and_links(html: str, page_url: str) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    for a in soup.find_all("a", href=True):
        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        href = (a.get("href") or "").strip()
        if href:
            abs_url = urljoin(page_url, href)
            abs_url, _ = urldefrag(abs_url)
            links.append((a_text, abs_url))

    text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    return text, links, title_like

# ============================
# Scanning + scoring
# ============================
@dataclass
class PageSignals:
    url: str
    year_hit: int
    undergrad_boost: int
    grad_penalty: int
    control_hits: Dict[str, int]
    struct_hits: List[str]
    any_control: int
    any_struct: int
    text_len: int

def compute_signals(url: str, text: str) -> PageSignals:
    control_hits = {k: 1 if pat.search(text) else 0 for k, pat in CONTROL_PATS.items()}
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(text)]
    year_hit = 1 if YEAR_REGEX.search(url) or YEAR_REGEX.search(text) else 0
    undergrad_boost = 1 if UNDERGRAD_BOOST_REGEX.search(url) or UNDERGRAD_BOOST_REGEX.search(text) else 0
    grad_penalty = 1 if GRAD_PENALTY_REGEX.search(url) or GRAD_PENALTY_REGEX.search(text) else 0
    any_control = 1 if sum(control_hits.values()) > 0 else 0
    any_struct = 1 if len(struct_hits) > 0 else 0
    return PageSignals(
        url=url,
        year_hit=year_hit,
        undergrad_boost=undergrad_boost,
        grad_penalty=grad_penalty,
        control_hits=control_hits,
        struct_hits=struct_hits,
        any_control=any_control,
        any_struct=any_struct,
        text_len=len(text),
    )

def score_page(sig: PageSignals) -> int:
    score = 0
    if sig.year_hit:
        score += 120
    if sig.undergrad_boost:
        score += 35
    if sig.grad_penalty:
        score -= 80
    score += 10 * len(sig.struct_hits)
    score += 6 * sum(sig.control_hits.values())

    u = sig.url.lower()
    if any(k in u for k in ("catalog", "catalogue", "bulletin")):
        score += 15
    if any(k in u for k in ("majors", "majors-minors", "minor", "program")):
        score += 10

    return score

def is_good_enough(sig: PageSignals) -> bool:
    if sig.grad_penalty:
        return False
    struct_n = len(sig.struct_hits)
    control_n = sum(sig.control_hits.values())

    if sig.year_hit and struct_n >= 2:
        return True
    if sig.undergrad_boost and struct_n >= 2 and control_n >= 1:
        return True
    if sig.year_hit and struct_n >= 1 and ("majors" in sig.struct_hits or "programs" in sig.struct_hits):
        return True
    return False

# ============================
# Candidate generation
# ============================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq

def prioritize_links(links: List[Tuple[str, str]], base_netloc: str) -> List[Tuple[str, str]]:
    def link_score(item: Tuple[str, str]) -> int:
        a_text, u = item
        ul = u.lower()
        at = a_text.lower()
        s = 0
        if not same_domain(u, base_netloc):
            s -= 999
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at):
            s += 100
        if "undergraduate" in ul or "majors-minors" in ul or "majors" in ul:
            s += 35
        if any(k in ul for k in ("catalog", "catalogue", "bulletin")):
            s += 25
        if any(k in ul for k in ("major", "program", "degree", "minor")):
            s += 10
        if "graduate" in ul or "graduate" in at or "grad" in ul:
            s -= 90
        return -s
    return sorted(links, key=link_score)

# ============================
# Fetch + parse (requests first; selenium fallback only if needed)
# ============================
def get_page_text_links_titles(url: str, session: requests.Session) -> Tuple[str, List[Tuple[str, str]], List[str], bool]:
    html = fetch_html_cached(url, session=session)
    text, links, title_like = parse_html_text_and_links(html, page_url=url)
    used_selenium = False

    if USE_SELENIUM_FALLBACK and len(text) < MIN_TEXT_LEN_FOR_TRUST:
        rendered = selenium_fetch_rendered(url)
        if rendered is not None:
            body_text, page_source_html = rendered
            _, links2, title_like2 = parse_html_text_and_links(page_source_html, page_url=url)
            text, links, title_like = body_text, links2, title_like2
            used_selenium = True

    return text, links, title_like, used_selenium

# ============================
# Discovery: two-phase search
# ============================
def find_best_page_for_institution(base_url: str, session: requests.Session) -> Tuple[str, PageSignals, bool]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    used_selenium_any = False

    best_url = ""
    best_sig = compute_signals("", "")

    def consider(url: str) -> Optional[PageSignals]:
        nonlocal best_url, best_sig, used_selenium_any
        if url in tried:
            return None
        tried.add(url)
        try:
            text, links, title_like, used_selenium = get_page_text_links_titles(url, session=session)
            used_selenium_any = used_selenium_any or used_selenium
            sig = compute_signals(url, text)
            if best_url == "" or score_page(sig) > score_page(best_sig):
                best_url, best_sig = url, sig
            return sig
        except Exception:
            return None
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # Phase 1: year candidates
    for url in build_year_candidates(base_url)[:PHASE1_MAX_CANDIDATES]:
        sig = consider(url)
        if sig and is_good_enough(sig):
            return url, sig, used_selenium_any

    # Phase 1: common paths
    for path in COMMON_PATHS:
        url = urljoin(base_url, path)
        sig = consider(url)
        if sig and is_good_enough(sig):
            return url, sig, used_selenium_any

    # Phase 1: homepage links sampling
    try:
        home_text, home_links, _, used_selenium = get_page_text_links_titles(base_url, session=session)
        used_selenium_any = used_selenium_any or used_selenium
        home_links = [(t, u) for (t, u) in home_links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc)
        for a_text, u in home_links[:PHASE1_MAX_CANDIDATES]:
            sig = consider(u)
            if sig and is_good_enough(sig):
                return u, sig, used_selenium_any
    except Exception:
        pass

    # Phase 2: shallow crawl
    queue: List[Tuple[str, int]] = [(base_url, 0)]
    seen: Set[str] = set()
    fetched = 0

    while queue and fetched < PHASE2_CRAWL_MAX_PAGES:
        url, depth = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)
        if not same_domain(url, base_netloc):
            continue

        sig = consider(url)
        fetched += 1
        if sig and is_good_enough(sig):
            return url, sig, used_selenium_any

        if depth >= PHASE2_CRAWL_MAX_DEPTH:
            continue

        try:
            _text, links, _title_like, used_selenium = get_page_text_links_titles(url, session=session)
            used_selenium_any = used_selenium_any or used_selenium
            links = [(t, u) for (t, u) in links if same_domain(u, base_netloc)]
            links = prioritize_links(links, base_netloc)
            for a_text, u in links[:60]:
                if u not in seen:
                    queue.append((u, depth + 1))
        except Exception:
            pass

    return best_url, best_sig, used_selenium_any

# ============================
# Program-title extraction
# ============================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_program_title(s: str) -> bool:
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False
    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        return False
    return TARGET_ANY_REGEX.search(s_norm) is not None

def extract_program_titles_from_page(
    url: str,
    text: str,
    links: List[Tuple[str, str]],
    title_like: List[str],
    base_netloc: str,
) -> Tuple[Set[str], List[Tuple[str, str]]]:
    titles: Set[str] = set()
    follow_links: List[Tuple[str, str]] = []

    for a_text, u in links:
        if not a_text or len(a_text) < ONE_HOP_MIN_ANCHOR_LEN:
            continue
        if not same_domain(u, base_netloc):
            continue

        if looks_like_program_title(a_text):
            titles.add(a_text)

        ul = u.lower()
        at = a_text.lower()
        # Follow links if they smell like program pages and/or include target terms; avoid obvious grad pages
        if "graduate" in ul or "grad" in ul:
            continue
        if TARGET_ANY_REGEX.search(a_text) or TARGET_ANY_REGEX.search(ul):
            follow_links.append((a_text, u))
        elif any(k in ul for k in ("major", "program", "degree", "department")) and TARGET_ANY_REGEX.search(at):
            follow_links.append((a_text, u))

    for t in title_like:
        if looks_like_program_title(t):
            titles.add(t)

    follow_links = prioritize_links(follow_links, base_netloc)
    uniq_links: List[Tuple[str, str]] = []
    seen_u: Set[str] = set()
    for a_text, u in follow_links:
        if u not in seen_u:
            seen_u.add(u)
            uniq_links.append((a_text, u))
    return titles, uniq_links[:ONE_HOP_MAX_LINKS_TOTAL]

def extract_h1_title(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    h1 = soup.find("h1")
    if h1:
        return normalize_unicode_text(h1.get_text(" ", strip=True))
    t = soup.find("title")
    if t:
        return normalize_unicode_text(t.get_text(" ", strip=True))
    return ""

def one_hop_confirm_program_titles(
    majors_url: str,
    base_netloc: str,
    session: requests.Session,
) -> List[str]:
    text, links, title_like, _ = get_page_text_links_titles(majors_url, session=session)
    titles, to_follow = extract_program_titles_from_page(
        majors_url, text=text, links=links, title_like=title_like, base_netloc=base_netloc
    )

    fetches = 0
    for a_text, u in to_follow:
        if fetches >= ONE_HOP_MAX_FETCHES:
            break
        try:
            html = fetch_html_cached(u, session=session)
            h1 = extract_h1_title(html)
            if h1 and looks_like_program_title(h1):
                titles.add(h1)
            fetches += 1
        except Exception:
            continue
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # Dedup by normalized key; keep best display (longer within reason)
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    return sorted(best_display.values())

# ============================
# Target token matches (evidence)
# ============================
def token_matches_from_text(text: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(text)))
    return out

# ============================
# Main
# ============================
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        majors_url = "N/A"
        used_selenium = 0

        any_control_found = 0
        struct_hits: List[str] = []
        any_struct_found = 0
        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0

        if base_url:
            base_netloc = urlparse(base_url).netloc

            best_url, best_sig, used_selenium_any = find_best_page_for_institution(base_url, session=session)
            used_selenium = 1 if used_selenium_any else 0

            if best_url:
                majors_url = best_url

                # fetch/parse best page once (from cache)
                text, links, title_like, _ = get_page_text_links_titles(best_url, session=session)
                sig = compute_signals(best_url, text)

                any_control_found = sig.any_control
                struct_hits = sig.struct_hits
                any_struct_found = sig.any_struct
                for k, v in sig.control_hits.items():
                    control_hit_cols[f"found_{k}"] = int(v)

                tm = token_matches_from_text(text)
                afri_matches = tm["afri"]
                ethnic_matches = tm["ethnic"]
                black_matches = tm["black"]

                # Only do 1-hop if page looks relevant (prevents drift)
                if any_struct_found or any_control_found:
                    program_titles = one_hop_confirm_program_titles(best_url, base_netloc, session=session)
                    program_title_count = len(program_titles)

                any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0

        rec: Dict[str, str] = dict(row)
        rec["majors_url"] = majors_url

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits"] = "|".join(struct_hits) if struct_hits else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["program_title_count"] = str(program_title_count)
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["used_selenium"] = str(used_selenium)

        rec.update({k: str(v) for k, v in control_hit_cols.items()})

        out_rows.append(rec)
        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL, "majors_url",
        "any_control_found", "struct_hits", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "used_selenium",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)

if __name__ == "__main__":
    main()

[1/5] unitid=188429 | Adelphi University
[2/5] unitid=138600 | Agnes Scott College
[3/5] unitid=168546 | Albion College
[4/5] unitid=164465 | Amherst College
[5/5] unitid=143084 | Augustana College

Wrote: Carnegie_Carla_2013_subset_majors_scan_v3.csv | rows,cols = (5, 35)
   unitid                 name          Web_address  \
0  188429   Adelphi University     www.adelphi.edu/   
1  138600  Agnes Scott College  www.agnesscott.edu/   
2  168546       Albion College      www.albion.edu/   
3  164465      Amherst College     www.amherst.edu/   
4  143084    Augustana College   www.augustana.edu/   

                                          majors_url any_control_found  \
0              https://www.adelphi.edu/bulletin/2526                 0   
1  https://www.agnesscott.edu/academics/majors-mi...                 1   
2                            https://www.albion.edu/                 0   
3  https://www.amherst.edu/academiclife/college-c...                 1   
4                   https

In [ ]:
# missing some pages, probably to high scoring structured pages
# still missing text when in links
# 1.	URL-tiered candidate ordering + homepage penalty
# 2.	bulletin demotion unless it leads to undergrad inventory
# 3.	anchor/aria-label corpus scanning + selective Selenium trigger

In [35]:
#!/usr/bin/env python3
"""
v4: 2025–26 UNDERGRAD majors/program discovery (inventory-first) + program-title extraction

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv
  Required cols: unitid, name, Web_address

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v4.csv

What changed vs v3 (to fix Albion / Agnes Scott / Adelphi issues):
1) INVENTORY-FIRST candidate ordering:
   - Prefer /academics/programs, /programs, /majors-minors, /majors before catalogs/bulletins/homepage.
2) Tiered scoring (page-type aware):
   - Inventory pages > Catalog pages > Bulletin pages > Homepage.
   - Bulletins demoted unless they look like *undergrad* inventory.
   - Homepage penalized so it won’t beat true inventory pages.
3) Stronger early-stop rules:
   - Only early-stop on inventory pages (or year+undergrad catalog pages) with sufficient structural evidence.
4) Better text corpus for matching (fixes “majors are links”):
   - Search corpus = visible page text + anchor text + aria-label/title attributes.
   - Selective Selenium fallback only when inventory-ish pages look JS/thin under requests.

Notes:
- requests/BeautifulSoup by default.
- Optional Selenium fallback ONLY when needed and USE_SELENIUM_FALLBACK=True.

TEST MODE:
- Set TEST_HEAD_N (e.g., 5). Set None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import hashlib
import re
import time
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================
# Config
# ============================
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v4.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

TEST_HEAD_N = 5  # set None for full run

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.35
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.35
MAX_BYTES_PER_PAGE = 2_000_000

# Cache (big speed win)
CACHE_DIR = Path(".cache_v4_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Two-phase budgets (per institution)
PHASE1_MAX_INVENTORY_CANDIDATES = 10
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 15

PHASE2_CRAWL_MAX_PAGES = 25
PHASE2_CRAWL_MAX_DEPTH = 2

# 1-hop expansion budgets
ONE_HOP_MAX_LINKS_TOTAL = 30
ONE_HOP_MAX_FETCHES = 14
ONE_HOP_MIN_ANCHOR_LEN = 4

# Optional Selenium fallback (only for thin/JS inventory-like pages)
USE_SELENIUM_FALLBACK = False
SELENIUM_MIN_SLEEP_SEC = 0.9
SELENIUM_WAIT_TIMEOUT_SEC = 15

# Heuristics for "thin" / maybe JS
MIN_VISIBLE_TEXT_LEN = 800
MIN_ANCHOR_CORPUS_LEN = 250

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/4.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================
# Prioritized URL templates
# ============================
INVENTORY_PATHS = [
    "/academics/programs",
    "/academics/programs/",
    "/programs",
    "/programs/",
    "/academics/majors-minors",
    "/academics/majors-minors/",
    "/majors-minors",
    "/majors-minors/",
    "/academics/majors",
    "/academics/majors/",
    "/majors",
    "/majors/",
    "/undergraduate/majors",
    "/undergraduate/majors/",
]

# Catalog/bulletin last-resort candidates (still useful for year pages)
CATALOG_PATHS = [
    "/college-catalog",
    "/college-catalog/",
    "/course-catalog",
    "/course-catalog/",
    "/catalog",
    "/catalog/",
    "/catalogue",
    "/catalogue/",
    "/bulletin",
    "/bulletin/",
    "/academics/catalog",
    "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",  # Amherst-like
]


# ============================
# Term sets / scoring signals
# ============================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

# Penalize grad-only pages hard
GRAD_PENALTY_REGEX = re.compile(
    r"\b(graduate|grad\s+studies|master'?s|ph\.?d|doctoral|mba|m\.?s\.?|m\.?a\.?|mfa|postbac|certificate)\b",
    flags=re.IGNORECASE,
)

# Boost undergrad pages
UNDERGRAD_BOOST_REGEX = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

# Control majors/program proxies (no art; include linguistics/chem/architect)
CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
}

# Structural hits treated separately (first-class signals)
STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

# Target token matches (evidence)
TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

# Program-title extraction: reject common non-program contexts
TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

# URL-tier indicators
INVENTORY_URL_HINTS = ("/programs", "/academics/programs", "/majors-minors", "/majors", "/undergraduate/majors")
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")
BULLETIN_URL_HINTS = ("bulletin",)


# ============================
# Unicode normalization
# ============================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ============================
# Pattern compilation
# ============================
def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}

CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================
# Cache + fetch
# ============================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.html", CACHE_DIR / f"{key}.meta"

def fetch_html_cached(url: str, session: requests.Session) -> str:
    html_path, meta_path = _cache_paths(url)

    if html_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return html_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]
    html = content.decode(r.encoding or "utf-8", errors="replace")

    try:
        html_path.write_text(html, encoding="utf-8", errors="replace")
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass

    return html


# ============================
# Optional Selenium fallback
# ============================
def selenium_fetch_rendered(url: str) -> Optional[Tuple[str, str]]:
    """
    Returns (rendered_body_text, page_source_html) or None if selenium unavailable/fails.
    Only used when USE_SELENIUM_FALLBACK=True.
    """
    if not USE_SELENIUM_FALLBACK:
        return None
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
    except Exception:
        return None

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--blink-settings=imagesEnabled=false")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_MIN_SLEEP_SEC)
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT_SEC).until(
            lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
        )
        body_text = normalize_unicode_text(driver.find_element(By.TAG_NAME, "body").text)
        html = driver.page_source
        return body_text, html
    except Exception:
        return None
    finally:
        driver.quit()


# ============================
# HTML parse helpers
# ============================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url

def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False

@dataclass
class ParsedPage:
    url: str
    visible_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]       # (anchor_text_norm, abs_url)
    title_like: List[str]              # h1/h2/h3/li text candidates
    corpus: str                        # search corpus: visible + anchors + aria/title attrs


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    # Search corpus (fixes “majors are links” false negatives)
    corpus = " ".join(
        x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x
    )
    corpus = normalize_unicode_text(corpus)

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        corpus=corpus,
    )


def get_parsed_page(url: str, session: requests.Session, allow_selenium: bool) -> ParsedPage:
    """
    requests first; selectively use selenium only if:
      - allow_selenium True
      - page looks inventory-ish
      - requests corpus looks thin (likely JS-rendered)
    """
    html = fetch_html_cached(url, session=session)
    parsed = parse_html_to_parsedpage(html, page_url=url)

    if allow_selenium and USE_SELENIUM_FALLBACK:
        ul = url.lower()
        is_inventoryish = any(h in ul for h in INVENTORY_URL_HINTS)
        thin = (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) or (len(" ".join(parsed.anchor_texts)) < MIN_ANCHOR_CORPUS_LEN)
        if is_inventoryish and thin:
            rendered = selenium_fetch_rendered(url)
            if rendered is not None:
                body_text, page_source_html = rendered
                parsed2 = parse_html_to_parsedpage(page_source_html, page_url=url)
                # preserve rendered body text as visible_text, but keep parsed links/attrs from HTML
                parsed2.visible_text = body_text
                parsed2.corpus = normalize_unicode_text(
                    " ".join([body_text, " ".join(parsed2.anchor_texts), " ".join(parsed2.anchor_attr_texts)])
                )
                return parsed2

    return parsed


# ============================
# Scanning + scoring (tiered)
# ============================
@dataclass
class PageSignals:
    url: str
    tier: str                    # inventory|catalog|bulletin|homepage|other
    year_hit: int
    undergrad_boost: int
    grad_penalty: int
    control_hits: Dict[str, int]
    struct_hits: List[str]
    any_control: int
    any_struct: int
    visible_len: int
    anchor_len: int

def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if any(h in u for h in INVENTORY_URL_HINTS):
        return "inventory"
    if any(h in u for h in BULLETIN_URL_HINTS):
        return "bulletin"
    if any(h in u for h in CATALOG_URL_HINTS):
        return "catalog"
    return "other"

def compute_signals(url: str, parsed: ParsedPage) -> PageSignals:
    corpus = parsed.corpus

    control_hits = {k: 1 if pat.search(corpus) else 0 for k, pat in CONTROL_PATS.items()}
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(corpus)]

    year_hit = 1 if YEAR_REGEX.search(url) or YEAR_REGEX.search(corpus) else 0
    undergrad_boost = 1 if UNDERGRAD_BOOST_REGEX.search(url) or UNDERGRAD_BOOST_REGEX.search(corpus) else 0
    grad_penalty = 1 if GRAD_PENALTY_REGEX.search(url) or GRAD_PENALTY_REGEX.search(corpus) else 0

    any_control = 1 if sum(control_hits.values()) > 0 else 0
    any_struct = 1 if len(struct_hits) > 0 else 0

    return PageSignals(
        url=url,
        tier=page_tier(url),
        year_hit=year_hit,
        undergrad_boost=undergrad_boost,
        grad_penalty=grad_penalty,
        control_hits=control_hits,
        struct_hits=struct_hits,
        any_control=any_control,
        any_struct=any_struct,
        visible_len=len(parsed.visible_text),
        anchor_len=len(" ".join(parsed.anchor_texts)),
    )

def score_page(sig: PageSignals) -> int:
    """
    Tiered scoring:
      inventory > catalog > bulletin > homepage.
    Year boost matters most in catalog tier, but never lets bulletin/homepage beat inventory.
    """
    score = 0

    # Tier bases
    if sig.tier == "inventory":
        score += 200
    elif sig.tier == "catalog":
        score += 120
    elif sig.tier == "bulletin":
        score += 60
    elif sig.tier == "homepage":
        score += 10
    else:
        score += 40

    # Year preference (dominant within catalog; helpful elsewhere)
    if sig.year_hit:
        score += 120 if sig.tier in ("catalog", "bulletin") else 40

    # Undergrad boost
    if sig.undergrad_boost:
        score += 45

    # Graduate penalty (strong)
    if sig.grad_penalty:
        score -= 120

    # Structural + control evidence
    score += 14 * len(sig.struct_hits)
    score += 7 * sum(sig.control_hits.values())

    # Demote homepages unless very strong
    if sig.tier == "homepage":
        score -= 40

    # Demote bulletins unless they also look like undergrad inventory
    if sig.tier == "bulletin":
        # If it doesn't look undergrad-ish + inventory-ish, push down hard
        inv_like = ("programs" in sig.struct_hits) or ("majors" in sig.struct_hits) or sig.undergrad_boost
        if not inv_like:
            score -= 80

    return score

def is_good_enough(sig: PageSignals) -> bool:
    """
    Early-stop only on:
    - inventory pages with strong structure and not grad
    - catalog pages that are year+undergrad with strong structure and not grad
    """
    if sig.grad_penalty:
        return False

    struct_n = len(sig.struct_hits)
    control_n = sum(sig.control_hits.values())

    if sig.tier == "inventory":
        return (struct_n >= 2) and (control_n >= 1 or sig.undergrad_boost == 1)

    if sig.tier in ("catalog", "bulletin"):
        return (sig.year_hit == 1) and (sig.undergrad_boost == 1) and (struct_n >= 2)

    return False


# ============================
# Candidate generation + link prioritization
# ============================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))
    # de-dup preserve order
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq

def prioritize_links(links: List[Tuple[str, str]], base_netloc: str) -> List[Tuple[str, str]]:
    """
    Prefer inventory-like and undergrad pages; demote graduate/bulletin drift.
    """
    def link_rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        ul = u.lower()
        at = (a_text or "").lower()
        s = 0

        if not same_domain(u, base_netloc):
            s -= 10_000

        # Inventory first
        if any(h in ul for h in INVENTORY_URL_HINTS):
            s += 220

        # Year & catalog hints (but below inventory)
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at):
            s += 120

        if "undergraduate" in ul or "majors-minors" in ul or "/programs" in ul or "/majors" in ul:
            s += 80

        if any(k in ul for k in ("catalog", "catalogue")):
            s += 40

        # Demotions
        if "bulletin" in ul:
            s -= 60
        if "graduate" in ul or "graduate" in at or "grad" in ul:
            s -= 140

        # Tiny/no anchor text is less helpful
        if len(a_text or "") < 3:
            s -= 10

        return -s  # ascending sort key

    return sorted(links, key=link_rank)


# ============================
# Discovery: inventory-first, tiered, early-stop
# ============================
def find_best_page_for_institution(base_url: str, session: requests.Session) -> Tuple[str, PageSignals, ParsedPage, bool]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    used_selenium_any = False

    best_url = ""
    best_sig = PageSignals(
        url="",
        tier="other",
        year_hit=0,
        undergrad_boost=0,
        grad_penalty=0,
        control_hits={k: 0 for k in CONTROL_TERMS},
        struct_hits=[],
        any_control=0,
        any_struct=0,
        visible_len=0,
        anchor_len=0,
    )
    best_parsed = ParsedPage(url="", visible_text="", anchor_texts=[], anchor_attr_texts=[], links=[], title_like=[], corpus="")

    def consider(url: str, allow_selenium: bool) -> Optional[Tuple[PageSignals, ParsedPage]]:
        nonlocal best_url, best_sig, best_parsed, used_selenium_any
        if url in tried:
            return None
        tried.add(url)

        try:
            parsed = get_parsed_page(url, session=session, allow_selenium=allow_selenium)
            if USE_SELENIUM_FALLBACK and allow_selenium:
                # heuristic: if corpus got "bigger", selenium might have been used (we don't track explicitly)
                pass
            sig = compute_signals(url, parsed)
            if best_url == "" or score_page(sig) > score_page(best_sig):
                best_url, best_sig, best_parsed = url, sig, parsed
            return sig, parsed
        except Exception:
            return None
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # ---- Phase 1A: inventory paths (highest priority)
    inv_candidates = [urljoin(base_url, p) for p in INVENTORY_PATHS]
    for url in inv_candidates[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        got = consider(url, allow_selenium=True)
        if got and is_good_enough(got[0]):
            return url, got[0], got[1], used_selenium_any

    # ---- Phase 1B: year-aware catalog templates (next)
    year_candidates = build_year_candidates(base_url)
    for url in year_candidates[:PHASE1_MAX_CATALOG_CANDIDATES]:
        got = consider(url, allow_selenium=True)
        if got and is_good_enough(got[0]):
            return url, got[0], got[1], used_selenium_any

    # ---- Phase 1C: generic catalog paths (last in phase 1)
    cat_candidates = [urljoin(base_url, p) for p in CATALOG_PATHS]
    for url in cat_candidates[:PHASE1_MAX_CATALOG_CANDIDATES]:
        got = consider(url, allow_selenium=False)
        if got and is_good_enough(got[0]):
            return url, got[0], got[1], used_selenium_any

    # ---- Phase 1D: homepage link sampling (careful: don’t let it drift to grad/bulletin)
    try:
        home_parsed = get_parsed_page(base_url, session=session, allow_selenium=False)
        home_links = [(t, u) for (t, u) in home_parsed.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc)

        for a_text, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            got = consider(u, allow_selenium=False)
            if got and is_good_enough(got[0]):
                return u, got[0], got[1], used_selenium_any
    except Exception:
        pass

    # ---- Phase 2: shallow crawl (fallback)
    queue: List[Tuple[str, int]] = [(base_url, 0)]
    seen: Set[str] = set()
    fetched = 0

    while queue and fetched < PHASE2_CRAWL_MAX_PAGES:
        url, depth = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)
        if not same_domain(url, base_netloc):
            continue

        got = consider(url, allow_selenium=False)
        fetched += 1
        if got and is_good_enough(got[0]):
            return url, got[0], got[1], used_selenium_any

        if depth >= PHASE2_CRAWL_MAX_DEPTH:
            continue

        # expand links, prioritized
        try:
            parsed = get_parsed_page(url, session=session, allow_selenium=False)
            links = [(t, u) for (t, u) in parsed.links if same_domain(u, base_netloc)]
            links = prioritize_links(links, base_netloc)
            for a_text, u in links[:70]:
                if u not in seen:
                    queue.append((u, depth + 1))
        except Exception:
            pass

    return best_url, best_sig, best_parsed, used_selenium_any


# ============================
# Program-title extraction (descriptive + dedup)
# ============================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_program_title(s: str) -> bool:
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False
    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        return False
    return TARGET_ANY_REGEX.search(s_norm) is not None

def extract_program_titles_from_parsedpage(parsed: ParsedPage, base_netloc: str) -> Tuple[Set[str], List[Tuple[str, str]]]:
    titles: Set[str] = set()
    follow_links: List[Tuple[str, str]] = []

    # From anchor text + aria/title attrs
    for txt in parsed.anchor_texts + parsed.anchor_attr_texts:
        if txt and looks_like_program_title(txt):
            titles.add(txt)

    # From headings/list items
    for t in parsed.title_like:
        if looks_like_program_title(t):
            titles.add(t)

    # Choose 1-hop links to follow (target-ish, inventory-ish, undergrad-ish; avoid grad/bulletin drift)
    for a_text, u in parsed.links:
        if not u or not same_domain(u, base_netloc):
            continue
        ul = u.lower()
        if "graduate" in ul or "grad" in ul:
            continue
        # follow if the anchor itself looks like a program title
        if looks_like_program_title(a_text):
            follow_links.append((a_text, u))
            continue
        # follow if the URL has target tokens (rare) or looks like a program page
        if TARGET_ANY_REGEX.search(ul) or any(k in ul for k in ("program", "major", "degree", "department")):
            follow_links.append((a_text, u))

    follow_links = prioritize_links(follow_links, base_netloc)

    # De-dup links
    uniq_links: List[Tuple[str, str]] = []
    seen_u: Set[str] = set()
    for a_text, u in follow_links[:ONE_HOP_MAX_LINKS_TOTAL]:
        if u not in seen_u:
            seen_u.add(u)
            uniq_links.append((a_text, u))

    return titles, uniq_links

def extract_h1_or_title(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    h1 = soup.find("h1")
    if h1:
        return normalize_unicode_text(h1.get_text(" ", strip=True))
    t = soup.find("title")
    if t:
        return normalize_unicode_text(t.get_text(" ", strip=True))
    return ""

def one_hop_confirm_program_titles(majors_url: str, parsed_majors: ParsedPage, base_netloc: str, session: requests.Session) -> List[str]:
    titles, to_follow = extract_program_titles_from_parsedpage(parsed_majors, base_netloc=base_netloc)

    fetches = 0
    for a_text, u in to_follow:
        if fetches >= ONE_HOP_MAX_FETCHES:
            break
        try:
            html = fetch_html_cached(u, session=session)
            t = extract_h1_or_title(html)
            if t and looks_like_program_title(t):
                titles.add(t)
            fetches += 1
        except Exception:
            continue
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # De-dup by normalized key; keep best display (longer within reason)
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    return sorted(best_display.values())


# ============================
# Target token matches (evidence)
# ============================
def token_matches_from_corpus(corpus: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(corpus)))
    return out


# ============================
# Main
# ============================
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        majors_url = "N/A"
        used_selenium = 0

        any_control_found = 0
        struct_hits: List[str] = []
        any_struct_found = 0
        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0

        if base_url:
            base_netloc = urlparse(base_url).netloc

            best_url, best_sig, best_parsed, used_selenium_any = find_best_page_for_institution(base_url, session=session)
            used_selenium = 1 if used_selenium_any else 0

            if best_url:
                majors_url = best_url

                # Signals from best_parsed
                any_control_found = best_sig.any_control
                struct_hits = best_sig.struct_hits
                any_struct_found = best_sig.any_struct
                for k, v in best_sig.control_hits.items():
                    control_hit_cols[f"found_{k}"] = int(v)

                tm = token_matches_from_corpus(best_parsed.corpus)
                afri_matches = tm["afri"]
                ethnic_matches = tm["ethnic"]
                black_matches = tm["black"]

                # Program titles: only when page looks relevant (prevents drift)
                if any_struct_found or any_control_found:
                    program_titles = one_hop_confirm_program_titles(
                        majors_url, parsed_majors=best_parsed, base_netloc=base_netloc, session=session
                    )
                    program_title_count = len(program_titles)

                any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0

        rec: Dict[str, str] = dict(row)
        rec["majors_url"] = majors_url

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits"] = "|".join(struct_hits) if struct_hits else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["program_title_count"] = str(program_title_count)
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["used_selenium"] = str(used_selenium)

        rec.update({k: str(v) for k, v in control_hit_cols.items()})

        out_rows.append(rec)
        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL, "majors_url",
        "any_control_found", "struct_hits", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "used_selenium",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)

if __name__ == "__main__":
    main()

[1/5] unitid=188429 | Adelphi University
[2/5] unitid=138600 | Agnes Scott College
[3/5] unitid=168546 | Albion College
[4/5] unitid=164465 | Amherst College


KeyboardInterrupt: 

In [33]:
# still not getting link text 
# check selenium backup logic and perhaps implement site index fallback for JS

In [36]:
#!/usr/bin/env python3
"""
v5: v4 + (A) sitemap-first discovery for JS-driven majors indexes, then (B) Selenium fallback (only if enabled)

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv
  Required cols: unitid, name, Web_address

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v5.csv

What changed vs v4:
1) If an INVENTORY-ish page (esp. /majors-minors) looks "thin" under requests (likely JS-rendered),
   we try a SITEMAP-FIRST fallback:
     - fetch /sitemap.xml (and sitemap indexes) for the domain
     - extract URLs matching likely inventory/program pages (esp. /academics/majors-minors/*/index.html)
     - evaluate those URLs as candidates (often finds Africana Studies pages without Selenium)
2) If sitemap fallback produces nothing useful and USE_SELENIUM_FALLBACK=True, then we retry that inventory
   page with Selenium to render link text.

Notes:
- requests/BeautifulSoup by default.
- Selenium is OFF by default. If you have selenium+chromedriver working, set USE_SELENIUM_FALLBACK=True.

TEST MODE:
- Set TEST_HEAD_N (e.g., 5). Set None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import hashlib
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================
# Config
# ============================
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v5.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

TEST_HEAD_N = 25  # set None for full run

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.35
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.35
MAX_BYTES_PER_PAGE = 2_000_000

# Cache
CACHE_DIR = Path(".cache_v5_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Two-phase budgets (per institution)
PHASE1_MAX_INVENTORY_CANDIDATES = 10
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 15

PHASE2_CRAWL_MAX_PAGES = 25
PHASE2_CRAWL_MAX_DEPTH = 2

# 1-hop expansion budgets
ONE_HOP_MAX_LINKS_TOTAL = 30
ONE_HOP_MAX_FETCHES = 14
ONE_HOP_MIN_ANCHOR_LEN = 4

# Sitemap budgets
SITEMAP_MAX_FETCHES = 4           # sitemap.xml + up to N nested sitemap files
SITEMAP_MAX_URLS = 1500           # cap total URLs parsed (per institution)
SITEMAP_CANDIDATE_CAP = 120       # cap candidate URLs considered (per institution)

# Optional Selenium fallback (only after sitemap-first)
USE_SELENIUM_FALLBACK = False
SELENIUM_MIN_SLEEP_SEC = 0.9
SELENIUM_WAIT_TIMEOUT_SEC = 15

# Heuristics for "thin" / maybe JS
MIN_VISIBLE_TEXT_LEN = 800
MIN_ANCHOR_CORPUS_LEN = 250

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/5.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================
# Prioritized URL templates
# ============================
INVENTORY_PATHS = [
    "/academics/programs",
    "/academics/programs/",
    "/programs",
    "/programs/",
    "/academics/majors-minors",
    "/academics/majors-minors/",
    "/majors-minors",
    "/majors-minors/",
    "/academics/majors",
    "/academics/majors/",
    "/majors",
    "/majors/",
    "/undergraduate/majors",
    "/undergraduate/majors/",
]

CATALOG_PATHS = [
    "/college-catalog",
    "/college-catalog/",
    "/course-catalog",
    "/course-catalog/",
    "/catalog",
    "/catalog/",
    "/catalogue",
    "/catalogue/",
    "/bulletin",
    "/bulletin/",
    "/academics/catalog",
    "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================
# Term sets / scoring signals
# ============================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_PENALTY_REGEX = re.compile(
    r"\b(graduate|grad\s+studies|master'?s|ph\.?d|doctoral|mba|m\.?s\.?|m\.?a\.?|mfa|postbac|certificate)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_BOOST_REGEX = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = ("/programs", "/academics/programs", "/majors-minors", "/majors", "/undergraduate/majors")
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")
BULLETIN_URL_HINTS = ("bulletin",)

# Sitemap candidate patterns (keep this focused; add more if needed)
SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================
# Unicode normalization
# ============================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ============================
# Pattern compilation
# ============================
def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}

CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================
# Cache + fetch
# ============================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.html", CACHE_DIR / f"{key}.meta"

def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    html_path, meta_path = _cache_paths(url)

    if html_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return html_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        html_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass

    return content

def fetch_text_cached(url: str, session: requests.Session) -> str:
    content = fetch_bytes_cached(url, session=session)
    # best-effort decode
    try:
        return content.decode("utf-8", errors="replace")
    except Exception:
        return content.decode("latin-1", errors="replace")


# ============================
# Optional Selenium fallback
# ============================
def selenium_fetch_rendered(url: str) -> Optional[Tuple[str, str]]:
    """
    Returns (rendered_body_text, page_source_html) or None.
    Only used when USE_SELENIUM_FALLBACK=True.
    """
    if not USE_SELENIUM_FALLBACK:
        return None
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
    except Exception:
        return None

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--blink-settings=imagesEnabled=false")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_MIN_SLEEP_SEC)
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT_SEC).until(
            lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
        )
        body_text = normalize_unicode_text(driver.find_element(By.TAG_NAME, "body").text)
        html = driver.page_source
        return body_text, html
    except Exception:
        return None
    finally:
        driver.quit()


# ============================
# HTML parse helpers
# ============================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url

def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False

@dataclass
class ParsedPage:
    url: str
    visible_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]       # (anchor_text_norm, abs_url)
    title_like: List[str]              # h1/h2/h3/li candidates
    corpus: str                        # visible + anchors + aria/title attrs


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus = " ".join(
        x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x
    )
    corpus = normalize_unicode_text(corpus)

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        corpus=corpus,
    )


def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)


def looks_thin_for_inventory(parsed: ParsedPage) -> bool:
    anchor_len = len(" ".join(parsed.anchor_texts))
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) or (anchor_len < MIN_ANCHOR_CORPUS_LEN)


def get_parsed_page_requests_only(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)


def get_parsed_page_with_optional_selenium(url: str, session: requests.Session, allow_selenium: bool) -> ParsedPage:
    """
    requests first. Selenium is only attempted if allow_selenium=True AND USE_SELENIUM_FALLBACK=True AND
    the page is inventory-ish and looks thin (likely JS).
    """
    parsed = get_parsed_page_requests_only(url, session=session)

    if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url) and looks_thin_for_inventory(parsed):
        rendered = selenium_fetch_rendered(url)
        if rendered is not None:
            body_text, page_source_html = rendered
            parsed2 = parse_html_to_parsedpage(page_source_html, page_url=url)
            parsed2.visible_text = body_text
            parsed2.corpus = normalize_unicode_text(
                " ".join([body_text, " ".join(parsed2.anchor_texts), " ".join(parsed2.anchor_attr_texts)])
            )
            return parsed2

    return parsed


# ============================
# Sitemap parsing (sitemap-first fallback)
# ============================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()

def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    """
    Fetch /sitemap.xml and possibly nested sitemap indexes.
    Returns a de-duplicated list of URLs (same domain).
    """
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        # sitemapindex -> contains <sitemap><loc>
        if root_tag == "sitemapindex":
            for sm in root.findall(".//"):
                if _strip_ns(sm.tag).lower() == "loc":
                    loc = _xml_text(sm)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        # urlset -> contains <url><loc>
        if root_tag == "urlset":
            for loc_el in root.findall(".//"):
                if _strip_ns(loc_el.tag).lower() == "loc":
                    loc = _xml_text(loc_el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls


def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    """
    Filter sitemap URLs down to likely program/majors pages.
    """
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()
        # quick exclude
        if "graduate" in ul or "/it-" in ul or "/news" in ul or "/event" in ul:
            continue
        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)
    # de-dup preserve order
    seen: Set[str] = set()
    out: List[str] = []
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break
    return out


# ============================
# Scanning + scoring (tiered)
# ============================
@dataclass
class PageSignals:
    url: str
    tier: str
    year_hit: int
    undergrad_boost: int
    grad_penalty: int
    control_hits: Dict[str, int]
    struct_hits: List[str]
    any_control: int
    any_struct: int
    visible_len: int
    anchor_len: int

def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if any(h in u for h in INVENTORY_URL_HINTS):
        return "inventory"
    if any(h in u for h in BULLETIN_URL_HINTS):
        return "bulletin"
    if any(h in u for h in CATALOG_URL_HINTS):
        return "catalog"
    return "other"

def compute_signals(url: str, parsed: ParsedPage) -> PageSignals:
    corpus = parsed.corpus

    control_hits = {k: 1 if pat.search(corpus) else 0 for k, pat in CONTROL_PATS.items()}
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(corpus)]

    year_hit = 1 if YEAR_REGEX.search(url) or YEAR_REGEX.search(corpus) else 0
    undergrad_boost = 1 if UNDERGRAD_BOOST_REGEX.search(url) or UNDERGRAD_BOOST_REGEX.search(corpus) else 0
    grad_penalty = 1 if GRAD_PENALTY_REGEX.search(url) or GRAD_PENALTY_REGEX.search(corpus) else 0

    any_control = 1 if sum(control_hits.values()) > 0 else 0
    any_struct = 1 if len(struct_hits) > 0 else 0

    return PageSignals(
        url=url,
        tier=page_tier(url),
        year_hit=year_hit,
        undergrad_boost=undergrad_boost,
        grad_penalty=grad_penalty,
        control_hits=control_hits,
        struct_hits=struct_hits,
        any_control=any_control,
        any_struct=any_struct,
        visible_len=len(parsed.visible_text),
        anchor_len=len(" ".join(parsed.anchor_texts)),
    )

def score_page(sig: PageSignals) -> int:
    score = 0
    if sig.tier == "inventory":
        score += 200
    elif sig.tier == "catalog":
        score += 120
    elif sig.tier == "bulletin":
        score += 60
    elif sig.tier == "homepage":
        score += 10
    else:
        score += 40

    if sig.year_hit:
        score += 120 if sig.tier in ("catalog", "bulletin") else 40
    if sig.undergrad_boost:
        score += 45
    if sig.grad_penalty:
        score -= 120

    score += 14 * len(sig.struct_hits)
    score += 7 * sum(sig.control_hits.values())

    if sig.tier == "homepage":
        score -= 40

    if sig.tier == "bulletin":
        inv_like = ("programs" in sig.struct_hits) or ("majors" in sig.struct_hits) or sig.undergrad_boost
        if not inv_like:
            score -= 80

    return score

def is_good_enough(sig: PageSignals) -> bool:
    if sig.grad_penalty:
        return False

    struct_n = len(sig.struct_hits)
    control_n = sum(sig.control_hits.values())

    if sig.tier == "inventory":
        return (struct_n >= 2) and (control_n >= 1 or sig.undergrad_boost == 1)

    if sig.tier in ("catalog", "bulletin"):
        return (sig.year_hit == 1) and (sig.undergrad_boost == 1) and (struct_n >= 2)

    return False


# ============================
# Link prioritization
# ============================
def prioritize_links(links: List[Tuple[str, str]], base_netloc: str) -> List[Tuple[str, str]]:
    def link_rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        ul = u.lower()
        at = (a_text or "").lower()
        s = 0

        if not same_domain(u, base_netloc):
            s -= 10_000

        if any(h in ul for h in INVENTORY_URL_HINTS):
            s += 220

        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at):
            s += 120

        if "undergraduate" in ul or "majors-minors" in ul or "/programs" in ul or "/majors" in ul:
            s += 80

        if any(k in ul for k in ("catalog", "catalogue")):
            s += 40

        if "bulletin" in ul:
            s -= 60
        if "graduate" in ul or "graduate" in at or "grad" in ul:
            s -= 140

        if len(a_text or "") < 3:
            s -= 10

        return -s

    return sorted(links, key=link_rank)


# ============================
# Candidate generation
# ============================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


# ============================
# Discovery: inventory-first + sitemap-first on thin inventory, then selenium fallback
# ============================
def find_best_page_for_institution(base_url: str, session: requests.Session) -> Tuple[str, PageSignals, ParsedPage, int]:
    """
    Returns: (best_url, best_sig, best_parsed, used_selenium_flag)
    """
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    used_selenium_flag = 0

    best_url = ""
    best_sig = PageSignals(
        url="",
        tier="other",
        year_hit=0,
        undergrad_boost=0,
        grad_penalty=0,
        control_hits={k: 0 for k in CONTROL_TERMS},
        struct_hits=[],
        any_control=0,
        any_struct=0,
        visible_len=0,
        anchor_len=0,
    )
    best_parsed = ParsedPage(url="", visible_text="", anchor_texts=[], anchor_attr_texts=[], links=[], title_like=[], corpus="")

    # cache sitemap per institution (lazy-loaded)
    sitemap_urls_cache: Optional[List[str]] = None
    sitemap_candidates_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_urls_cache, sitemap_candidates_cache
        if sitemap_candidates_cache is not None:
            return sitemap_candidates_cache
        sitemap_urls_cache = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_candidates_cache = sitemap_candidate_urls(sitemap_urls_cache)
        return sitemap_candidates_cache

    def consider(url: str, allow_selenium: bool) -> Optional[Tuple[PageSignals, ParsedPage]]:
        nonlocal best_url, best_sig, best_parsed, used_selenium_flag
        if url in tried:
            return None
        tried.add(url)

        try:
            parsed = get_parsed_page_with_optional_selenium(url, session=session, allow_selenium=allow_selenium)
            # Detect selenium usage approximately: if allow_selenium and still thin under requests,
            # the selenium fetch would typically increase anchor/visible content; we just flag on allow_selenium here.
            if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url):
                # If page was thin, selenium was attempted; even if it failed, it's okay to mark.
                used_selenium_flag = 1 if used_selenium_flag == 0 else used_selenium_flag

            sig = compute_signals(url, parsed)
            if best_url == "" or score_page(sig) > score_page(best_sig):
                best_url, best_sig, best_parsed = url, sig, parsed

            return sig, parsed
        except Exception:
            return None
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # ---- Phase 1A: inventory paths (requests first)
    inv_candidates = [urljoin(base_url, p) for p in INVENTORY_PATHS]
    for url in inv_candidates[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        got = consider(url, allow_selenium=False)
        if got:
            sig, parsed = got
            if is_good_enough(sig):
                return url, sig, parsed, used_selenium_flag

            # If this looks like a JS-driven majors index (thin), do sitemap-first, then selenium fallback.
            if "majors-minors" in url.lower() and looks_thin_for_inventory(parsed):
                # 1) sitemap-first
                for u2 in load_sitemap_candidates():
                    got2 = consider(u2, allow_selenium=False)
                    if got2 and is_good_enough(got2[0]):
                        return u2, got2[0], got2[1], used_selenium_flag

                # 2) selenium fallback for the original inventory page (if enabled)
                if USE_SELENIUM_FALLBACK:
                    got3 = consider(url, allow_selenium=True)
                    if got3 and is_good_enough(got3[0]):
                        return url, got3[0], got3[1], used_selenium_flag

    # ---- Phase 1B: year-aware catalog templates
    for url in build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES]:
        got = consider(url, allow_selenium=False)
        if got and is_good_enough(got[0]):
            return url, got[0], got[1], used_selenium_flag

    # ---- Phase 1C: generic catalog paths
    cat_candidates = [urljoin(base_url, p) for p in CATALOG_PATHS]
    for url in cat_candidates[:PHASE1_MAX_CATALOG_CANDIDATES]:
        got = consider(url, allow_selenium=False)
        if got and is_good_enough(got[0]):
            return url, got[0], got[1], used_selenium_flag

    # ---- Phase 1D: homepage link sampling
    try:
        home_parsed = get_parsed_page_requests_only(base_url, session=session)
        home_links = [(t, u) for (t, u) in home_parsed.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc)

        for a_text, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            got = consider(u, allow_selenium=False)
            if got and is_good_enough(got[0]):
                return u, got[0], got[1], used_selenium_flag
    except Exception:
        pass

    # ---- Phase 2: shallow crawl
    queue: List[Tuple[str, int]] = [(base_url, 0)]
    seen: Set[str] = set()
    fetched = 0

    while queue and fetched < PHASE2_CRAWL_MAX_PAGES:
        url, depth = queue.pop(0)
        if url in seen:
            continue
        seen.add(url)
        if not same_domain(url, base_netloc):
            continue

        got = consider(url, allow_selenium=False)
        fetched += 1
        if got and is_good_enough(got[0]):
            return url, got[0], got[1], used_selenium_flag

        if depth >= PHASE2_CRAWL_MAX_DEPTH:
            continue

        try:
            parsed = get_parsed_page_requests_only(url, session=session)
            links = [(t, u) for (t, u) in parsed.links if same_domain(u, base_netloc)]
            links = prioritize_links(links, base_netloc)
            for a_text, u in links[:70]:
                if u not in seen:
                    queue.append((u, depth + 1))
        except Exception:
            pass

    return best_url, best_sig, best_parsed, used_selenium_flag


# ============================
# Program-title extraction (sitemap helps when majors index has no link text)
# ============================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_program_title(s: str) -> bool:
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False
    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        return False
    return TARGET_ANY_REGEX.search(s_norm) is not None

def extract_h1_or_title(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    h1 = soup.find("h1")
    if h1:
        return normalize_unicode_text(h1.get_text(" ", strip=True))
    t = soup.find("title")
    if t:
        return normalize_unicode_text(t.get_text(" ", strip=True))
    return ""

def one_hop_confirm_program_titles(
    majors_url: str,
    parsed_majors: ParsedPage,
    base_url: str,
    base_netloc: str,
    session: requests.Session,
) -> List[str]:
    titles: Set[str] = set()
    follow_urls: List[str] = []

    # From anchor text + aria/title attrs
    for txt in parsed_majors.anchor_texts + parsed_majors.anchor_attr_texts + parsed_majors.title_like:
        if txt and looks_like_program_title(txt):
            titles.add(txt)

    # Candidate follow links (from parsed page)
    for a_text, u in parsed_majors.links:
        if not u or not same_domain(u, base_netloc):
            continue
        ul = u.lower()
        if "graduate" in ul or "grad" in ul:
            continue
        if looks_like_program_title(a_text) or TARGET_ANY_REGEX.search(ul):
            follow_urls.append(u)

    # If the majors-minors index is JS-thin and gave us no useful follow links, use sitemap
    if (not follow_urls) and ("majors-minors" in majors_url.lower()):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        cands = sitemap_candidate_urls(all_urls)
        # Prefer target-y program pages first
        target_first = []
        other = []
        for u in cands:
            if TARGET_ANY_REGEX.search(u):
                target_first.append(u)
            else:
                other.append(u)
        follow_urls = target_first + other

    # De-dup follow URLs
    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    # Fetch a small number of follow URLs and extract canonical titles
    fetches = 0
    for u in uniq_follow:
        if fetches >= ONE_HOP_MAX_FETCHES:
            break
        try:
            html = fetch_text_cached(u, session=session)
            t = extract_h1_or_title(html)
            if t and looks_like_program_title(t):
                titles.add(t)
            fetches += 1
        except Exception:
            continue
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # De-dup by normalized key; keep best display
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    return sorted(best_display.values())


# ============================
# Target token matches (evidence)
# ============================
def token_matches_from_corpus(corpus: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(corpus)))
    return out


# ============================
# Main
# ============================
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        majors_url = "N/A"
        used_selenium = 0

        any_control_found = 0
        struct_hits: List[str] = []
        any_struct_found = 0
        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0

        if base_url:
            base_netloc = urlparse(base_url).netloc

            best_url, best_sig, best_parsed, used_selenium = find_best_page_for_institution(
                base_url, session=session
            )

            if best_url:
                majors_url = best_url

                any_control_found = best_sig.any_control
                struct_hits = best_sig.struct_hits
                any_struct_found = best_sig.any_struct
                for k, v in best_sig.control_hits.items():
                    control_hit_cols[f"found_{k}"] = int(v)

                tm = token_matches_from_corpus(best_parsed.corpus)
                afri_matches = tm["afri"]
                ethnic_matches = tm["ethnic"]
                black_matches = tm["black"]

                if any_struct_found or any_control_found:
                    program_titles = one_hop_confirm_program_titles(
                        majors_url=majors_url,
                        parsed_majors=best_parsed,
                        base_url=base_url,
                        base_netloc=base_netloc,
                        session=session,
                    )
                    program_title_count = len(program_titles)

                any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0

        rec: Dict[str, str] = dict(row)
        rec["majors_url"] = majors_url

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits"] = "|".join(struct_hits) if struct_hits else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["program_title_count"] = str(program_title_count)
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["used_selenium"] = str(used_selenium)

        rec.update({k: str(v) for k, v in control_hit_cols.items()})

        out_rows.append(rec)
        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL, "majors_url",
        "any_control_found", "struct_hits", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "used_selenium",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)

if __name__ == "__main__":
    main()

[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

This is a good test case. We now have new instances where we do not find the correct majors page:
1) We find the page https://www.barnard.edu/majors-minors where we should have found https://barnard.edu/departments-and-programs.
2) We find https://www.barry.edu/majors-minors where we should have found https://www.barry.edu/en/find-your-program (where there is in-site navigation needed to find undergraduate programs).
3) we find https://www.boisestate.edu/news/2025/12/20/university-marks-117th-commencement-ceremonies/ where we should have found https://majors.boisestate.edu/
4) we find https://www.bowdoin.edu/admissions/index.html where we should have found https://www.bowdoin.edu/academics/departments-and-programs/index.html

There are many more instances where we failed to find the correct majors/programs page, but this is a good evaluation set. Some of these indicate we should adjust our scoring schema to de prioritize "2025" and similar slightly and allow for "majors"/"programs"/etc to be before our base url in addition to after it (point 3 above). Others of these seem as though they did not look deep enough or had non-standard organization. 

In addition to adding to our non-standard site organization regex search, I think we want to allow for deeper searching of each site to find the right page, and also scrape multiple high-scoring pages for our search terms. While we will lose speed, hopefully this will increase sensitivity as positives from our search or control set are highly indicative that we've found the correct page.

In [38]:
#!/usr/bin/env python3
"""
v6: Sensitivity upgrades (Barnard/Barry/Boise/Bowdoin fixes) on top of v5:
  1) Conditional year boosting + strong URL penalties for news/admissions/etc (prevents Boise news drift).
  2) Expanded inventory URL patterns (e.g., /departments-and-programs, /areas-of-study, etc.).
  3) Subdomain probing fallback (majors.<root>, programs.<root>, catalog.<root>) if no strong page found.
  4) Deeper, guided search + multi-page scanning:
       - best-first crawl with URL-prior queue
       - keep TOP_K candidate pages
       - aggregate matches/program-titles across TOP_K (deduped), not just one page

Sitemap-first then Selenium fallback:
  - If an inventory-ish page (esp. majors-minors/program finder) is "thin" under requests:
      * try sitemap-first to discover program pages and inventory pages
      * if still thin and USE_SELENIUM_FALLBACK=True, then Selenium render that page
  - Selenium remains OFF by default.

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v6.csv

TEST MODE:
- Set TEST_HEAD_N (e.g., 5). Set None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================
# Config
# ============================
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v6.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

TEST_HEAD_N = 25  # set None for full run

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.30
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.25
MAX_BYTES_PER_PAGE = 2_000_000

# Cache
CACHE_DIR = Path(".cache_v6_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Search budgets (per institution) — increased a bit for sensitivity
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 12
PHASE1_MAX_HOMEPAGE_LINKS = 25

PHASE2_BESTFIRST_MAX_PAGES = 60
PHASE2_MAX_DEPTH = 3

# Keep top candidates; aggregate matches across them
TOP_K = 5

# 1-hop expansion budgets
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 20

# Sitemap budgets
SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

# Optional Selenium fallback (only after sitemap-first)
USE_SELENIUM_FALLBACK = False
SELENIUM_MIN_SLEEP_SEC = 0.9
SELENIUM_WAIT_TIMEOUT_SEC = 15

# Heuristics for "thin" / maybe JS
MIN_VISIBLE_TEXT_LEN = 900
MIN_ANCHOR_CORPUS_LEN = 300

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/6.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================
# Prioritized URL templates (expanded)
# ============================
INVENTORY_PATHS = [
    # common majors/programs paths
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    # non-standard but common inventory conventions
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================
# Term sets / scoring signals
# ============================
# IMPORTANT: year regex is now *conditional* (see score_page)
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_PENALTY_REGEX = re.compile(
    r"\b(graduate|grad\s+studies|master'?s|ph\.?d|doctoral|mba|m\.?s\.?|m\.?a\.?|mfa|postbac|certificate)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_BOOST_REGEX = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

# Strong “irrelevant page” signals (penalize)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

# URL-tier indicators (expanded)
INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")
BULLETIN_URL_HINTS = ("bulletin",)

# Sitemap candidate patterns (expanded)
SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================
# Unicode normalization
# ============================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ============================
# Pattern compilation
# ============================
def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}

CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================
# Cache + fetch
# ============================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"

def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)

    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass

    return content

def fetch_text_cached(url: str, session: requests.Session) -> str:
    content = fetch_bytes_cached(url, session=session)
    try:
        return content.decode("utf-8", errors="replace")
    except Exception:
        return content.decode("latin-1", errors="replace")


# ============================
# Optional Selenium fallback
# ============================
def selenium_fetch_rendered(url: str) -> Optional[Tuple[str, str]]:
    """
    Returns (rendered_body_text, page_source_html) or None.
    Only used when USE_SELENIUM_FALLBACK=True.
    """
    if not USE_SELENIUM_FALLBACK:
        return None
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
    except Exception:
        return None

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--blink-settings=imagesEnabled=false")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_MIN_SLEEP_SEC)
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT_SEC).until(
            lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
        )
        body_text = normalize_unicode_text(driver.find_element(By.TAG_NAME, "body").text)
        html = driver.page_source
        return body_text, html
    except Exception:
        return None
    finally:
        driver.quit()


# ============================
# HTML parse helpers
# ============================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url

def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False

def root_domain(netloc: str) -> str:
    """
    Heuristic: strip common prefixes like www., m., web., etc.
    Keeps the remaining netloc. Works well for typical .edu.
    """
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n

@dataclass
class ParsedPage:
    url: str
    visible_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]       # (anchor_text, abs_url)
    title_like: List[str]              # h1/h2/h3/li candidates
    corpus: str                        # visible + anchors + aria/title attrs


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus = normalize_unicode_text(
        " ".join(x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x)
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        corpus=corpus,
    )


def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)

def looks_thin_for_inventory(parsed: ParsedPage) -> bool:
    anchor_len = len(" ".join(parsed.anchor_texts))
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) or (anchor_len < MIN_ANCHOR_CORPUS_LEN)

def get_parsed_page_requests_only(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)

def get_parsed_page_with_optional_selenium(url: str, session: requests.Session, allow_selenium: bool) -> ParsedPage:
    parsed = get_parsed_page_requests_only(url, session=session)
    if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url) and looks_thin_for_inventory(parsed):
        rendered = selenium_fetch_rendered(url)
        if rendered is not None:
            body_text, page_source_html = rendered
            parsed2 = parse_html_to_parsedpage(page_source_html, page_url=url)
            parsed2.visible_text = body_text
            parsed2.corpus = normalize_unicode_text(
                " ".join([body_text, " ".join(parsed2.anchor_texts), " ".join(parsed2.anchor_attr_texts)])
            )
            return parsed2
    return parsed


# ============================
# Sitemap parsing (sitemap-first fallback)
# ============================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()

def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls

def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue
        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    seen: Set[str] = set()
    out: List[str] = []
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break
    return out


# ============================
# Scanning + scoring (tiered + penalties + conditional year)
# ============================
@dataclass
class PageSignals:
    url: str
    tier: str
    year_hit: int
    undergrad_boost: int
    grad_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    control_hits: Dict[str, int]
    struct_hits: List[str]
    any_control: int
    any_struct: int
    visible_len: int
    anchor_len: int
    inventory_signature: int  # heuristic: "looks like an index" via link/list density


def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if any(h in u for h in INVENTORY_URL_HINTS):
        return "inventory"
    if any(h in u for h in BULLETIN_URL_HINTS):
        return "bulletin"
    if any(h in u for h in CATALOG_URL_HINTS):
        return "catalog"
    return "other"


def compute_inventory_signature(parsed: ParsedPage, base_netloc: str) -> int:
    """
    A cheap, structural "inventory index" detector:
    - Many internal links with non-trivial anchor text
    - Many list-like title candidates (li/h2/h3)
    """
    internal_links = 0
    program_like_links = 0
    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        internal_links += 1
        t = (a_text or "").strip()
        if len(t) >= 6 and not re.search(r"\b(apply|visit|donate|giving|news|event|calendar|admissions)\b", t, re.I):
            program_like_links += 1

    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)
    # signature thresholds tuned to be forgiving
    sig = 0
    if program_like_links >= 18:
        sig += 1
    if internal_links >= 40:
        sig += 1
    if li_like >= 25:
        sig += 1
    return sig


def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    corpus = parsed.corpus

    control_hits = {k: 1 if pat.search(corpus) else 0 for k, pat in CONTROL_PATS.items()}
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(corpus)]

    year_hit = 1 if YEAR_REGEX.search(url) or YEAR_REGEX.search(corpus) else 0
    undergrad_boost = 1 if UNDERGRAD_BOOST_REGEX.search(url) or UNDERGRAD_BOOST_REGEX.search(corpus) else 0
    grad_penalty = 1 if GRAD_PENALTY_REGEX.search(url) or GRAD_PENALTY_REGEX.search(corpus) else 0

    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(url.lower()) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(url.lower()) else 0

    any_control = 1 if sum(control_hits.values()) > 0 else 0
    any_struct = 1 if len(struct_hits) > 0 else 0

    inv_sig = compute_inventory_signature(parsed, base_netloc=base_netloc)

    return PageSignals(
        url=url,
        tier=page_tier(url),
        year_hit=year_hit,
        undergrad_boost=undergrad_boost,
        grad_penalty=grad_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        control_hits=control_hits,
        struct_hits=struct_hits,
        any_control=any_control,
        any_struct=any_struct,
        visible_len=len(parsed.visible_text),
        anchor_len=len(" ".join(parsed.anchor_texts)),
        inventory_signature=inv_sig,
    )


def score_page(sig: PageSignals) -> int:
    """
    Key changes:
    - Year boost is CONDITIONAL: only meaningful for catalog/inventory-ish pages AND not news/admissions.
    - Strong penalties for irrelevant/admissions URLs unless inventory signature is strong.
    - Inventory signature boosts pages that actually *list* programs.
    """
    score = 0

    # Tier bases
    if sig.tier == "inventory":
        score += 220
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 70
    elif sig.tier == "homepage":
        score += 10
    else:
        score += 45

    # Structural + control evidence
    score += 16 * len(sig.struct_hits)
    score += 8 * sum(sig.control_hits.values())

    # Inventory signature is a big deal (prevents shallow pages from winning)
    score += 55 * sig.inventory_signature

    # Undergrad boost
    if sig.undergrad_boost:
        score += 55

    # Graduate penalty (strong)
    if sig.grad_penalty:
        score -= 160

    # Penalize irrelevant/admissions content unless page strongly looks like inventory
    if sig.irrelevant_penalty and sig.inventory_signature == 0:
        score -= 220
    if sig.admissions_penalty and sig.inventory_signature == 0:
        score -= 120

    # Homepage demotion
    if sig.tier == "homepage":
        score -= 50

    # Bulletins demotion unless they look undergrad inventory-ish
    if sig.tier == "bulletin":
        inv_like = ("programs" in sig.struct_hits) or ("majors" in sig.struct_hits) or sig.undergrad_boost or (sig.inventory_signature > 0)
        if not inv_like:
            score -= 110

    # CONDITIONAL year boost
    if sig.year_hit:
        allow_year = (
            (sig.tier in ("catalog", "bulletin")) or
            (sig.tier == "inventory") or
            (len(sig.struct_hits) >= 2)  # fallback: it already looks academic/inventory-like
        )
        if allow_year and not sig.irrelevant_penalty and not sig.admissions_penalty:
            score += 120 if sig.tier in ("catalog", "bulletin") else 40

    return score


def is_good_enough(sig: PageSignals) -> bool:
    """
    Early-stop is stricter now:
    - Prefer pages that actually list programs (inventory_signature)
    - Avoid grad/admissions/news drift
    """
    if sig.grad_penalty:
        return False
    if sig.irrelevant_penalty and sig.inventory_signature == 0:
        return False
    if sig.admissions_penalty and sig.inventory_signature == 0 and sig.tier != "inventory":
        return False

    struct_n = len(sig.struct_hits)
    control_n = sum(sig.control_hits.values())

    if sig.tier == "inventory":
        return (sig.inventory_signature >= 1) and (struct_n >= 2) and (control_n >= 1 or sig.undergrad_boost == 1)

    if sig.tier in ("catalog", "bulletin"):
        return (sig.year_hit == 1) and (sig.undergrad_boost == 1) and (struct_n >= 2) and (sig.inventory_signature >= 1)

    return False


# ============================
# Link prioritization / best-first URL prior
# ============================
def url_prior(url: str) -> int:
    """
    Cheap prior without fetching:
    Higher is better (we'll use negative for heap).
    """
    u = url.lower()
    s = 0
    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 220
    if any(k in u for k in ("academics", "academic", "program", "majors", "minors", "departments", "areas-of-study", "fields-of-study")):
        s += 80
    if any(k in u for k in ("catalog", "catalogue")):
        s += 30
    if "bulletin" in u:
        s -= 40
    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 250
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 140
    if "graduate" in u or "grad" in u:
        s -= 200
    return s

def prioritize_links(links: List[Tuple[str, str]], base_netloc: str) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000
        # combine URL prior + mild anchor hint
        s = url_prior(u)
        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 40
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 80
        if len(a_text or "") < 3:
            s -= 10
        return -s
    return sorted(links, key=rank)


# ============================
# Candidate generation
# ============================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


def build_subdomain_candidates(base_url: str) -> List[str]:
    """
    Subdomain probing fallback for cases like majors.boisestate.edu.
    """
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)

    # Only attempt for plausible .edu-like domains
    if "." not in rd:
        return []

    bases = [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]
    return bases


# ============================
# Discovery: inventory-first + sitemap-first + best-first crawl + top-K
# ============================
@dataclass
class CandidatePage:
    score: int
    sig: PageSignals
    parsed: ParsedPage

def push_topk(topk: List[CandidatePage], cand: CandidatePage) -> None:
    """
    Maintain a min-heap of size TOP_K by score.
    """
    if len(topk) < TOP_K:
        heapq.heappush(topk, cand)
        return
    if cand.score > topk[0].score:
        heapq.heapreplace(topk, cand)

def topk_sorted(topk: List[CandidatePage]) -> List[CandidatePage]:
    return sorted(topk, key=lambda c: c.score, reverse=True)


def find_top_pages_for_institution(base_url: str, session: requests.Session) -> Tuple[List[CandidatePage], int]:
    """
    Returns: (top_pages_sorted, used_selenium_flag)
    """
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    used_selenium_flag = 0
    topk: List[CandidatePage] = []

    sitemap_urls_cache: Optional[List[str]] = None
    sitemap_candidates_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_urls_cache, sitemap_candidates_cache
        if sitemap_candidates_cache is not None:
            return sitemap_candidates_cache
        sitemap_urls_cache = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_candidates_cache = sitemap_candidate_urls(sitemap_urls_cache)
        return sitemap_candidates_cache

    def consider(url: str, allow_selenium: bool) -> Optional[CandidatePage]:
        nonlocal used_selenium_flag
        if not url or url in tried:
            return None
        tried.add(url)
        try:
            parsed = get_parsed_page_with_optional_selenium(url, session=session, allow_selenium=allow_selenium)
            if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url):
                used_selenium_flag = 1
            sig = compute_signals(url, parsed, base_netloc=base_netloc)
            s = score_page(sig)
            cand = CandidatePage(score=s, sig=sig, parsed=parsed)
            push_topk(topk, cand)
            return cand
        except Exception:
            return None
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # ---- Phase 1A: explicit inventory paths
    inv_candidates = [urljoin(base_url, p) for p in INVENTORY_PATHS]
    for url in inv_candidates[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        got = consider(url, allow_selenium=False)
        if got and is_good_enough(got.sig):
            return topk_sorted(topk), used_selenium_flag

        # Sitemap-first then selenium fallback for thin inventory-ish pages
        if got and is_inventoryish_url(url) and looks_thin_for_inventory(got.parsed):
            # sitemap-first
            for u2 in load_sitemap_candidates():
                got2 = consider(u2, allow_selenium=False)
                if got2 and is_good_enough(got2.sig):
                    return topk_sorted(topk), used_selenium_flag

            # then selenium fallback (only if enabled)
            if USE_SELENIUM_FALLBACK:
                got3 = consider(url, allow_selenium=True)
                if got3 and is_good_enough(got3.sig):
                    return topk_sorted(topk), used_selenium_flag

    # ---- Phase 1B: year-aware catalog templates
    for url in build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES]:
        got = consider(url, allow_selenium=False)
        if got and is_good_enough(got.sig):
            return topk_sorted(topk), used_selenium_flag

    # ---- Phase 1C: generic catalog paths
    for url in [urljoin(base_url, p) for p in CATALOG_PATHS][:PHASE1_MAX_CATALOG_CANDIDATES]:
        got = consider(url, allow_selenium=False)
        if got and is_good_enough(got.sig):
            return topk_sorted(topk), used_selenium_flag

    # ---- Phase 1D: sample prioritized homepage links
    try:
        home = get_parsed_page_requests_only(base_url, session=session)
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc)
        for a_text, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            got = consider(u, allow_selenium=False)
            if got and is_good_enough(got.sig):
                return topk_sorted(topk), used_selenium_flag
    except Exception:
        pass

    # ---- Phase 2: best-first crawl (guided deeper search)
    # Priority queue: (-prior, depth, url)
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u), depth, u))

    pq_push(base_url, 0)
    pages_fetched = 0

    while pq and pages_fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        got = consider(u, allow_selenium=False)
        pages_fetched += 1
        if got and is_good_enough(got.sig):
            return topk_sorted(topk), used_selenium_flag

        if depth >= PHASE2_MAX_DEPTH:
            continue

        # expand links from this page
        try:
            parsed = got.parsed if got is not None else get_parsed_page_requests_only(u, session=session)
            links = [(t, link_u) for (t, link_u) in parsed.links if same_domain(link_u, base_netloc)]
            links = prioritize_links(links, base_netloc)
            for a_text, link_u in links[:80]:
                pq_push(link_u, depth + 1)
        except Exception:
            pass

    # ---- Subdomain probing fallback (only if we didn't find a strong page)
    # Try a few high-signal subdomains and include them in topK if reachable.
    for sub in build_subdomain_candidates(base_url):
        got = consider(sub, allow_selenium=False)
        if got and is_good_enough(got.sig):
            return topk_sorted(topk), used_selenium_flag

        # also try common inventory endpoints on that subdomain
        for p in ("/", "/programs", "/majors", "/majors-minors"):
            got2 = consider(urljoin(sub, p), allow_selenium=False)
            if got2 and is_good_enough(got2.sig):
                return topk_sorted(topk), used_selenium_flag

    return topk_sorted(topk), used_selenium_flag


# ============================
# Program-title extraction (aggregate across topK)
# ============================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_program_title(s: str) -> bool:
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False
    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        return False
    return TARGET_ANY_REGEX.search(s_norm) is not None

def extract_h1_or_title(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    h1 = soup.find("h1")
    if h1:
        return normalize_unicode_text(h1.get_text(" ", strip=True))
    t = soup.find("title")
    if t:
        return normalize_unicode_text(t.get_text(" ", strip=True))
    return ""

def token_matches_from_corpus(corpus: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(corpus)))
    return out

def aggregate_program_titles_and_tokens(
    top_pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
) -> Tuple[List[str], Dict[str, List[str]]]:
    """
    Aggregate:
      - program titles (deduped)
      - token matches (afri/ethnic/black) across top pages
    Also uses sitemap candidate program URLs if a majors-minors index is thin.
    """
    base_netloc = urlparse(base_url).netloc

    titles: Set[str] = set()
    tokens_agg: Dict[str, Set[str]] = {"afri": set(), "ethnic": set(), "black": set()}

    # collect quick tokens + title candidates from top pages
    for cand in top_pages:
        tm = token_matches_from_corpus(cand.parsed.corpus)
        for k in tm:
            tokens_agg[k].update(tm[k])

        for txt in cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like:
            if txt and looks_like_program_title(txt):
                titles.add(txt)

    # 1-hop confirm: follow a bounded set of likely program links from top pages
    follow_urls: List[str] = []
    for cand in top_pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if "graduate" in ul or "grad" in ul:
                continue
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul):
                continue
            if looks_like_program_title(a_text) or TARGET_ANY_REGEX.search(ul):
                follow_urls.append(u)

    # If we have a thin majors-minors index among top pages, add sitemap program URLs
    thin_inventory_detected = any(
        is_inventoryish_url(c.sig.url) and looks_thin_for_inventory(c.parsed) for c in top_pages
    )
    if thin_inventory_detected:
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    # de-dup and cap
    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    fetches = 0
    for u in uniq_follow:
        if fetches >= ONE_HOP_MAX_FETCHES:
            break
        try:
            html = fetch_text_cached(u, session=session)
            t = extract_h1_or_title(html)
            if t and looks_like_program_title(t):
                titles.add(t)
            # also pick up tokens from these pages cheaply
            parsed = parse_html_to_parsedpage(html, page_url=u)
            tm = token_matches_from_corpus(parsed.corpus)
            for k in tm:
                tokens_agg[k].update(tm[k])
            fetches += 1
        except Exception:
            continue
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # Dedup titles by normalized key (keep best display)
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    titles_out = sorted(best_display.values())
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}
    return titles_out, tokens_out


# ============================
# Main
# ============================
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        best_guess_url = "N/A"
        alt_candidate_urls = ""
        used_selenium = 0

        any_control_found = 0
        any_struct_found = 0
        struct_hits_union: Set[str] = set()
        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0

        if base_url:
            top_pages, used_selenium = find_top_pages_for_institution(base_url, session=session)
            if top_pages:
                best_guess_url = top_pages[0].sig.url
                if len(top_pages) > 1:
                    alt_candidate_urls = "|".join(c.sig.url for c in top_pages[1:])

                # union signals across TOP_K
                for c in top_pages:
                    any_control_found = max(any_control_found, c.sig.any_control)
                    any_struct_found = max(any_struct_found, c.sig.any_struct)
                    struct_hits_union.update(c.sig.struct_hits)
                    for k, v in c.sig.control_hits.items():
                        control_hit_cols[f"found_{k}"] = max(control_hit_cols[f"found_{k}"], int(v))

                # aggregate titles + tokens across TOP_K (+ limited 1-hop)
                program_titles, tokens = aggregate_program_titles_and_tokens(top_pages, base_url=base_url, session=session)
                program_title_count = len(program_titles)
                any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0

                afri_matches = tokens.get("afri", [])
                ethnic_matches = tokens.get("ethnic", [])
                black_matches = tokens.get("black", [])

        rec: Dict[str, str] = dict(row)
        rec["best_guess_inventory_url"] = best_guess_url
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["program_title_count"] = str(program_title_count)
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["used_selenium"] = str(used_selenium)

        rec.update({k: str(v) for k, v in control_hit_cols.items()})

        out_rows.append(rec)
        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "used_selenium",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)


if __name__ == "__main__":
    main()

[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

In [1]:
#!/usr/bin/env python3
"""
v7: Two-tier page selection (Inventory vs Evidence) + expanded control terms + edge-case guardrails.

Key changes vs v6:
A) Two-tier outputs
   - best_guess_inventory_url: selected from pages that look like an *inventory/index* (high inventory signature)
   - best_evidence_url: selected from pages that maximize *unique control-term hits* (majors evidence), with guardrails

B) Expanded control-term set (unique-count matters a lot for evidence selection)
   Added: economics, psychology, sociology, history, english, political science, philosophy,
          computer science, engineering, physics, geology, statistics, neuroscience

C) Guardrails to avoid bizarre picks
   - Profile/faculty URL/text penalties (people/faculty-directory/profile/bio, Email:, Phone:, Office:, etc.)
   - News/admissions penalties remain strong
   - Conditional year boosting remains (only matters if page already looks academic/inventory-ish)
   - Soft-404 detection (pages that return 200 but are effectively "Not Found" get zeroed)

D) Still: Sitemap-first then Selenium fallback (only if USE_SELENIUM_FALLBACK=True)
   - especially for thin majors/minors / program-finder pages

Optional PDF fallback (rare cases like Fresno PDF):
   - if a candidate page contains year-ish PDF links and we can't find strong inventory,
     try downloading and scanning PDF text (best-effort; skips if pdfminer isn't available).

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v7.csv

TEST MODE:
- Set TEST_HEAD_N (e.g., 5). Set None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================
# Config
# ============================
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v7.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

TEST_HEAD_N = 25  # set None for full run

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.30
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.25
MAX_BYTES_PER_PAGE = 2_000_000

CACHE_DIR = Path(".cache_v7_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Search budgets (per institution)
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 12
PHASE1_MAX_HOMEPAGE_LINKS = 25

PHASE2_BESTFIRST_MAX_PAGES = 65
PHASE2_MAX_DEPTH = 3

TOP_K = 6  # keep more candidates; we'll aggregate extraction across these

# 1-hop expansion budgets (for program title confirmation)
ONE_HOP_MAX_LINKS_TOTAL = 45
ONE_HOP_MAX_FETCHES = 22

# Sitemap budgets
SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 260

# Optional Selenium fallback
USE_SELENIUM_FALLBACK = False
SELENIUM_MIN_SLEEP_SEC = 0.9
SELENIUM_WAIT_TIMEOUT_SEC = 15

# Heuristics for "thin" / maybe JS
MIN_VISIBLE_TEXT_LEN = 900
MIN_ANCHOR_CORPUS_LEN = 300

# Optional PDF scanning fallback
ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/7.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================
# URL templates (expanded)
# ============================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================
# Patterns / scoring signals
# ============================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_PENALTY_REGEX = re.compile(
    r"\b(graduate|grad\s+studies|master'?s|ph\.?d|doctoral|mba|m\.?s\.?|m\.?a\.?|mfa|postbac|certificate)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_BOOST_REGEX = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)

ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)

PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)

PROFILE_TEXT_PENALTY = re.compile(
    r"\b(email|e-mail|telephone|phone|office|cv|curriculum\s+vitae|publications|research\s+interests)\b",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

ARCHIVE_TEXT_PENALTY = re.compile(r"\barchive\b|\bpast\s+catalogs\b", flags=re.IGNORECASE)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    # original controls
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    # added controls
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    # careful ones
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    # more STEM
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")
BULLETIN_URL_HINTS = ("bulletin",)

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)


# ============================
# Unicode normalization
# ============================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ============================
# Pattern compilation
# ============================
def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}

CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================
# Cache + fetch
# ============================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"

def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)

    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass

    return content

def fetch_text_cached(url: str, session: requests.Session) -> str:
    content = fetch_bytes_cached(url, session=session)
    try:
        return content.decode("utf-8", errors="replace")
    except Exception:
        return content.decode("latin-1", errors="replace")


# ============================
# Optional Selenium fallback
# ============================
def selenium_fetch_rendered(url: str) -> Optional[Tuple[str, str]]:
    if not USE_SELENIUM_FALLBACK:
        return None
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
    except Exception:
        return None

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--blink-settings=imagesEnabled=false")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_MIN_SLEEP_SEC)
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT_SEC).until(
            lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
        )
        body_text = normalize_unicode_text(driver.find_element(By.TAG_NAME, "body").text)
        html = driver.page_source
        return body_text, html
    except Exception:
        return None
    finally:
        driver.quit()


# ============================
# HTML parse helpers
# ============================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url

def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False

def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n

@dataclass
class ParsedPage:
    url: str
    visible_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]       # (anchor_text, abs_url)
    title_like: List[str]              # h1/h2/h3/li candidates
    corpus: str                        # visible + anchors + aria/title attrs

def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus = normalize_unicode_text(
        " ".join(x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x)
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        corpus=corpus,
    )

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)

def looks_thin_for_inventory(parsed: ParsedPage) -> bool:
    anchor_len = len(" ".join(parsed.anchor_texts))
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) or (anchor_len < MIN_ANCHOR_CORPUS_LEN)

def is_soft_404(parsed: ParsedPage) -> bool:
    # If the page is very small and contains common "not found" language, treat as soft-404
    if len(parsed.visible_text) < 700 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    if SOFT_404_TEXT.search(parsed.visible_text) and SOFT_404_TEXT.search(" ".join(parsed.title_like[:5])):
        return True
    return False

def get_parsed_page_requests_only(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)

def get_parsed_page_with_optional_selenium(url: str, session: requests.Session, allow_selenium: bool) -> ParsedPage:
    parsed = get_parsed_page_requests_only(url, session=session)
    if is_soft_404(parsed):
        return parsed
    if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url) and looks_thin_for_inventory(parsed):
        rendered = selenium_fetch_rendered(url)
        if rendered is not None:
            body_text, page_source_html = rendered
            parsed2 = parse_html_to_parsedpage(page_source_html, page_url=url)
            parsed2.visible_text = body_text
            parsed2.corpus = normalize_unicode_text(
                " ".join([body_text, " ".join(parsed2.anchor_texts), " ".join(parsed2.anchor_attr_texts)])
            )
            return parsed2
    return parsed


# ============================
# Sitemap parsing
# ============================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()

def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls

def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue
        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    seen: Set[str] = set()
    out: List[str] = []
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break
    return out


# ============================
# Scoring
# ============================
@dataclass
class PageSignals:
    url: str
    tier: str
    year_hit: int
    undergrad_boost: int
    grad_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    profile_url_penalty: int
    profile_text_penalty: int
    archive_penalty: int
    soft404: int

    control_hits: Dict[str, int]
    control_unique: int
    struct_hits: List[str]

    any_control: int
    any_struct: int

    visible_len: int
    anchor_len: int
    inventory_signature: int

def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if any(h in u for h in INVENTORY_URL_HINTS):
        return "inventory"
    if any(h in u for h in BULLETIN_URL_HINTS):
        return "bulletin"
    if any(h in u for h in CATALOG_URL_HINTS):
        return "catalog"
    return "other"

def compute_inventory_signature(parsed: ParsedPage, base_netloc: str) -> int:
    internal_links = 0
    program_like_links = 0

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        internal_links += 1
        t = (a_text or "").strip()
        if len(t) >= 6 and not re.search(
            r"\b(apply|visit|donate|giving|news|event|calendar|admissions|faculty|staff|directory)\b",
            t,
            re.I,
        ):
            program_like_links += 1

    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if program_like_links >= 18:
        sig += 1
    if internal_links >= 45:
        sig += 1
    if li_like >= 25:
        sig += 1
    return sig

def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    corpus = parsed.corpus

    control_hits = {k: 1 if pat.search(corpus) else 0 for k, pat in CONTROL_PATS.items()}
    control_unique = sum(control_hits.values())
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(corpus)]

    year_hit = 1 if YEAR_REGEX.search(url) or YEAR_REGEX.search(corpus) else 0
    undergrad_boost = 1 if UNDERGRAD_BOOST_REGEX.search(url) or UNDERGRAD_BOOST_REGEX.search(corpus) else 0
    grad_penalty = 1 if GRAD_PENALTY_REGEX.search(url) or GRAD_PENALTY_REGEX.search(corpus) else 0

    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(url.lower()) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(url.lower()) else 0
    profile_url_penalty = 1 if PROFILE_URL_PENALTY.search(url.lower()) else 0
    profile_text_penalty = 1 if PROFILE_TEXT_PENALTY.search(parsed.visible_text) else 0
    archive_penalty = 1 if (ARCHIVE_URL_PENALTY.search(url.lower()) or ARCHIVE_TEXT_PENALTY.search(parsed.visible_text)) else 0
    soft404 = 1 if is_soft_404(parsed) else 0

    any_control = 1 if control_unique > 0 else 0
    any_struct = 1 if len(struct_hits) > 0 else 0

    inv_sig = compute_inventory_signature(parsed, base_netloc=base_netloc)

    return PageSignals(
        url=url,
        tier=page_tier(url),
        year_hit=year_hit,
        undergrad_boost=undergrad_boost,
        grad_penalty=grad_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        profile_url_penalty=profile_url_penalty,
        profile_text_penalty=profile_text_penalty,
        archive_penalty=archive_penalty,
        soft404=soft404,
        control_hits=control_hits,
        control_unique=control_unique,
        struct_hits=struct_hits,
        any_control=any_control,
        any_struct=any_struct,
        visible_len=len(parsed.visible_text),
        anchor_len=len(" ".join(parsed.anchor_texts)),
        inventory_signature=inv_sig,
    )

def is_academicish(sig: PageSignals) -> bool:
    """
    Guardrail for evidence-tier:
    - Must look somewhat academic (struct hits, inventory signature, or academic-ish URL hints).
    - Reject obvious irrelevant/profile/admissions unless the page *strongly* looks like inventory.
    """
    ul = sig.url.lower()
    url_academic_hint = any(k in ul for k in ("academics", "program", "major", "minor", "department", "catalog", "bulletin"))
    baseline = (sig.inventory_signature >= 1) or (len(sig.struct_hits) >= 1) or url_academic_hint

    if not baseline:
        return False

    if sig.soft404:
        return False

    if sig.profile_url_penalty or sig.profile_text_penalty:
        return sig.inventory_signature >= 2  # extremely rare; otherwise reject

    if sig.irrelevant_penalty or sig.admissions_penalty:
        return sig.inventory_signature >= 2  # allow only if truly a big inventory page

    if sig.archive_penalty:
        return False

    return True

def score_for_inventory(sig: PageSignals) -> int:
    """
    Score for selecting the *inventory/index* page.
    Control hits matter, but structure dominates.
    """
    if sig.soft404:
        return -10_000

    score = 0
    # tier base
    if sig.tier == "inventory":
        score += 230
    elif sig.tier == "catalog":
        score += 120
    elif sig.tier == "bulletin":
        score += 70
    elif sig.tier == "homepage":
        score += 0
    else:
        score += 40

    # structural evidence
    score += 65 * sig.inventory_signature
    score += 16 * len(sig.struct_hits)

    # modest control contribution (do not let it dominate inventory selection)
    score += 6 * sig.control_unique

    # undergrad boost
    if sig.undergrad_boost:
        score += 45

    # penalties
    if sig.grad_penalty:
        score -= 170
    if sig.profile_url_penalty or sig.profile_text_penalty:
        score -= 250
    if sig.irrelevant_penalty:
        score -= 230
    if sig.admissions_penalty:
        score -= 140
    if sig.archive_penalty:
        score -= 180

    # conditional year boost
    if sig.year_hit:
        allow_year = (
            (sig.tier in ("catalog", "bulletin", "inventory")) or
            (sig.inventory_signature >= 1) or
            (len(sig.struct_hits) >= 2)
        )
        if allow_year and not (sig.irrelevant_penalty or sig.admissions_penalty or sig.profile_url_penalty):
            score += 80 if sig.tier in ("catalog", "bulletin") else 30

    # prefer true index hubs over detail pages for inventory selection
    # If URL looks like /majors-minors/<slug>/... then demote relative to /majors-minors root.
    path = urlparse(sig.url).path.lower().strip("/")
    if "majors-minors" in path and path.count("/") >= 1 and not path.endswith("majors-minors"):
        score -= 60
    if "programs" in path and path.count("/") >= 1 and not path.endswith("programs"):
        score -= 25

    return score

def score_for_evidence(sig: PageSignals) -> int:
    """
    Score for selecting the *evidence* page (max unique controls), with guardrails.
    """
    if not is_academicish(sig):
        return -10_000

    score = 0
    # Strong emphasis on unique control hits
    score += 35 * sig.control_unique

    # still reward inventory-ness
    score += 30 * sig.inventory_signature
    score += 10 * len(sig.struct_hits)

    # small tier preference
    if sig.tier == "inventory":
        score += 40
    elif sig.tier in ("catalog", "bulletin"):
        score += 15

    # undergrad helps, grad hurts
    if sig.undergrad_boost:
        score += 20
    if sig.grad_penalty:
        score -= 90

    # keep hard penalties hard (guardrails already enforce most)
    if sig.profile_url_penalty or sig.profile_text_penalty:
        score -= 250
    if sig.irrelevant_penalty or sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 200

    # conditional year
    if sig.year_hit and sig.inventory_signature >= 1 and sig.tier in ("catalog", "bulletin"):
        score += 35

    return score

def is_good_inventory_candidate(sig: PageSignals) -> bool:
    if sig.soft404:
        return False
    if sig.grad_penalty:
        return False
    if sig.profile_url_penalty or sig.profile_text_penalty:
        return False
    if sig.archive_penalty:
        return False
    if sig.irrelevant_penalty and sig.inventory_signature == 0:
        return False
    if sig.admissions_penalty and sig.inventory_signature == 0:
        return False
    # Must look like a list/index
    return sig.inventory_signature >= 1 and (len(sig.struct_hits) >= 2 or sig.control_unique >= 4 or sig.undergrad_boost == 1)


# ============================
# Link prioritization / best-first URL prior
# ============================
def url_prior(url: str) -> int:
    u = url.lower()
    s = 0
    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 220
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 30
    if "bulletin" in u:
        s -= 40
    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if "graduate" in u or "grad" in u:
        s -= 200
    return s

def prioritize_links(links: List[Tuple[str, str]], base_netloc: str) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000
        s = url_prior(u)
        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 40
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 80
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 120
        if len(a_text or "") < 3:
            s -= 10
        return -s
    return sorted(links, key=rank)


# ============================
# Candidate generation
# ============================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq

def build_subdomain_candidates(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    # include root of each as first-choice; do not blindly append paths before scoring root
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================
# PDF fallback (best-effort)
# ============================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    """
    Best-effort PDF text extraction.
    Uses pdfminer if available; otherwise returns "".
    """
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""

def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    """
    Return PDF URLs that look related to majors/minors and 25-26-ish.
    """
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        # year-ish or majors/minors-ish signals in href or anchor
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)
    # de-dup preserve order
    seen: Set[str] = set()
    out: List[str] = []
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================
# Discovery: collect candidates + score both tiers
# ============================
@dataclass
class CandidatePage:
    url: str
    inv_score: int
    ev_score: int
    sig: PageSignals
    parsed: ParsedPage

def push_topk(topk: List[CandidatePage], cand: CandidatePage) -> None:
    # min-heap by inv_score (store as (score, idx, cand) pattern)
    # We'll keep TOP_K by inv_score for breadth; evidence selection comes separately.
    if len(topk) < TOP_K:
        heapq.heappush(topk, (cand.inv_score, len(topk), cand))
        return
    if cand.inv_score > topk[0][0]:
        heapq.heapreplace(topk, (cand.inv_score, topk[0][1], cand))

def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    return [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]

def fetch_and_score(url: str, base_netloc: str, session: requests.Session, allow_selenium: bool) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page_with_optional_selenium(url, session=session, allow_selenium=allow_selenium)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        inv_score = score_for_inventory(sig)
        ev_score = score_for_evidence(sig)
        return CandidatePage(url=url, inv_score=inv_score, ev_score=ev_score, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

def find_candidates_for_institution(base_url: str, session: requests.Session) -> Tuple[List[CandidatePage], int]:
    """
    Returns:
      candidates_sorted_by_inv_score, used_selenium_flag
    """
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    used_selenium_flag = 0

    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_candidates_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_candidates_cache
        if sitemap_candidates_cache is not None:
            return sitemap_candidates_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_candidates_cache = sitemap_candidate_urls(all_urls)
        return sitemap_candidates_cache

    def consider(url: str, allow_selenium: bool) -> Optional[CandidatePage]:
        nonlocal used_selenium_flag
        if not url or url in tried:
            return None
        tried.add(url)

        cand = fetch_and_score(url, base_netloc=base_netloc, session=session, allow_selenium=allow_selenium)
        if cand is None:
            return None

        if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url):
            used_selenium_flag = 1

        push_topk(topk, cand)
        return cand

    # Phase 0: always consider base_url itself (especially important for majors.<domain> roots)
    consider(base_url, allow_selenium=False)

    # Phase 1A: explicit inventory paths
    inv_candidates = [urljoin(base_url, p) for p in INVENTORY_PATHS]
    for url in inv_candidates[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        cand = consider(url, allow_selenium=False)
        if cand and is_inventoryish_url(url) and looks_thin_for_inventory(cand.parsed):
            # Sitemap-first
            for u2 in load_sitemap_candidates():
                consider(u2, allow_selenium=False)
            # Selenium fallback if enabled
            if USE_SELENIUM_FALLBACK:
                consider(url, allow_selenium=True)

    # Phase 1B: year-aware catalog templates
    for url in build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES]:
        consider(url, allow_selenium=False)

    # Phase 1C: generic catalog paths
    for url in [urljoin(base_url, p) for p in CATALOG_PATHS][:PHASE1_MAX_CATALOG_CANDIDATES]:
        consider(url, allow_selenium=False)

    # Phase 1D: sample prioritized homepage links
    try:
        home = get_parsed_page_requests_only(base_url, session=session)
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc)
        for a_text, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            consider(u, allow_selenium=False)
    except Exception:
        pass

    # Phase 2: best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u), depth, u))

    pq_push(base_url, 0)
    pages_fetched = 0

    while pq and pages_fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u, allow_selenium=False)
        pages_fetched += 1

        if depth >= PHASE2_MAX_DEPTH:
            continue

        try:
            parsed = cand.parsed if cand is not None else get_parsed_page_requests_only(u, session=session)
            links = [(t, link_u) for (t, link_u) in parsed.links if same_domain(link_u, base_netloc)]
            links = prioritize_links(links, base_netloc)
            for a_text, link_u in links[:80]:
                pq_push(link_u, depth + 1)
        except Exception:
            pass

    # Subdomain probing fallback
    for sub in build_subdomain_candidates(base_url):
        # IMPORTANT: consider root first; only append paths later if root looks weak
        c_root = consider(sub, allow_selenium=False)
        if c_root and c_root.sig.inventory_signature >= 1:
            # root already good; do not force /majors-minors etc.
            continue

        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(sub, p), allow_selenium=False)

    return topk_sorted(topk), used_selenium_flag


# ============================
# Program-title extraction / token aggregation
# ============================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_program_title(s: str) -> bool:
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False
    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        return False
    return TARGET_ANY_REGEX.search(s_norm) is not None

def extract_h1_or_title(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    h1 = soup.find("h1")
    if h1:
        return normalize_unicode_text(h1.get_text(" ", strip=True))
    t = soup.find("title")
    if t:
        return normalize_unicode_text(t.get_text(" ", strip=True))
    return ""

def token_matches_from_corpus(corpus: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(corpus)))
    return out

def aggregate_program_titles_and_tokens(
    candidates: List[CandidatePage],
    base_url: str,
    session: requests.Session,
) -> Tuple[List[str], Dict[str, List[str]], List[str]]:
    """
    Aggregate across candidate pages:
      - program titles (deduped)
      - token matches (afri/ethnic/black)
      - pdf_hits (best-effort)
    """
    base_netloc = urlparse(base_url).netloc

    titles: Set[str] = set()
    tokens_agg: Dict[str, Set[str]] = {"afri": set(), "ethnic": set(), "black": set()}
    pdf_hits: List[str] = []

    # quick tokens + title candidates from candidate pages
    for cand in candidates:
        tm = token_matches_from_corpus(cand.parsed.corpus)
        for k in tm:
            tokens_agg[k].update(tm[k])

        for txt in cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like:
            if txt and looks_like_program_title(txt):
                titles.add(txt)

    # 1-hop follow likely program links
    follow_urls: List[str] = []
    for cand in candidates:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            if looks_like_program_title(a_text) or TARGET_ANY_REGEX.search(ul):
                follow_urls.append(u)

    # thin inventory pages: optionally add sitemap candidates
    thin_inventory_detected = any(is_inventoryish_url(c.url) and looks_thin_for_inventory(c.parsed) for c in candidates)
    if thin_inventory_detected:
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    # de-dup cap
    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    fetches = 0
    for u in uniq_follow:
        if fetches >= ONE_HOP_MAX_FETCHES:
            break
        try:
            html = fetch_text_cached(u, session=session)
            t = extract_h1_or_title(html)
            if t and looks_like_program_title(t):
                titles.add(t)

            parsed = parse_html_to_parsedpage(html, page_url=u)
            tm = token_matches_from_corpus(parsed.corpus)
            for k in tm:
                tokens_agg[k].update(tm[k])

            fetches += 1
        except Exception:
            continue
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # PDF fallback (rare): look for year-ish major/minor PDFs in candidate pages
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in candidates:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            pdf_links = find_yearish_major_pdf_links(cand.parsed)
            for pdf_url in pdf_links:
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    # treat as evidence hit if it contains majors/minors/program + any controls
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        # update controls/tokens from PDF text
                        # tokens:
                        tm = token_matches_from_corpus(txt)
                        for k in tm:
                            tokens_agg[k].update(tm[k])
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Dedup titles by normalized key
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    titles_out = sorted(best_display.values())
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}
    return titles_out, tokens_out, pdf_hits


# ============================
# Main
# ============================
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        best_guess_inventory_url = "N/A"
        best_evidence_url = "N/A"
        alt_candidate_urls = ""
        used_selenium = 0

        any_control_found = 0
        any_struct_found = 0
        struct_hits_union: Set[str] = set()

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        # Evidence reporting
        evidence_control_unique = 0
        evidence_controls_matched = ""

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0

        pdf_hits: List[str] = []

        if base_url:
            candidates, used_selenium = find_candidates_for_institution(base_url, session=session)

            if candidates:
                # Build alt list (top by inventory score)
                alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K])

                # Choose inventory URL: highest inventory score among good inventory candidates;
                # fallback to top inventory-score overall.
                inv_candidates = [c for c in candidates if is_good_inventory_candidate(c.sig)]
                inv_pick = max(inv_candidates, key=lambda c: c.inv_score) if inv_candidates else max(candidates, key=lambda c: c.inv_score)
                best_guess_inventory_url = inv_pick.url

                # Choose evidence URL: max evidence score (guardrails inside score)
                ev_pick = max(candidates, key=lambda c: c.ev_score)
                if ev_pick.ev_score > -9000:
                    best_evidence_url = ev_pick.url
                    evidence_control_unique = ev_pick.sig.control_unique
                    evidence_controls_matched = "|".join([k for k, v in ev_pick.sig.control_hits.items() if v == 1])

                # Union signals across top candidates (for quick summary columns)
                for c in candidates[:TOP_K]:
                    any_control_found = max(any_control_found, 1 if c.sig.control_unique > 0 else 0)
                    any_struct_found = max(any_struct_found, 1 if len(c.sig.struct_hits) > 0 else 0)
                    struct_hits_union.update(c.sig.struct_hits)
                    for k, v in c.sig.control_hits.items():
                        control_hit_cols[f"found_{k}"] = max(control_hit_cols[f"found_{k}"], int(v))

                # Aggregate program titles/tokens across a small set:
                # include top-K (by inv_score) + ensure evidence page included if distinct.
                agg_set: List[CandidatePage] = candidates[:TOP_K].copy()
                if best_evidence_url != "N/A" and all(c.url != best_evidence_url for c in agg_set):
                    # append the evidence page if present
                    for c in candidates:
                        if c.url == best_evidence_url:
                            agg_set.append(c)
                            break

                program_titles, tokens, pdf_hits = aggregate_program_titles_and_tokens(agg_set, base_url=base_url, session=session)
                program_title_count = len(program_titles)
                any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0

                afri_matches = tokens.get("afri", [])
                ethnic_matches = tokens.get("ethnic", [])
                black_matches = tokens.get("black", [])

        rec: Dict[str, str] = dict(row)
        rec["best_guess_inventory_url"] = best_guess_inventory_url
        rec["best_evidence_url"] = best_evidence_url
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["evidence_control_unique"] = str(evidence_control_unique)
        rec["evidence_controls_matched"] = evidence_controls_matched

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["program_title_count"] = str(program_title_count)
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        rec["used_selenium"] = str(used_selenium)

        rec.update({k: str(v) for k, v in control_hit_cols.items()})

        out_rows.append(rec)
        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_evidence_url", "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "evidence_control_unique", "evidence_controls_matched",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "pdf_hits",
        "used_selenium",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)


if __name__ == "__main__":
    main()

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

Clear page-selection improvements: Barnard, Bowdoin, Cal Poly SLO, Boise (partial), Barry (partial)
Clear page-selection regressions: Agnes Scott, Boston College, Amherst (moved off correct catalog), Brandeis, Bradley, CPP, CSUEB (picked a single major page), Brown evidence page oddity
Extraction quality regressions (noise): Amherst, Brandeis, Fresno, BC, BGSU

•	Split control hits into “structured” (anchors/headings/list items) vs “any text”, and use structured-only control hits for evidence-tier scoring.
•	Strengthen inventory signature (more list/hub indicators: higher program-like link counts, distinct anchors, slug diversity) so hub pages dominate.
•	Replace fixed detail-page penalties with a scaled penalty: penalize program/major slug pages heavily unless they also look like a hub (high inventory signature).
•	Add a canonical hub URL bonus for exact endpoints like /academics/programs, /programs, /academics/majors-minors, /departments-and-programs, etc.
•	Add a targeted compliance/CIP/consumer-info penalty so disclosure pages can remain alternates/evidence but don’t become the primary inventory URL.
•	Tighten year boosting: only boost year tokens for catalog/bulletin-like contexts and add an awards/honors penalty to avoid “2024 awards” style mispicks.
•	Make evidence-tier scoring “list-aware”: require some list-ness (inventory signature or strong struct hits) and heavily penalize news/admissions/profile pages.
•	Fix subdomain probing (e.g., Boise): always score the subdomain root first and only append paths if root is weak; improve soft-404 detection on “200 OK but not found” templates.
•	Harden program-title extraction filters: drop course codes/units/semester language and reject person-name-only strings unless they contain program keywords.
•	Restrict program titles to title-like markup sources (anchors/headings/list items/H1-title), treating token matches as diagnostics rather than titles.
•	Add an output “reason” field (e.g., best_guess_inventory_url_reason) logging key signals/penalties so mis-selections are explainable and debuggable quickly.

In [2]:
#!/usr/bin/env python3
"""
v8: Two-tier selection (Inventory vs Evidence) with safer scoring + cleaner program-title extraction.

Major deltas vs v7:
- Structured-only control hits for evidence-tier (anchors/headings/list items), not full body text.
- Stronger inventory signature (hub-ness): program-like links, distinct anchors, slug diversity.
- Scaled detail-page penalty (slug pages can't beat hubs unless they truly look like hubs).
- Canonical hub URL bonus (exact /programs, /academics/programs, /majors-minors, /departments-and-programs, etc.).
- Compliance/CIP pages penalized for inventory pick (still allowed as alternates/evidence).
- Year boosting tightened (catalog/bulletin only) + awards/honors penalty.
- Evidence-tier requires list-ness (inventory signature or strong struct hits or canonical hub).
- Subdomain probing root-first; only append paths if root looks weak; improved soft-404 detection.
- Program-title extraction hardened (drop course codes/units/semester, drop person-name-only).
- Program titles extracted only from title-like markup sources (anchors/headings/list items/H1-title).

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv  (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v8.csv

TEST MODE:
- Set TEST_HEAD_N (e.g., 5). Set None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================
# Config
# ============================
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v8.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

TEST_HEAD_N = 25  # set None for full run

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.30
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.25
MAX_BYTES_PER_PAGE = 2_000_000

CACHE_DIR = Path(".cache_v8_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# budgets per institution
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 12
PHASE1_MAX_HOMEPAGE_LINKS = 25

PHASE2_BESTFIRST_MAX_PAGES = 65
PHASE2_MAX_DEPTH = 3

TOP_K = 6  # candidates to aggregate from
ONE_HOP_MAX_LINKS_TOTAL = 45
ONE_HOP_MAX_FETCHES = 22

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 260

# Optional Selenium fallback (kept off by default for speed)
USE_SELENIUM_FALLBACK = False
SELENIUM_MIN_SLEEP_SEC = 0.9
SELENIUM_WAIT_TIMEOUT_SEC = 15

MIN_VISIBLE_TEXT_LEN = 900
MIN_ANCHOR_CORPUS_LEN = 300

# Optional PDF scanning fallback (rare)
ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/8.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================
# URL templates (expanded)
# ============================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================
# Patterns / scoring signals
# ============================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_PENALTY_REGEX = re.compile(
    r"\b(graduate|grad\s+studies|master'?s|ph\.?d|doctoral|mba|m\.?s\.?|m\.?a\.?|mfa|postbac|certificate)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_BOOST_REGEX = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)

ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)

PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)

PROFILE_TEXT_PENALTY = re.compile(
    r"\b(email|e-mail|telephone|phone|office|cv|curriculum\s+vitae|publications|research\s+interests)\b",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

ARCHIVE_TEXT_PENALTY = re.compile(r"\barchive\b|\bpast\s+catalogs\b", flags=re.IGNORECASE)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)

AWARDS_PENALTY = re.compile(r"\b(awards?|honors?|recognition|prizes?)\b", flags=re.IGNORECASE)

COMPLIANCE_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    # base controls
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    # added controls
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    # careful ones
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    # more STEM
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}\s*[-]?\s*\d{2,3}[A-Z]?\b")
COURSE_WORDS = re.compile(r"\b(units?:|credits?:|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
RECOMMENDED_COURSE = re.compile(r"\brecommended\s+(first\s+)?course(s)?\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")

PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================
# Unicode normalization
# ============================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)

    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")

    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))

    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}

CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================
# Cache + fetch
# ============================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"

def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)

    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass

    return content

def fetch_text_cached(url: str, session: requests.Session) -> str:
    content = fetch_bytes_cached(url, session=session)
    try:
        return content.decode("utf-8", errors="replace")
    except Exception:
        return content.decode("latin-1", errors="replace")


# ============================
# Optional Selenium fallback
# ============================
def selenium_fetch_rendered(url: str) -> Optional[Tuple[str, str]]:
    if not USE_SELENIUM_FALLBACK:
        return None
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
    except Exception:
        return None

    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--blink-settings=imagesEnabled=false")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_MIN_SLEEP_SEC)
        WebDriverWait(driver, SELENIUM_WAIT_TIMEOUT_SEC).until(
            lambda d: len(d.find_element(By.TAG_NAME, "body").text.strip()) > 0
        )
        body_text = normalize_unicode_text(driver.find_element(By.TAG_NAME, "body").text)
        html = driver.page_source
        return body_text, html
    except Exception:
        return None
    finally:
        driver.quit()


# ============================
# HTML parse helpers
# ============================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url

def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False

def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)

def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)

def is_canonical_hub_url(url: str) -> bool:
    """
    Exact “hub endpoints” should win inventory selection.
    """
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical


@dataclass
class ParsedPage:
    url: str
    visible_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]       # (anchor_text, abs_url)
    title_like: List[str]              # h1/h2/h3/li candidates
    html_title: str                    # <title> text
    corpus_any: str                    # visible + anchors + attrs
    corpus_structured: str             # anchors + attrs + title_like (NO visible)


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    # capture <title> before stripping
    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x)
    )

    corpus_structured = normalize_unicode_text(
        " ".join(x for x in [" ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)] if x)
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
    )

def looks_thin_for_inventory(parsed: ParsedPage) -> bool:
    anchor_len = len(" ".join(parsed.anchor_texts))
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) or (anchor_len < MIN_ANCHOR_CORPUS_LEN)

def is_soft_404(parsed: ParsedPage) -> bool:
    title_blob = normalize_unicode_text(" ".join([parsed.html_title] + parsed.title_like[:5]))
    # strong 404 signal in title/h1
    if SOFT_404_TEXT.search(title_blob):
        return True
    # small + not found in body
    if len(parsed.visible_text) < 800 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    # explicit "not found" anywhere and extremely low structured content
    if SOFT_404_TEXT.search(parsed.corpus_any) and len(parsed.corpus_structured) < 250:
        return True
    return False

def get_parsed_page_requests_only(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)

def get_parsed_page_with_optional_selenium(url: str, session: requests.Session, allow_selenium: bool) -> ParsedPage:
    parsed = get_parsed_page_requests_only(url, session=session)
    if is_soft_404(parsed):
        return parsed
    if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url) and looks_thin_for_inventory(parsed):
        rendered = selenium_fetch_rendered(url)
        if rendered is not None:
            body_text, page_source_html = rendered
            parsed2 = parse_html_to_parsedpage(page_source_html, page_url=url)
            parsed2.visible_text = body_text
            parsed2.corpus_any = normalize_unicode_text(
                " ".join([body_text, " ".join(parsed2.anchor_texts), " ".join(parsed2.anchor_attr_texts)])
            )
            parsed2.corpus_structured = normalize_unicode_text(
                " ".join([" ".join(parsed2.anchor_texts), " ".join(parsed2.anchor_attr_texts), " ".join(parsed2.title_like)])
            )
            return parsed2
    return parsed


# ============================
# Sitemap parsing
# ============================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()

def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls

def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue
        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    seen: Set[str] = set()
    out: List[str] = []
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break
    return out


# ============================
# Inventory signature (stronger)
# ============================
def compute_inventory_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    internal_links = 0
    program_like_links = 0

    distinct_anchor: Set[str] = set()
    slug_prefixes: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        internal_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        # program-like anchor text heuristic
        if len(t) >= 6 and not re.search(
            r"\b(apply|visit|donate|giving|news|event|calendar|admissions|faculty|staff|directory|alumni|give)\b",
            t,
            re.I,
        ):
            program_like_links += 1

        # slug diversity under common hubs
        path = (urlparse(u).path or "").strip("/").lower()
        if not path:
            continue
        m = re.match(r"^(majors-minors|programs|departments-and-programs|departments)/([^/]+)/", path)
        if m:
            slug_prefixes.add(f"{m.group(1)}/{m.group(2)}")

    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    # original-ish thresholds (kept)
    if program_like_links >= 18:
        sig += 1
    if internal_links >= 45:
        sig += 1
    if li_like >= 25:
        sig += 1

    # stronger thresholds (new)
    if program_like_links >= 30:
        sig += 1
    if len(distinct_anchor) >= 35:
        sig += 1
    if len(slug_prefixes) >= 12:
        sig += 1

    counts = {
        "internal_links": internal_links,
        "program_like_links": program_like_links,
        "distinct_anchor": len(distinct_anchor),
        "slug_diversity": len(slug_prefixes),
        "li_like": li_like,
    }
    return sig, counts


# ============================
# Scoring
# ============================
@dataclass
class PageSignals:
    url: str
    tier: str

    year_hit: int
    undergrad_boost: int
    grad_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    profile_url_penalty: int
    profile_text_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int
    soft404: int

    control_hits_any: Dict[str, int]
    control_unique_any: int
    control_hits_structured: Dict[str, int]
    control_unique_structured: int

    struct_hits: List[str]
    any_control: int
    any_struct: int

    inventory_signature: int
    inv_counts: Dict[str, int]

def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if any(h in u for h in INVENTORY_URL_HINTS):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"

def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    control_hits_any = {k: 1 if pat.search(parsed.corpus_any) else 0 for k, pat in CONTROL_PATS.items()}
    control_unique_any = sum(control_hits_any.values())

    control_hits_structured = {k: 1 if pat.search(parsed.corpus_structured) else 0 for k, pat in CONTROL_PATS.items()}
    control_unique_structured = sum(control_hits_structured.values())

    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    year_hit = 1 if YEAR_REGEX.search(url) or YEAR_REGEX.search(parsed.corpus_any) else 0
    undergrad_boost = 1 if UNDERGRAD_BOOST_REGEX.search(url) or UNDERGRAD_BOOST_REGEX.search(parsed.corpus_any) else 0
    grad_penalty = 1 if GRAD_PENALTY_REGEX.search(url) or GRAD_PENALTY_REGEX.search(parsed.corpus_any) else 0

    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(url.lower()) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(url.lower()) else 0
    profile_url_penalty = 1 if PROFILE_URL_PENALTY.search(url.lower()) else 0
    profile_text_penalty = 1 if PROFILE_TEXT_PENALTY.search(parsed.visible_text) else 0
    archive_penalty = 1 if (ARCHIVE_URL_PENALTY.search(url.lower()) or ARCHIVE_TEXT_PENALTY.search(parsed.visible_text)) else 0
    awards_penalty = 1 if (AWARDS_PENALTY.search(url.lower()) or AWARDS_PENALTY.search(parsed.html_title) or AWARDS_PENALTY.search(parsed.visible_text)) else 0
    compliance_penalty = 1 if COMPLIANCE_PENALTY.search(url.lower()) else 0
    soft404 = 1 if is_soft_404(parsed) else 0

    any_control = 1 if control_unique_any > 0 else 0
    any_struct = 1 if len(struct_hits) > 0 else 0

    inv_sig, inv_counts = compute_inventory_signature(parsed, base_netloc=base_netloc)

    return PageSignals(
        url=url,
        tier=page_tier(url),
        year_hit=year_hit,
        undergrad_boost=undergrad_boost,
        grad_penalty=grad_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        profile_url_penalty=profile_url_penalty,
        profile_text_penalty=profile_text_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
        soft404=soft404,
        control_hits_any=control_hits_any,
        control_unique_any=control_unique_any,
        control_hits_structured=control_hits_structured,
        control_unique_structured=control_unique_structured,
        struct_hits=struct_hits,
        any_control=any_control,
        any_struct=any_struct,
        inventory_signature=inv_sig,
        inv_counts=inv_counts,
    )

def is_detail_slug_page(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return re.search(r"/(majors-minors|programs|departments-and-programs|departments)/[^/]+/", path) is not None

def score_for_inventory(sig: PageSignals) -> int:
    if sig.soft404:
        return -10_000

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 45

    # canonical hub bonus
    if is_canonical_hub_url(sig.url):
        score += 160

    # structural evidence dominates
    score += 75 * sig.inventory_signature
    score += 18 * len(sig.struct_hits)

    # modest control contribution (use structured; safer)
    score += 5 * sig.control_unique_structured

    # boosts
    if sig.undergrad_boost:
        score += 45

    # penalties (hard)
    if sig.grad_penalty:
        score -= 190
    if sig.profile_url_penalty or sig.profile_text_penalty:
        score -= 270
    if sig.irrelevant_penalty:
        score -= 250
    if sig.admissions_penalty:
        score -= 150
    if sig.archive_penalty:
        score -= 220

    # new targeted penalties
    if sig.awards_penalty:
        score -= 220
    if sig.compliance_penalty:
        score -= 220

    # tightened year boost: only for catalog/bulletin contexts
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        # require it to still look academic-ish
        if sig.inventory_signature >= 1 or len(sig.struct_hits) >= 2:
            score += 85

    # scaled detail penalty
    if is_detail_slug_page(sig.url):
        if sig.inventory_signature == 0:
            score -= 180
        elif sig.inventory_signature == 1:
            score -= 90
        else:
            score -= 20

    return score

def evidence_guardrail(sig: PageSignals) -> bool:
    if sig.soft404 or sig.archive_penalty:
        return False
    if sig.profile_url_penalty or sig.profile_text_penalty:
        return False
    if sig.irrelevant_penalty or sig.admissions_penalty:
        return False
    if sig.grad_penalty and sig.inventory_signature == 0 and not is_canonical_hub_url(sig.url):
        return False

    # Must be list-ish: inventory signature OR strong struct hits OR canonical hub
    if sig.inventory_signature >= 1:
        return True
    if len(sig.struct_hits) >= 3 and sig.control_unique_structured >= 2:
        return True
    if is_canonical_hub_url(sig.url) and len(sig.struct_hits) >= 2:
        return True
    return False

def score_for_evidence(sig: PageSignals) -> int:
    if not evidence_guardrail(sig):
        return -10_000

    score = 0

    # Structured-only controls dominate evidence scoring
    score += 50 * sig.control_unique_structured

    # still reward hub-ness
    score += 40 * sig.inventory_signature
    score += 10 * len(sig.struct_hits)

    if sig.tier == "inventory":
        score += 35
    elif sig.tier in ("catalog", "bulletin"):
        score += 10

    if is_canonical_hub_url(sig.url):
        score += 40

    if sig.undergrad_boost:
        score += 15
    if sig.grad_penalty:
        score -= 90

    # keep additional penalties
    if sig.awards_penalty:
        score -= 220
    if sig.compliance_penalty:
        score -= 80  # allow compliance pages to still serve as “evidence” if structured controls are high

    # no year bonus here (avoid awards pages etc.)
    return score

def is_good_inventory_candidate(sig: PageSignals) -> bool:
    if sig.soft404 or sig.archive_penalty:
        return False
    if sig.profile_url_penalty or sig.profile_text_penalty:
        return False
    if sig.grad_penalty:
        return False
    if sig.irrelevant_penalty and sig.inventory_signature == 0:
        return False
    if sig.admissions_penalty and sig.inventory_signature == 0:
        return False
    if sig.awards_penalty and sig.inventory_signature <= 1:
        return False
    # must look like a list/hub
    return (sig.inventory_signature >= 1) and (len(sig.struct_hits) >= 2 or sig.control_unique_structured >= 4 or is_canonical_hub_url(sig.url))


# ============================
# Link prioritization / best-first URL prior
# ============================
def url_prior(url: str) -> int:
    u = url.lower()
    s = 0
    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 220
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 30
    if "bulletin" in u:
        s -= 40
    if AWARDS_PENALTY.search(u):
        s -= 180
    if COMPLIANCE_PENALTY.search(u):
        s -= 90
    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 340
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if "graduate" in u or "grad" in u:
        s -= 200
    # canonical hub bump
    if is_canonical_hub_url(url):
        s += 160
    return s

def prioritize_links(links: List[Tuple[str, str]], base_netloc: str) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000
        s = url_prior(u)
        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 40
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 80
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140
        if len(a_text or "") < 3:
            s -= 10
        return -s
    return sorted(links, key=rank)


# ============================
# Candidate generation helpers
# ============================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq

def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================
# PDF fallback (best-effort)
# ============================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""

def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)
    seen: Set[str] = set()
    out: List[str] = []
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================
# Discovery: collect candidates + score both tiers
# ============================
@dataclass
class CandidatePage:
    url: str
    inv_score: int
    ev_score: int
    sig: PageSignals
    parsed: ParsedPage

def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.inv_score, len(topk), cand))
        return
    if cand.inv_score > topk[0][0]:
        heapq.heapreplace(topk, (cand.inv_score, topk[0][1], cand))

def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    return [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]

def fetch_and_score(url: str, base_netloc: str, session: requests.Session, allow_selenium: bool) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page_with_optional_selenium(url, session=session, allow_selenium=allow_selenium)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        inv_score = score_for_inventory(sig)
        ev_score = score_for_evidence(sig)
        return CandidatePage(url=url, inv_score=inv_score, ev_score=ev_score, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

def find_candidates_for_institution(base_url: str, session: requests.Session) -> Tuple[List[CandidatePage], int]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    used_selenium_flag = 0
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_candidates_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_candidates_cache
        if sitemap_candidates_cache is not None:
            return sitemap_candidates_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_candidates_cache = sitemap_candidate_urls(all_urls)
        return sitemap_candidates_cache

    def consider(url: str, allow_selenium: bool) -> Optional[CandidatePage]:
        nonlocal used_selenium_flag
        if not url or url in tried:
            return None
        tried.add(url)

        cand = fetch_and_score(url, base_netloc=base_netloc, session=session, allow_selenium=allow_selenium)
        if cand is None:
            return None

        if allow_selenium and USE_SELENIUM_FALLBACK and is_inventoryish_url(url):
            used_selenium_flag = 1

        push_topk(topk, cand, k=TOP_K)
        return cand

    # Phase 0: base root
    consider(base_url, allow_selenium=False)

    # Phase 1A: explicit inventory paths
    for url in [urljoin(base_url, p) for p in INVENTORY_PATHS][:PHASE1_MAX_INVENTORY_CANDIDATES]:
        cand = consider(url, allow_selenium=False)
        if cand and is_inventoryish_url(url) and looks_thin_for_inventory(cand.parsed):
            # sitemap-first then optional selenium
            for u2 in load_sitemap_candidates():
                consider(u2, allow_selenium=False)
            if USE_SELENIUM_FALLBACK:
                consider(url, allow_selenium=True)

    # Phase 1B: year-aware catalog templates
    for url in build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES]:
        consider(url, allow_selenium=False)

    # Phase 1C: generic catalog paths
    for url in [urljoin(base_url, p) for p in CATALOG_PATHS][:PHASE1_MAX_CATALOG_CANDIDATES]:
        consider(url, allow_selenium=False)

    # Phase 1D: sample prioritized homepage links
    try:
        home = get_parsed_page_requests_only(base_url, session=session)
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            consider(u, allow_selenium=False)
    except Exception:
        pass

    # Phase 2: best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u), depth, u))

    pq_push(base_url, 0)
    pages_fetched = 0

    while pq and pages_fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u, allow_selenium=False)
        pages_fetched += 1

        if depth >= PHASE2_MAX_DEPTH:
            continue

        try:
            parsed = cand.parsed if cand is not None else get_parsed_page_requests_only(u, session=session)
            links = [(t, link_u) for (t, link_u) in parsed.links if same_domain(link_u, base_netloc)]
            links = prioritize_links(links, base_netloc)
            for _, link_u in links[:80]:
                pq_push(link_u, depth + 1)
        except Exception:
            pass

    # Subdomain probing: ROOT FIRST, only append paths if root looks weak
    for root in build_subdomain_roots(base_url):
        c_root = consider(root, allow_selenium=False)
        if not c_root:
            continue

        # If root is already list-ish, do NOT force path appends (fixes Boise-like behavior)
        root_is_good = (c_root.sig.inventory_signature >= 1) or (len(c_root.sig.struct_hits) >= 2) or is_canonical_hub_url(root)
        if root_is_good and not is_soft_404(c_root.parsed):
            continue

        # Otherwise try appended paths
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p), allow_selenium=False)

    return topk_sorted(topk), used_selenium_flag


# ============================
# Program-title extraction (hardened)
# ============================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_program_title(s: str) -> bool:
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False
    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        return False

    # must mention target tokens
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False

    # reject course-like lines
    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm) or RECOMMENDED_COURSE.search(s_norm):
        return False

    # reject person-name-only unless program keywords appear
    if PERSON_NAME_ONLY.match(s_norm) and not PROGRAM_KEYWORDS.search(s_norm):
        return False

    return True

def extract_h1_or_title_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    h1 = soup.find("h1")
    if h1:
        return normalize_unicode_text(h1.get_text(" ", strip=True))
    t = soup.find("title")
    if t:
        return normalize_unicode_text(t.get_text(" ", strip=True))
    return ""

def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out

def aggregate_program_titles_and_tokens(
    candidates: List[CandidatePage],
    base_url: str,
    session: requests.Session,
) -> Tuple[List[str], Dict[str, List[str]], List[str]]:
    base_netloc = urlparse(base_url).netloc

    titles: Set[str] = set()
    tokens_agg: Dict[str, Set[str]] = {"afri": set(), "ethnic": set(), "black": set()}
    pdf_hits: List[str] = []

    # tokens: use ANY text (diagnostic)
    for cand in candidates:
        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k in tm:
            tokens_agg[k].update(tm[k])

    # program titles: ONLY from structured sources
    for cand in candidates:
        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt):
                titles.add(txt)

    # 1-hop follow likely program links (still bounded)
    follow_urls: List[str] = []
    for cand in candidates:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            # follow if anchor text looks like a target-program title, or URL contains tokens
            if looks_like_program_title(a_text) or TARGET_ANY_REGEX.search(ul):
                follow_urls.append(u)

    # If thin inventory pages detected, add sitemap candidates
    if any(is_inventoryish_url(c.url) and looks_thin_for_inventory(c.parsed) for c in candidates):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    # de-dup cap
    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    fetches = 0
    for u in uniq_follow:
        if fetches >= ONE_HOP_MAX_FETCHES:
            break
        try:
            html = fetch_text_cached(u, session=session)
            # H1/title candidate
            h1t = extract_h1_or_title_from_html(html)
            if h1t and looks_like_program_title(h1t):
                titles.add(h1t)

            parsed = parse_html_to_parsedpage(html, page_url=u)

            # tokens (diagnostic)
            tm = token_matches_from_text(parsed.corpus_any)
            for k in tm:
                tokens_agg[k].update(tm[k])

            # titles (structured only)
            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt):
                    titles.add(txt)

            fetches += 1
        except Exception:
            continue
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in candidates:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k in tm:
                            tokens_agg[k].update(tm[k])
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Dedup titles by normalized key; keep best display
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    titles_out = sorted(best_display.values())
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}
    return titles_out, tokens_out, pdf_hits


# ============================
# Inventory reason string
# ============================
def inventory_reason(c: CandidatePage) -> str:
    sig = c.sig
    parts = [
        f"inv_score={c.inv_score}",
        f"tier={sig.tier}",
        f"invSig={sig.inventory_signature}",
        f"progLinks={sig.inv_counts.get('program_like_links', 0)}",
        f"distinctA={sig.inv_counts.get('distinct_anchor', 0)}",
        f"slugDiv={sig.inv_counts.get('slug_diversity', 0)}",
        f"structHits={len(sig.struct_hits)}",
        f"ctrlStructured={sig.control_unique_structured}",
        f"canonicalHub={int(is_canonical_hub_url(sig.url))}",
    ]
    flags = []
    if sig.grad_penalty: flags.append("grad")
    if sig.profile_url_penalty or sig.profile_text_penalty: flags.append("profile")
    if sig.irrelevant_penalty: flags.append("news/events/etc")
    if sig.admissions_penalty: flags.append("admissions")
    if sig.archive_penalty: flags.append("archive")
    if sig.awards_penalty: flags.append("awards")
    if sig.compliance_penalty: flags.append("compliance")
    if sig.year_hit: flags.append("yearHit")
    if sig.soft404: flags.append("soft404")
    if flags:
        parts.append("penalties=" + ",".join(flags))
    return ";".join(parts)


# ============================
# Main
# ============================
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        best_guess_inventory_url = "N/A"
        best_evidence_url = "N/A"
        alt_candidate_urls = ""
        used_selenium = 0

        best_guess_inventory_reason = ""

        any_control_found = 0
        any_struct_found = 0
        struct_hits_union: Set[str] = set()

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        evidence_control_unique = 0
        evidence_controls_matched = ""

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0

        pdf_hits: List[str] = []

        if base_url:
            candidates, used_selenium = find_candidates_for_institution(base_url, session=session)

            if candidates:
                alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K])

                # Inventory pick: highest inv_score among good inventory candidates; fallback to max inv_score overall
                inv_candidates = [c for c in candidates if is_good_inventory_candidate(c.sig)]
                inv_pick = max(inv_candidates, key=lambda c: c.inv_score) if inv_candidates else max(candidates, key=lambda c: c.inv_score)
                best_guess_inventory_url = inv_pick.url
                best_guess_inventory_reason = inventory_reason(inv_pick)

                # Evidence pick: max ev_score among guarded pages
                ev_pick = max(candidates, key=lambda c: c.ev_score)
                if ev_pick.ev_score > -9000:
                    best_evidence_url = ev_pick.url
                    evidence_control_unique = ev_pick.sig.control_unique_structured
                    evidence_controls_matched = "|".join([k for k, v in ev_pick.sig.control_hits_structured.items() if v == 1])

                # union signals across top candidates
                for c in candidates[:TOP_K]:
                    any_control_found = max(any_control_found, 1 if c.sig.control_unique_any > 0 else 0)
                    any_struct_found = max(any_struct_found, 1 if len(c.sig.struct_hits) > 0 else 0)
                    struct_hits_union.update(c.sig.struct_hits)
                    for k, v in c.sig.control_hits_any.items():
                        control_hit_cols[f"found_{k}"] = max(control_hit_cols[f"found_{k}"], int(v))

                # aggregate across top candidates + ensure evidence page included if distinct
                agg_set: List[CandidatePage] = candidates[:TOP_K].copy()
                if best_evidence_url != "N/A" and all(c.url != best_evidence_url for c in agg_set):
                    for c in candidates:
                        if c.url == best_evidence_url:
                            agg_set.append(c)
                            break

                program_titles, tokens, pdf_hits = aggregate_program_titles_and_tokens(agg_set, base_url=base_url, session=session)
                program_title_count = len(program_titles)
                any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0

                afri_matches = tokens.get("afri", [])
                ethnic_matches = tokens.get("ethnic", [])
                black_matches = tokens.get("black", [])

        rec: Dict[str, str] = dict(row)

        rec["best_guess_inventory_url"] = best_guess_inventory_url
        rec["best_guess_inventory_reason"] = best_guess_inventory_reason
        rec["best_evidence_url"] = best_evidence_url
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["evidence_control_unique_structured"] = str(evidence_control_unique)
        rec["evidence_controls_matched_structured"] = evidence_controls_matched

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""
        rec["used_selenium"] = str(used_selenium)

        rec.update({k: str(v) for k, v in control_hit_cols.items()})

        out_rows.append(rec)
        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "best_evidence_url", "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "evidence_control_unique_structured", "evidence_controls_matched_structured",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "pdf_hits",
        "used_selenium",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)


if __name__ == "__main__":
    main()

[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

•	Drop evidence-tier selection → compute all hits from the chosen inventory page + top alternates.
•	Redefine hubness → count only program-listing-style links (and slug diversity), ignoring nav/footer noise where possible.
•	Make penalties URL/title/H1-based (not full-body) to avoid boilerplate false penalties.
•	Broaden detail-page penalties for common “single-major page” URL schemes so hubs win.
•	Gate canonical-hub bonus behind “page looks real + list-like” checks; strengthen soft-404/thin-page detection.

In [3]:
#!/usr/bin/env python3
"""
v9: Simplify + improve majors/programs page selection.

Key changes vs v8:
- Remove evidence-tier selection entirely.
- Choose best inventory URL via improved hubness (program-listing link patterns + slug diversity).
- Penalties are URL/title/H1 based (not full body) to avoid boilerplate false flags.
- Broader detail-page detection so single-major pages don’t beat hubs.
- Canonical hub bonus gated behind minimal "real page" + list-like checks.
- All term/program extraction is computed from: inventory pick + top-K alternates (+ limited 1-hop).

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv  (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v9.csv

TEST MODE:
- Set TEST_HEAD_N to small integer; set None for full run.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================
# Config
# ============================
INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v9.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

TEST_HEAD_N = 25  # set None for full run

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.25
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.20
MAX_BYTES_PER_PAGE = 2_000_000

CACHE_DIR = Path(".cache_v9_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# per institution budgets
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 22
PHASE2_BESTFIRST_MAX_PAGES = 55
PHASE2_MAX_DEPTH = 3

TOP_K_ALTS = 6
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 18

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

MIN_VISIBLE_TEXT_LEN = 900
MIN_MAIN_TEXT_LEN = 650

ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/9.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================
# URL templates
# ============================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================
# Patterns / scoring signals
# ============================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

# URL-only / title-only penalties
GRAD_URL_PENALTY = re.compile(
    r"\b(graduate|grad-studies|gradstudies|master|masters|doctoral|phd|mba|mfa|ms|ma)\b",
    flags=re.IGNORECASE,
)
PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)
AWARDS_URL_PENALTY = re.compile(r"\b(awards?|honors?)\b", flags=re.IGNORECASE)
COMPLIANCE_URL_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_TITLE_BOOST = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}\s*[-]?\s*\d{2,3}[A-Z]?\b")
COURSE_WORDS = re.compile(r"\b(units?:|credits?:|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")
PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

# "Listing-like" URL patterns for anchors we count toward hubness
LISTING_LINK_PATTERNS = [
    re.compile(r"/programs/[^/]+/?$", re.I),
    re.compile(r"/majors-minors/[^/]+/?$", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/departments/[^/]+/?$", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
]

# Broader detail-page detection
DETAIL_URL_PATTERNS = [
    re.compile(r"/programs/[^/]+/.+", re.I),
    re.compile(r"/majors-minors/[^/]+/.+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/.+", re.I),
    re.compile(r"/departments/[^/]+/.+", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
    re.compile(r"/content/.*/departments/[^/]+\.html$", re.I),
]

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================
# Unicode normalization
# ============================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}

CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================
# Cache + fetch
# ============================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()

def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"

def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)
    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass
    return content

def fetch_text_cached(url: str, session: requests.Session) -> str:
    b = fetch_bytes_cached(url, session=session)
    try:
        return b.decode("utf-8", errors="replace")
    except Exception:
        return b.decode("latin-1", errors="replace")


# ============================
# URL helpers
# ============================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url

def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False

def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)

def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)

def is_canonical_hub_url(url: str) -> bool:
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical

def looks_like_detail_url(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return any(p.search(path) for p in DETAIL_URL_PATTERNS)


# ============================
# HTML parse + main-content extraction
# ============================
@dataclass
class ParsedPage:
    url: str
    visible_text: str        # whole page (scripts removed)
    main_text: str           # best-effort main content only
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]   # (anchor_text, abs_url)
    title_like: List[str]
    html_title: str
    h1: str
    corpus_any: str
    corpus_structured: str


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    h1_tag = soup.find("h1")
    h1 = normalize_unicode_text(h1_tag.get_text(" ", strip=True)) if h1_tag else ""

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    # Whole-page visible text
    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    # Attempt to get main content text (reduce nav/footer impact)
    main_node = soup.find("main") or soup.find(id=re.compile(r"content|main", re.I)) or soup.find("article")
    if main_node:
        main_text = normalize_unicode_text(main_node.get_text(separator=" ", strip=True))
    else:
        # fallback: remove header/nav/footer if present
        soup2 = BeautifulSoup(str(soup), "html.parser")
        for tag in soup2(["header", "footer", "nav"]):
            tag.decompose()
        main_text = normalize_unicode_text(soup2.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x)
    )
    corpus_structured = normalize_unicode_text(
        " ".join(x for x in [" ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)] if x)
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        main_text=main_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        h1=h1,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
    )

def is_soft_404(parsed: ParsedPage) -> bool:
    blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1] + parsed.title_like[:5]))
    if SOFT_404_TEXT.search(blob):
        return True
    # thin + 404-ish anywhere
    if len(parsed.main_text) < 600 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    # extremely thin and almost no links/anchors
    if len(parsed.main_text) < 250 and len(parsed.anchor_texts) < 5:
        return True
    return False

def page_is_thin(parsed: ParsedPage) -> bool:
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) and (len(parsed.main_text) < MIN_MAIN_TEXT_LEN)


# ============================
# Sitemap parsing
# ============================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()

def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break
    return urls

def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALALTY.search(ul) if False else False:
            pass
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue
        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break
    return out


# ============================
# Hubness: program-listing-only link counts
# ============================
def is_listing_link(url: str) -> bool:
    path = (urlparse(url).path or "")
    return any(p.search(path) for p in LISTING_LINK_PATTERNS)

def hubness_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    listing_links = 0
    slug_set: Set[str] = set()
    distinct_anchor: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        if not is_listing_link(u):
            continue
        listing_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        path = (urlparse(u).path or "").strip("/").lower()
        # slug = last segment for listing link
        segs = [s for s in path.split("/") if s]
        if segs:
            slug_set.add(segs[-1])

    # list-like items from title_like are a weak secondary signal
    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if listing_links >= 12:
        sig += 1
    if listing_links >= 25:
        sig += 1
    if len(slug_set) >= 8:
        sig += 1
    if len(slug_set) >= 16:
        sig += 1
    if len(distinct_anchor) >= 18:
        sig += 1
    if li_like >= 40 and (listing_links >= 10):
        sig += 1

    counts = {
        "listing_links": listing_links,
        "slug_diversity": len(slug_set),
        "distinct_anchor": len(distinct_anchor),
        "li_like": li_like,
    }
    return sig, counts


# ============================
# Scoring
# ============================
@dataclass
class PageSignals:
    url: str
    tier: str
    soft404: int
    thin: int

    canonical_hub: int
    hub_sig: int
    hub_counts: Dict[str, int]

    struct_hits: List[str]
    control_unique_structured: int

    year_hit: int
    undergrad_title_boost: int

    # URL/title/H1-only penalty flags
    grad_penalty: int
    profile_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int

def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if is_inventoryish_url(url):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"

def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    soft404 = 1 if is_soft_404(parsed) else 0
    thin = 1 if page_is_thin(parsed) else 0

    # structured-only controls: anchors + attrs + title_like (not body)
    control_unique_structured = sum(1 for pat in CONTROL_PATS.values() if pat.search(parsed.corpus_structured))
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    # year hit: URL or title/H1 (not whole body)
    title_blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1]))
    year_hit = 1 if (YEAR_REGEX.search(url) or YEAR_REGEX.search(title_blob)) else 0
    undergrad_title_boost = 1 if UNDERGRAD_TITLE_BOOST.search(title_blob) else 0

    # URL / title / H1 penalties
    u_low = url.lower()
    grad_penalty = 1 if (GRAD_URL_PENALTY.search(u_low) or re.search(r"\bgraduate\b", title_blob, re.I)) else 0
    profile_penalty = 1 if (PROFILE_URL_PENALTY.search(u_low) or re.search(r"\bfaculty directory\b", title_blob, re.I)) else 0
    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(u_low) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(u_low) else 0
    archive_penalty = 1 if ARCHIVE_URL_PENALTY.search(u_low) else 0
    awards_penalty = 1 if AWARDS_URL_PENALTY.search(u_low) else 0
    compliance_penalty = 1 if COMPLIANCE_URL_PENALTY.search(u_low) else 0

    hub_sig, hub_counts = hubness_signature(parsed, base_netloc=base_netloc)
    canonical_hub = 1 if is_canonical_hub_url(url) else 0

    return PageSignals(
        url=url,
        tier=page_tier(url),
        soft404=soft404,
        thin=thin,
        canonical_hub=canonical_hub,
        hub_sig=hub_sig,
        hub_counts=hub_counts,
        struct_hits=struct_hits,
        control_unique_structured=control_unique_structured,
        year_hit=year_hit,
        undergrad_title_boost=undergrad_title_boost,
        grad_penalty=grad_penalty,
        profile_penalty=profile_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
    )

def canonical_bonus_allowed(sig: PageSignals) -> bool:
    # gate canonical bonus behind "real page" and at least somewhat list-like
    if sig.soft404:
        return False
    if sig.thin and sig.hub_sig == 0 and len(sig.struct_hits) < 2:
        return False
    return True

def score_inventory(sig: PageSignals) -> int:
    if sig.soft404:
        return -10_000

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 40

    # hubness dominates
    score += 85 * sig.hub_sig
    score += 10 * len(sig.struct_hits)
    score += 4 * sig.control_unique_structured

    # canonical hub bonus (gated)
    if sig.canonical_hub and canonical_bonus_allowed(sig):
        score += 160

    # undergrad title boost
    if sig.undergrad_title_boost:
        score += 35

    # year boost only for catalog/bulletin pages
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        score += 70

    # penalties (URL/title/H1)
    if sig.profile_penalty:
        score -= 260
    if sig.grad_penalty:
        score -= 170
    if sig.irrelevant_penalty:
        score -= 260
    if sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 220
    if sig.awards_penalty:
        score -= 200
    if sig.compliance_penalty:
        score -= 150

    # broad detail penalty so hubs win
    if looks_like_detail_url(sig.url):
        # allow detail to win only if it truly behaves like a hub
        if sig.hub_sig <= 1:
            score -= 220
        elif sig.hub_sig == 2:
            score -= 90
        else:
            score -= 25

    # thin page penalty
    if sig.thin and sig.hub_sig == 0:
        score -= 180

    return score


def url_prior(url: str) -> int:
    u = url.lower()
    s = 0
    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 200
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 35
    if "bulletin" in u:
        s -= 20
    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if COMPLIANCE_URL_PENALTY.search(u):
        s -= 90
    if AWARDS_URL_PENALTY.search(u):
        s -= 160
    if GRAD_URL_PENALTY.search(u):
        s -= 180
    if is_canonical_hub_url(url):
        s += 160
    return s

def prioritize_links(links: List[Tuple[str, str]], base_netloc: str) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000
        s = url_prior(u)
        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 45
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 90
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140
        return -s
    return sorted(links, key=rank)


# ============================
# Candidate structure
# ============================
@dataclass
class CandidatePage:
    url: str
    score: int
    sig: PageSignals
    parsed: ParsedPage


def get_parsed_page(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)

def fetch_and_score(url: str, session: requests.Session, base_netloc: str) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page(url, session=session)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        sc = score_inventory(sig)
        return CandidatePage(url=url, score=sc, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.score, len(topk), cand))
        return
    if cand.score > topk[0][0]:
        heapq.heapreplace(topk, (cand.score, topk[0][1], cand))

def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    return [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]


# ============================
# Candidate generation
# ============================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq

def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================
# Program title extraction
# ============================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_program_title(s: str) -> bool:
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False
    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        return False
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False
    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm):
        return False
    if PERSON_NAME_ONLY.match(s_norm) and not PROGRAM_KEYWORDS.search(s_norm):
        return False
    return True

def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out


# ============================
# PDF fallback (best-effort)
# ============================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""

def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)
    out: List[str] = []
    seen: Set[str] = set()
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================
# Aggregate extraction from inventory + alts
# ============================
def aggregate_outputs(
    pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
) -> Tuple[
    int, Set[str], Dict[str, int],  # any_control_found, struct_hits_union, control_hit_cols
    Dict[str, List[str]],           # tokens (afri/ethnic/black)
    List[str],                      # program_titles
    List[str],                      # pdf_hits
]:
    base_netloc = urlparse(base_url).netloc

    struct_hits_union: Set[str] = set()
    control_hit_cols = {k: 0 for k in CONTROL_TERMS}
    any_control_found = 0

    tokens_agg: Dict[str, Set[str]] = {"afri": set(), "ethnic": set(), "black": set()}
    titles: Set[str] = set()

    pdf_hits: List[str] = []

    # from chosen pages
    for cand in pages:
        any_control_found = max(any_control_found, 1 if cand.sig.control_unique_structured > 0 else 0)
        struct_hits_union.update(cand.sig.struct_hits)

        for ck, pat in CONTROL_PATS.items():
            if pat.search(cand.parsed.corpus_structured):
                control_hit_cols[ck] = 1

        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k, vals in tm.items():
            tokens_agg[k].update(vals)

        # program titles from structured sources only
        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt):
                titles.add(txt)

    # 1-hop follow target-looking links from the best inventory page + top alts (bounded)
    follow_urls: List[str] = []
    for cand in pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            if looks_like_program_title(a_text) or TARGET_ANY_REGEX.search(ul):
                follow_urls.append(u)

    # Add sitemap candidates if the inventory page is thin
    if pages and page_is_thin(pages[0].parsed):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    fetches = 0
    for u in uniq_follow:
        if fetches >= ONE_HOP_MAX_FETCHES:
            break
        try:
            parsed = get_parsed_page(u, session=session)

            # tokens
            tm = token_matches_from_text(parsed.corpus_any)
            for k, vals in tm.items():
                tokens_agg[k].update(vals)

            # titles from structured only
            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt):
                    titles.add(txt)

            fetches += 1
        except Exception:
            continue
        finally:
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in pages:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k, vals in tm.items():
                            tokens_agg[k].update(vals)
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Dedup titles by normalized key
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t
    titles_out = sorted(best_display.values())

    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}

    return any_control_found, struct_hits_union, control_hit_cols, tokens_out, titles_out, pdf_hits


# ============================
# Discovery: find candidate pages
# ============================
def find_candidates_for_institution(base_url: str, session: requests.Session) -> List[CandidatePage]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_cache
        if sitemap_cache is not None:
            return sitemap_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_cache = sitemap_candidate_urls(all_urls)
        return sitemap_cache

    def consider(u: str) -> Optional[CandidatePage]:
        if not u or u in tried:
            return None
        tried.add(u)
        cand = fetch_and_score(u, session=session, base_netloc=base_netloc)
        if cand is None:
            return None
        push_topk(topk, cand, k=TOP_K_ALTS)
        return cand

    # Phase 0: base root
    consider(base_url)

    # Phase 1: explicit inventory paths
    for p in INVENTORY_PATHS[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        c = consider(urljoin(base_url, p))
        # if thin, try sitemap candidates early
        if c and (c.sig.thin or c.sig.hub_sig == 0) and is_inventoryish_url(c.url):
            for u2 in load_sitemap_candidates()[:50]:
                consider(u2)

    # Year-aware catalog templates
    for u in build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES]:
        consider(u)

    # Generic catalog paths
    for p in CATALOG_PATHS[:PHASE1_MAX_CATALOG_CANDIDATES]:
        consider(urljoin(base_url, p))

    # Prioritized homepage links
    try:
        home = get_parsed_page(base_url, session=session)
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            consider(u)
    except Exception:
        pass

    # Best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u), depth, u))

    pq_push(base_url, 0)
    fetched = 0
    while pq and fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u)
        fetched += 1
        if depth >= PHASE2_MAX_DEPTH:
            continue
        try:
            parsed = cand.parsed if cand is not None else get_parsed_page(u, session=session)
            links = [(t, lu) for (t, lu) in parsed.links if same_domain(lu, base_netloc)]
            links = prioritize_links(links, base_netloc)
            for _, lu in links[:70]:
                pq_push(lu, depth + 1)
        except Exception:
            pass

    # Subdomain roots (root-first; don’t append unless root looks weak)
    for root in build_subdomain_roots(base_url):
        c_root = consider(root)
        if not c_root:
            continue
        root_is_good = (c_root.sig.hub_sig >= 1) or (len(c_root.sig.struct_hits) >= 2) or (not c_root.sig.thin)
        if root_is_good and not c_root.sig.soft404:
            continue
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p))

    return topk_sorted(topk)


# ============================
# Reason string
# ============================
def inventory_reason(c: CandidatePage) -> str:
    s = c.sig
    flags = []
    if s.grad_penalty: flags.append("grad")
    if s.profile_penalty: flags.append("profile")
    if s.irrelevant_penalty: flags.append("irrelevant")
    if s.admissions_penalty: flags.append("admissions")
    if s.archive_penalty: flags.append("archive")
    if s.awards_penalty: flags.append("awards")
    if s.compliance_penalty: flags.append("compliance")
    if s.year_hit: flags.append("yearHit")
    if s.thin: flags.append("thin")
    if s.soft404: flags.append("soft404")

    return ";".join([
        f"score={c.score}",
        f"tier={s.tier}",
        f"hubSig={s.hub_sig}",
        f"listingLinks={s.hub_counts.get('listing_links', 0)}",
        f"slugDiv={s.hub_counts.get('slug_diversity', 0)}",
        f"structHits={len(s.struct_hits)}",
        f"ctrlStructured={s.control_unique_structured}",
        f"canonicalHub={s.canonical_hub}",
        ("flags=" + ",".join(flags)) if flags else "flags=",
    ])


# ============================
# Main
# ============================
def main() -> None:
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {INPUT_PATH}: {sorted(missing)}")

    if TEST_HEAD_N is not None:
        df = df.head(TEST_HEAD_N).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        best_inventory_url = "N/A"
        best_inventory_reason = ""
        alt_candidate_urls = ""

        any_control_found = 0
        struct_hits_union: Set[str] = set()
        any_struct_found = 0

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        any_potential_AfrAmr_found = 0
        program_title_count = 0
        pdf_hits: List[str] = []

        if base_url:
            candidates = find_candidates_for_institution(base_url, session=session)

            if candidates:
                best = candidates[0]
                best_inventory_url = best.url
                best_inventory_reason = inventory_reason(best)
                alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K_ALTS])

                # aggregate from best + alts
                any_control_found, struct_hits_union, control_hits, tokens, titles, pdf_hits = aggregate_outputs(
                    pages=candidates[:TOP_K_ALTS],
                    base_url=base_url,
                    session=session,
                )

                any_struct_found = 1 if len(struct_hits_union) > 0 else 0

                for k, v in control_hits.items():
                    control_hit_cols[f"found_{k}"] = int(v)

                afri_matches = tokens.get("afri", [])
                ethnic_matches = tokens.get("ethnic", [])
                black_matches = tokens.get("black", [])

                program_titles = titles
                program_title_count = len(program_titles)
                any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0

        rec: Dict[str, str] = dict(row)

        rec["best_guess_inventory_url"] = best_inventory_url
        rec["best_guess_inventory_reason"] = best_inventory_reason
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        rec.update({k: str(v) for k, v in control_hit_cols.items()})
        out_rows.append(rec)

        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "pdf_hits",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nWrote: {OUTPUT_PATH} | rows,cols = {out_df.shape}")
    print(out_df.head())
    print(out_df.shape)


if __name__ == "__main__":
    main()

[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

parse to fine tune scoring

In [8]:
#!/usr/bin/env python3
"""
v10: scoring knobs + row tagging + notebook sweep support (clean)

Key additions vs v9:
- Row tagging for best_guess_inventory_url:
    * Correct hub
    * Correct-ish but too specific
    * Wrong subsite
    * Wrong non-academic
- CLI knobs for sweep runs from ipynb:
    --subsite-penalty (int): penalty applied to subsite-like pages
    --progtitle-strictness (int): controls how strict program title extraction is
    --head (int): limit rows for quick testing
    --out-suffix (str): appended to output filename stem
    --input / --output: file paths

Notebook-friendly:
- TEST_HEAD_N provides default subset size when --head is not provided.
  Use --head 0 for full run.
- DEFAULT_WORKERS provides a top-level default for --workers.

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v10.csv (default)
"""

from __future__ import annotations

import argparse
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ============================================================
# Thread-local requests sessions (for safe parallel fetching)
# ============================================================
_thread_local = threading.local()


def get_thread_session() -> requests.Session:
    """Return a per-thread requests.Session (requests.Session is not thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        _thread_local.session = s
    return s



# ============================================================
# Notebook-friendly default test mode
# ============================================================
# If you run from an ipynb as:  !python v10_program_inventory.py
# it will default to processing TEST_HEAD_N rows unless you pass --head.
# Use --head 0 for full run.
TEST_HEAD_N = 25

# ============================================================
# Top-level parallelism defaults
# ============================================================
# You can override via CLI: --workers
# Keep MAX_WORKERS small to stay polite to university sites.
DEFAULT_WORKERS = 4
MAX_WORKERS = 8


# ============================================================
# Defaults (override via CLI)
# ============================================================
DEFAULT_INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
DEFAULT_OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v10.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.25
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.20
MAX_BYTES_PER_PAGE = 2_000_000

CACHE_DIR = Path(".cache_v10_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Per-institution budgets
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 22
PHASE2_BESTFIRST_MAX_PAGES = 55
PHASE2_MAX_DEPTH = 3

TOP_K_ALTS = 6
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 18

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

MIN_VISIBLE_TEXT_LEN = 900
MIN_MAIN_TEXT_LEN = 650

ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/10.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================================================
# URL templates
# ============================================================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================================================
# Patterns / scoring signals
# ============================================================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_URL_PENALTY = re.compile(
    r"\b(graduate|grad-studies|gradstudies|master|masters|doctoral|phd|mba|mfa|ms|ma)\b",
    flags=re.IGNORECASE,
)
PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)
AWARDS_URL_PENALTY = re.compile(r"\b(awards?|honors?)\b", flags=re.IGNORECASE)
COMPLIANCE_URL_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_TITLE_BOOST = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}\s*[-]?\s*\d{2,3}[A-Z]?\b")
COURSE_WORDS = re.compile(r"\b(units?:|credits?:|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")
PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

# "Listing-like" URL patterns for anchors we count toward hubness
LISTING_LINK_PATTERNS = [
    re.compile(r"/programs/[^/]+/?$", re.I),
    re.compile(r"/majors-minors/[^/]+/?$", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/departments/[^/]+/?$", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
]

# Broader detail-page detection (so single-major pages don’t beat hubs)
DETAIL_URL_PATTERNS = [
    re.compile(r"/programs/[^/]+/.+", re.I),
    re.compile(r"/majors-minors/[^/]+/.+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/.+", re.I),
    re.compile(r"/departments/[^/]+/.+", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
    re.compile(r"/content/.*/departments/[^/]+\.html$", re.I),
]

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================================================
# Row tagging heuristics
# ============================================================
HUB_PATH_HINTS = [
    "majors", "minors", "programs", "degrees", "department", "departments",
    "departments-and-programs", "academics", "academic", "catalog", "bulletin",
    "undergraduate", "undergraduate-programs", "courses", "areas-of-study"
]

NON_ACADEMIC_HINTS = [
    "news", "event", "calendar", "giving", "alumni", "advancement",
    "athletics", "sports", "tickets", "shop", "bookstore",
    "it", "its", "help", "support", "login", "privacy", "accessibility",
    "admissions", "apply", "visit", "tuition", "financial-aid",
    "student-life", "housing", "dining", "library",
    "about", "contact", "directory", "people", "faculty", "staff",
    "press", "media", "magazine", "blog", "policy"
]

SUBSITE_HINTS = [
    "center", "centre", "institute", "office", "community", "belonging",
    "extension", "continuing-ed", "continuing-education",
    "professional", "outreach",
    # dataset-specific common traps
    "nwo", "cge",
]

TOO_SPECIFIC_PATTERNS = [
    r"/departments?/[a-z0-9_-]+",
    r"/departments-and-programs/[a-z0-9_-]+",
    r"/programs?/[a-z0-9_-]+",
    r"/academics/.+/(major|minors|programs)\b",
]


# ============================================================
# Unicode normalization
# ============================================================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================================================
# Cache + fetch
# ============================================================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"


def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)
    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass
    return content


def fetch_text_cached(url: str, session: requests.Session) -> str:
    b = fetch_bytes_cached(url, session=session)
    try:
        return b.decode("utf-8", errors="replace")
    except Exception:
        return b.decode("latin-1", errors="replace")


# ============================================================
# URL helpers
# ============================================================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n


def rootish_domain(netloc: str) -> str:
    """
    Lightweight "registrable-ish" domain: last two labels if possible.
    This is imperfect for .edu subdomains but works well enough for drift detection.
    """
    n = root_domain(netloc)
    parts = [p for p in n.split(".") if p]
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return n


ALLOWED_ACADEMIC_SUBDOMAINS = {"www", "majors", "programs", "catalog", "bulletin", "courses"}


def is_subsite_like(url: str, base_netloc: str) -> bool:
    """
    Heuristic: "subsite-like" means the URL is likely an office/center/outreach subsite,
    or lives on a non-standard subdomain that frequently hosts non-inventory pages.

    Rules:
    - Different rootish domain => subsite-like
    - Subdomain first-label not in allowlist => subsite-like
      (unless it's literally the base host itself)
    - Path contains subsite hint words => subsite-like
    """
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Offsite-ish drift
    if rootish_domain(host) != rootish_domain(base):
        return True

    # Subdomain gating: majors.<root>, catalog.<root>, etc allowed; others penalized
    if host != base:
        # get first label (e.g., "catalog" in "catalog.bu.edu")
        parts = [p for p in host.split(".") if p]
        sub = parts[0] if parts else ""
        if sub and sub not in ALLOWED_ACADEMIC_SUBDOMAINS:
            return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in SUBSITE_HINTS):
        return True

    return False


def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)


def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)


def is_canonical_hub_url(url: str) -> bool:
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical


def looks_like_detail_url(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return any(p.search(path) for p in DETAIL_URL_PATTERNS)


# ============================================================
# Row tagging
# ============================================================
def _norm_host_from_web_address(web_address: str) -> str:
    u = ensure_scheme(web_address)
    if not u:
        return ""
    return (urlparse(u).netloc or "").lower()


def tag_row(web_address: str, best_url: str) -> str:
    """
    Returns one of:
      - 'Correct hub'
      - 'Correct-ish but too specific'
      - 'Wrong subsite'
      - 'Wrong non-academic'
    """
    base_host = _norm_host_from_web_address(web_address)
    u = ensure_scheme(best_url)
    if not u or best_url == "N/A":
        return "Wrong non-academic"

    host = (urlparse(u).netloc or "").lower()
    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in NON_ACADEMIC_HINTS):
        return "Wrong non-academic"

    # off-root-domain or obvious subsite hints
    if base_host and rootish_domain(host) != rootish_domain(base_host):
        return "Wrong subsite"
    if any(x in full for x in SUBSITE_HINTS):
        return "Wrong subsite"

    # hub-ish path
    if any(x in path for x in HUB_PATH_HINTS):
        for pat in TOO_SPECIFIC_PATTERNS:
            if re.search(pat, path):
                return "Correct-ish but too specific"
        return "Correct hub"

    # weaker academic fallback
    if "academ" in path or "catalog" in path or "bulletin" in path:
        return "Correct-ish but too specific"

    return "Wrong subsite"


# ============================================================
# HTML parse + main-content extraction
# ============================================================
@dataclass
class ParsedPage:
    url: str
    visible_text: str
    main_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]  # (anchor_text, abs_url)
    title_like: List[str]
    html_title: str
    h1: str
    corpus_any: str
    corpus_structured: str


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    h1_tag = soup.find("h1")
    h1 = normalize_unicode_text(h1_tag.get_text(" ", strip=True)) if h1_tag else ""

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    main_node = soup.find("main") or soup.find(id=re.compile(r"content|main", re.I)) or soup.find("article")
    if main_node:
        main_text = normalize_unicode_text(main_node.get_text(separator=" ", strip=True))
    else:
        soup2 = BeautifulSoup(str(soup), "html.parser")
        for tag in soup2(["header", "footer", "nav"]):
            tag.decompose()
        main_text = normalize_unicode_text(soup2.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x)
    )
    corpus_structured = normalize_unicode_text(
        " ".join(x for x in [" ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)] if x)
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        main_text=main_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        h1=h1,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
    )


def is_soft_404(parsed: ParsedPage) -> bool:
    blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1] + parsed.title_like[:5]))
    if SOFT_404_TEXT.search(blob):
        return True
    if len(parsed.main_text) < 600 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    if len(parsed.main_text) < 250 and len(parsed.anchor_texts) < 5:
        return True
    return False


def page_is_thin(parsed: ParsedPage) -> bool:
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) and (len(parsed.main_text) < MIN_MAIN_TEXT_LEN)


# ============================================================
# Sitemap parsing
# ============================================================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()


def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls


def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()

        # hard skips
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue

        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break

    return out


# ============================================================
# Hubness: program-listing-only link counts
# ============================================================
def is_listing_link(url: str) -> bool:
    path = (urlparse(url).path or "")
    return any(p.search(path) for p in LISTING_LINK_PATTERNS)


def hubness_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    listing_links = 0
    slug_set: Set[str] = set()
    distinct_anchor: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        if not is_listing_link(u):
            continue

        listing_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        path = (urlparse(u).path or "").strip("/").lower()
        segs = [s for s in path.split("/") if s]
        if segs:
            slug_set.add(segs[-1])

    # weak secondary signal
    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if listing_links >= 12:
        sig += 1
    if listing_links >= 25:
        sig += 1
    if len(slug_set) >= 8:
        sig += 1
    if len(slug_set) >= 16:
        sig += 1
    if len(distinct_anchor) >= 18:
        sig += 1
    if li_like >= 40 and (listing_links >= 10):
        sig += 1

    counts = {
        "listing_links": listing_links,
        "slug_diversity": len(slug_set),
        "distinct_anchor": len(distinct_anchor),
        "li_like": li_like,
    }
    return sig, counts


# ============================================================
# Scoring
# ============================================================
@dataclass
class PageSignals:
    url: str
    tier: str
    soft404: int
    thin: int

    canonical_hub: int
    hub_sig: int
    hub_counts: Dict[str, int]

    struct_hits: List[str]
    control_unique_structured: int

    year_hit: int
    undergrad_title_boost: int

    grad_penalty: int
    profile_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int

    subsite_like: int  # configurable penalty via knob


def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if is_inventoryish_url(url):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"


def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    soft404 = 1 if is_soft_404(parsed) else 0
    thin = 1 if page_is_thin(parsed) else 0

    control_unique_structured = sum(1 for pat in CONTROL_PATS.values() if pat.search(parsed.corpus_structured))
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    title_blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1]))
    year_hit = 1 if (YEAR_REGEX.search(url) or YEAR_REGEX.search(title_blob)) else 0
    undergrad_title_boost = 1 if UNDERGRAD_TITLE_BOOST.search(title_blob) else 0

    u_low = url.lower()
    grad_penalty = 1 if (GRAD_URL_PENALTY.search(u_low) or re.search(r"\bgraduate\b", title_blob, re.I)) else 0
    profile_penalty = 1 if (PROFILE_URL_PENALTY.search(u_low) or re.search(r"\bfaculty directory\b", title_blob, re.I)) else 0
    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(u_low) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(u_low) else 0
    archive_penalty = 1 if ARCHIVE_URL_PENALTY.search(u_low) else 0
    awards_penalty = 1 if AWARDS_URL_PENALTY.search(u_low) else 0
    compliance_penalty = 1 if COMPLIANCE_URL_PENALTY.search(u_low) else 0

    hub_sig, hub_counts = hubness_signature(parsed, base_netloc=base_netloc)
    canonical_hub = 1 if is_canonical_hub_url(url) else 0

    subsite_like = 1 if is_subsite_like(url, base_netloc=base_netloc) else 0

    return PageSignals(
        url=url,
        tier=page_tier(url),
        soft404=soft404,
        thin=thin,
        canonical_hub=canonical_hub,
        hub_sig=hub_sig,
        hub_counts=hub_counts,
        struct_hits=struct_hits,
        control_unique_structured=control_unique_structured,
        year_hit=year_hit,
        undergrad_title_boost=undergrad_title_boost,
        grad_penalty=grad_penalty,
        profile_penalty=profile_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
        subsite_like=subsite_like,
    )


def canonical_bonus_allowed(sig: PageSignals) -> bool:
    if sig.soft404:
        return False
    if sig.thin and sig.hub_sig == 0 and len(sig.struct_hits) < 2:
        return False
    return True


def score_inventory(sig: PageSignals, subsite_penalty: int) -> int:
    if sig.soft404:
        return -10_000

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 40

    # hubness + structured hits
    score += 85 * sig.hub_sig
    score += 10 * len(sig.struct_hits)
    score += 4 * sig.control_unique_structured

    # canonical hub bonus (gated)
    if sig.canonical_hub and canonical_bonus_allowed(sig):
        score += 160

    # undergrad title boost
    if sig.undergrad_title_boost:
        score += 35

    # year boost only for catalog/bulletin pages
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        score += 70

    # penalties (URL/title/H1)
    if sig.profile_penalty:
        score -= 260
    if sig.grad_penalty:
        score -= 170
    if sig.irrelevant_penalty:
        score -= 260
    if sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 220
    if sig.awards_penalty:
        score -= 200
    if sig.compliance_penalty:
        score -= 150

    # v10: configurable subsite penalty
    if sig.subsite_like:
        score -= int(subsite_penalty)

    # broad detail penalty so hubs win
    if looks_like_detail_url(sig.url):
        if sig.hub_sig <= 1:
            score -= 220
        elif sig.hub_sig == 2:
            score -= 90
        else:
            score -= 25

    # thin penalty
    if sig.thin and sig.hub_sig == 0:
        score -= 180

    return score


def url_prior(url: str, base_netloc: str, subsite_penalty: int) -> int:
    u = url.lower()
    s = 0

    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 200
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 35
    if "bulletin" in u:
        s -= 20

    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if COMPLIANCE_URL_PENALTY.search(u):
        s -= 90
    if AWARDS_URL_PENALTY.search(u):
        s -= 160
    if GRAD_URL_PENALTY.search(u):
        s -= 180

    if is_subsite_like(url, base_netloc=base_netloc):
        s -= int(subsite_penalty)

    if is_canonical_hub_url(url):
        s += 160

    return s


def prioritize_links(links: List[Tuple[str, str]], base_netloc: str, subsite_penalty: int) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000

        s = url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty)

        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 45
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 90
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140

        return -s

    return sorted(links, key=rank)


# ============================================================
# Candidate structure
# ============================================================
@dataclass
class CandidatePage:
    url: str
    score: int
    sig: PageSignals
    parsed: ParsedPage


def get_parsed_page(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)


def fetch_and_score(url: str, session: requests.Session, base_netloc: str, subsite_penalty: int) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page(url, session=session)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        sc = score_inventory(sig, subsite_penalty=subsite_penalty)
        return CandidatePage(url=url, score=sc, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)


# ============================================================
# Parallel batch fetch+score helper
# ============================================================
def fetch_and_score_many(
    urls: List[str],
    base_netloc: str,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    """Fetch+score a batch of URLs concurrently (one Session per thread)."""
    # de-dupe while preserving order
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            uniq.append(u)

    if workers <= 1:
        out: List[CandidatePage] = []
        s = requests.Session()
        for u in uniq:
            cand = fetch_and_score(u, session=s, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
            if cand is not None:
                out.append(cand)
        return out

    out: List[CandidatePage] = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [
            ex.submit(fetch_and_score, u, get_thread_session(), base_netloc, subsite_penalty)
            for u in uniq
        ]
        for fut in as_completed(futs):
            cand = fut.result()
            if cand is not None:
                out.append(cand)
    return out


def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.score, len(topk), cand))
        return
    if cand.score > topk[0][0]:
        heapq.heapreplace(topk, (cand.score, topk[0][1], cand))


def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    return [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]


# ============================================================
# Candidate generation
# ============================================================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))

    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================================================
# Program title extraction
# ============================================================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def looks_like_program_title(s: str, progtitle_strictness: int) -> bool:
    """
    Strictness guidance:
      1-2: permissive (still requires TARGET_ANY)
      3: current default behavior
      4-5: stricter (prefer explicit program-ish keywords; tighter filters)
    """
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False

    # always require the target token family (keeps extraction scoped)
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False

    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm):
        return False

    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        if progtitle_strictness >= 3:
            return False

    if PERSON_NAME_ONLY.match(s_norm):
        if progtitle_strictness >= 3 and not PROGRAM_KEYWORDS.search(s_norm):
            return False
        if progtitle_strictness >= 5:
            return False

    if progtitle_strictness >= 4:
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_norm.lower():
            return False

    return True


def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out


# ============================================================
# PDF fallback (best-effort)
# ============================================================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""


def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================================================
# Aggregate extraction from inventory + alts (+ limited 1-hop)
# ============================================================
def aggregate_outputs(
    pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
    progtitle_strictness: int,
    workers: int,
) -> Tuple[
    int, Set[str], Dict[str, int],
    Dict[str, List[str]],
    List[str],
    List[str],
]:
    base_netloc = urlparse(base_url).netloc

    struct_hits_union: Set[str] = set()
    control_hit_cols = {k: 0 for k in CONTROL_TERMS}
    any_control_found = 0

    tokens_agg: Dict[str, Set[str]] = {"afri": set(), "ethnic": set(), "black": set()}
    titles: Set[str] = set()

    pdf_hits: List[str] = []

    # from chosen pages
    for cand in pages:
        any_control_found = max(any_control_found, 1 if cand.sig.control_unique_structured > 0 else 0)
        struct_hits_union.update(cand.sig.struct_hits)

        for ck, pat in CONTROL_PATS.items():
            if pat.search(cand.parsed.corpus_structured):
                control_hit_cols[ck] = 1

        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k, vals in tm.items():
            tokens_agg[k].update(vals)

        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness):
                titles.add(txt)

    # 1-hop follow
    follow_urls: List[str] = []
    for cand in pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            if looks_like_program_title(a_text or "", progtitle_strictness=progtitle_strictness) or TARGET_ANY_REGEX.search(ul):
                follow_urls.append(u)

    # sitemap assist if best looks thin
    if pages and page_is_thin(pages[0].parsed):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    # Fetch a limited number of follow URLs (optionally in parallel)
    follow_batch = uniq_follow[:ONE_HOP_MAX_FETCHES]

    def _safe_fetch_parse(u: str) -> Optional[ParsedPage]:
        try:
            # Use per-thread sessions when parallel
            s = get_thread_session() if workers and workers > 1 else session
            return get_parsed_page(u, session=s)
        except Exception:
            return None
        finally:
            # Keep some politeness throttling even when parallel
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    if workers and workers > 1 and len(follow_batch) > 1:
        max_w = min(int(workers), 8, len(follow_batch))
        with ThreadPoolExecutor(max_workers=max_w) as ex:
            futs = [ex.submit(_safe_fetch_parse, u) for u in follow_batch]
            for fut in as_completed(futs):
                parsed = fut.result()
                if not parsed:
                    continue

                tm = token_matches_from_text(parsed.corpus_any)
                for k, vals in tm.items():
                    tokens_agg[k].update(vals)

                for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                    if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness):
                        titles.add(txt)
    else:
        for u in follow_batch:
            parsed = _safe_fetch_parse(u)
            if not parsed:
                continue

            tm = token_matches_from_text(parsed.corpus_any)
            for k, vals in tm.items():
                tokens_agg[k].update(vals)

            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness):
                    titles.add(txt)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in pages:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k, vals in tm.items():
                            tokens_agg[k].update(vals)
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Dedup titles by normalized key
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    titles_out = sorted(best_display.values())
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}

    return any_control_found, struct_hits_union, control_hit_cols, tokens_out, titles_out, pdf_hits


#
# ============================================================
# Discovery: find candidate pages
# ============================================================
def find_candidates_for_institution(
    base_url: str,
    session: requests.Session,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_cache
        if sitemap_cache is not None:
            return sitemap_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_cache = sitemap_candidate_urls(all_urls)
        return sitemap_cache

    def consider(u: str) -> Optional[CandidatePage]:
        if not u or u in tried:
            return None
        tried.add(u)
        cand = fetch_and_score(u, session=session, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
        if cand is None:
            return None
        push_topk(topk, cand, k=TOP_K_ALTS)
        return cand

    # Phase 0
    consider(base_url)

    # ------------------------------------------------------------
    # Phase 1 (parallelizable): explicit paths + year/catalog + home links
    # ------------------------------------------------------------
    batch_urls: List[str] = []

    for p in INVENTORY_PATHS[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    batch_urls.extend(build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES])

    for p in CATALOG_PATHS[:PHASE1_MAX_CATALOG_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    # homepage links require parsing homepage once (sequential), but the fetched URLs can be scored in batch
    try:
        home = get_parsed_page(base_url, session=session)
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc, subsite_penalty=subsite_penalty)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            batch_urls.append(u)
    except Exception:
        pass

    # score the batch in parallel (if workers>1)
    batch_cands = fetch_and_score_many(
        batch_urls,
        base_netloc=base_netloc,
        subsite_penalty=subsite_penalty,
        workers=workers,
    )
    for cand in batch_cands:
        tried.add(cand.url)
        push_topk(topk, cand, k=TOP_K_ALTS)

    # Optional: if best so far looks weak, pull some sitemap candidates (also batchable)
    if topk:
        best_so_far = max(topk, key=lambda x: x[0])[2]
        if (best_so_far.sig.thin or best_so_far.sig.hub_sig == 0) and is_inventoryish_url(best_so_far.url):
            try:
                sm_urls = load_sitemap_candidates()[:50]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
            except Exception:
                pass

    # Best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty), depth, u))

    pq_push(base_url, 0)
    fetched = 0
    while pq and fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u)
        fetched += 1
        if depth >= PHASE2_MAX_DEPTH:
            continue
        try:
            parsed = cand.parsed if cand is not None else get_parsed_page(u, session=session)
            links = [(t, lu) for (t, lu) in parsed.links if same_domain(lu, base_netloc)]
            links = prioritize_links(links, base_netloc, subsite_penalty=subsite_penalty)
            for _, lu in links[:70]:
                pq_push(lu, depth + 1)
        except Exception:
            pass

    # Subdomain roots
    for root in build_subdomain_roots(base_url):
        c_root = consider(root)
        if not c_root:
            continue
        root_is_good = (c_root.sig.hub_sig >= 1) or (len(c_root.sig.struct_hits) >= 2) or (not c_root.sig.thin)
        if root_is_good and not c_root.sig.soft404:
            continue
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p))

    return topk_sorted(topk)


# ============================================================
# Reason string
# ============================================================
def inventory_reason(c: CandidatePage) -> str:
    s = c.sig
    flags = []
    if s.grad_penalty:
        flags.append("grad")
    if s.profile_penalty:
        flags.append("profile")
    if s.irrelevant_penalty:
        flags.append("irrelevant")
    if s.admissions_penalty:
        flags.append("admissions")
    if s.archive_penalty:
        flags.append("archive")
    if s.awards_penalty:
        flags.append("awards")
    if s.compliance_penalty:
        flags.append("compliance")
    if s.year_hit:
        flags.append("yearHit")
    if s.thin:
        flags.append("thin")
    if s.soft404:
        flags.append("soft404")
    if s.subsite_like:
        flags.append("subsiteLike")

    return ";".join([
        f"score={c.score}",
        f"tier={s.tier}",
        f"hubSig={s.hub_sig}",
        f"listingLinks={s.hub_counts.get('listing_links', 0)}",
        f"slugDiv={s.hub_counts.get('slug_diversity', 0)}",
        f"structHits={len(s.struct_hits)}",
        f"ctrlStructured={s.control_unique_structured}",
        f"canonicalHub={s.canonical_hub}",
        ("flags=" + ",".join(flags)) if flags else "flags=",
    ])


# ============================================================
# CLI
# ============================================================
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--input", type=str, default=str(DEFAULT_INPUT_PATH))
    ap.add_argument("--output", type=str, default=str(DEFAULT_OUTPUT_PATH))

    # If omitted, default to TEST_HEAD_N (notebook-friendly). Use --head 0 for full run.
    ap.add_argument("--head", type=int, default=None, help="If >0, limit rows for testing; if omitted defaults to TEST_HEAD_N")

    ap.add_argument("--out-suffix", type=str, default="", help="Append to output filename stem")
    ap.add_argument("--subsite-penalty", type=int, default=10)
    ap.add_argument("--progtitle-strictness", type=int, default=3)

    # Parallelism (keep small to be polite)
    ap.add_argument(
        "--workers",
        type=int,
        default=DEFAULT_WORKERS,
        help="Parallel fetch workers for candidate scoring (1 = off). Try 4.",
    )

    # IMPORTANT: ignore Jupyter/ipykernel injected args like --f=...
    args, _unknown = ap.parse_known_args(argv)

    if args.head is None:
        args.head = TEST_HEAD_N

    # sanity clamp
    if args.workers is None or args.workers < 1:
        args.workers = 1
    if args.workers > MAX_WORKERS:
        args.workers = MAX_WORKERS

    return args


def apply_out_suffix(out_path: Path, suffix: str) -> Path:
    if not suffix:
        return out_path
    return out_path.with_name(out_path.stem + suffix + out_path.suffix)


# ============================================================
# Main
# ============================================================
def main() -> None:
    args = parse_args()
    input_path = Path(args.input)
    output_path = apply_out_suffix(Path(args.output), args.out_suffix)

    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {input_path}: {sorted(missing)}")

    if args.head and args.head > 0:
        df = df.head(args.head).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        best_inventory_url = "N/A"
        best_inventory_reason = ""
        alt_candidate_urls = ""

        any_control_found = 0
        struct_hits_union: Set[str] = set()
        any_struct_found = 0

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0
        pdf_hits: List[str] = []

        try:
            if base_url:
                candidates = find_candidates_for_institution(
                    base_url,
                    session=session,
                    subsite_penalty=args.subsite_penalty,
                    workers=args.workers,
                )

                if candidates:
                    best = candidates[0]
                    best_inventory_url = best.url
                    best_inventory_reason = inventory_reason(best)
                    alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K_ALTS])

                    any_control_found, struct_hits_union, control_hits, tokens, titles, pdf_hits = aggregate_outputs(
                        pages=candidates[:TOP_K_ALTS],
                        base_url=base_url,
                        session=session,
                        progtitle_strictness=args.progtitle_strictness,
                        workers=args.workers,
                    )

                    any_struct_found = 1 if struct_hits_union else 0

                    for k, v in control_hits.items():
                        control_hit_cols[f"found_{k}"] = int(v)

                    afri_matches = tokens.get("afri", [])
                    ethnic_matches = tokens.get("ethnic", [])
                    black_matches = tokens.get("black", [])

                    program_titles = titles
                    program_title_count = len(program_titles)
                    any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0
        except Exception:
            # keep row, but leave defaults
            pass

        rec: Dict[str, str] = dict(row)

        rec["best_guess_inventory_url"] = best_inventory_url
        rec["best_guess_inventory_reason"] = best_inventory_reason
        rec["url_tag"] = tag_row(rec.get(WEB_COL, ""), best_inventory_url)
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        rec.update({k: str(v) for k, v in control_hit_cols.items()})
        out_rows.append(rec)

        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "url_tag",
        "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "pdf_hits",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(output_path, index=False)
    print(f"\nWrote: {output_path} | rows,cols = {out_df.shape}")
    print(out_df.head())


if __name__ == "__main__":
    main()

[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

run sweeps repeadedly 

In [12]:
#!/usr/bin/env python3
"""
v10 notebook-friendly sweep runner

Runs v10_program_inventory.py across a grid of:
  - --subsite-penalty
  - --progtitle-strictness
  - (optional) --workers

Writes one CSV per run, embedding params in the filename.

Example:
  python sweep_v10.py --script v10_program_inventory.py --head 25
"""

from __future__ import annotations

import argparse
import itertools
import subprocess
from pathlib import Path
from typing import List, Optional


def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(add_help=True)

    ap.add_argument(
        "--script",
        type=str,
        default="v10_program_inventory.py",
        help="Path to v10 script (default: v10_program_inventory.py)",
    )
    ap.add_argument(
        "--input",
        type=str,
        default="Carnegie_Carla_2013_subset_with_web.csv",
        help="Input CSV passed to v10 script",
    )
    ap.add_argument(
        "--output-base",
        type=str,
        default="Carnegie_Carla_v10_eval.csv",
        help="Base output filename; params will be appended to stem",
    )
    ap.add_argument(
        "--head",
        type=int,
        default=25,
        help="Row limit for fast tests (0 = full). Overrides v10 default.",
    )

    ap.add_argument(
        "--subsite-penalties",
        type=int,
        nargs="*",
        default=[5, 10, 15],
        help="Grid for --subsite-penalty (space-separated ints)",
    )
    ap.add_argument(
        "--progtitle-strictnesses",
        type=int,
        nargs="*",
        default=[2, 3, 4],
        help="Grid for --progtitle-strictness (space-separated ints)",
    )

    # Optional: sweep parallelism too (since v10 supports --workers)
    ap.add_argument(
        "--workers-grid",
        type=int,
        nargs="*",
        default=[1],
        help="Grid for --workers (default: 1 only). Try: 1 2 4",
    )

    ap.add_argument(
        "--out-dir",
        type=str,
        default=".",
        help="Directory to write outputs",
    )
    ap.add_argument(
        "--python",
        type=str,
        default="python",
        help="Python executable to use (e.g., python3)",
    )
    ap.add_argument(
        "--dry-run",
        action="store_true",
        help="Print commands but do not execute",
    )

    # IMPORTANT: ignore Jupyter/ipykernel injected args like --f=...
    args, _unknown = ap.parse_known_args(argv)
    return args


def main() -> None:
    args = parse_args()

    script_path = Path(args.script)
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    base_out = Path(args.output_base)
    stem = base_out.stem
    suffix = base_out.suffix if base_out.suffix else ".csv"

    runs = list(
        itertools.product(
            args.subsite_penalties,
            args.progtitle_strictnesses,
            args.workers_grid,
        )
    )

    print(f"Script : {script_path}")
    print(f"Input  : {args.input}")
    print(f"OutDir : {out_dir.resolve()}")
    print(f"Runs   : {len(runs)}")
    print("Grid   : (subsite_penalty, progtitle_strictness, workers)")
    print(runs)
    print()

    for sub_pen, pts, workers in runs:
        run_tag = f"_v10_subsite{sub_pen}_progtitle{pts}_w{workers}"
        out_file = out_dir / f"{stem}{run_tag}{suffix}"

        cmd = [
            args.python,
            str(script_path),
            "--input", args.input,
            "--output", str(out_file),
            "--out-suffix", "",  # already embedded in output name
            "--subsite-penalty", str(sub_pen),
            "--progtitle-strictness", str(pts),
            "--workers", str(workers),
            "--head", str(args.head),
        ]

        print("RUN :", " ".join(cmd))
        if args.dry_run:
            continue

        subprocess.run(cmd, check=True)
        print("WROTE:", out_file)
        print()

    print("Done.")


if __name__ == "__main__":
    main()

Script : v10_program_inventory.py
Input  : Carnegie_Carla_2013_subset_with_web.csv
OutDir : /Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard
Runs   : 9
Grid   : (subsite_penalty, progtitle_strictness, workers)
[(5, 2, 1), (5, 3, 1), (5, 4, 1), (10, 2, 1), (10, 3, 1), (10, 4, 1), (15, 2, 1), (15, 3, 1), (15, 4, 1)]

RUN : python v10_program_inventory.py --input Carnegie_Carla_2013_subset_with_web.csv --output Carnegie_Carla_v10_eval_v10_subsite5_progtitle2_w1.csv --out-suffix  --subsite-penalty 5 --progtitle-strictness 2 --workers 1 --head 25


/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

...

[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

Notes on variable sweep:
best_guess_inventory_url selection was effectively identical across all parameter combos
4 incorrect cases are (representative row view; same across runs):
	•	Correct-ish but too specific (2): the script picked a department/program detail-ish page rather than a programs hub.
	•	Wrong subsite (1): the script picked a subsite/center/programs page off the main academic hub.
	•	Wrong non-academic (1): the script picked something “programs-looking” but functionally not the intended academics programs hub.

small set of path tokens you already know are traps (/nwo/, /cge/, etc.), treat them like “irrelevant/non-academic”:
	•	Either return score = -10_000 (like soft404), or apply a very large penalty (e.g., -800)

Add an explicit penalty for “too specific” department/detail pages (separate from the current detail penalty):
	•	If TOO_SPECIFIC_PATTERNS matches and NOT canonical_hub: apply -250 (or -400)
	•	Exception: if hubness is extremely strong and URL contains canonical hub paths, don’t apply.

Prefer canonical hubs more aggressively when tie-ish
	•	Increase canonical hub bonus (e.g., +220)

•	--subsite-penalty: try 80 as new baseline for sweeps
•	--progtitle-strictness: leave at 3 

In [1]:
#!/usr/bin/env python3
"""
v11: program-inventory discovery + scoring + tagging (clean)

Key additions vs v10:
- Row tagging for best_guess_inventory_url:
    * Correct hub
    * Correct-ish but too specific
    * Wrong subsite
    * Wrong non-academic
- CLI knobs for sweep runs from ipynb:
    --subsite-penalty (int): penalty applied to subsite-like pages
    --progtitle-strictness (int): controls how strict program title extraction is
    --head (int): limit rows for quick testing
    --out-suffix (str): appended to output filename stem
    --input / --output: file paths

Notebook-friendly:
- TEST_HEAD_N provides default subset size when --head is not provided.
  Use --head 0 for full run.
- DEFAULT_WORKERS provides a top-level default for --workers.

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v11.csv (default)
"""

from __future__ import annotations

import argparse
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ============================================================
# Thread-local requests sessions (for safe parallel fetching)
# ============================================================
_thread_local = threading.local()


def get_thread_session() -> requests.Session:
    """Return a per-thread requests.Session (requests.Session is not thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        _thread_local.session = s
    return s



# ============================================================
# Notebook-friendly default test mode
# ============================================================
# If you run from an ipynb as:  !python v11_program_inventory.py
# it will default to processing TEST_HEAD_N rows unless you pass --head.
# Use --head 0 for full run.
TEST_HEAD_N = 25

# ============================================================
# Top-level parallelism defaults
# ============================================================
# You can override via CLI: --workers
# Keep MAX_WORKERS small to stay polite to university sites.
DEFAULT_WORKERS = 4
MAX_WORKERS = 8


# ============================================================
# Defaults (override via CLI)
# ============================================================
DEFAULT_INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
DEFAULT_OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v11.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.25
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.20
MAX_BYTES_PER_PAGE = 2_000_000

CACHE_DIR = Path(".cache_v11_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Per-institution budgets
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 22
PHASE2_BESTFIRST_MAX_PAGES = 55
PHASE2_MAX_DEPTH = 3

TOP_K_ALTS = 6
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 18

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

MIN_VISIBLE_TEXT_LEN = 900
MIN_MAIN_TEXT_LEN = 650

ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/11.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================================================
# URL templates
# ============================================================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================================================
# Patterns / scoring signals
# ============================================================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_URL_PENALTY = re.compile(
    r"\b(graduate|grad-studies|gradstudies|master|masters|doctoral|phd|mba|mfa|ms|ma)\b",
    flags=re.IGNORECASE,
)
PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)
AWARDS_URL_PENALTY = re.compile(r"\b(awards?|honors?)\b", flags=re.IGNORECASE)
COMPLIANCE_URL_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_TITLE_BOOST = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

TARGET_TOKEN_REGEX = {
    "afri": re.compile(r"\bafri[a-z\-]*\b", flags=re.IGNORECASE),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
}
TARGET_ANY_REGEX = re.compile(r"\b(afri[a-z\-]*|ethnic[a-z\-]*|black[a-z\-]*)\b", flags=re.IGNORECASE)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}\s*[-]?\s*\d{2,3}[A-Z]?\b")
COURSE_WORDS = re.compile(r"\b(units?:|credits?:|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")
PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

# "Listing-like" URL patterns for anchors we count toward hubness
LISTING_LINK_PATTERNS = [
    re.compile(r"/programs/[^/]+/?$", re.I),
    re.compile(r"/majors-minors/[^/]+/?$", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/departments/[^/]+/?$", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
]

# Broader detail-page detection (so single-major pages don’t beat hubs)
DETAIL_URL_PATTERNS = [
    re.compile(r"/programs/[^/]+/.+", re.I),
    re.compile(r"/majors-minors/[^/]+/.+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/.+", re.I),
    re.compile(r"/departments/[^/]+/.+", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
    re.compile(r"/content/.*/departments/[^/]+\.html$", re.I),
]

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================================================
# Row tagging heuristics
# ============================================================
HUB_PATH_HINTS = [
    "majors", "minors", "programs", "degrees", "department", "departments",
    "departments-and-programs", "academics", "academic", "catalog", "bulletin",
    "undergraduate", "undergraduate-programs", "courses", "areas-of-study"
]

NON_ACADEMIC_HINTS = [
    "news", "event", "calendar", "giving", "alumni", "advancement",
    "athletics", "sports", "tickets", "shop", "bookstore",
    "it", "its", "help", "support", "login", "privacy", "accessibility",
    "admissions", "apply", "visit", "tuition", "financial-aid",
    "student-life", "housing", "dining", "library",
    "about", "contact", "directory", "people", "faculty", "staff",
    "press", "media", "magazine", "blog", "policy"
]

SUBSITE_HINTS = [
    "center", "centre", "institute", "office", "community", "belonging",
    "extension", "continuing-ed", "continuing-education",
    "professional", "outreach",
    # dataset-specific common traps
    "nwo", "cge",
]

# High-precision traps: if these appear in host+path, treat page as an automatic loss.
HARD_SUBSITE_BLOCK_TOKENS = [
    "nwo",  # dataset-specific trap
    "cge",  # dataset-specific trap
]

# Separate from the broad detail penalty: this targets department/program/detail pages that can look
# academically relevant but are still not the intended "programs hub" landing page.
# Starting value calibrated from sweep snapshots: “Correct-ish but too specific” was rare (~2/25 rows, ~8%),
# and usually didn’t have an obvious canonical hub alternative in the top-K alts; use a moderate penalty
# (similar scale to other URL penalties) to nudge toward hubs without overcorrecting.
TOO_SPECIFIC_PENALTY = 240

# If a canonical hub is within this many points of best, prefer it.
CANONICAL_TIE_THRESHOLD = 60

TOO_SPECIFIC_PATTERNS = [
    r"/departments?/[a-z0-9_-]+",
    r"/departments-and-programs/[a-z0-9_-]+",
    r"/programs?/[a-z0-9_-]+",
    r"/academics/.+/(major|minors|programs)\b",
]


# ============================================================
# Unicode normalization
# ============================================================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================================================
# Cache + fetch
# ============================================================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"


def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)
    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass
    return content


def fetch_text_cached(url: str, session: requests.Session) -> str:
    b = fetch_bytes_cached(url, session=session)
    try:
        return b.decode("utf-8", errors="replace")
    except Exception:
        return b.decode("latin-1", errors="replace")


# ============================================================
# URL helpers
# ============================================================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n


def rootish_domain(netloc: str) -> str:
    """
    Lightweight "registrable-ish" domain: last two labels if possible.
    This is imperfect for .edu subdomains but works well enough for drift detection.
    """
    n = root_domain(netloc)
    parts = [p for p in n.split(".") if p]
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return n


ALLOWED_ACADEMIC_SUBDOMAINS = {"www", "majors", "programs", "catalog", "bulletin", "courses"}


def is_subsite_like(url: str, base_netloc: str) -> bool:
    """
    Heuristic: "subsite-like" means the URL is likely an office/center/outreach subsite,
    or lives on a non-standard subdomain that frequently hosts non-inventory pages.

    Rules:
    - Different rootish domain => subsite-like
    - Subdomain first-label not in allowlist => subsite-like
      (unless it's literally the base host itself)
    - Path contains subsite hint words => subsite-like
    """
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Offsite-ish drift
    if rootish_domain(host) != rootish_domain(base):
        return True

    # Subdomain gating: majors.<root>, catalog.<root>, etc allowed; others penalized
    if host != base:
        # get first label (e.g., "catalog" in "catalog.bu.edu")
        parts = [p for p in host.split(".") if p]
        sub = parts[0] if parts else ""
        if sub and sub not in ALLOWED_ACADEMIC_SUBDOMAINS:
            return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in SUBSITE_HINTS):
        return True

    return False

def is_hard_subsite_block(url: str, base_netloc: str) -> bool:
    """True if URL is a known high-precision subsite trap (auto-loss)."""
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Drift to different registrable-ish domain => hard-block
    if rootish_domain(host) != rootish_domain(base):
        return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    return any(tok in full for tok in HARD_SUBSITE_BLOCK_TOKENS)

def is_too_specific_url(url: str) -> bool:
    """True if URL path matches TOO_SPECIFIC_PATTERNS."""
    path = (urlparse(url).path or "").lower()
    for pat in TOO_SPECIFIC_PATTERNS:
        if re.search(pat, path):
            return True
    return False

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)


def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)


def is_canonical_hub_url(url: str) -> bool:
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical


def looks_like_detail_url(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return any(p.search(path) for p in DETAIL_URL_PATTERNS)


# ============================================================
# Row tagging
# ============================================================
def _norm_host_from_web_address(web_address: str) -> str:
    u = ensure_scheme(web_address)
    if not u:
        return ""
    return (urlparse(u).netloc or "").lower()


def tag_row(web_address: str, best_url: str) -> str:
    """
    Returns one of:
      - 'Correct hub'
      - 'Correct-ish but too specific'
      - 'Wrong subsite'
      - 'Wrong non-academic'
    """
    base_host = _norm_host_from_web_address(web_address)
    u = ensure_scheme(best_url)
    if not u or best_url == "N/A":
        return "Wrong non-academic"

    host = (urlparse(u).netloc or "").lower()
    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in NON_ACADEMIC_HINTS):
        return "Wrong non-academic"

    # off-root-domain or obvious subsite hints
    if base_host and rootish_domain(host) != rootish_domain(base_host):
        return "Wrong subsite"
    if any(x in full for x in SUBSITE_HINTS):
        return "Wrong subsite"

    # hub-ish path
    if any(x in path for x in HUB_PATH_HINTS):
        for pat in TOO_SPECIFIC_PATTERNS:
            if re.search(pat, path):
                return "Correct-ish but too specific"
        return "Correct hub"

    # weaker academic fallback
    if "academ" in path or "catalog" in path or "bulletin" in path:
        return "Correct-ish but too specific"

    return "Wrong subsite"


# ============================================================
# HTML parse + main-content extraction
# ============================================================
@dataclass
class ParsedPage:
    url: str
    visible_text: str
    main_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]  # (anchor_text, abs_url)
    title_like: List[str]
    html_title: str
    h1: str
    corpus_any: str
    corpus_structured: str


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    h1_tag = soup.find("h1")
    h1 = normalize_unicode_text(h1_tag.get_text(" ", strip=True)) if h1_tag else ""

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    main_node = soup.find("main") or soup.find(id=re.compile(r"content|main", re.I)) or soup.find("article")
    if main_node:
        main_text = normalize_unicode_text(main_node.get_text(separator=" ", strip=True))
    else:
        soup2 = BeautifulSoup(str(soup), "html.parser")
        for tag in soup2(["header", "footer", "nav"]):
            tag.decompose()
        main_text = normalize_unicode_text(soup2.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(x for x in [visible_text, " ".join(anchor_texts), " ".join(anchor_attr_texts)] if x)
    )
    corpus_structured = normalize_unicode_text(
        " ".join(x for x in [" ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)] if x)
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        main_text=main_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        h1=h1,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
    )


def is_soft_404(parsed: ParsedPage) -> bool:
    blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1] + parsed.title_like[:5]))
    if SOFT_404_TEXT.search(blob):
        return True
    if len(parsed.main_text) < 600 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    if len(parsed.main_text) < 250 and len(parsed.anchor_texts) < 5:
        return True
    return False


def page_is_thin(parsed: ParsedPage) -> bool:
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) and (len(parsed.main_text) < MIN_MAIN_TEXT_LEN)


# ============================================================
# Sitemap parsing
# ============================================================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()


def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls


def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()

        # hard skips
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue

        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break

    return out


# ============================================================
# Hubness: program-listing-only link counts
# ============================================================
def is_listing_link(url: str) -> bool:
    path = (urlparse(url).path or "")
    return any(p.search(path) for p in LISTING_LINK_PATTERNS)


def hubness_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    listing_links = 0
    slug_set: Set[str] = set()
    distinct_anchor: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        if not is_listing_link(u):
            continue

        listing_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        path = (urlparse(u).path or "").strip("/").lower()
        segs = [s for s in path.split("/") if s]
        if segs:
            slug_set.add(segs[-1])

    # weak secondary signal
    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if listing_links >= 12:
        sig += 1
    if listing_links >= 25:
        sig += 1
    if len(slug_set) >= 8:
        sig += 1
    if len(slug_set) >= 16:
        sig += 1
    if len(distinct_anchor) >= 18:
        sig += 1
    if li_like >= 40 and (listing_links >= 10):
        sig += 1

    counts = {
        "listing_links": listing_links,
        "slug_diversity": len(slug_set),
        "distinct_anchor": len(distinct_anchor),
        "li_like": li_like,
    }
    return sig, counts


# ============================================================
# Scoring
# ============================================================
@dataclass
class PageSignals:
    url: str
    tier: str
    soft404: int
    thin: int

    canonical_hub: int
    hub_sig: int
    hub_counts: Dict[str, int]

    struct_hits: List[str]
    control_unique_structured: int

    year_hit: int
    undergrad_title_boost: int

    grad_penalty: int
    profile_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int

    subsite_like: int  # configurable penalty via knob
    hard_subsite_block: int  # auto-loss for known traps
    too_specific: int  # explicit penalty separate from generic detail penalty


def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if is_inventoryish_url(url):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"


def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    soft404 = 1 if is_soft_404(parsed) else 0
    thin = 1 if page_is_thin(parsed) else 0

    control_unique_structured = sum(1 for pat in CONTROL_PATS.values() if pat.search(parsed.corpus_structured))
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    title_blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1]))
    year_hit = 1 if (YEAR_REGEX.search(url) or YEAR_REGEX.search(title_blob)) else 0
    undergrad_title_boost = 1 if UNDERGRAD_TITLE_BOOST.search(title_blob) else 0

    u_low = url.lower()
    grad_penalty = 1 if (GRAD_URL_PENALTY.search(u_low) or re.search(r"\bgraduate\b", title_blob, re.I)) else 0
    profile_penalty = 1 if (PROFILE_URL_PENALTY.search(u_low) or re.search(r"\bfaculty directory\b", title_blob, re.I)) else 0
    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(u_low) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(u_low) else 0
    archive_penalty = 1 if ARCHIVE_URL_PENALTY.search(u_low) else 0
    awards_penalty = 1 if AWARDS_URL_PENALTY.search(u_low) else 0
    compliance_penalty = 1 if COMPLIANCE_URL_PENALTY.search(u_low) else 0

    hub_sig, hub_counts = hubness_signature(parsed, base_netloc=base_netloc)
    canonical_hub = 1 if is_canonical_hub_url(url) else 0

    subsite_like = 1 if is_subsite_like(url, base_netloc=base_netloc) else 0
    hard_subsite_block = 1 if is_hard_subsite_block(url, base_netloc=base_netloc) else 0
    too_specific = 1 if (is_too_specific_url(url) and not is_canonical_hub_url(url)) else 0

    return PageSignals(
        url=url,
        tier=page_tier(url),
        soft404=soft404,
        thin=thin,
        canonical_hub=canonical_hub,
        hub_sig=hub_sig,
        hub_counts=hub_counts,
        struct_hits=struct_hits,
        control_unique_structured=control_unique_structured,
        year_hit=year_hit,
        undergrad_title_boost=undergrad_title_boost,
        grad_penalty=grad_penalty,
        profile_penalty=profile_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
        subsite_like=subsite_like,
        hard_subsite_block=hard_subsite_block,
        too_specific=too_specific,
    )


def canonical_bonus_allowed(sig: PageSignals) -> bool:
    if sig.soft404:
        return False
    if sig.thin and sig.hub_sig == 0 and len(sig.struct_hits) < 2:
        return False
    return True


def score_inventory(sig: PageSignals, subsite_penalty: int) -> int:
    if sig.soft404:
        return -10_000
    
    # hard subsite blocks are automatic losses
    if getattr(sig, "hard_subsite_block", 0):
        return -10_000    

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 40

    # hubness + structured hits
    score += 85 * sig.hub_sig
    score += 10 * len(sig.struct_hits)
    score += 4 * sig.control_unique_structured

    # canonical hub bonus (gated)
    if sig.canonical_hub and canonical_bonus_allowed(sig):
        score += 160

    # undergrad title boost
    if sig.undergrad_title_boost:
        score += 35

    # year boost only for catalog/bulletin pages
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        score += 70

    # penalties (URL/title/H1)
    if sig.profile_penalty:
        score -= 260
    if sig.grad_penalty:
        score -= 170
    if sig.irrelevant_penalty:
        score -= 260
    if sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 220
    if sig.awards_penalty:
        score -= 200
    if sig.compliance_penalty:
        score -= 150

    # configurable subsite penalty
    if sig.subsite_like:
        score -= int(subsite_penalty)

    # explicit penalty for "too specific" department/program pages
    if getattr(sig, "too_specific", 0):
        score -= int(TOO_SPECIFIC_PENALTY)    

    # broad detail penalty so hubs win
    if looks_like_detail_url(sig.url):
        if sig.hub_sig <= 1:
            score -= 220
        elif sig.hub_sig == 2:
            score -= 90
        else:
            score -= 25

    # thin penalty
    if sig.thin and sig.hub_sig == 0:
        score -= 180

    return score


def url_prior(url: str, base_netloc: str, subsite_penalty: int) -> int:
    u = url.lower()
    s = 0

    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 200
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 35
    if "bulletin" in u:
        s -= 20

    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if COMPLIANCE_URL_PENALTY.search(u):
        s -= 90
    if AWARDS_URL_PENALTY.search(u):
        s -= 160
    if GRAD_URL_PENALTY.search(u):
        s -= 180

    # hard subsite blocks should be avoided aggressively
    if is_hard_subsite_block(url, base_netloc=base_netloc):
        s -= 10_000
    if is_subsite_like(url, base_netloc=base_netloc):
        s -= int(subsite_penalty)

    if is_canonical_hub_url(url):
        s += 160

    return s


def prioritize_links(links: List[Tuple[str, str]], base_netloc: str, subsite_penalty: int) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000

        s = url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty)

        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 45
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 90
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140

        return -s

    return sorted(links, key=rank)


# ============================================================
# Candidate structure
# ============================================================
@dataclass
class CandidatePage:
    url: str
    score: int
    sig: PageSignals
    parsed: ParsedPage


def get_parsed_page(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)


def fetch_and_score(url: str, session: requests.Session, base_netloc: str, subsite_penalty: int) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page(url, session=session)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        sc = score_inventory(sig, subsite_penalty=subsite_penalty)
        return CandidatePage(url=url, score=sc, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)


# ============================================================
# Parallel batch fetch+score helper
# ============================================================
def fetch_and_score_many(
    urls: List[str],
    base_netloc: str,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    """Fetch+score a batch of URLs concurrently (one Session per thread)."""
    # de-dupe while preserving order
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            uniq.append(u)

    if workers <= 1:
        out: List[CandidatePage] = []
        s = requests.Session()
        for u in uniq:
            cand = fetch_and_score(u, session=s, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
            if cand is not None:
                out.append(cand)
        return out

    out: List[CandidatePage] = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [
            ex.submit(fetch_and_score, u, get_thread_session(), base_netloc, subsite_penalty)
            for u in uniq
        ]
        for fut in as_completed(futs):
            cand = fut.result()
            if cand is not None:
                out.append(cand)
    return out


def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.score, len(topk), cand))
        return
    if cand.score > topk[0][0]:
        heapq.heapreplace(topk, (cand.score, topk[0][1], cand))


def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    """Return candidates sorted by score (desc), with an optional canonical-hub tie-break."""
    cands = [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]

    # C: Prefer canonical hubs more aggressively when tie-ish.
    if cands:
        best_score = cands[0].score

        canonical_best: Optional[CandidatePage] = None
        for c in cands:
            if c.sig.canonical_hub and (best_score - c.score) <= CANONICAL_TIE_THRESHOLD:
                if canonical_best is None or c.score > canonical_best.score:
                    canonical_best = c

        if canonical_best is not None and canonical_best is not cands[0]:
            cands = [canonical_best] + [x for x in cands if x is not canonical_best]

    return cands


# ============================================================
# Candidate generation
# ============================================================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))

    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================================================
# Program title extraction
# ============================================================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def looks_like_program_title(s: str, progtitle_strictness: int) -> bool:
    """
    Strictness guidance:
      1-2: permissive (still requires TARGET_ANY)
      3: current default behavior
      4-5: stricter (prefer explicit program-ish keywords; tighter filters)
    """
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False

    # always require the target token family (keeps extraction scoped)
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False

    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm):
        return False

    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        if progtitle_strictness >= 3:
            return False

    if PERSON_NAME_ONLY.match(s_norm):
        if progtitle_strictness >= 3 and not PROGRAM_KEYWORDS.search(s_norm):
            return False
        if progtitle_strictness >= 5:
            return False

    if progtitle_strictness >= 4:
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_norm.lower():
            return False

    return True


def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out


# ============================================================
# PDF fallback (best-effort)
# ============================================================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""


def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================================================
# Aggregate extraction from inventory + alts (+ limited 1-hop)
# ============================================================
def aggregate_outputs(
    pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
    progtitle_strictness: int,
    workers: int,
) -> Tuple[
    int, Set[str], Dict[str, int],
    Dict[str, List[str]],
    List[str],
    List[str],
]:
    base_netloc = urlparse(base_url).netloc

    struct_hits_union: Set[str] = set()
    control_hit_cols = {k: 0 for k in CONTROL_TERMS}
    any_control_found = 0

    tokens_agg: Dict[str, Set[str]] = {"afri": set(), "ethnic": set(), "black": set()}
    titles: Set[str] = set()

    pdf_hits: List[str] = []

    # from chosen pages
    for cand in pages:
        any_control_found = max(any_control_found, 1 if cand.sig.control_unique_structured > 0 else 0)
        struct_hits_union.update(cand.sig.struct_hits)

        for ck, pat in CONTROL_PATS.items():
            if pat.search(cand.parsed.corpus_structured):
                control_hit_cols[ck] = 1

        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k, vals in tm.items():
            tokens_agg[k].update(vals)

        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness):
                titles.add(txt)

    # 1-hop follow
    follow_urls: List[str] = []
    for cand in pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            if looks_like_program_title(a_text or "", progtitle_strictness=progtitle_strictness) or TARGET_ANY_REGEX.search(ul):
                follow_urls.append(u)

    # sitemap assist if best looks thin
    if pages and page_is_thin(pages[0].parsed):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    # Fetch a limited number of follow URLs (optionally in parallel)
    follow_batch = uniq_follow[:ONE_HOP_MAX_FETCHES]

    def _safe_fetch_parse(u: str) -> Optional[ParsedPage]:
        try:
            # Use per-thread sessions when parallel
            s = get_thread_session() if workers and workers > 1 else session
            return get_parsed_page(u, session=s)
        except Exception:
            return None
        finally:
            # Keep some politeness throttling even when parallel
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    if workers and workers > 1 and len(follow_batch) > 1:
        max_w = min(int(workers), 8, len(follow_batch))
        with ThreadPoolExecutor(max_workers=max_w) as ex:
            futs = [ex.submit(_safe_fetch_parse, u) for u in follow_batch]
            for fut in as_completed(futs):
                parsed = fut.result()
                if not parsed:
                    continue

                tm = token_matches_from_text(parsed.corpus_any)
                for k, vals in tm.items():
                    tokens_agg[k].update(vals)

                for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                    if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness):
                        titles.add(txt)
    else:
        for u in follow_batch:
            parsed = _safe_fetch_parse(u)
            if not parsed:
                continue

            tm = token_matches_from_text(parsed.corpus_any)
            for k, vals in tm.items():
                tokens_agg[k].update(vals)

            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness):
                    titles.add(txt)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in pages:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k, vals in tm.items():
                            tokens_agg[k].update(vals)
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Dedup titles by normalized key
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    titles_out = sorted(best_display.values())
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}

    return any_control_found, struct_hits_union, control_hit_cols, tokens_out, titles_out, pdf_hits


#
# ============================================================
# Discovery: find candidate pages
# ============================================================
def find_candidates_for_institution(
    base_url: str,
    session: requests.Session,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_cache
        if sitemap_cache is not None:
            return sitemap_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_cache = sitemap_candidate_urls(all_urls)
        return sitemap_cache

    def consider(u: str) -> Optional[CandidatePage]:
        if not u or u in tried:
            return None
        tried.add(u)
        cand = fetch_and_score(u, session=session, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
        if cand is None:
            return None
        push_topk(topk, cand, k=TOP_K_ALTS)
        return cand

    # Phase 0
    consider(base_url)

    # ------------------------------------------------------------
    # Phase 1 (parallelizable): explicit paths + year/catalog + home links
    # ------------------------------------------------------------
    batch_urls: List[str] = []

    for p in INVENTORY_PATHS[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    batch_urls.extend(build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES])

    for p in CATALOG_PATHS[:PHASE1_MAX_CATALOG_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    # homepage links require parsing homepage once (sequential), but the fetched URLs can be scored in batch
    try:
        home = get_parsed_page(base_url, session=session)
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc, subsite_penalty=subsite_penalty)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            batch_urls.append(u)
    except Exception:
        pass

    # score the batch in parallel (if workers>1)
    batch_cands = fetch_and_score_many(
        batch_urls,
        base_netloc=base_netloc,
        subsite_penalty=subsite_penalty,
        workers=workers,
    )
    for cand in batch_cands:
        tried.add(cand.url)
        push_topk(topk, cand, k=TOP_K_ALTS)

    # Optional: if best so far looks weak, pull some sitemap candidates (also batchable)
    if topk:
        best_so_far = max(topk, key=lambda x: x[0])[2]
        if (best_so_far.sig.thin or best_so_far.sig.hub_sig == 0) and is_inventoryish_url(best_so_far.url):
            try:
                sm_urls = load_sitemap_candidates()[:50]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
            except Exception:
                pass

    # Best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty), depth, u))

    pq_push(base_url, 0)
    fetched = 0
    while pq and fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u)
        fetched += 1
        if depth >= PHASE2_MAX_DEPTH:
            continue
        try:
            parsed = cand.parsed if cand is not None else get_parsed_page(u, session=session)
            links = [(t, lu) for (t, lu) in parsed.links if same_domain(lu, base_netloc)]
            links = prioritize_links(links, base_netloc, subsite_penalty=subsite_penalty)
            for _, lu in links[:70]:
                pq_push(lu, depth + 1)
        except Exception:
            pass

    # Subdomain roots
    for root in build_subdomain_roots(base_url):
        c_root = consider(root)
        if not c_root:
            continue
        root_is_good = (c_root.sig.hub_sig >= 1) or (len(c_root.sig.struct_hits) >= 2) or (not c_root.sig.thin)
        if root_is_good and not c_root.sig.soft404:
            continue
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p))

    return topk_sorted(topk)


# ============================================================
# Reason string
# ============================================================
def inventory_reason(c: CandidatePage) -> str:
    s = c.sig
    flags = []
    if s.grad_penalty:
        flags.append("grad")
    if s.profile_penalty:
        flags.append("profile")
    if s.irrelevant_penalty:
        flags.append("irrelevant")
    if s.admissions_penalty:
        flags.append("admissions")
    if s.archive_penalty:
        flags.append("archive")
    if s.awards_penalty:
        flags.append("awards")
    if s.compliance_penalty:
        flags.append("compliance")
    if s.year_hit:
        flags.append("yearHit")
    if s.thin:
        flags.append("thin")
    if s.soft404:
        flags.append("soft404")
    if s.subsite_like:
        flags.append("subsiteLike")
    if getattr(s, "hard_subsite_block", 0):
        flags.append("hardSubsiteBlock")
    if getattr(s, "too_specific", 0):
        flags.append("tooSpecific")

    return ";".join([
        f"score={c.score}",
        f"tier={s.tier}",
        f"hubSig={s.hub_sig}",
        f"listingLinks={s.hub_counts.get('listing_links', 0)}",
        f"slugDiv={s.hub_counts.get('slug_diversity', 0)}",
        f"structHits={len(s.struct_hits)}",
        f"ctrlStructured={s.control_unique_structured}",
        f"canonicalHub={s.canonical_hub}",
        ("flags=" + ",".join(flags)) if flags else "flags=",
    ])


# ============================================================
# CLI
# ============================================================
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--input", type=str, default=str(DEFAULT_INPUT_PATH))
    ap.add_argument("--output", type=str, default=str(DEFAULT_OUTPUT_PATH))

    # If omitted, default to TEST_HEAD_N (notebook-friendly). Use --head 0 for full run.
    ap.add_argument("--head", type=int, default=None, help="If >0, limit rows for testing; if omitted defaults to TEST_HEAD_N")

    ap.add_argument("--out-suffix", type=str, default="", help="Append to output filename stem")
    ap.add_argument("--subsite-penalty", type=int, default=80)
    ap.add_argument("--progtitle-strictness", type=int, default=3)

    # Parallelism (keep small to be polite)
    ap.add_argument(
        "--workers",
        type=int,
        default=DEFAULT_WORKERS,
        help="Parallel fetch workers for candidate scoring (1 = off). Try 4.",
    )

    # IMPORTANT: ignore Jupyter/ipykernel injected args like --f=...
    args, _unknown = ap.parse_known_args(argv)

    if args.head is None:
        args.head = TEST_HEAD_N

    # sanity clamp
    if args.workers is None or args.workers < 1:
        args.workers = 1
    if args.workers > MAX_WORKERS:
        args.workers = MAX_WORKERS

    return args


def apply_out_suffix(out_path: Path, suffix: str) -> Path:
    if not suffix:
        return out_path
    return out_path.with_name(out_path.stem + suffix + out_path.suffix)


# ============================================================
# Main
# ============================================================
def main() -> None:
    args = parse_args()
    input_path = Path(args.input)
    output_path = apply_out_suffix(Path(args.output), args.out_suffix)

    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {input_path}: {sorted(missing)}")

    if args.head and args.head > 0:
        df = df.head(args.head).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        best_inventory_url = "N/A"
        best_inventory_reason = ""
        alt_candidate_urls = ""

        any_control_found = 0
        struct_hits_union: Set[str] = set()
        any_struct_found = 0

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0
        pdf_hits: List[str] = []

        try:
            if base_url:
                candidates = find_candidates_for_institution(
                    base_url,
                    session=session,
                    subsite_penalty=args.subsite_penalty,
                    workers=args.workers,
                )

                if candidates:
                    best = candidates[0]
                    best_inventory_url = best.url
                    best_inventory_reason = inventory_reason(best)
                    alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K_ALTS])

                    any_control_found, struct_hits_union, control_hits, tokens, titles, pdf_hits = aggregate_outputs(
                        pages=candidates[:TOP_K_ALTS],
                        base_url=base_url,
                        session=session,
                        progtitle_strictness=args.progtitle_strictness,
                        workers=args.workers,
                    )

                    any_struct_found = 1 if struct_hits_union else 0

                    for k, v in control_hits.items():
                        control_hit_cols[f"found_{k}"] = int(v)

                    afri_matches = tokens.get("afri", [])
                    ethnic_matches = tokens.get("ethnic", [])
                    black_matches = tokens.get("black", [])

                    program_titles = titles
                    program_title_count = len(program_titles)
                    any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0
        except Exception:
            # keep row, but leave defaults
            pass

        rec: Dict[str, str] = dict(row)

        rec["best_guess_inventory_url"] = best_inventory_url
        rec["best_guess_inventory_reason"] = best_inventory_reason
        rec["url_tag"] = tag_row(rec.get(WEB_COL, ""), best_inventory_url)
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        rec.update({k: str(v) for k, v in control_hit_cols.items()})
        out_rows.append(rec)

        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "url_tag",
        "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches",
        "any_potential_AfrAmr_found",
        "pdf_hits",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(output_path, index=False)
    print(f"\nWrote: {output_path} | rows,cols = {out_df.shape}")
    print(out_df.head())


if __name__ == "__main__":
    main()

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

	•	Boise State (majors.boisestate.edu): likely treated as subsite-like (non-allowlisted subdomain) and/or JS-rendered hub, so it got penalized and/or looked “thin,” preventing it from winning.
	•	Agnes Scott: you found the correct hub, but the Africana Studies program list appears to be JS/embedded-data driven and/or not present in the text you currently harvest (anchors/title-like), so extraction didn’t capture “Africana Studies.”
	•	Barry (find-your-program): the hub is a JS app and your pipeline doesn’t pull embedded JSON / API-fed program lists, so there’s nothing “real” to extract unless you expand via sitemap or JSON.
	•	Cal Poly Pomona (cpp.edu/programs): the page is clearly template/JS-driven (placeholders like {{ ... }} appear in the HTML), so your HTML-only extraction won’t see the actual program records.  ￼
	•	Cal Poly SLO: hub is fine, but your one-hop follow is pulling in non-title prose/course/GE fragments and your title filter is too permissive for catalog/detail pages (e.g., it’s letting sentence-like strings through).  ￼
	•	Bates: hub is fine, but you’re capturing navigation/prose (“About the Program…”, mission statements) because “title-like” harvesting + one-hop follow isn’t sufficiently constrained to “program-name-ish” items.  ￼
	•	Fresno CHHS (chhs.fresnostate.edu/academics/index.html): it likely lost because the chhs subdomain is penalized as subsite-like (not in ALLOWED_ACADEMIC_SUBDOMAINS), so catalog-ish pages on the main domain outscored it.

In [1]:
#!/usr/bin/env python3
"""
v11: program-inventory discovery + scoring + tagging (clean)

Key additions vs v10:
- Row tagging for best_guess_inventory_url:
    * Correct hub
    * Correct-ish but too specific
    * Wrong subsite
    * Wrong non-academic
- CLI knobs for sweep runs from ipynb:
    --subsite-penalty (int): penalty applied to subsite-like pages
    --progtitle-strictness (int): controls how strict program title extraction is
    --head (int): limit rows for quick testing
    --out-suffix (str): appended to output filename stem
    --input / --output: file paths

Notebook-friendly:
- TEST_HEAD_N provides default subset size when --head is not provided.
  Use --head 0 for full run.
- DEFAULT_WORKERS provides a top-level default for --workers.

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v11.csv (default)
"""

from __future__ import annotations

import argparse
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ============================================================
# Thread-local requests sessions (for safe parallel fetching)
# ============================================================
_thread_local = threading.local()


def get_thread_session() -> requests.Session:
    """Return a per-thread requests.Session (requests.Session is not thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        _thread_local.session = s
    return s



# ============================================================
# Notebook-friendly default test mode
# ============================================================
# If you run from an ipynb as:  !python v11_program_inventory.py
# it will default to processing TEST_HEAD_N rows unless you pass --head.
# Use --head 0 for full run.
TEST_HEAD_N = 25

# ============================================================
# Top-level parallelism defaults
# ============================================================
# You can override via CLI: --workers
# Keep MAX_WORKERS small to stay polite to university sites.
DEFAULT_WORKERS = 6
MAX_WORKERS = 8


# ============================================================
# Defaults (override via CLI)
# ============================================================
DEFAULT_INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
DEFAULT_OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v11.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.25
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.20

MAX_BYTES_PER_PAGE = 2_000_000
MAX_JSON_BLOB_CHARS = 200_000

CACHE_DIR = Path(".cache_v11_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Per-institution budgets
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 22
PHASE2_BESTFIRST_MAX_PAGES = 55
PHASE2_MAX_DEPTH = 3

TOP_K_ALTS = 6
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 18

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

MIN_VISIBLE_TEXT_LEN = 900
MIN_MAIN_TEXT_LEN = 650

ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/11.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================================================
# URL templates
# ============================================================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================================================
# Patterns / scoring signals
# ============================================================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_URL_PENALTY = re.compile(
    r"\b(graduate|grad-studies|gradstudies|master|masters|doctoral|phd|mba|mfa|ms|ma)\b",
    flags=re.IGNORECASE,
)
PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)
AWARDS_URL_PENALTY = re.compile(r"\b(awards?|honors?)\b", flags=re.IGNORECASE)
COMPLIANCE_URL_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_TITLE_BOOST = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

#
# Expanded target token family (v12): includes Africa/African/Africana, Pan-African,
# African American, Afro-American, Africology, Race, plus existing afri/ethnic/black.
TARGET_TOKEN_REGEX = {
    # Broaden "afri" bucket to cover common program-name variants beyond just "afri*".
    "afri": re.compile(
        r"\b(afri[a-z\-]*|africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology)\b",
        flags=re.IGNORECASE,
    ),
    "ethnic": re.compile(r"\bethnic[a-z\-]*\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
    # New bucket: race / racial
    "race": re.compile(r"\b(race|racial)\b", flags=re.IGNORECASE),
}

# Trigger regex used to decide whether text is in-scope for extraction.
# Keep this broad (union of all target families) so we don't miss relevant programs.
TARGET_ANY_REGEX = re.compile(
    r"\b(afri[a-z\-]*|africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology|"
    r"ethnic[a-z\-]*|ethnicity|black[a-z\-]*|race|racial)\b",
    flags=re.IGNORECASE,
)

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}\s*[-]?\s*\d{2,3}[A-Z]?\b")
COURSE_WORDS = re.compile(r"\b(units?|credits?|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")
PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

# --- Title junk / prose detection (v12 D) ---
URL_LIKE_TEXT = re.compile(r"(https?://|www\.|\b[a-z0-9\-]+\.(edu|com|org|net)(/|\b))", flags=re.IGNORECASE)
PROSE_VERBS = re.compile(
    r"\b(define|apply|enrich|explore|learn|prepare|prepares|focus(es)?|offers|provides|designed|helps|aims)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

# "Listing-like" URL patterns for anchors we count toward hubness
LISTING_LINK_PATTERNS = [
    re.compile(r"/programs/[^/]+/?$", re.I),
    re.compile(r"/majors-minors/[^/]+/?$", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/departments/[^/]+/?$", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
]

# Broader detail-page detection (so single-major pages don’t beat hubs)
DETAIL_URL_PATTERNS = [
    re.compile(r"/programs/[^/]+/.+", re.I),
    re.compile(r"/majors-minors/[^/]+/.+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/.+", re.I),
    re.compile(r"/departments/[^/]+/.+", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
    re.compile(r"/content/.*/departments/[^/]+\.html$", re.I),
]

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================================================
# Row tagging heuristics
# ============================================================
HUB_PATH_HINTS = [
    "majors", "minors", "programs", "degrees", "department", "departments",
    "departments-and-programs", "academics", "academic", "catalog", "bulletin",
    "undergraduate", "undergraduate-programs", "courses", "areas-of-study"
]

NON_ACADEMIC_HINTS = [
    "news", "event", "calendar", "giving", "alumni", "advancement",
    "athletics", "sports", "tickets", "shop", "bookstore",
    "it", "its", "help", "support", "login", "privacy", "accessibility",
    "admissions", "apply", "visit", "tuition", "financial-aid",
    "student-life", "housing", "dining", "library",
    "about", "contact", "directory", "people", "faculty", "staff",
    "press", "media", "magazine", "blog", "policy"
]

SUBSITE_HINTS = [
    "center", "centre", "institute", "office", "community", "belonging",
    "extension", "continuing-ed", "continuing-education",
    "professional", "outreach",
    # dataset-specific common traps
    "nwo", "cge",
]

# High-precision traps: if these appear in host+path, treat page as an automatic loss.
HARD_SUBSITE_BLOCK_TOKENS = [
    "nwo",  # dataset-specific trap
    "cge",  # dataset-specific trap
]

# Separate from the broad detail penalty: this targets department/program/detail pages that can look
# academically relevant but are still not the intended "programs hub" landing page.
# Starting value calibrated from sweep snapshots: “Correct-ish but too specific” was rare (~2/25 rows, ~8%),
# and usually didn’t have an obvious canonical hub alternative in the top-K alts; use a moderate penalty
# (similar scale to other URL penalties) to nudge toward hubs without overcorrecting.
TOO_SPECIFIC_PENALTY = 240

# If a canonical hub is within this many points of best, prefer it.
CANONICAL_TIE_THRESHOLD = 60

TOO_SPECIFIC_PATTERNS = [
    r"/departments?/[a-z0-9_-]+",
    r"/departments-and-programs/[a-z0-9_-]+",
    r"/programs?/[a-z0-9_-]+",
    r"/academics/.+/(major|minors|programs)\b",
]


# ============================================================
# Unicode normalization
# ============================================================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================================================
# Cache + fetch
# ============================================================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"


def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)
    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass
    return content


def fetch_text_cached(url: str, session: requests.Session) -> str:
    b = fetch_bytes_cached(url, session=session)
    try:
        return b.decode("utf-8", errors="replace")
    except Exception:
        return b.decode("latin-1", errors="replace")


# ============================================================
# URL helpers
# ============================================================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n


def rootish_domain(netloc: str) -> str:
    """
    Lightweight "registrable-ish" domain: last two labels if possible.
    This is imperfect for .edu subdomains but works well enough for drift detection.
    """
    n = root_domain(netloc)
    parts = [p for p in n.split(".") if p]
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return n


ALLOWED_ACADEMIC_SUBDOMAINS = {"www", "majors", "programs", "catalog", "bulletin", "courses"}


def is_subsite_like(url: str, base_netloc: str) -> bool:
    """
    Heuristic: "subsite-like" means the URL is likely an office/center/outreach subsite,
    or lives on a non-standard subdomain that frequently hosts non-inventory pages.

    Rules:
    - Different rootish domain => subsite-like
    - Subdomain first-label not in allowlist => subsite-like
      (unless it's literally the base host itself)
    - Path contains subsite hint words => subsite-like
    """
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Offsite-ish drift
    if rootish_domain(host) != rootish_domain(base):
        return True

    # Subdomain gating: majors.<root>, catalog.<root>, etc allowed; others penalized
    if host != base:
        # get first label (e.g., "catalog" in "catalog.bu.edu")
        parts = [p for p in host.split(".") if p]
        sub = parts[0] if parts else ""
        if sub and sub not in ALLOWED_ACADEMIC_SUBDOMAINS:
            return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in SUBSITE_HINTS):
        return True

    return False

def is_hard_subsite_block(url: str, base_netloc: str) -> bool:
    """True if URL is a known high-precision subsite trap (auto-loss)."""
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Drift to different registrable-ish domain => hard-block
    if rootish_domain(host) != rootish_domain(base):
        return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    return any(tok in full for tok in HARD_SUBSITE_BLOCK_TOKENS)

def is_too_specific_url(url: str) -> bool:
    """True if URL path matches TOO_SPECIFIC_PATTERNS."""
    path = (urlparse(url).path or "").lower()
    for pat in TOO_SPECIFIC_PATTERNS:
        if re.search(pat, path):
            return True
    return False

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)


def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)


def is_canonical_hub_url(url: str) -> bool:
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical


def looks_like_detail_url(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return any(p.search(path) for p in DETAIL_URL_PATTERNS)


# ============================================================
# Row tagging
# ============================================================
def _norm_host_from_web_address(web_address: str) -> str:
    u = ensure_scheme(web_address)
    if not u:
        return ""
    return (urlparse(u).netloc or "").lower()


def tag_row(web_address: str, best_url: str) -> str:
    """
    Returns one of:
      - 'Correct hub'
      - 'Correct-ish but too specific'
      - 'Wrong subsite'
      - 'Wrong non-academic'
    """
    base_host = _norm_host_from_web_address(web_address)
    u = ensure_scheme(best_url)
    if not u or best_url == "N/A":
        return "Wrong non-academic"

    host = (urlparse(u).netloc or "").lower()
    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in NON_ACADEMIC_HINTS):
        return "Wrong non-academic"

    # off-root-domain or obvious subsite hints
    if base_host and rootish_domain(host) != rootish_domain(base_host):
        return "Wrong subsite"
    if any(x in full for x in SUBSITE_HINTS):
        return "Wrong subsite"

    # hub-ish path
    if any(x in path for x in HUB_PATH_HINTS):
        for pat in TOO_SPECIFIC_PATTERNS:
            if re.search(pat, path):
                return "Correct-ish but too specific"
        return "Correct hub"

    # weaker academic fallback
    if "academ" in path or "catalog" in path or "bulletin" in path:
        return "Correct-ish but too specific"

    return "Wrong subsite"


# ============================================================
# HTML parse + main-content extraction
# ============================================================
@dataclass
class ParsedPage:
    url: str
    visible_text: str
    main_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]  # (anchor_text, abs_url)
    title_like: List[str]
    html_title: str
    h1: str
    corpus_any: str
    corpus_structured: str
    json_blob: str
    js_hint: int


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    h1_tag = soup.find("h1")
    h1 = normalize_unicode_text(h1_tag.get_text(" ", strip=True)) if h1_tag else ""

    # --- Embedded JSON support (v12 B): keep machine-readable program lists ---
    json_chunks: List[str] = []
    for sc in soup.find_all("script"):
        typ = (sc.get("type") or "").strip().lower()
        sid = (sc.get("id") or "").strip().lower()
        # Many modern hubs embed the program list in JSON (ld+json, app/json, or Next.js __NEXT_DATA__)
        if typ in ("application/ld+json", "application/json") or sid == "__next_data__":
            raw = sc.string or sc.get_text(" ", strip=True)
            raw = normalize_unicode_text(raw)
            if raw:
                json_chunks.append(raw)

    json_blob = normalize_unicode_text(" ".join(json_chunks))
    if len(json_blob) > MAX_JSON_BLOB_CHARS:
        json_blob = json_blob[:MAX_JSON_BLOB_CHARS]

    # Heuristic: if page looks like a JS-rendered app shell, mark it so scoring/candidate expansion can adapt
    raw_lower = (html or "").lower()
    js_hint = 1 if (
        "__next_data__" in raw_lower
        or "window.__nuxt__" in raw_lower
        or "data-reactroot" in raw_lower
        or "id=\"root\"" in raw_lower
        or "id=\"app\"" in raw_lower
        or "ng-version" in raw_lower
    ) else 0

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    main_node = soup.find("main") or soup.find(id=re.compile(r"content|main", re.I)) or soup.find("article")
    if main_node:
        main_text = normalize_unicode_text(main_node.get_text(separator=" ", strip=True))
    else:
        soup2 = BeautifulSoup(str(soup), "html.parser")
        for tag in soup2(["header", "footer", "nav"]):
            tag.decompose()
        main_text = normalize_unicode_text(soup2.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(
            x
            for x in [visible_text, json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts)]
            if x
        )
    )
    corpus_structured = normalize_unicode_text(
        " ".join(
            x
            for x in [json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)]
            if x
        )
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        main_text=main_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        h1=h1,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
        json_blob=json_blob,
        js_hint=js_hint,
    )


def is_soft_404(parsed: ParsedPage) -> bool:
    blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1] + parsed.title_like[:5]))
    if SOFT_404_TEXT.search(blob):
        return True
    if len(parsed.main_text) < 600 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    if len(parsed.main_text) < 250 and len(parsed.anchor_texts) < 5:
        return True
    return False


def page_is_thin(parsed: ParsedPage) -> bool:
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) and (len(parsed.main_text) < MIN_MAIN_TEXT_LEN)


# ============================================================
# Sitemap parsing
# ============================================================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()


def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls


def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()

        # hard skips
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue

        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break

    return out


# ============================================================
# Hubness: program-listing-only link counts
# ============================================================
def is_listing_link(url: str) -> bool:
    path = (urlparse(url).path or "")
    return any(p.search(path) for p in LISTING_LINK_PATTERNS)


def hubness_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    listing_links = 0
    slug_set: Set[str] = set()
    distinct_anchor: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        if not is_listing_link(u):
            continue

        listing_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        path = (urlparse(u).path or "").strip("/").lower()
        segs = [s for s in path.split("/") if s]
        if segs:
            slug_set.add(segs[-1])

    # weak secondary signal
    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if listing_links >= 12:
        sig += 1
    if listing_links >= 25:
        sig += 1
    if len(slug_set) >= 8:
        sig += 1
    if len(slug_set) >= 16:
        sig += 1
    if len(distinct_anchor) >= 18:
        sig += 1
    if li_like >= 40 and (listing_links >= 10):
        sig += 1

    counts = {
        "listing_links": listing_links,
        "slug_diversity": len(slug_set),
        "distinct_anchor": len(distinct_anchor),
        "li_like": li_like,
    }
    return sig, counts


# ============================================================
# Scoring
# ============================================================
@dataclass
class PageSignals:
    url: str
    tier: str
    soft404: int
    thin: int

    canonical_hub: int
    hub_sig: int
    hub_counts: Dict[str, int]

    struct_hits: List[str]
    control_unique_structured: int

    year_hit: int
    undergrad_title_boost: int

    grad_penalty: int
    profile_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int

    subsite_like: int  # configurable penalty via knob
    hard_subsite_block: int  # auto-loss for known traps
    too_specific: int  # explicit penalty separate from generic detail penalty
    js_hub_hint: int


def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if is_inventoryish_url(url):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"


def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    soft404 = 1 if is_soft_404(parsed) else 0
    thin = 1 if page_is_thin(parsed) else 0

    control_unique_structured = sum(1 for pat in CONTROL_PATS.values() if pat.search(parsed.corpus_structured))
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    title_blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1]))
    year_hit = 1 if (YEAR_REGEX.search(url) or YEAR_REGEX.search(title_blob)) else 0
    undergrad_title_boost = 1 if UNDERGRAD_TITLE_BOOST.search(title_blob) else 0

    u_low = url.lower()
    grad_penalty = 1 if (GRAD_URL_PENALTY.search(u_low) or re.search(r"\bgraduate\b", title_blob, re.I)) else 0
    profile_penalty = 1 if (PROFILE_URL_PENALTY.search(u_low) or re.search(r"\bfaculty directory\b", title_blob, re.I)) else 0
    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(u_low) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(u_low) else 0
    archive_penalty = 1 if ARCHIVE_URL_PENALTY.search(u_low) else 0
    awards_penalty = 1 if AWARDS_URL_PENALTY.search(u_low) else 0
    compliance_penalty = 1 if COMPLIANCE_URL_PENALTY.search(u_low) else 0

    hub_sig, hub_counts = hubness_signature(parsed, base_netloc=base_netloc)
    canonical_hub = 1 if is_canonical_hub_url(url) else 0

    subsite_like = 1 if is_subsite_like(url, base_netloc=base_netloc) else 0
    hard_subsite_block = 1 if is_hard_subsite_block(url, base_netloc=base_netloc) else 0
    too_specific = 1 if (is_too_specific_url(url) and not is_canonical_hub_url(url)) else 0

    js_hub_hint = 1 if (getattr(parsed, "js_hint", 0) and (len(getattr(parsed, "json_blob", "")) >= 200)) else 0

    return PageSignals(
        url=url,
        tier=page_tier(url),
        soft404=soft404,
        thin=thin,
        canonical_hub=canonical_hub,
        hub_sig=hub_sig,
        hub_counts=hub_counts,
        struct_hits=struct_hits,
        control_unique_structured=control_unique_structured,
        year_hit=year_hit,
        undergrad_title_boost=undergrad_title_boost,
        grad_penalty=grad_penalty,
        profile_penalty=profile_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
        subsite_like=subsite_like,
        hard_subsite_block=hard_subsite_block,
        too_specific=too_specific,
        js_hub_hint=js_hub_hint,
    )


def canonical_bonus_allowed(sig: PageSignals) -> bool:
    if sig.soft404:
        return False
    if sig.thin and sig.hub_sig == 0 and len(sig.struct_hits) < 2:
        return False
    return True


def score_inventory(sig: PageSignals, subsite_penalty: int) -> int:
    if sig.soft404:
        return -10_000
    
    # hard subsite blocks are automatic losses
    if getattr(sig, "hard_subsite_block", 0):
        return -10_000    

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 40

    # hubness + structured hits
    score += 85 * sig.hub_sig
    score += 10 * len(sig.struct_hits)
    score += 4 * sig.control_unique_structured

    # canonical hub bonus (gated)
    if sig.canonical_hub and canonical_bonus_allowed(sig):
        score += 160

    # undergrad title boost
    if sig.undergrad_title_boost:
        score += 35

    # year boost only for catalog/bulletin pages
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        score += 70

    # penalties (URL/title/H1)
    if sig.profile_penalty:
        score -= 260
    if sig.grad_penalty:
        score -= 170
    if sig.irrelevant_penalty:
        score -= 260
    if sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 220
    if sig.awards_penalty:
        score -= 200
    if sig.compliance_penalty:
        score -= 150

    # configurable subsite penalty
    if sig.subsite_like:
        score -= int(subsite_penalty)

    # explicit penalty for "too specific" department/program pages
    if getattr(sig, "too_specific", 0):
        score -= int(TOO_SPECIFIC_PENALTY)    

    # broad detail penalty so hubs win
    if looks_like_detail_url(sig.url):
        if sig.hub_sig <= 1:
            score -= 220
        elif sig.hub_sig == 2:
            score -= 90
        else:
            score -= 25

    # thin penalty (C2: JS hubs can be thin app shells; barely penalize)
    if sig.thin and sig.hub_sig == 0:
        if getattr(sig, "js_hub_hint", 0) and is_inventoryish_url(sig.url):
            score -= 20
        else:
            score -= 180

    return score
# ============================================================
# Candidate generation
# ============================================================
def looks_like_js_hub(parsed: ParsedPage, url: str) -> bool:
    """Detect JS-rendered program hubs early so we can expand via sitemap sooner."""
    if not page_is_thin(parsed):
        return False

    u = (url or "").lower()
    host = (urlparse(url).netloc or "").lower()
    inventoryish = is_inventoryish_url(url) or (host.startswith("majors.") or host.startswith("programs."))
    if not inventoryish and "find-your-program" not in u:
        return False

    # Typical app shells have few usable anchors and often embed data in JSON
    if len(parsed.links) > 60:
        return False
    if getattr(parsed, "js_hint", 0):
        return True
    if len(getattr(parsed, "json_blob", "")) >= 200:
        return True
    return False



def url_prior(url: str, base_netloc: str, subsite_penalty: int) -> int:
    u = url.lower()
    s = 0

    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 200
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 35
    if "bulletin" in u:
        s -= 20

    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if COMPLIANCE_URL_PENALTY.search(u):
        s -= 90
    if AWARDS_URL_PENALTY.search(u):
        s -= 160
    if GRAD_URL_PENALTY.search(u):
        s -= 180

    # hard subsite blocks should be avoided aggressively
    if is_hard_subsite_block(url, base_netloc=base_netloc):
        s -= 10_000
    if is_subsite_like(url, base_netloc=base_netloc):
        s -= int(subsite_penalty)

    if is_canonical_hub_url(url):
        s += 160

    return s


def prioritize_links(links: List[Tuple[str, str]], base_netloc: str, subsite_penalty: int) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000

        s = url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty)

        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 45
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 90
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140

        return -s

    return sorted(links, key=rank)


# ============================================================
# Candidate structure
# ============================================================
@dataclass
class CandidatePage:
    url: str
    score: int
    sig: PageSignals
    parsed: ParsedPage


def get_parsed_page(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)


def fetch_and_score(url: str, session: requests.Session, base_netloc: str, subsite_penalty: int) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page(url, session=session)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        sc = score_inventory(sig, subsite_penalty=subsite_penalty)
        return CandidatePage(url=url, score=sc, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)


# ============================================================
# Parallel batch fetch+score helper
# ============================================================
def fetch_and_score_many(
    urls: List[str],
    base_netloc: str,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    """Fetch+score a batch of URLs concurrently (one Session per thread)."""
    # de-dupe while preserving order
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            uniq.append(u)

    if workers <= 1:
        out: List[CandidatePage] = []
        s = requests.Session()
        for u in uniq:
            cand = fetch_and_score(u, session=s, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
            if cand is not None:
                out.append(cand)
        return out

    out: List[CandidatePage] = []

    def _task(u: str) -> Optional[CandidatePage]:
        # IMPORTANT: call get_thread_session() inside the worker thread
        return fetch_and_score(
            u,
            session=get_thread_session(),
            base_netloc=base_netloc,
            subsite_penalty=subsite_penalty,
        )

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_task, u) for u in uniq]
        for fut in as_completed(futs):
            cand = fut.result()
            if cand is not None:
                out.append(cand)
    return out


def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.score, len(topk), cand))
        return
    if cand.score > topk[0][0]:
        heapq.heapreplace(topk, (cand.score, topk[0][1], cand))


def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    """Return candidates sorted by score (desc), with an optional canonical-hub tie-break."""
    cands = [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]

    # C: Prefer canonical hubs more aggressively when tie-ish.
    if cands:
        best_score = cands[0].score

        canonical_best: Optional[CandidatePage] = None
        for c in cands:
            if c.sig.canonical_hub and (best_score - c.score) <= CANONICAL_TIE_THRESHOLD:
                if canonical_best is None or c.score > canonical_best.score:
                    canonical_best = c

        if canonical_best is not None and canonical_best is not cands[0]:
            cands = [canonical_best] + [x for x in cands if x is not canonical_best]

    return cands


# ============================================================
# Candidate generation
# ============================================================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))

    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================================================
# Program title extraction
# ============================================================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def looks_like_program_title(s: str, progtitle_strictness: int, context: str = "hub") -> bool:
    """
    Strictness guidance:
      1-2: permissive (still requires TARGET_ANY)
      3: current default behavior
      4-5: stricter (prefer explicit program-ish keywords; tighter filters)
    context: "hub" (default) or "follow" (detail)
    """
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False

    s_low = s_norm.lower()

    # D1: obvious junk rejects
    if URL_LIKE_TEXT.search(s_norm):
        return False
    if "toggle" in s_low:
        return False

    # always require the target token family (keeps extraction scoped)
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False

    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm):
        return False

    # D1: sentence-like / prose-ish rejects
    if ". " in s_norm and len(s_norm) >= 40:
        return False
    if ": " in s_norm and len(s_norm) >= 55:
        # allow short degree-style labels, reject long descriptive clauses
        if PROGRAM_KEYWORDS.search(s_norm) is None and not re.search(r"\b(B\.?A\.?|B\.?S\.?|BA|BS)\b", s_norm, re.I):
            return False
    if PROSE_VERBS.search(s_norm) and len(s_norm) >= 45:
        # follow/detail pages are stricter; hubs can still reject obvious prose
        if context == "follow" or progtitle_strictness >= 3:
            return False

    # D1: long multi-clause strings are usually prose or navigation
    if len(s_norm) > 90:
        comma_ct = s_norm.count(",")
        if comma_ct >= 2 and PROGRAM_KEYWORDS.search(s_norm) is None:
            return False

    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        if progtitle_strictness >= 3:
            return False

    # D2: follow/detail pages must show stronger "program" intent
    if context == "follow":
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    if PERSON_NAME_ONLY.match(s_norm):
        if progtitle_strictness >= 3 and not PROGRAM_KEYWORDS.search(s_norm):
            return False
        if progtitle_strictness >= 5:
            return False

    if progtitle_strictness >= 4:
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    return True
def is_program_detailish_url(url: str) -> bool:
    """Tight allowlist for one-hop follow: only follow likely program/major detail pages."""
    path = (urlparse(url).path or "")
    if any(p.search(path) for p in LISTING_LINK_PATTERNS):
        return True
    # also allow common majors-programs hubs/details
    if re.search(r"/majors-programs/[^/]+", path, re.I):
        return True
    return False


def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out


# ============================================================
# PDF fallback (best-effort)
# ============================================================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""


def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================================================
# Aggregate extraction from inventory + alts (+ limited 1-hop)
# ============================================================
def aggregate_outputs(
    pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
    progtitle_strictness: int,
    workers: int,
) -> Tuple[
    int, Set[str], Dict[str, int],
    Dict[str, List[str]],
    List[str],
    List[str],
]:
    base_netloc = urlparse(base_url).netloc

    struct_hits_union: Set[str] = set()
    control_hit_cols = {k: 0 for k in CONTROL_TERMS}
    any_control_found = 0

    tokens_agg: Dict[str, Set[str]] = {"afri": set(), "ethnic": set(), "black": set(), "race": set()}
    titles: Set[str] = set()

    pdf_hits: List[str] = []

    # from chosen pages
    for cand in pages:
        any_control_found = max(any_control_found, 1 if cand.sig.control_unique_structured > 0 else 0)
        struct_hits_union.update(cand.sig.struct_hits)

        for ck, pat in CONTROL_PATS.items():
            if pat.search(cand.parsed.corpus_structured):
                control_hit_cols[ck] = 1

        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k, vals in tm.items():
            tokens_agg[k].update(vals)

        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
                titles.add(txt)

    # 1-hop follow
    follow_urls: List[str] = []
    for cand in pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            # D3: follow only if anchor text looks like a title, OR URL looks like a program detail page
            if looks_like_program_title(a_text or "", progtitle_strictness=progtitle_strictness, context="hub") or is_program_detailish_url(u):
                follow_urls.append(u)

    # sitemap assist if best looks thin
    if pages and page_is_thin(pages[0].parsed):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    # Fetch a limited number of follow URLs (optionally in parallel)
    follow_batch = uniq_follow[:ONE_HOP_MAX_FETCHES]

    def _safe_fetch_parse(u: str) -> Optional[ParsedPage]:
        try:
            # Use per-thread sessions when parallel
            s = get_thread_session() if workers and workers > 1 else session
            return get_parsed_page(u, session=s)
        except Exception:
            return None
        finally:
            # Keep some politeness throttling even when parallel
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    if workers and workers > 1 and len(follow_batch) > 1:
        max_w = min(int(workers), 8, len(follow_batch))
        with ThreadPoolExecutor(max_workers=max_w) as ex:
            futs = [ex.submit(_safe_fetch_parse, u) for u in follow_batch]
            for fut in as_completed(futs):
                parsed = fut.result()
                if not parsed:
                    continue

                tm = token_matches_from_text(parsed.corpus_any)
                for k, vals in tm.items():
                    tokens_agg[k].update(vals)

                for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                    if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                        titles.add(txt)
    else:
        for u in follow_batch:
            parsed = _safe_fetch_parse(u)
            if not parsed:
                continue

            tm = token_matches_from_text(parsed.corpus_any)
            for k, vals in tm.items():
                tokens_agg[k].update(vals)

            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                    titles.add(txt)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in pages:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k, vals in tm.items():
                            tokens_agg[k].update(vals)
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Dedup titles by normalized key
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    titles_out = sorted(best_display.values())
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}

    return any_control_found, struct_hits_union, control_hit_cols, tokens_out, titles_out, pdf_hits


#
# ============================================================
# Discovery: find candidate pages
# ============================================================
def find_candidates_for_institution(
    base_url: str,
    session: requests.Session,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_cache
        if sitemap_cache is not None:
            return sitemap_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_cache = sitemap_candidate_urls(all_urls)
        return sitemap_cache

    def consider(u: str) -> Optional[CandidatePage]:
        if not u or u in tried:
            return None
        tried.add(u)
        cand = fetch_and_score(u, session=session, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
        if cand is None:
            return None
        push_topk(topk, cand, k=TOP_K_ALTS)
        return cand

    # Phase 0
    consider(base_url)

    # ------------------------------------------------------------
    # Phase 1 (parallelizable): explicit paths + year/catalog + home links
    # ------------------------------------------------------------
    batch_urls: List[str] = []

    for p in INVENTORY_PATHS[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    batch_urls.extend(build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES])

    for p in CATALOG_PATHS[:PHASE1_MAX_CATALOG_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    # homepage links require parsing homepage once (sequential), but the fetched URLs can be scored in batch
    try:
        home = get_parsed_page(base_url, session=session)
        # C1: If the homepage/programs page is a JS hub (thin app shell), expand candidates via sitemap immediately
        if looks_like_js_hub(home, base_url):
            try:
                batch_urls.extend(load_sitemap_candidates()[:80])
            except Exception:
                pass
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc, subsite_penalty=subsite_penalty)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            batch_urls.append(u)
    except Exception:
        pass

    # score the batch in parallel (if workers>1)
    batch_cands = fetch_and_score_many(
        batch_urls,
        base_netloc=base_netloc,
        subsite_penalty=subsite_penalty,
        workers=workers,
    )
    for cand in batch_cands:
        tried.add(cand.url)
        push_topk(topk, cand, k=TOP_K_ALTS)

    # C2: If ANY early candidate looks like a JS-rendered hub, expand via sitemap NOW
    # (even if the thin hub didn't win the scoring yet).
    try:
        if sitemap_cache is None:
            js_hub_seen = any(looks_like_js_hub(c.parsed, c.url) for c in batch_cands)
            if js_hub_seen:
                sm_urls = load_sitemap_candidates()[:120]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
    except Exception:
        pass

    # Optional: if best so far looks weak, pull some sitemap candidates (also batchable)
    if topk:
        best_so_far = max(topk, key=lambda x: x[0])[2]
        if (best_so_far.sig.thin or best_so_far.sig.hub_sig == 0) and is_inventoryish_url(best_so_far.url):
            try:
                sm_urls = load_sitemap_candidates()[:50]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
            except Exception:
                pass

    # Best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty), depth, u))

    pq_push(base_url, 0)
    fetched = 0
    while pq and fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u)
        fetched += 1
        if depth >= PHASE2_MAX_DEPTH:
            continue
        try:
            parsed = cand.parsed if cand is not None else get_parsed_page(u, session=session)
            links = [(t, lu) for (t, lu) in parsed.links if same_domain(lu, base_netloc)]
            links = prioritize_links(links, base_netloc, subsite_penalty=subsite_penalty)
            for _, lu in links[:70]:
                pq_push(lu, depth + 1)
        except Exception:
            pass

    # Subdomain roots
    for root in build_subdomain_roots(base_url):
        c_root = consider(root)
        if not c_root:
            continue
        root_is_good = (c_root.sig.hub_sig >= 1) or (len(c_root.sig.struct_hits) >= 2) or (not c_root.sig.thin)
        if root_is_good and not c_root.sig.soft404:
            continue
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p))

    return topk_sorted(topk)


# ============================================================
# Reason string
# ============================================================
def inventory_reason(c: CandidatePage) -> str:
    s = c.sig
    flags = []
    if s.grad_penalty:
        flags.append("grad")
    if s.profile_penalty:
        flags.append("profile")
    if s.irrelevant_penalty:
        flags.append("irrelevant")
    if s.admissions_penalty:
        flags.append("admissions")
    if s.archive_penalty:
        flags.append("archive")
    if s.awards_penalty:
        flags.append("awards")
    if s.compliance_penalty:
        flags.append("compliance")
    if s.year_hit:
        flags.append("yearHit")
    if s.thin:
        flags.append("thin")
    if s.soft404:
        flags.append("soft404")
    if s.subsite_like:
        flags.append("subsiteLike")
    if getattr(s, "hard_subsite_block", 0):
        flags.append("hardSubsiteBlock")
    if getattr(s, "too_specific", 0):
        flags.append("tooSpecific")

    return ";".join([
        f"score={c.score}",
        f"tier={s.tier}",
        f"hubSig={s.hub_sig}",
        f"listingLinks={s.hub_counts.get('listing_links', 0)}",
        f"slugDiv={s.hub_counts.get('slug_diversity', 0)}",
        f"structHits={len(s.struct_hits)}",
        f"ctrlStructured={s.control_unique_structured}",
        f"canonicalHub={s.canonical_hub}",
        ("flags=" + ",".join(flags)) if flags else "flags=",
    ])


# ============================================================
# CLI
# ============================================================
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--input", type=str, default=str(DEFAULT_INPUT_PATH))
    ap.add_argument("--output", type=str, default=str(DEFAULT_OUTPUT_PATH))

    # If omitted, default to TEST_HEAD_N (notebook-friendly). Use --head 0 for full run.
    ap.add_argument("--head", type=int, default=None, help="If >0, limit rows for testing; if omitted defaults to TEST_HEAD_N")

    ap.add_argument("--out-suffix", type=str, default="", help="Append to output filename stem")
    ap.add_argument("--subsite-penalty", type=int, default=80)
    ap.add_argument("--progtitle-strictness", type=int, default=3)

    # Parallelism (keep small to be polite)
    ap.add_argument(
        "--workers",
        type=int,
        default=DEFAULT_WORKERS,
        help="Parallel fetch workers for candidate scoring (1 = off). Try 4.",
    )

    # IMPORTANT: ignore Jupyter/ipykernel injected args like --f=...
    args, _unknown = ap.parse_known_args(argv)

    if args.head is None:
        args.head = TEST_HEAD_N

    # sanity clamp
    if args.workers is None or args.workers < 1:
        args.workers = 1
    if args.workers > MAX_WORKERS:
        args.workers = MAX_WORKERS

    return args


def apply_out_suffix(out_path: Path, suffix: str) -> Path:
    if not suffix:
        return out_path
    return out_path.with_name(out_path.stem + suffix + out_path.suffix)


# ============================================================
# Main
# ============================================================
def main() -> None:
    args = parse_args()
    input_path = Path(args.input)
    output_path = apply_out_suffix(Path(args.output), args.out_suffix)

    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {input_path}: {sorted(missing)}")

    if args.head and args.head > 0:
        df = df.head(args.head).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")

        best_inventory_url = "N/A"
        best_inventory_reason = ""
        alt_candidate_urls = ""

        any_control_found = 0
        struct_hits_union: Set[str] = set()
        any_struct_found = 0

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        afri_matches: List[str] = []
        ethnic_matches: List[str] = []
        black_matches: List[str] = []
        race_matches: List[str] = []

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0
        pdf_hits: List[str] = []

        try:
            if base_url:
                candidates = find_candidates_for_institution(
                    base_url,
                    session=session,
                    subsite_penalty=args.subsite_penalty,
                    workers=args.workers,
                )

                if candidates:
                    best = candidates[0]
                    best_inventory_url = best.url
                    best_inventory_reason = inventory_reason(best)
                    alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K_ALTS])

                    any_control_found, struct_hits_union, control_hits, tokens, titles, pdf_hits = aggregate_outputs(
                        pages=candidates[:TOP_K_ALTS],
                        base_url=base_url,
                        session=session,
                        progtitle_strictness=args.progtitle_strictness,
                        workers=args.workers,
                    )

                    any_struct_found = 1 if struct_hits_union else 0

                    for k, v in control_hits.items():
                        control_hit_cols[f"found_{k}"] = int(v)

                    afri_matches = tokens.get("afri", [])
                    ethnic_matches = tokens.get("ethnic", [])
                    black_matches = tokens.get("black", [])
                    race_matches = tokens.get("race", [])

                    program_titles = titles
                    program_title_count = len(program_titles)
                    any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0
        except Exception:
            # keep row, but leave defaults
            pass

        rec: Dict[str, str] = dict(row)

        rec["best_guess_inventory_url"] = best_inventory_url
        rec["best_guess_inventory_reason"] = best_inventory_reason
        rec["url_tag"] = tag_row(rec.get(WEB_COL, ""), best_inventory_url)
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        rec["afri_matches"] = "|".join(afri_matches) if afri_matches else ""
        rec["ethnic_matches"] = "|".join(ethnic_matches) if ethnic_matches else ""
        rec["black_matches"] = "|".join(black_matches) if black_matches else ""
        rec["race_matches"] = "|".join(race_matches) if race_matches else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        rec.update({k: str(v) for k, v in control_hit_cols.items()})
        out_rows.append(rec)

        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "url_tag",
        "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        "afri_matches", "ethnic_matches", "black_matches", "race_matches",
        "any_potential_AfrAmr_found",
        "pdf_hits",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(output_path, index=False)
    print(f"\nWrote: {output_path} | rows,cols = {out_df.shape}")
    print(out_df.head())


if __name__ == "__main__":
    main()

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

overwrote v11 output on accident

A) Wrong hub selected
	•	Boise: missed the majors subdomain canonical hub
	•	Barry: missed the canonical find-your-program hub
	•	Chico: selected a subset portal (“online”) that happens to look inventory-ish
	•	Fresno: selected a catalog detail page rather than CHHS academics hub

B) Correct hub but zero titles
	•	Agnes Scott / CPP: strongly points to JS-rendered program directory where titles live in embedded JSON / API (or don’t match token-trigger rules in the captured text)

C) Correct hub but titles are noisy / not canonical
	•	Bates / CSUDH: title extraction still allows non-program headings and “topic” content; likely need stronger “hub list item” heuristics + better junk rejection.

In [2]:
#!/usr/bin/env python3
"""
v13: program-inventory discovery + scoring + tagging (clean)

Key additions vs v10:
- Row tagging for best_guess_inventory_url:
    * Correct hub
    * Correct-ish but too specific
    * Wrong subsite
    * Wrong non-academic
- CLI knobs for sweep runs from ipynb:
    --subsite-penalty (int): penalty applied to subsite-like pages
    --progtitle-strictness (int): controls how strict program title extraction is
    --head (int): limit rows for quick testing
    --out-suffix (str): appended to output filename stem
    --input / --output: file paths

Notebook-friendly:
- TEST_HEAD_N provides default subset size when --head is not provided.
  Use --head 0 for full run.
- DEFAULT_WORKERS provides a top-level default for --workers.

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v11.csv (default)
"""

from __future__ import annotations

import argparse
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ============================================================
# Thread-local requests sessions (for safe parallel fetching)
# ============================================================
_thread_local = threading.local()


def get_thread_session() -> requests.Session:
    """Return a per-thread requests.Session (requests.Session is not thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        _thread_local.session = s
    return s

# ============================================================
# Per-institution fetch-status tally (403-aware)
# ============================================================
# We process institutions sequentially, but fetch pages in parallel per institution.
# This shared tally lets us tag rows when any attempted fetch hit a 403 (or similar).
_STATUS_LOCK = threading.Lock()
_STATUS_COUNTS: Dict[int, int] = {}


def reset_status_tally() -> None:
    with _STATUS_LOCK:
        _STATUS_COUNTS.clear()


def record_http_status(code: int) -> None:
    if not code:
        return
    with _STATUS_LOCK:
        _STATUS_COUNTS[code] = _STATUS_COUNTS.get(code, 0) + 1


def fetch_status_tag(best_url: str) -> str:
    """Row tag describing fetch status / blocks encountered during candidate discovery."""
    with _STATUS_LOCK:
        counts = dict(_STATUS_COUNTS)

    # Prioritize explicit blocks
    if counts.get(403, 0) > 0:
        return "403_forbidden_seen"
    if counts.get(429, 0) > 0:
        return "429_rate_limited_seen"

    # Any server errors encountered
    if any(500 <= c <= 599 for c in counts.keys()):
        return "5xx_seen"

    # If nothing worked
    if not best_url or best_url == "N/A":
        return "no_candidate"

    return "ok"

# ============================================================
# Notebook-friendly default test mode
# ============================================================
# If you run from an ipynb as:  !python v11_program_inventory.py
# it will default to processing TEST_HEAD_N rows unless you pass --head.
# Use --head 0 for full run.
TEST_HEAD_N = 40

# ============================================================
# Top-level parallelism defaults
# ============================================================
# You can override via CLI: --workers
# Keep MAX_WORKERS small to stay polite to university sites.
DEFAULT_WORKERS = 6
MAX_WORKERS = 8


# ============================================================
# Defaults (override via CLI)
# ============================================================
DEFAULT_INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
DEFAULT_OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v13.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.25
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.20

MAX_BYTES_PER_PAGE = 2_000_000
MAX_JSON_BLOB_CHARS = 200_000

CACHE_DIR = Path(".cache_v13_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Per-institution budgets
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 22
PHASE2_BESTFIRST_MAX_PAGES = 55
PHASE2_MAX_DEPTH = 3

TOP_K_ALTS = 6
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 18

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

MIN_VISIBLE_TEXT_LEN = 900
MIN_MAIN_TEXT_LEN = 650

ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/13.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================================================
# URL templates
# ============================================================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================================================
# Patterns / scoring signals
# ============================================================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_URL_PENALTY = re.compile(
    r"\b(graduate|grad-studies|gradstudies|master|masters|doctoral|phd|mba|mfa|ms|ma)\b",
    flags=re.IGNORECASE,
)
PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)
AWARDS_URL_PENALTY = re.compile(r"\b(awards?|honors?)\b", flags=re.IGNORECASE)
COMPLIANCE_URL_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_TITLE_BOOST = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    "bio": r"\bbio[a-z]*\b",
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

#
# Expanded target token family (v12): includes Africa/African/Africana, Pan-African,
# African American, Afro-American, Africology, Race, plus existing afri/ethnic/black.
#
# NOTE: Keep "TARGET_ANY_REGEX" broad (union) so extraction triggers properly,
# but keep per-bucket regexes interpretable for output columns.
TARGET_TOKEN_REGEX = {
    # Broaden "afri" bucket to cover common program-name variants beyond just "afri*".
    "afri": re.compile(
        r"\b(afri[a-z\-]*|africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology)\b",
        flags=re.IGNORECASE,
    ),
    # Include both "ethnic*" and "ethnicity" so the bucket reflects real page phrasing.
    "ethnic": re.compile(r"\b(ethnic[a-z\-]*|ethnicity)\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
    # New bucket: race / racial
    "race": re.compile(r"\b(race|racial)\b", flags=re.IGNORECASE),
}

# Trigger regex used to decide whether text is in-scope for extraction.
# Keep this broad (union of all target families) so we don't miss relevant programs.
TARGET_ANY_REGEX = re.compile(
    r"\b(afri[a-z\-]*|africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology|"
    r"ethnic[a-z\-]*|ethnicity|black[a-z\-]*|race|racial)\b",
    flags=re.IGNORECASE,
)

# Output column stability: keep these buckets as fixed output columns.
# If TARGET_TOKEN_REGEX changes, these columns will still be present (empty if missing).
OUTPUT_TOKEN_BUCKETS = ["afri", "ethnic", "black", "race"]

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}\s*[-]?\s*\d{2,3}[A-Z]?\b")
COURSE_WORDS = re.compile(r"\b(units?|credits?|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")
PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

# --- Title junk / prose detection (v12 D) ---
URL_LIKE_TEXT = re.compile(r"(https?://|www\.|\b[a-z0-9\-]+\.(edu|com|org|net)(/|\b))", flags=re.IGNORECASE)
PROSE_VERBS = re.compile(
    r"\b(define|apply|enrich|explore|learn|prepare|prepares|focus(es)?|offers|provides|designed|helps|aims)\b",
    flags=re.IGNORECASE,
)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

# "Listing-like" URL patterns for anchors we count toward hubness
LISTING_LINK_PATTERNS = [
    re.compile(r"/programs/[^/]+/?$", re.I),
    re.compile(r"/majors-minors/[^/]+/?$", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/departments/[^/]+/?$", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
]

# Broader detail-page detection (so single-major pages don’t beat hubs)
DETAIL_URL_PATTERNS = [
    re.compile(r"/programs/[^/]+/.+", re.I),
    re.compile(r"/majors-minors/[^/]+/.+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/.+", re.I),
    re.compile(r"/departments/[^/]+/.+", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
    re.compile(r"/content/.*/departments/[^/]+\.html$", re.I),
]

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================================================
# Row tagging heuristics
# ============================================================
HUB_PATH_HINTS = [
    "majors", "minors", "programs", "degrees", "department", "departments",
    "departments-and-programs", "academics", "academic", "catalog", "bulletin",
    "undergraduate", "undergraduate-programs", "courses", "areas-of-study"
]

NON_ACADEMIC_HINTS = [
    "news", "event", "calendar", "giving", "alumni", "advancement",
    "athletics", "sports", "tickets", "shop", "bookstore",
    "it", "its", "help", "support", "login", "privacy", "accessibility",
    "admissions", "apply", "visit", "tuition", "financial-aid",
    "student-life", "housing", "dining", "library",
    "about", "contact", "directory", "people", "faculty", "staff",
    "press", "media", "magazine", "blog", "policy"
]

SUBSITE_HINTS = [
    "center", "centre", "institute", "office", "community", "belonging",
    "extension", "continuing-ed", "continuing-education",
    "professional", "outreach",
    # dataset-specific common traps
    "nwo", "cge",
]

# High-precision traps: if these appear in host+path, treat page as an automatic loss.
HARD_SUBSITE_BLOCK_TOKENS = [
    "nwo",  # dataset-specific trap
    "cge",  # dataset-specific trap
]

# Separate from the broad detail penalty: this targets department/program/detail pages that can look
# academically relevant but are still not the intended "programs hub" landing page.
# Starting value calibrated from sweep snapshots: “Correct-ish but too specific” was rare (~2/25 rows, ~8%),
# and usually didn’t have an obvious canonical hub alternative in the top-K alts; use a moderate penalty
# (similar scale to other URL penalties) to nudge toward hubs without overcorrecting.
TOO_SPECIFIC_PENALTY = 240

# If a canonical hub is within this many points of best, prefer it.
CANONICAL_TIE_THRESHOLD = 60

TOO_SPECIFIC_PATTERNS = [
    r"/departments?/[a-z0-9_-]+",
    r"/departments-and-programs/[a-z0-9_-]+",
    r"/programs?/[a-z0-9_-]+",
    r"/academics/.+/(major|minors|programs)\b",
]


# ============================================================
# Unicode normalization
# ============================================================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================================================
# Cache + fetch
# ============================================================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"


def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)
    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    # Record fetch status for row-level tagging (403-aware)
    try:
        record_http_status(getattr(r, "status_code", 0) or 0)
    except Exception:
        pass
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass
    return content


def fetch_text_cached(url: str, session: requests.Session) -> str:
    b = fetch_bytes_cached(url, session=session)
    try:
        return b.decode("utf-8", errors="replace")
    except Exception:
        return b.decode("latin-1", errors="replace")


# ============================================================
# URL helpers
# ============================================================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n


def rootish_domain(netloc: str) -> str:
    """
    Lightweight "registrable-ish" domain: last two labels if possible.
    This is imperfect for .edu subdomains but works well enough for drift detection.
    """
    n = root_domain(netloc)
    parts = [p for p in n.split(".") if p]
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return n


ALLOWED_ACADEMIC_SUBDOMAINS = {"www", "majors", "programs", "catalog", "bulletin", "courses"}


def is_subsite_like(url: str, base_netloc: str) -> bool:
    """
    Heuristic: "subsite-like" means the URL is likely an office/center/outreach subsite,
    or lives on a non-standard subdomain that frequently hosts non-inventory pages.

    Rules:
    - Different rootish domain => subsite-like
    - Subdomain first-label not in allowlist => subsite-like
      (unless it's literally the base host itself)
    - Path contains subsite hint words => subsite-like
    """
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Offsite-ish drift
    if rootish_domain(host) != rootish_domain(base):
        return True

    # Subdomain gating: majors.<root>, catalog.<root>, etc allowed; others penalized
    if host != base:
        # get first label (e.g., "catalog" in "catalog.bu.edu")
        parts = [p for p in host.split(".") if p]
        sub = parts[0] if parts else ""
        if sub and sub not in ALLOWED_ACADEMIC_SUBDOMAINS:
            return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in SUBSITE_HINTS):
        return True

    return False

def is_hard_subsite_block(url: str, base_netloc: str) -> bool:
    """True if URL is a known high-precision subsite trap (auto-loss)."""
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Drift to different registrable-ish domain => hard-block
    if rootish_domain(host) != rootish_domain(base):
        return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    return any(tok in full for tok in HARD_SUBSITE_BLOCK_TOKENS)

def is_too_specific_url(url: str) -> bool:
    """True if URL path matches TOO_SPECIFIC_PATTERNS."""
    path = (urlparse(url).path or "").lower()
    for pat in TOO_SPECIFIC_PATTERNS:
        if re.search(pat, path):
            return True
    return False

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)


def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)


def is_canonical_hub_url(url: str) -> bool:
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical


def looks_like_detail_url(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return any(p.search(path) for p in DETAIL_URL_PATTERNS)


# ============================================================
# Row tagging
# ============================================================
def _norm_host_from_web_address(web_address: str) -> str:
    u = ensure_scheme(web_address)
    if not u:
        return ""
    return (urlparse(u).netloc or "").lower()


def tag_row(web_address: str, best_url: str) -> str:
    """
    Returns one of:
      - 'Correct hub'
      - 'Correct-ish but too specific'
      - 'Wrong subsite'
      - 'Wrong non-academic'
    """
    base_host = _norm_host_from_web_address(web_address)
    u = ensure_scheme(best_url)
    if not u or best_url == "N/A":
        return "Wrong non-academic"

    host = (urlparse(u).netloc or "").lower()
    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in NON_ACADEMIC_HINTS):
        return "Wrong non-academic"

    # off-root-domain or obvious subsite hints
    if base_host and rootish_domain(host) != rootish_domain(base_host):
        return "Wrong subsite"
    if any(x in full for x in SUBSITE_HINTS):
        return "Wrong subsite"

    # hub-ish path
    if any(x in path for x in HUB_PATH_HINTS):
        for pat in TOO_SPECIFIC_PATTERNS:
            if re.search(pat, path):
                return "Correct-ish but too specific"
        return "Correct hub"

    # weaker academic fallback
    if "academ" in path or "catalog" in path or "bulletin" in path:
        return "Correct-ish but too specific"

    return "Wrong subsite"


# ============================================================
# HTML parse + main-content extraction
# ============================================================
@dataclass
class ParsedPage:
    url: str
    visible_text: str
    main_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]  # (anchor_text, abs_url)
    title_like: List[str]
    html_title: str
    h1: str
    corpus_any: str
    corpus_structured: str
    json_blob: str
    js_hint: int


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    h1_tag = soup.find("h1")
    h1 = normalize_unicode_text(h1_tag.get_text(" ", strip=True)) if h1_tag else ""

    # --- Embedded JSON support (v12 B): keep machine-readable program lists ---
    # Many modern hubs embed program lists as JSON (ld+json, app/json, Next/Nuxt payloads).
    # We extract likely-JSON script contents into `json_blob` and then remove scripts so visible text stays clean.
    json_chunks: List[str] = []
    for sc in soup.find_all("script"):
        typ = (sc.get("type") or "").strip().lower()
        sid = (sc.get("id") or "").strip().lower()

        is_json_type = ("json" in typ)  # includes application/ld+json, application/json, etc.
        is_framework_payload = sid in ("__next_data__", "__nuxt__")

        if not (is_json_type or is_framework_payload):
            continue

        raw = sc.string if sc.string is not None else sc.get_text(" ", strip=True)
        raw = raw.strip() if raw else ""
        if not raw:
            continue

        raw_norm = normalize_unicode_text(raw)
        if not raw_norm:
            continue

        # Keep only content that looks plausibly JSON-like to avoid pulling in inline JS.
        lstripped = raw_norm.lstrip()
        if not (
            lstripped.startswith("{")
            or lstripped.startswith("[")
            or '"@context"' in raw_norm
            or '"@type"' in raw_norm
            or '"@graph"' in raw_norm
            or '"itemListElement"' in raw_norm
        ):
            continue

        json_chunks.append(raw_norm)

    json_blob = normalize_unicode_text(" ".join(json_chunks))
    if len(json_blob) > MAX_JSON_BLOB_CHARS:
        json_blob = json_blob[:MAX_JSON_BLOB_CHARS]

    # Heuristic: if page looks like a JS-rendered app shell, mark it so scoring/candidate expansion can adapt
    raw_lower = (html or "").lower()
    js_hint = 1 if (
        "__next_data__" in raw_lower
        or "window.__nuxt__" in raw_lower
        or "data-reactroot" in raw_lower
        or "id=\"root\"" in raw_lower
        or "id=\"app\"" in raw_lower
        or "ng-version" in raw_lower
    ) else 0

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    main_node = soup.find("main") or soup.find(id=re.compile(r"content|main", re.I)) or soup.find("article")
    if main_node:
        main_text = normalize_unicode_text(main_node.get_text(separator=" ", strip=True))
    else:
        soup2 = BeautifulSoup(str(soup), "html.parser")
        for tag in soup2(["header", "footer", "nav"]):
            tag.decompose()
        main_text = normalize_unicode_text(soup2.get_text(separator=" ", strip=True))

    title_like: List[str] = []
    for tag in soup.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(
            x
            for x in [visible_text, json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts)]
            if x
        )
    )
    corpus_structured = normalize_unicode_text(
        " ".join(
            x
            for x in [json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)]
            if x
        )
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        main_text=main_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        h1=h1,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
        json_blob=json_blob,
        js_hint=js_hint,
    )


def is_soft_404(parsed: ParsedPage) -> bool:
    blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1] + parsed.title_like[:5]))
    if SOFT_404_TEXT.search(blob):
        return True
    if len(parsed.main_text) < 600 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    if len(parsed.main_text) < 250 and len(parsed.anchor_texts) < 5:
        return True
    return False


def page_is_thin(parsed: ParsedPage) -> bool:
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) and (len(parsed.main_text) < MIN_MAIN_TEXT_LEN)


# ============================================================
# Sitemap parsing
# ============================================================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()


def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls


def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()

        # hard skips
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue

        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break

    return out


# ============================================================
# Hubness: program-listing-only link counts
# ============================================================
def is_listing_link(url: str) -> bool:
    path = (urlparse(url).path or "")
    return any(p.search(path) for p in LISTING_LINK_PATTERNS)


def hubness_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    listing_links = 0
    slug_set: Set[str] = set()
    distinct_anchor: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        if not is_listing_link(u):
            continue

        listing_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        path = (urlparse(u).path or "").strip("/").lower()
        segs = [s for s in path.split("/") if s]
        if segs:
            slug_set.add(segs[-1])

    # weak secondary signal
    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if listing_links >= 12:
        sig += 1
    if listing_links >= 25:
        sig += 1
    if len(slug_set) >= 8:
        sig += 1
    if len(slug_set) >= 16:
        sig += 1
    if len(distinct_anchor) >= 18:
        sig += 1
    if li_like >= 40 and (listing_links >= 10):
        sig += 1

    counts = {
        "listing_links": listing_links,
        "slug_diversity": len(slug_set),
        "distinct_anchor": len(distinct_anchor),
        "li_like": li_like,
    }
    return sig, counts


# ============================================================
# Scoring
# ============================================================
@dataclass
class PageSignals:
    url: str
    tier: str
    soft404: int
    thin: int

    canonical_hub: int
    hub_sig: int
    hub_counts: Dict[str, int]

    struct_hits: List[str]
    control_unique_structured: int

    year_hit: int
    undergrad_title_boost: int

    grad_penalty: int
    profile_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int

    subsite_like: int  # configurable penalty via knob
    hard_subsite_block: int  # auto-loss for known traps
    too_specific: int  # explicit penalty separate from generic detail penalty
    js_hub_hint: int


def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if is_inventoryish_url(url):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"


def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    soft404 = 1 if is_soft_404(parsed) else 0
    thin = 1 if page_is_thin(parsed) else 0

    control_unique_structured = sum(1 for pat in CONTROL_PATS.values() if pat.search(parsed.corpus_structured))
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    title_blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1]))
    year_hit = 1 if (YEAR_REGEX.search(url) or YEAR_REGEX.search(title_blob)) else 0
    undergrad_title_boost = 1 if UNDERGRAD_TITLE_BOOST.search(title_blob) else 0

    u_low = url.lower()
    grad_penalty = 1 if (GRAD_URL_PENALTY.search(u_low) or re.search(r"\bgraduate\b", title_blob, re.I)) else 0
    profile_penalty = 1 if (PROFILE_URL_PENALTY.search(u_low) or re.search(r"\bfaculty directory\b", title_blob, re.I)) else 0
    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(u_low) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(u_low) else 0
    archive_penalty = 1 if ARCHIVE_URL_PENALTY.search(u_low) else 0
    awards_penalty = 1 if AWARDS_URL_PENALTY.search(u_low) else 0
    compliance_penalty = 1 if COMPLIANCE_URL_PENALTY.search(u_low) else 0

    hub_sig, hub_counts = hubness_signature(parsed, base_netloc=base_netloc)
    canonical_hub = 1 if is_canonical_hub_url(url) else 0

    subsite_like = 1 if is_subsite_like(url, base_netloc=base_netloc) else 0
    hard_subsite_block = 1 if is_hard_subsite_block(url, base_netloc=base_netloc) else 0
    too_specific = 1 if (is_too_specific_url(url) and not is_canonical_hub_url(url)) else 0

    js_hub_hint = 1 if (getattr(parsed, "js_hint", 0) and (len(getattr(parsed, "json_blob", "")) >= 200)) else 0
    return PageSignals(
        url=url,
        tier=page_tier(url),
        soft404=soft404,
        thin=thin,
        canonical_hub=canonical_hub,
        hub_sig=hub_sig,
        hub_counts=hub_counts,
        struct_hits=struct_hits,
        control_unique_structured=control_unique_structured,
        year_hit=year_hit,
        undergrad_title_boost=undergrad_title_boost,
        grad_penalty=grad_penalty,
        profile_penalty=profile_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
        subsite_like=subsite_like,
        hard_subsite_block=hard_subsite_block,
        too_specific=too_specific,
        js_hub_hint=js_hub_hint,
    )


def canonical_bonus_allowed(sig: PageSignals) -> bool:
    if sig.soft404:
        return False
    if sig.thin and sig.hub_sig == 0 and len(sig.struct_hits) < 2:
        return False
    return True


def score_inventory(sig: PageSignals, subsite_penalty: int) -> int:
    if sig.soft404:
        return -10_000
    
    # hard subsite blocks are automatic losses
    if getattr(sig, "hard_subsite_block", 0):
        return -10_000    

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 40

    # hubness + structured hits
    score += 85 * sig.hub_sig
    score += 10 * len(sig.struct_hits)
    score += 4 * sig.control_unique_structured

    # canonical hub bonus (gated)
    if sig.canonical_hub and canonical_bonus_allowed(sig):
        score += 160

    # undergrad title boost
    if sig.undergrad_title_boost:
        score += 35

    # year boost only for catalog/bulletin pages
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        score += 70

    # penalties (URL/title/H1)
    if sig.profile_penalty:
        score -= 260
    if sig.grad_penalty:
        score -= 170
    if sig.irrelevant_penalty:
        score -= 260
    if sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 220
    if sig.awards_penalty:
        score -= 200
    if sig.compliance_penalty:
        score -= 150

    # configurable subsite penalty
    if sig.subsite_like:
        score -= int(subsite_penalty)

    # explicit penalty for "too specific" department/program pages
    if getattr(sig, "too_specific", 0):
        score -= int(TOO_SPECIFIC_PENALTY)    

    # broad detail penalty so hubs win
    if looks_like_detail_url(sig.url):
        if sig.hub_sig <= 1:
            score -= 220
        elif sig.hub_sig == 2:
            score -= 90
        else:
            score -= 25

    # thin penalty (C2: JS hubs can be thin app shells; barely penalize)
    if sig.thin and sig.hub_sig == 0:
        if getattr(sig, "js_hub_hint", 0) and is_inventoryish_url(sig.url):
            score -= 20
        else:
            score -= 180

    return score
# ============================================================
# Candidate generation
# ============================================================

def looks_like_js_hub(parsed: ParsedPage, url: str) -> bool:
    """Detect JS-rendered program hubs early so we can expand via sitemap sooner."""
    if not page_is_thin(parsed):
        return False

    u = (url or "").lower()
    host = (urlparse(url).netloc or "").lower()
    inventoryish = is_inventoryish_url(url) or (host.startswith("majors.") or host.startswith("programs."))
    if not inventoryish and "find-your-program" not in u:
        return False

    # Typical app shells have few usable anchors and often embed data in JSON
    if len(parsed.links) > 60:
        return False
    if getattr(parsed, "js_hint", 0):
        return True
    if len(getattr(parsed, "json_blob", "")) >= 200:
        return True
    return False



def url_prior(url: str, base_netloc: str, subsite_penalty: int) -> int:
    u = url.lower()
    s = 0

    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 200
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 35
    if "bulletin" in u:
        s -= 20

    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if COMPLIANCE_URL_PENALTY.search(u):
        s -= 90
    if AWARDS_URL_PENALTY.search(u):
        s -= 160
    if GRAD_URL_PENALTY.search(u):
        s -= 180

    # hard subsite blocks should be avoided aggressively
    if is_hard_subsite_block(url, base_netloc=base_netloc):
        s -= 10_000
    if is_subsite_like(url, base_netloc=base_netloc):
        s -= int(subsite_penalty)

    if is_canonical_hub_url(url):
        s += 160

    return s


def prioritize_links(links: List[Tuple[str, str]], base_netloc: str, subsite_penalty: int) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000

        s = url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty)

        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 45
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 90
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140

        return -s

    return sorted(links, key=rank)


# ============================================================
# Candidate structure
# ============================================================
@dataclass
class CandidatePage:
    url: str
    score: int
    sig: PageSignals
    parsed: ParsedPage


def get_parsed_page(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)


def fetch_and_score(url: str, session: requests.Session, base_netloc: str, subsite_penalty: int) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page(url, session=session)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        sc = score_inventory(sig, subsite_penalty=subsite_penalty)
        return CandidatePage(url=url, score=sc, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)


# ============================================================
# Parallel batch fetch+score helper
# ============================================================
def fetch_and_score_many(
    urls: List[str],
    base_netloc: str,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    """Fetch+score a batch of URLs concurrently (one Session per thread)."""
    # de-dupe while preserving order
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            uniq.append(u)

    if workers <= 1:
        out: List[CandidatePage] = []
        s = requests.Session()
        for u in uniq:
            cand = fetch_and_score(u, session=s, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
            if cand is not None:
                out.append(cand)
        return out

    out: List[CandidatePage] = []

    def _task(u: str) -> Optional[CandidatePage]:
        # IMPORTANT: call get_thread_session() inside the worker thread
        return fetch_and_score(
            u,
            session=get_thread_session(),
            base_netloc=base_netloc,
            subsite_penalty=subsite_penalty,
        )

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_task, u) for u in uniq]
        for fut in as_completed(futs):
            cand = fut.result()
            if cand is not None:
                out.append(cand)
    return out


def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.score, len(topk), cand))
        return
    if cand.score > topk[0][0]:
        heapq.heapreplace(topk, (cand.score, topk[0][1], cand))


def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    """Return candidates sorted by score (desc), with an optional canonical-hub tie-break."""
    cands = [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]

    # C: Prefer canonical hubs more aggressively when tie-ish.
    if cands:
        best_score = cands[0].score

        canonical_best: Optional[CandidatePage] = None
        for c in cands:
            if c.sig.canonical_hub and (best_score - c.score) <= CANONICAL_TIE_THRESHOLD:
                if canonical_best is None or c.score > canonical_best.score:
                    canonical_best = c

        if canonical_best is not None and canonical_best is not cands[0]:
            cands = [canonical_best] + [x for x in cands if x is not canonical_best]

    return cands


# ============================================================
# Candidate generation
# ============================================================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))

    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================================================
# Program title extraction
# ============================================================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def looks_like_program_title(s: str, progtitle_strictness: int, context: str = "hub") -> bool:
    """
    Strictness guidance:
      1-2: permissive (still requires TARGET_ANY)
      3: current default behavior
      4-5: stricter (prefer explicit program-ish keywords; tighter filters)
    context: "hub" (default) or "follow" (detail)
    """
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False

    s_low = s_norm.lower()

    # D1: obvious junk rejects
    if URL_LIKE_TEXT.search(s_norm):
        return False
    if "toggle" in s_low:
        return False

    # always require the target token family (keeps extraction scoped)
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False

    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm):
        return False

    # D1: sentence-like / prose-ish rejects
    if ". " in s_norm and len(s_norm) >= 40:
        return False
    if ": " in s_norm and len(s_norm) >= 55:
        # allow short degree-style labels, reject long descriptive clauses
        if PROGRAM_KEYWORDS.search(s_norm) is None and not re.search(r"\b(B\.?A\.?|B\.?S\.?|BA|BS)\b", s_norm, re.I):
            return False
    if PROSE_VERBS.search(s_norm) and len(s_norm) >= 45:
        # follow/detail pages are stricter; hubs can still reject obvious prose
        if context == "follow" or progtitle_strictness >= 3:
            return False

    # D1: long multi-clause strings are usually prose or navigation
    if len(s_norm) > 90:
        comma_ct = s_norm.count(",")
        if comma_ct >= 2 and PROGRAM_KEYWORDS.search(s_norm) is None:
            return False

    if TITLE_NEGATIVE_CONTEXT.search(s_norm):
        if progtitle_strictness >= 3:
            return False

    # D2: follow/detail pages must show stronger "program" intent
    if context == "follow":
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    if PERSON_NAME_ONLY.match(s_norm):
        if progtitle_strictness >= 3 and not PROGRAM_KEYWORDS.search(s_norm):
            return False
        if progtitle_strictness >= 5:
            return False

    if progtitle_strictness >= 4:
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    return True
def is_program_detailish_url(url: str) -> bool:
    """Tight allowlist for one-hop follow: only follow likely program/major detail pages."""
    path = (urlparse(url).path or "")
    if any(p.search(path) for p in LISTING_LINK_PATTERNS):
        return True
    # also allow common majors-programs hubs/details
    if re.search(r"/majors-programs/[^/]+", path, re.I):
        return True
    return False


def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out


# ============================================================
# PDF fallback (best-effort)
# ============================================================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""


def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================================================
# Aggregate extraction from inventory + alts (+ limited 1-hop)
# ============================================================
def aggregate_outputs(
    pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
    progtitle_strictness: int,
    workers: int,
) -> Tuple[
    int, Set[str], Dict[str, int],
    Dict[str, List[str]],
    List[str],
    List[str],
]:
    base_netloc = urlparse(base_url).netloc

    struct_hits_union: Set[str] = set()
    control_hit_cols = {k: 0 for k in CONTROL_TERMS}
    any_control_found = 0

    tokens_agg: Dict[str, Set[str]] = {k: set() for k in TARGET_TOKEN_REGEX.keys()}
    titles: Set[str] = set()

    pdf_hits: List[str] = []

    # from chosen pages
    for cand in pages:
        any_control_found = max(any_control_found, 1 if cand.sig.control_unique_structured > 0 else 0)
        struct_hits_union.update(cand.sig.struct_hits)

        for ck, pat in CONTROL_PATS.items():
            if pat.search(cand.parsed.corpus_structured):
                control_hit_cols[ck] = 1

        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k, vals in tm.items():
            tokens_agg[k].update(vals)

        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
                titles.add(txt)

    # 1-hop follow
    follow_urls: List[str] = []
    for cand in pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            # D3: follow only if anchor text looks like a title, OR URL looks like a program detail page
            if looks_like_program_title(a_text or "", progtitle_strictness=progtitle_strictness, context="hub") or is_program_detailish_url(u):
                follow_urls.append(u)

    # sitemap assist if best looks thin
    if pages and page_is_thin(pages[0].parsed):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    # Fetch a limited number of follow URLs (optionally in parallel)
    follow_batch = uniq_follow[:ONE_HOP_MAX_FETCHES]

    def _safe_fetch_parse(u: str) -> Optional[ParsedPage]:
        try:
            # Use per-thread sessions when parallel
            s = get_thread_session() if workers and workers > 1 else session
            return get_parsed_page(u, session=s)
        except Exception:
            return None
        finally:
            # Keep some politeness throttling even when parallel
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    if workers and workers > 1 and len(follow_batch) > 1:
        max_w = min(int(workers), 8, len(follow_batch))
        with ThreadPoolExecutor(max_workers=max_w) as ex:
            futs = [ex.submit(_safe_fetch_parse, u) for u in follow_batch]
            for fut in as_completed(futs):
                parsed = fut.result()
                if not parsed:
                    continue

                tm = token_matches_from_text(parsed.corpus_any)
                for k, vals in tm.items():
                    tokens_agg[k].update(vals)

                for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                    if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                        titles.add(txt)
    else:
        for u in follow_batch:
            parsed = _safe_fetch_parse(u)
            if not parsed:
                continue

            tm = token_matches_from_text(parsed.corpus_any)
            for k, vals in tm.items():
                tokens_agg[k].update(vals)

            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                    titles.add(txt)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in pages:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k, vals in tm.items():
                            tokens_agg[k].update(vals)
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Dedup titles by normalized key
    best_display: Dict[str, str] = {}
    for t in titles:
        k = norm_title_key(t)
        if not k:
            continue
        cur = best_display.get(k)
        if cur is None or (len(t) > len(cur) and len(t) <= 120):
            best_display[k] = t

    titles_out = sorted(best_display.values())
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}

    return any_control_found, struct_hits_union, control_hit_cols, tokens_out, titles_out, pdf_hits


#
# ============================================================
# Discovery: find candidate pages
# ============================================================
def find_candidates_for_institution(
    base_url: str,
    session: requests.Session,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_cache
        if sitemap_cache is not None:
            return sitemap_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_cache = sitemap_candidate_urls(all_urls)
        return sitemap_cache

    def consider(u: str) -> Optional[CandidatePage]:
        if not u or u in tried:
            return None
        tried.add(u)
        cand = fetch_and_score(u, session=session, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
        if cand is None:
            return None
        push_topk(topk, cand, k=TOP_K_ALTS)
        return cand

    # Phase 0
    consider(base_url)

    # ------------------------------------------------------------
    # Phase 1 (parallelizable): explicit paths + year/catalog + home links
    # ------------------------------------------------------------
    batch_urls: List[str] = []

    for p in INVENTORY_PATHS[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    batch_urls.extend(build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES])

    for p in CATALOG_PATHS[:PHASE1_MAX_CATALOG_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    # homepage links require parsing homepage once (sequential), but the fetched URLs can be scored in batch
    try:
        home = get_parsed_page(base_url, session=session)
        
        # C1: If the homepage/programs page is a JS hub (thin app shell), expand candidates via sitemap immediately
        if looks_like_js_hub(home, base_url):
            try:
                batch_urls.extend(load_sitemap_candidates()[:80])
            except Exception:
                pass
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc, subsite_penalty=subsite_penalty)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            batch_urls.append(u)
    except Exception:
        pass

    # score the batch in parallel (if workers>1)
    batch_cands = fetch_and_score_many(
        batch_urls,
        base_netloc=base_netloc,
        subsite_penalty=subsite_penalty,
        workers=workers,
    )
    for cand in batch_cands:
        tried.add(cand.url)
        push_topk(topk, cand, k=TOP_K_ALTS)

    # C2: If ANY early candidate looks like a JS-rendered hub, expand via sitemap NOW
    # (even if the thin hub didn't win the scoring yet).
    try:
        if sitemap_cache is None:
            js_hub_seen = any(looks_like_js_hub(c.parsed, c.url) for c in batch_cands)
            if js_hub_seen:
                sm_urls = load_sitemap_candidates()[:120]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
    except Exception:
        pass

    # Optional: if best so far looks weak, pull some sitemap candidates (also batchable)
    if topk:
        best_so_far = max(topk, key=lambda x: x[0])[2]
        if (best_so_far.sig.thin or best_so_far.sig.hub_sig == 0) and is_inventoryish_url(best_so_far.url):
            try:
                sm_urls = load_sitemap_candidates()[:50]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
            except Exception:
                pass

    # Best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty), depth, u))

    pq_push(base_url, 0)
    fetched = 0
    while pq and fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u)
        fetched += 1
        if depth >= PHASE2_MAX_DEPTH:
            continue
        try:
            parsed = cand.parsed if cand is not None else get_parsed_page(u, session=session)
            links = [(t, lu) for (t, lu) in parsed.links if same_domain(lu, base_netloc)]
            links = prioritize_links(links, base_netloc, subsite_penalty=subsite_penalty)
            for _, lu in links[:70]:
                pq_push(lu, depth + 1)
        except Exception:
            pass

    # Subdomain roots
    for root in build_subdomain_roots(base_url):
        c_root = consider(root)
        if not c_root:
            continue
        root_is_good = (c_root.sig.hub_sig >= 1) or (len(c_root.sig.struct_hits) >= 2) or (not c_root.sig.thin)
        if root_is_good and not c_root.sig.soft404:
            continue
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p))

    return topk_sorted(topk)


# ============================================================
# Reason string
# ============================================================
def inventory_reason(c: CandidatePage) -> str:
    s = c.sig
    flags = []
    if s.grad_penalty:
        flags.append("grad")
    if s.profile_penalty:
        flags.append("profile")
    if s.irrelevant_penalty:
        flags.append("irrelevant")
    if s.admissions_penalty:
        flags.append("admissions")
    if s.archive_penalty:
        flags.append("archive")
    if s.awards_penalty:
        flags.append("awards")
    if s.compliance_penalty:
        flags.append("compliance")
    if s.year_hit:
        flags.append("yearHit")
    if s.thin:
        flags.append("thin")
    if s.soft404:
        flags.append("soft404")
    if s.subsite_like:
        flags.append("subsiteLike")
    if getattr(s, "hard_subsite_block", 0):
        flags.append("hardSubsiteBlock")
    if getattr(s, "too_specific", 0):
        flags.append("tooSpecific")

    return ";".join([
        f"score={c.score}",
        f"tier={s.tier}",
        f"hubSig={s.hub_sig}",
        f"listingLinks={s.hub_counts.get('listing_links', 0)}",
        f"slugDiv={s.hub_counts.get('slug_diversity', 0)}",
        f"structHits={len(s.struct_hits)}",
        f"ctrlStructured={s.control_unique_structured}",
        f"canonicalHub={s.canonical_hub}",
        ("flags=" + ",".join(flags)) if flags else "flags=",
    ])


# ============================================================
# CLI
# ============================================================
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--input", type=str, default=str(DEFAULT_INPUT_PATH))
    ap.add_argument("--output", type=str, default=str(DEFAULT_OUTPUT_PATH))

    # If omitted, default to TEST_HEAD_N (notebook-friendly). Use --head 0 for full run.
    ap.add_argument("--head", type=int, default=None, help="If >0, limit rows for testing; if omitted defaults to TEST_HEAD_N")

    ap.add_argument("--out-suffix", type=str, default="", help="Append to output filename stem")
    ap.add_argument("--subsite-penalty", type=int, default=80)
    ap.add_argument("--progtitle-strictness", type=int, default=3)

    # Parallelism (keep small to be polite)
    ap.add_argument(
        "--workers",
        type=int,
        default=DEFAULT_WORKERS,
        help="Parallel fetch workers for candidate scoring (1 = off). Try 4.",
    )

    ap.add_argument(
        "--debug-jsonlen",
        action="store_true",
        help="Debug: print json_blob length (chars) for the best candidate per school",
    )

    # IMPORTANT: ignore Jupyter/ipykernel injected args like --f=...
    args, _unknown = ap.parse_known_args(argv)

    if args.head is None:
        args.head = TEST_HEAD_N

    # sanity clamp
    if args.workers is None or args.workers < 1:
        args.workers = 1
    if args.workers > MAX_WORKERS:
        args.workers = MAX_WORKERS

    return args


def apply_out_suffix(out_path: Path, suffix: str) -> Path:
    if not suffix:
        return out_path
    return out_path.with_name(out_path.stem + suffix + out_path.suffix)


# ============================================================
# Main
# ============================================================
def main() -> None:
    args = parse_args()
    input_path = Path(args.input)
    output_path = apply_out_suffix(Path(args.output), args.out_suffix)

    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {input_path}: {sorted(missing)}")

    if args.head and args.head > 0:
        df = df.head(args.head).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")
        # Reset per-institution status tally for fetch-status tagging (403-aware)
        reset_status_tally()

        best_inventory_url = "N/A"
        best_inventory_reason = ""
        alt_candidate_urls = ""

        any_control_found = 0
        struct_hits_union: Set[str] = set()
        any_struct_found = 0

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        token_matches: Dict[str, List[str]] = {k: [] for k in OUTPUT_TOKEN_BUCKETS}

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0
        pdf_hits: List[str] = []

        try:
            if base_url:
                candidates = find_candidates_for_institution(
                    base_url,
                    session=session,
                    subsite_penalty=args.subsite_penalty,
                    workers=args.workers,
                )

                if candidates:
                    best = candidates[0]
                    best_inventory_url = best.url
                    best_inventory_reason = inventory_reason(best)
                    alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K_ALTS])
                    if getattr(args, "debug_jsonlen", False):
                        jb_len = len(getattr(best.parsed, "json_blob", "") or "")
                        print(f"  [debug-jsonlen] best_url={best.url} | json_blob_chars={jb_len}")

                    any_control_found, struct_hits_union, control_hits, tokens, titles, pdf_hits = aggregate_outputs(
                        pages=candidates[:TOP_K_ALTS],
                        base_url=base_url,
                        session=session,
                        progtitle_strictness=args.progtitle_strictness,
                        workers=args.workers,
                    )

                    any_struct_found = 1 if struct_hits_union else 0

                    for k, v in control_hits.items():
                        control_hit_cols[f"found_{k}"] = int(v)

                    for k in OUTPUT_TOKEN_BUCKETS:
                        token_matches[k] = tokens.get(k, [])

                    program_titles = titles
                    program_title_count = len(program_titles)
                    any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0
        except Exception:
            # keep row, but leave defaults
            pass

        rec: Dict[str, str] = dict(row)

        rec["best_guess_inventory_url"] = best_inventory_url
        rec["best_guess_inventory_reason"] = best_inventory_reason
        rec["url_tag"] = fetch_status_tag(best_inventory_url)
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        for k in OUTPUT_TOKEN_BUCKETS:
            vals = token_matches.get(k, [])
            rec[f"{k}_matches"] = "|".join(vals) if vals else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        rec.update({k: str(v) for k, v in control_hit_cols.items()})
        out_rows.append(rec)

        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "url_tag",
        "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        *[f"{k}_matches" for k in OUTPUT_TOKEN_BUCKETS],
        "any_potential_AfrAmr_found",
        "pdf_hits",
    ]
    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]
    remaining_cols = [c for c in out_df.columns if c not in set(front_cols + control_cols)]
    out_df = out_df[front_cols + control_cols + remaining_cols]

    out_df.to_csv(output_path, index=False)
    print(f"\nWrote: {output_path} | rows,cols = {out_df.shape}")
    print(out_df.head())


if __name__ == "__main__":
    main()

[1/40] unitid=188429 | Adelphi University
[2/40] unitid=138600 | Agnes Scott College
[3/40] unitid=168546 | Albion College
[4/40] unitid=164465 | Amherst College
[5/40] unitid=143084 | Augustana College
[6/40] unitid=189097 | Barnard College
[7/40] unitid=132471 | Barry University
[8/40] unitid=160977 | Bates College
[9/40] unitid=156295 | Berea College
[10/40] unitid=142115 | Boise State University
[11/40] unitid=164924 | Boston College
[12/40] unitid=164988 | Boston University
[13/40] unitid=161004 | Bowdoin College
[14/40] unitid=201441 | Bowling Green State University-Main Campus
[15/40] unitid=143358 | Bradley University
[16/40] unitid=165015 | Brandeis University
[17/40] unitid=217156 | Brown University
[18/40] unitid=211273 | Bryn Mawr College
[19/40] unitid=211291 | Bucknell University
[20/40] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/40] unitid=110529 | California State Polytechnic University-Pomona
[22/40] unitid=110538 | California State Uni

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/bs4/__init__.py:473: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self._feed()
/var/folders/0_/vt5npj3x5wq9q_4bzlj28tqw0000gn/T/ipykernel_7104/361789366.py:821: XMLParsedAsHTMLWarning: It looks like you're using

[32/40] unitid=110495 | California State University-Stanislaus
[33/40] unitid=173258 | Carleton College
[34/40] unitid=201645 | Case Western Reserve University
[35/40] unitid=128771 | Central Connecticut State University
[36/40] unitid=234827 | Central Washington University
[37/40] unitid=156408 | Centre College
[38/40] unitid=144005 | Chicago State University
[39/40] unitid=217864 | Citadel Military College of South Carolina
[40/40] unitid=138947 | Clark Atlanta University

Wrote: Carnegie_Carla_2013_subset_majors_scan_v13.csv | rows,cols = (40, 52)
   unitid                 name          Web_address  \
0  188429   Adelphi University     www.adelphi.edu/   
1  138600  Agnes Scott College  www.agnesscott.edu/   
2  168546       Albion College      www.albion.edu/   
3  164465      Amherst College     www.amherst.edu/   
4  143084    Augustana College   www.augustana.edu/   

                            best_guess_inventory_url  \
0                  https://www.adelphi.edu/programs/   


need to adjust program search and postprocessing to clean found programs. 
integrate colelge vine search
comparison with 2013 set

In [4]:
#!/usr/bin/env python3
"""
v14: program-inventory discovery + scoring + tagging (clean)

v14 additions / changes (relative to v13):
- Robust fetch-status tagging via a per-institution status tally:
    * Deterministic row-level `url_tag` derived from all attempted institution-site fetches
    * Explicit buckets for 403, 429, any 5xx, request exceptions (timeouts/connection errors),
      and a catch-all for other 4xx (excluding 403/429)
    * Status tally is reset between the institution crawl and CollegeVine to keep `url_tag`
      institution-only

- Program-title extraction accuracy improvements (cleaning/postprocessing):
    * Stronger course detection (e.g., "AAAS 115a", "AAAS/WGS 125a")
    * Negative-context / announcement filtering (news/events/job postings/platform links)
    * Salvage of common boilerplate prefixes ("Welcome to", "About the program in", etc.)
    * Breadcrumb splitting, degree-suffix normalization, scoring, and deduping

- Expanded target token family for matching and outputs:
    * Africa/African/Africana, Pan-African, African-American, Afro-American, Africology,
      plus Ethnic/Ethnicity, Black, Race/Racial
    * Fixed output token buckets: afri / ethnic / black / race

- Hub confidence proxy from control majors:
    * `total_controls_found` = sum(found_<control>) across the CONTROL_TERMS list
    * `controls_sufficiency` = "warning_<X>_MajorsFound" if total_controls_found < X else "sufficient majors"
    * X is controlled by `MIN_CONTROL_HITS_FOR_CONFIDENT_HUB`

- Canonical-hub preference + JS-hub handling:
    * Canonical hubs can win tie-ish scores (CANONICAL_TIE_THRESHOLD)
    * JS-rendered thin hubs trigger earlier sitemap expansion (looks_like_js_hub)

- CollegeVine (3rd-party truth source) integration:
    * Builds a small slug candidate set for each school name and probes
      `https://www.collegevine.com/schools/hub/all/d/<slug>/majors`
    * Outputs `college_vine_url` (traceability), `college_vine_site` (1/0),
      `college_vine_control_hits`, and CollegeVine-derived program titles
      (extracted with the same heuristics + cleaning as institution pages)
    * Optional control threshold flag: `college_vine_controls_ge_<X>`

- Concordance checks (exact normalized matches; may be expanded later):
    * institution programs vs CollegeVine programs
    * institution programs vs `2013_program_name`
    * CollegeVine programs vs `2013_program_name`

Key features (overall):
- Candidate discovery: path templates + homepage link prioritization + best-first crawl + sitemap assist
- Page scoring: hubness signatures + structured term hits + control-term proxy + penalties
- Row tagging for best_guess_inventory_url: Correct hub / Correct-ish but too specific / Wrong subsite / Wrong non-academic
- CLI knobs for sweep runs from ipynb:
    --subsite-penalty (int): penalty applied to subsite-like pages
    --progtitle-strictness (int): controls how strict program title extraction is
    --head (int): limit rows for quick testing
    --out-suffix (str): appended to output filename stem
    --input / --output: file paths

Notebook-friendly:
- TEST_HEAD_N provides default subset size when --head is not provided. Use --head 0 for full run.
- DEFAULT_WORKERS provides a top-level default for --workers.

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v14.csv (default)
"""

from __future__ import annotations

import argparse
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup

import difflib

# ============================================================
# Thread-local requests sessions (for safe parallel fetching)
# ============================================================
_thread_local = threading.local()


def get_thread_session() -> requests.Session:
    """Return a per-thread requests.Session (requests.Session is not thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        _thread_local.session = s
    return s

# ============================================================
# Per-institution fetch-status tally (403-aware)
# ============================================================
# We process institutions sequentially, but fetch pages in parallel per institution.
# This shared tally lets us tag rows when any attempted fetch hit a 403 (or similar).
_STATUS_LOCK = threading.Lock()
_STATUS_COUNTS: Dict[int, int] = {}


def reset_status_tally() -> None:
    with _STATUS_LOCK:
        _STATUS_COUNTS.clear()


def record_http_status(code: int) -> None:
    if not code:
        return
    with _STATUS_LOCK:
        _STATUS_COUNTS[code] = _STATUS_COUNTS.get(code, 0) + 1


# New: Record request exceptions for row-level tagging
def record_request_exception() -> None:
    """Record that a request raised an exception (timeout, connection error, etc.)."""
    with _STATUS_LOCK:
        _STATUS_COUNTS[-1] = _STATUS_COUNTS.get(-1, 0) + 1


def fetch_status_tag(best_url: str) -> str:
    """Row tag describing fetch status / blocks encountered during candidate discovery."""
    with _STATUS_LOCK:
        counts = dict(_STATUS_COUNTS)

    # Prioritize explicit blocks
    if counts.get(403, 0) > 0:
        return "403_forbidden_seen"
    if counts.get(429, 0) > 0:
        return "429_rate_limited_seen"

    # Any server errors encountered
    if any(isinstance(c, int) and 500 <= c <= 599 for c in counts.keys()):
        return "5xx_seen"

    # Request exceptions (timeouts, connection errors, etc.)
    if counts.get(-1, 0) > 0:
        return "exceptions_seen"

    # Any other 4xx (excluding 403/429)
    if any(isinstance(c, int) and 400 <= c <= 499 and c not in (403, 429) for c in counts.keys()):
        return "4xx_seen"

    # If nothing worked
    if not best_url or best_url == "N/A":
        return "no_candidate"

    return "ok"

# ============================================================
# Notebook-friendly default test mode
# ============================================================
# If you run from an ipynb as:  !python v14_program_inventory.py
# it will default to processing TEST_HEAD_N rows unless you pass --head.
# Use --head 0 for full run.
TEST_HEAD_N = 25

# ============================================================
# Top-level parallelism defaults
# ============================================================
# You can override via CLI: --workers
# Keep MAX_WORKERS small to stay polite to university sites.

DEFAULT_WORKERS = 6
MAX_WORKERS = 8

# ============================================================
# Control-term coverage threshold (majors proxy)
# ============================================================
# We use the number of CONTROL_TERMS matched as a rough proxy for whether we landed on a
# real programs/majors hub page. If total controls found is below this threshold, we flag.
# (Do NOT reintroduce found_bio; this counts only the current CONTROL_TERMS list.)
MIN_CONTROL_HITS_FOR_CONFIDENT_HUB = 10

# ============================================================
# CollegeVine (3rd-party truth source): majors page presence + control hits
# ============================================================
# CollegeVine majors pages follow this pattern:
#   https://www.collegevine.com/schools/hub/all/d/<school-slug>/majors
# We use this as an alternate public listing to sanity-check that a school exists
# and that the majors listing surface contains our CONTROL_TERMS (majors proxy).

COLLEGEVINE_BASE = "https://www.collegevine.com/schools/hub/all/d"
COLLEGEVINE_TIMEOUT_SEC = 20


def slugify_collegevine_school_name(name: str) -> str:
    """Convert a school name to CollegeVine's expected hyphenated slug."""
    s = normalize_unicode_text(name or "").lower()

    # Replace '&' with 'and' to match common slug conventions
    s = s.replace("&", " and ")

    # Drop apostrophes/quotes; keep alnum + spaces + hyphens
    s = re.sub(r"[\"']", "", s)

    # Collapse non-alphanumeric into spaces, then hyphenate
    s = re.sub(r"[^a-z0-9\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ", "-")

    # Collapse duplicate hyphens
    s = re.sub(r"-+", "-", s).strip("-")
    return s

def collegevine_slug_candidates(name: str) -> List[str]:
    """Generate a small set of plausible CollegeVine slugs for a school name.

    We start with the direct slugified name, then try a few high-yield variants that
    commonly differ across third-party directory conventions.
    """
    base = normalize_unicode_text(name or "")
    if not base:
        return []

    cands: List[str] = []

    def _add(nm: str) -> None:
        s = slugify_collegevine_school_name(nm)
        if s and s not in cands:
            cands.append(s)

    # 1) direct
    _add(base)

    # 2) drop common campus qualifiers / parentheticals
    nm2 = re.sub(r"\s*\([^)]*\)\s*", " ", base)
    nm2 = re.sub(r"\b(main\s+campus|downtown\s+campus|online|global\s+campus)\b", " ", nm2, flags=re.I)
    nm2 = normalize_unicode_text(nm2)
    _add(nm2)

    # 3) remove common suffix tokens (but keep order)
    nm3 = re.sub(
        r"\b(university|college|institute|school|state\s+university|community\s+college|polytechnic|campus)\b",
        " ",
        base,
        flags=re.I,
    )
    nm3 = normalize_unicode_text(nm3)
    _add(nm3)

    # 4) remove leading 'the'
    nm4 = re.sub(r"^the\s+", "", base, flags=re.I)
    nm4 = normalize_unicode_text(nm4)
    _add(nm4)

    # Return only the first few to avoid excessive probing
    return cands[:4]

def count_control_hits_in_text(text: str) -> int:
    """Count how many distinct CONTROL_TERMS match in the provided text."""
    blob = normalize_unicode_text(text or "")
    return int(sum(1 for pat in CONTROL_PATS.values() if pat.search(blob)))


def extract_strings_from_json_blob(json_blob: str, *, max_strings: int = 6000) -> List[str]:
    """Extract candidate text strings from embedded JSON payloads.

    CollegeVine (and similar JS apps) often embed major lists inside Next/Nuxt payloads.
    We parse JSON when possible; otherwise we fall back to a lightweight quoted-string scan.
    """
    blob = (json_blob or "").strip()
    if not blob:
        return []

    out: List[str] = []

    def _push(s: str) -> None:
        if not s:
            return
        if len(out) >= max_strings:
            return
        s2 = normalize_unicode_text(s)
        if s2 and s2 not in out:
            out.append(s2)

    # 1) Try structured JSON parsing first
    try:
        obj = json.loads(blob)

        def _walk(o) -> None:
            if len(out) >= max_strings:
                return
            if isinstance(o, str):
                _push(o)
                return
            if isinstance(o, dict):
                for v in o.values():
                    _walk(v)
                return
            if isinstance(o, list):
                for v in o:
                    _walk(v)
                return

        _walk(obj)
        return out
    except Exception:
        pass

    # 2) Fallback: extract quoted strings (best-effort)
    # Keep it conservative to avoid huge noisy dumps.
    try:
        for m in re.finditer(r'"([^"\\]{3,140})"', blob):
            _push(m.group(1))
            if len(out) >= max_strings:
                break
    except Exception:
        return []

    return out


def fetch_collegevine_majors_page(
    school_name: str,
    session: requests.Session,
    progtitle_strictness: int,
) -> Tuple[str, int, int, int, List[str]]:
    """
    Return: (collegevine_url, college_vine_site, college_vine_control_hits,
             college_vine_program_title_count, college_vine_program_titles)

    - college_vine_site is 1 if the majors page returns a 200 and is not a soft-404.
    - college_vine_control_hits counts distinct CONTROL_TERMS found on that page.
    - college_vine_program_titles are extracted using the same TARGET_ANY + title heuristics
      and then cleaned with the same postprocessing pipeline as institution pages.

    NOTE: This fetch is intentionally isolated from the per-institution _STATUS_COUNTS
    used for tagging university-site 403/429s.
    """
    slugs = collegevine_slug_candidates(school_name)
    if not slugs:
        return "", 0, 0, 0, []

    last_url = f"{COLLEGEVINE_BASE}/{slugs[0]}/majors"

    html = ""
    parsed: Optional[ParsedPage] = None
    used_url = ""

    for slug in slugs:
        cv_url = f"{COLLEGEVINE_BASE}/{slug}/majors"
        last_url = cv_url

        try:
            r = session.get(cv_url, headers=HEADERS, timeout=COLLEGEVINE_TIMEOUT_SEC, allow_redirects=True)
        except Exception:
            continue

        if getattr(r, "status_code", 0) != 200:
            continue

        try:
            html = r.text or ""
        except Exception:
            try:
                html = (r.content or b"").decode("utf-8", errors="replace")
            except Exception:
                html = ""

        if not html:
            continue

        parsed = parse_html_to_parsedpage(html, page_url=cv_url)
        if is_soft_404(parsed):
            parsed = None
            continue

        used_url = cv_url
        break

    if not used_url or parsed is None:
        return last_url, 0, 0, 0, []

    cv_url = used_url

    # Use visible text (after script/style removal) as the control-term surface.
    ctrl_hits = count_control_hits_in_text(parsed.corpus_any)  # includes json_blob + anchors + visible
    

    # Extract program-title candidates using the same heuristics as the institution crawl,
    # then run the same cleaning/scoring/deduping pipeline.
    raw_titles: Set[str] = set()

    # 1) Visible / anchor-derived candidates
    for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
        if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
            raw_titles.add(txt)

    # 2) Title/H1 can carry the program framing on some CV pages
    for txt in [getattr(parsed, "h1", ""), getattr(parsed, "html_title", "")]:
        if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
            raw_titles.add(txt)

    # 3) Embedded JSON payloads (Next/Nuxt/ld+json) often contain the major list
    for txt in extract_strings_from_json_blob(getattr(parsed, "json_blob", "")):
        # Cheap prefilter first to avoid expensive title heuristics on irrelevant strings
        if TARGET_ANY_REGEX.search(txt) is None:
            continue
        if looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
            raw_titles.add(txt)

    cleaned_titles = clean_program_titles(list(raw_titles), progtitle_strictness=progtitle_strictness)
    title_count = int(len(cleaned_titles))
    return cv_url, 1, ctrl_hits, title_count, cleaned_titles

# ============================================================
# Defaults (override via CLI)
# ============================================================
DEFAULT_INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
DEFAULT_OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v14.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.25
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.20

MAX_BYTES_PER_PAGE = 2_000_000
MAX_JSON_BLOB_CHARS = 200_000

CACHE_DIR = Path(".cache_v14_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Per-institution budgets
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 22
PHASE2_BESTFIRST_MAX_PAGES = 55
PHASE2_MAX_DEPTH = 3

TOP_K_ALTS = 6
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 18

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

MIN_VISIBLE_TEXT_LEN = 900
MIN_MAIN_TEXT_LEN = 650

ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/14.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================================================
# URL templates
# ============================================================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================================================
# Patterns / scoring signals
# ============================================================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_URL_PENALTY = re.compile(
    r"\b(graduate|grad-studies|gradstudies|master|masters|doctoral|phd|mba|mfa|ms|ma)\b",
    flags=re.IGNORECASE,
)
PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)
AWARDS_URL_PENALTY = re.compile(r"\b(awards?|honors?)\b", flags=re.IGNORECASE)
COMPLIANCE_URL_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_TITLE_BOOST = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    # "bio": r"\bbio[a-z]*\b", # not a good control due to "biography" and other simple matches
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

#
# Expanded target token family (v12): includes Africa/African/Africana, Pan-African,
# African American, Afro-American, Africology, Race, plus existing afri/ethnic/black.
#
# NOTE: Keep "TARGET_ANY_REGEX" broad (union) so extraction triggers properly,
# but keep per-bucket regexes interpretable for output columns.
TARGET_TOKEN_REGEX = {
    # "afri" bucket (kept for output column stability) WITHOUT broad afri* wildcard.
    "afri": re.compile(
        r"\b(africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology)\b",
        flags=re.IGNORECASE,
    ),
    # Include both "ethnic*" and "ethnicity" so the bucket reflects real page phrasing.
    "ethnic": re.compile(r"\b(ethnic[a-z\-]*|ethnicity)\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
    # New bucket: race / racial
    "race": re.compile(r"\b(race|racial)\b", flags=re.IGNORECASE),
}

# Trigger regex used to decide whether text is in-scope for extraction.
# Keep this broad (union of all target families) so we don't miss relevant programs.
TARGET_ANY_REGEX = re.compile(
    r"\b(africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology|"
    r"ethnic[a-z\-]*|ethnicity|black[a-z\-]*|race|racial)\b",
    flags=re.IGNORECASE,
)

# Output column stability: keep these buckets as fixed output columns.
# If TARGET_TOKEN_REGEX changes, these columns will still be present (empty if missing).
OUTPUT_TOKEN_BUCKETS = ["afri", "ethnic", "black", "race"]

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}(?:/[A-Z]{2,6})*\s*[-]?\s*\d{1,3}[A-Z]?\b")
COURSE_WORDS = re.compile(r"\b(units?|credits?|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")
PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

# --- Title junk / prose detection (v12 D) ---
URL_LIKE_TEXT = re.compile(r"(https?://|www\.|\b[a-z0-9\-]+\.(edu|com|org|net)(/|\b))", flags=re.IGNORECASE)
PROSE_VERBS = re.compile(
    r"\b(define|apply|enrich|explore|learn|prepare|prepares|focus(es)?|offers|provides|designed|helps|aims)\b",
    flags=re.IGNORECASE,
)

# --- Program-title cleaning / postprocessing (v14) ---
TITLE_PLATFORM_KEYWORDS = re.compile(
    r"\b(facebook|instagram|twitter|linkedin|youtube|blackboard|canvas|email|login)\b",
    flags=re.IGNORECASE,
)
TITLE_SLUG_LIKE = re.compile(r"^[a-z0-9\-]{10,}$")
TITLE_LEADING_SLASH = re.compile(r"^/\s*")

TITLE_STRIP_PREFIXES = [
    # salvage patterns (capture group is the likely program name)
    re.compile(r"^\s*welcome\s+to\s+(.+)$", re.I),
    re.compile(r"^\s*about\s+(?:the\s+)?(?:program|department)\s+(?:in|of)\s+(.+)$", re.I),
    re.compile(r"^\s*view\s+(.+)$", re.I),
    re.compile(r"^\s*connect\s+with\s+(.+)$", re.I),
    re.compile(r"^\s*prospective\s+students\s*:\s*(.+)$", re.I),
    re.compile(r"^\s*learn\s+more\s+about\s+(.+)$", re.I),
    re.compile(r"^\s*explore\s+(.+)$", re.I),
]

TITLE_HARD_DROPS = re.compile(
    r"\b(lecturer\s+pool|job|position|apply\b|hiring\b|employment\b)\b",
    flags=re.IGNORECASE,
)

TITLE_CONTENTY_WORDS = re.compile(
    r"\b(discusses|announces|becomes|celebrate|event|workshop|lecture|seminar|news|story|profile|alumni|upcoming)\b",
    flags=re.IGNORECASE,
)

TITLE_YEARISH = re.compile(r"\b(20\d{2}|\d{2}\s*[-–—/]\s*\d{2})\b")


def _has_program_intent(s: str) -> bool:
    s_low = (s or "").lower()
    # Strong program intent: explicit keywords OR common framing
    if PROGRAM_KEYWORDS.search(s) is not None:
        return True
    if s_low.endswith(" studies") or s_low.endswith(" studies program"):
        return True
    if s_low.startswith("department of ") or s_low.startswith("center for ") or s_low.startswith("centre for "):
        return True
    return False


def _split_title_candidates(s: str) -> List[str]:
    """Split breadcrumb-ish or concatenated titles into candidate chunks."""
    s = normalize_unicode_text(s)
    if not s:
        return []

    parts: List[str] = [s]
    for sep in ["|", "•", "·", "–", "—"]:
        if sep in s:
            tmp: List[str] = []
            for p in parts:
                tmp.extend([x.strip() for x in p.split(sep) if x.strip()])
            parts = tmp

    # Light split on " / " only (avoid breaking AAAS/WGS course codes)
    tmp2: List[str] = []
    for p in parts:
        if " / " in p:
            tmp2.extend([x.strip() for x in p.split(" / ") if x.strip()])
        else:
            tmp2.append(p)
    parts = tmp2

    return [normalize_unicode_text(p) for p in parts if p]


def _salvage_prefix(s: str) -> List[str]:
    s = normalize_unicode_text(s)
    if not s:
        return []
    out = [s]
    for rx in TITLE_STRIP_PREFIXES:
        m = rx.match(s)
        if m:
            g = normalize_unicode_text(m.group(1))
            if g and g != s:
                out.append(g)
    return out


def _normalize_degree_suffix(s: str) -> str:
    # Normalize common degree-label patterns that appear after colons or at end.
    s = normalize_unicode_text(s)
    s = re.sub(r"\s*:\s*(B\.?A\.?|B\.?S\.?|BA|BS)\b\s*", " ", s, flags=re.I)
    s = re.sub(r"\s+\b(B\.?A\.?|B\.?S\.?|BA|BS)\b\s*$", "", s, flags=re.I)
    return normalize_unicode_text(s)


def _score_title_variant(s: str) -> int:
    """Higher is better."""
    s_norm = normalize_unicode_text(s)
    s_low = s_norm.lower()
    score = 0

    if _has_program_intent(s_norm):
        score += 40
    if re.search(r"\b(major|minor|concentration|certificate)\b", s_norm, re.I):
        score += 15
    if s_low.startswith("department of "):
        score += 8
    if s_low.endswith(" studies"):
        score += 10

    # Penalize content-y / announcement-y strings
    if TITLE_HARD_DROPS.search(s_norm):
        score -= 200
    if TITLE_YEARISH.search(s_norm):
        score -= 80
    if TITLE_CONTENTY_WORDS.search(s_norm) and not _has_program_intent(s_norm):
        score -= 120

    # Mild penalty for boilerplate verbs/prefixes that survived
    if re.match(r"^(about|welcome|view|connect|prospective\s+students)\b", s_low):
        score -= 40

    # Prefer reasonably short, name-like strings
    if 8 <= len(s_norm) <= 80:
        score += 5
    if len(s_norm) > 110:
        score -= 25

    return score


def clean_program_titles(raw_titles: List[str], progtitle_strictness: int) -> List[str]:
    """Clean, salvage, split, score, and dedupe extracted program-title candidates."""
    expanded: List[str] = []
    for t in raw_titles:
        for s in _salvage_prefix(t):
            expanded.extend(_split_title_candidates(s))

    keep: List[str] = []
    for t in expanded:
        t = _normalize_degree_suffix(t)
        if not t:
            continue

        # Hard drops
        if TITLE_PLATFORM_KEYWORDS.search(t):
            continue
        if TITLE_LEADING_SLASH.search(t):
            continue
        if TITLE_SLUG_LIKE.match(t):
            continue
        if URL_LIKE_TEXT.search(t):
            continue
        if TITLE_HARD_DROPS.search(t):
            continue

        # Course detection (must drop)
        if COURSE_CODE.match(t):
            continue
        if COURSE_WORDS.search(t):
            continue

        # Always require target token family (keeps extraction scoped)
        if TARGET_ANY_REGEX.search(t) is None:
            continue

        # Contenty strings are only acceptable if they ALSO show program intent
        if (TITLE_CONTENTY_WORDS.search(t) or PROSE_VERBS.search(t)) and not _has_program_intent(t):
            if progtitle_strictness >= 3:
                continue

        # Drop sentence-like cases
        if ". " in t and len(t) >= 40:
            continue
        if ": " in t and len(t) >= 60 and not _has_program_intent(t):
            continue

        keep.append(t)

    # Score + dedupe by normalized key
    best_by_key: Dict[str, Tuple[int, str]] = {}
    for t in keep:
        k = norm_title_key(t)
        if not k:
            continue
        sc = _score_title_variant(t)
        cur = best_by_key.get(k)
        if cur is None or sc > cur[0] or (sc == cur[0] and len(t) < len(cur[1])):
            best_by_key[k] = (sc, t)

    out = [v[1] for v in best_by_key.values()]
    return sorted(out)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

# "Listing-like" URL patterns for anchors we count toward hubness
LISTING_LINK_PATTERNS = [
    re.compile(r"/programs/[^/]+/?$", re.I),
    re.compile(r"/majors-minors/[^/]+/?$", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/departments/[^/]+/?$", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
]

# Broader detail-page detection (so single-major pages don’t beat hubs)
DETAIL_URL_PATTERNS = [
    re.compile(r"/programs/[^/]+/.+", re.I),
    re.compile(r"/majors-minors/[^/]+/.+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/.+", re.I),
    re.compile(r"/departments/[^/]+/.+", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
    re.compile(r"/content/.*/departments/[^/]+\.html$", re.I),
]

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================================================
# Row tagging heuristics
# ============================================================
HUB_PATH_HINTS = [
    "majors", "minors", "programs", "degrees", "department", "departments",
    "departments-and-programs", "academics", "academic", "catalog", "bulletin",
    "undergraduate", "undergraduate-programs", "courses", "areas-of-study"
]

NON_ACADEMIC_HINTS = [
    "news", "event", "calendar", "giving", "alumni", "advancement",
    "athletics", "sports", "tickets", "shop", "bookstore",
    "it", "its", "help", "support", "login", "privacy", "accessibility",
    "admissions", "apply", "visit", "tuition", "financial-aid",
    "student-life", "housing", "dining", "library",
    "about", "contact", "directory", "people", "faculty", "staff",
    "press", "media", "magazine", "blog", "policy"
]

SUBSITE_HINTS = [
    "center", "centre", "institute", "office", "community", "belonging",
    "extension", "continuing-ed", "continuing-education",
    "professional", "outreach",
    # dataset-specific common traps
    "nwo", "cge",
]

# High-precision traps: if these appear in host+path, treat page as an automatic loss.
HARD_SUBSITE_BLOCK_TOKENS = [
    "nwo",  # dataset-specific trap
    "cge",  # dataset-specific trap
]

# Separate from the broad detail penalty: this targets department/program/detail pages that can look
# academically relevant but are still not the intended "programs hub" landing page.
# Starting value calibrated from sweep snapshots: “Correct-ish but too specific” was rare (~2/25 rows, ~8%),
# and usually didn’t have an obvious canonical hub alternative in the top-K alts; use a moderate penalty
# (similar scale to other URL penalties) to nudge toward hubs without overcorrecting.
TOO_SPECIFIC_PENALTY = 240

# If a canonical hub is within this many points of best, prefer it.
CANONICAL_TIE_THRESHOLD = 60

TOO_SPECIFIC_PATTERNS = [
    r"/departments?/[a-z0-9_-]+",
    r"/departments-and-programs/[a-z0-9_-]+",
    r"/programs?/[a-z0-9_-]+",
    r"/academics/.+/(major|minors|programs)\b",
]


# ============================================================
# Unicode normalization
# ============================================================
def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================================================
# Cache + fetch
# ============================================================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"


def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)
    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    try:
        r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    except Exception:
        # Record request exceptions for row-level tagging (timeouts, connection errors, etc.)
        try:
            record_request_exception()
        except Exception:
            pass
        raise
    # Record fetch status for row-level tagging (403-aware)
    try:
        record_http_status(getattr(r, "status_code", 0) or 0)
    except Exception:
        pass
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass
    return content


def fetch_text_cached(url: str, session: requests.Session) -> str:
    b = fetch_bytes_cached(url, session=session)
    try:
        return b.decode("utf-8", errors="replace")
    except Exception:
        return b.decode("latin-1", errors="replace")


# ============================================================
# URL helpers
# ============================================================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n


def rootish_domain(netloc: str) -> str:
    """
    Lightweight "registrable-ish" domain: last two labels if possible.
    This is imperfect for .edu subdomains but works well enough for drift detection.
    """
    n = root_domain(netloc)
    parts = [p for p in n.split(".") if p]
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return n


ALLOWED_ACADEMIC_SUBDOMAINS = {"www", "majors", "programs", "catalog", "bulletin", "courses"}


def is_subsite_like(url: str, base_netloc: str) -> bool:
    """
    Heuristic: "subsite-like" means the URL is likely an office/center/outreach subsite,
    or lives on a non-standard subdomain that frequently hosts non-inventory pages.

    Rules:
    - Different rootish domain => subsite-like
    - Subdomain first-label not in allowlist => subsite-like
      (unless it's literally the base host itself)
    - Path contains subsite hint words => subsite-like
    """
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Offsite-ish drift
    if rootish_domain(host) != rootish_domain(base):
        return True

    # Subdomain gating: majors.<root>, catalog.<root>, etc allowed; others penalized
    if host != base:
        # get first label (e.g., "catalog" in "catalog.bu.edu")
        parts = [p for p in host.split(".") if p]
        sub = parts[0] if parts else ""
        if sub and sub not in ALLOWED_ACADEMIC_SUBDOMAINS:
            return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in SUBSITE_HINTS):
        return True

    return False

def is_hard_subsite_block(url: str, base_netloc: str) -> bool:
    """True if URL is a known high-precision subsite trap (auto-loss)."""
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Drift to different registrable-ish domain => hard-block
    if rootish_domain(host) != rootish_domain(base):
        return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    return any(tok in full for tok in HARD_SUBSITE_BLOCK_TOKENS)

def is_too_specific_url(url: str) -> bool:
    """True if URL path matches TOO_SPECIFIC_PATTERNS."""
    path = (urlparse(url).path or "").lower()
    for pat in TOO_SPECIFIC_PATTERNS:
        if re.search(pat, path):
            return True
    return False

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)


def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)


def is_canonical_hub_url(url: str) -> bool:
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical


def looks_like_detail_url(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return any(p.search(path) for p in DETAIL_URL_PATTERNS)


# ============================================================
# Row tagging
# ============================================================
def _norm_host_from_web_address(web_address: str) -> str:
    u = ensure_scheme(web_address)
    if not u:
        return ""
    return (urlparse(u).netloc or "").lower()


def tag_row(web_address: str, best_url: str) -> str:
    """
    Returns one of:
      - 'Correct hub'
      - 'Correct-ish but too specific'
      - 'Wrong subsite'
      - 'Wrong non-academic'
    """
    base_host = _norm_host_from_web_address(web_address)
    u = ensure_scheme(best_url)
    if not u or best_url == "N/A":
        return "Wrong non-academic"

    host = (urlparse(u).netloc or "").lower()
    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in NON_ACADEMIC_HINTS):
        return "Wrong non-academic"

    # off-root-domain or obvious subsite hints
    if base_host and rootish_domain(host) != rootish_domain(base_host):
        return "Wrong subsite"
    if any(x in full for x in SUBSITE_HINTS):
        return "Wrong subsite"

    # hub-ish path
    if any(x in path for x in HUB_PATH_HINTS):
        for pat in TOO_SPECIFIC_PATTERNS:
            if re.search(pat, path):
                return "Correct-ish but too specific"
        return "Correct hub"

    # weaker academic fallback
    if "academ" in path or "catalog" in path or "bulletin" in path:
        return "Correct-ish but too specific"

    return "Wrong subsite"


# ============================================================
# HTML parse + main-content extraction
# ============================================================
@dataclass
class ParsedPage:
    url: str
    visible_text: str
    main_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]  # (anchor_text, abs_url)
    title_like: List[str]
    html_title: str
    h1: str
    corpus_any: str
    corpus_structured: str
    json_blob: str
    js_hint: int


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    h1_tag = soup.find("h1")
    h1 = normalize_unicode_text(h1_tag.get_text(" ", strip=True)) if h1_tag else ""

    # --- Embedded JSON support (v12 B): keep machine-readable program lists ---
    # Many modern hubs embed program lists as JSON (ld+json, app/json, Next/Nuxt payloads).
    # We extract likely-JSON script contents into `json_blob` and then remove scripts so visible text stays clean.
    json_chunks: List[str] = []
    for sc in soup.find_all("script"):
        typ = (sc.get("type") or "").strip().lower()
        sid = (sc.get("id") or "").strip().lower()

        is_json_type = ("json" in typ)  # includes application/ld+json, application/json, etc.
        is_framework_payload = sid in ("__next_data__", "__nuxt__")

        if not (is_json_type or is_framework_payload):
            continue

        raw = sc.string if sc.string is not None else sc.get_text(" ", strip=True)
        raw = raw.strip() if raw else ""
        if not raw:
            continue

        raw_norm = normalize_unicode_text(raw)
        if not raw_norm:
            continue

        # Keep only content that looks plausibly JSON-like to avoid pulling in inline JS.
        lstripped = raw_norm.lstrip()
        if not (
            lstripped.startswith("{")
            or lstripped.startswith("[")
            or '"@context"' in raw_norm
            or '"@type"' in raw_norm
            or '"@graph"' in raw_norm
            or '"itemListElement"' in raw_norm
        ):
            continue

        json_chunks.append(raw_norm)

    json_blob = normalize_unicode_text(" ".join(json_chunks))
    if len(json_blob) > MAX_JSON_BLOB_CHARS:
        json_blob = json_blob[:MAX_JSON_BLOB_CHARS]

    # Heuristic: if page looks like a JS-rendered app shell, mark it so scoring/candidate expansion can adapt
    raw_lower = (html or "").lower()
    js_hint = 1 if (
        "__next_data__" in raw_lower
        or "window.__nuxt__" in raw_lower
        or "data-reactroot" in raw_lower
        or "id=\"root\"" in raw_lower
        or "id=\"app\"" in raw_lower
        or "ng-version" in raw_lower
    ) else 0

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    main_node = soup.find("main") or soup.find(id=re.compile(r"content|main", re.I)) or soup.find("article")
    if main_node:
        main_text = normalize_unicode_text(main_node.get_text(separator=" ", strip=True))
    else:
        soup2 = BeautifulSoup(str(soup), "html.parser")
        for tag in soup2(["header", "footer", "nav"]):
            tag.decompose()
        main_text = normalize_unicode_text(soup2.get_text(separator=" ", strip=True))

    # Title-like candidates should be pulled from "main-ish" content when possible.
    # This reduces nav/news/social noise before later postprocessing.
    title_like: List[str] = []
    title_scope = main_node
    if title_scope is None:
        soup_scope = BeautifulSoup(str(soup), "html.parser")
        for tag in soup_scope(["header", "footer", "nav", "aside"]):
            tag.decompose()
        title_scope = soup_scope

    for tag in title_scope.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(
            x
            for x in [visible_text, json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts)]
            if x
        )
    )
    corpus_structured = normalize_unicode_text(
        " ".join(
            x
            for x in [json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)]
            if x
        )
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        main_text=main_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        h1=h1,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
        json_blob=json_blob,
        js_hint=js_hint,
    )


def is_soft_404(parsed: ParsedPage) -> bool:
    blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1] + parsed.title_like[:5]))
    if SOFT_404_TEXT.search(blob):
        return True
    if len(parsed.main_text) < 600 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    if len(parsed.main_text) < 250 and len(parsed.anchor_texts) < 5:
        return True
    return False


def page_is_thin(parsed: ParsedPage) -> bool:
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) and (len(parsed.main_text) < MIN_MAIN_TEXT_LEN)


# ============================================================
# Sitemap parsing
# ============================================================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()


def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls


def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()

        # hard skips
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue

        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break

    return out


# ============================================================
# Hubness: program-listing-only link counts
# ============================================================
def is_listing_link(url: str) -> bool:
    path = (urlparse(url).path or "")
    return any(p.search(path) for p in LISTING_LINK_PATTERNS)


def hubness_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    listing_links = 0
    slug_set: Set[str] = set()
    distinct_anchor: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        if not is_listing_link(u):
            continue

        listing_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        path = (urlparse(u).path or "").strip("/").lower()
        segs = [s for s in path.split("/") if s]
        if segs:
            slug_set.add(segs[-1])

    # weak secondary signal
    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if listing_links >= 12:
        sig += 1
    if listing_links >= 25:
        sig += 1
    if len(slug_set) >= 8:
        sig += 1
    if len(slug_set) >= 16:
        sig += 1
    if len(distinct_anchor) >= 18:
        sig += 1
    if li_like >= 40 and (listing_links >= 10):
        sig += 1

    counts = {
        "listing_links": listing_links,
        "slug_diversity": len(slug_set),
        "distinct_anchor": len(distinct_anchor),
        "li_like": li_like,
    }
    return sig, counts


# ============================================================
# Scoring
# ============================================================
@dataclass
class PageSignals:
    url: str
    tier: str
    soft404: int
    thin: int

    canonical_hub: int
    hub_sig: int
    hub_counts: Dict[str, int]

    struct_hits: List[str]
    control_unique_structured: int

    year_hit: int
    undergrad_title_boost: int

    grad_penalty: int
    profile_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int

    subsite_like: int  # configurable penalty via knob
    hard_subsite_block: int  # auto-loss for known traps
    too_specific: int  # explicit penalty separate from generic detail penalty
    js_hub_hint: int


def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if is_inventoryish_url(url):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"


def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    soft404 = 1 if is_soft_404(parsed) else 0
    thin = 1 if page_is_thin(parsed) else 0

    control_unique_structured = sum(1 for pat in CONTROL_PATS.values() if pat.search(parsed.corpus_structured))
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    title_blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1]))
    year_hit = 1 if (YEAR_REGEX.search(url) or YEAR_REGEX.search(title_blob)) else 0
    undergrad_title_boost = 1 if UNDERGRAD_TITLE_BOOST.search(title_blob) else 0

    u_low = url.lower()
    grad_penalty = 1 if (GRAD_URL_PENALTY.search(u_low) or re.search(r"\bgraduate\b", title_blob, re.I)) else 0
    profile_penalty = 1 if (PROFILE_URL_PENALTY.search(u_low) or re.search(r"\bfaculty directory\b", title_blob, re.I)) else 0
    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(u_low) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(u_low) else 0
    archive_penalty = 1 if ARCHIVE_URL_PENALTY.search(u_low) else 0
    awards_penalty = 1 if AWARDS_URL_PENALTY.search(u_low) else 0
    compliance_penalty = 1 if COMPLIANCE_URL_PENALTY.search(u_low) else 0

    hub_sig, hub_counts = hubness_signature(parsed, base_netloc=base_netloc)
    canonical_hub = 1 if is_canonical_hub_url(url) else 0

    subsite_like = 1 if is_subsite_like(url, base_netloc=base_netloc) else 0
    hard_subsite_block = 1 if is_hard_subsite_block(url, base_netloc=base_netloc) else 0
    too_specific = 1 if (is_too_specific_url(url) and not is_canonical_hub_url(url)) else 0

    js_hub_hint = 1 if (getattr(parsed, "js_hint", 0) and (len(getattr(parsed, "json_blob", "")) >= 200)) else 0
    return PageSignals(
        url=url,
        tier=page_tier(url),
        soft404=soft404,
        thin=thin,
        canonical_hub=canonical_hub,
        hub_sig=hub_sig,
        hub_counts=hub_counts,
        struct_hits=struct_hits,
        control_unique_structured=control_unique_structured,
        year_hit=year_hit,
        undergrad_title_boost=undergrad_title_boost,
        grad_penalty=grad_penalty,
        profile_penalty=profile_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
        subsite_like=subsite_like,
        hard_subsite_block=hard_subsite_block,
        too_specific=too_specific,
        js_hub_hint=js_hub_hint,
    )


def canonical_bonus_allowed(sig: PageSignals) -> bool:
    if sig.soft404:
        return False
    if sig.thin and sig.hub_sig == 0 and len(sig.struct_hits) < 2:
        return False
    return True


def score_inventory(sig: PageSignals, subsite_penalty: int) -> int:
    if sig.soft404:
        return -10_000
    
    # hard subsite blocks are automatic losses
    if getattr(sig, "hard_subsite_block", 0):
        return -10_000    

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 40

    # hubness + structured hits
    score += 85 * sig.hub_sig
    score += 10 * len(sig.struct_hits)
    score += 4 * sig.control_unique_structured

    # canonical hub bonus (gated)
    if sig.canonical_hub and canonical_bonus_allowed(sig):
        score += 160

    # undergrad title boost
    if sig.undergrad_title_boost:
        score += 35

    # year boost only for catalog/bulletin pages
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        score += 70

    # penalties (URL/title/H1)
    if sig.profile_penalty:
        score -= 260
    if sig.grad_penalty:
        score -= 170
    if sig.irrelevant_penalty:
        score -= 260
    if sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 220
    if sig.awards_penalty:
        score -= 200
    if sig.compliance_penalty:
        score -= 150

    # configurable subsite penalty
    if sig.subsite_like:
        score -= int(subsite_penalty)

    # explicit penalty for "too specific" department/program pages
    if getattr(sig, "too_specific", 0):
        score -= int(TOO_SPECIFIC_PENALTY)    

    # broad detail penalty so hubs win
    if looks_like_detail_url(sig.url):
        if sig.hub_sig <= 1:
            score -= 220
        elif sig.hub_sig == 2:
            score -= 90
        else:
            score -= 25

    # thin penalty (C2: JS hubs can be thin app shells; barely penalize)
    if sig.thin and sig.hub_sig == 0:
        if getattr(sig, "js_hub_hint", 0) and is_inventoryish_url(sig.url):
            score -= 20
        else:
            score -= 180

    return score
# ============================================================
# Candidate generation
# ============================================================

def looks_like_js_hub(parsed: ParsedPage, url: str) -> bool:
    """Detect JS-rendered program hubs early so we can expand via sitemap sooner."""
    if not page_is_thin(parsed):
        return False

    u = (url or "").lower()
    host = (urlparse(url).netloc or "").lower()
    inventoryish = is_inventoryish_url(url) or (host.startswith("majors.") or host.startswith("programs."))
    if not inventoryish and "find-your-program" not in u:
        return False

    # Typical app shells have few usable anchors and often embed data in JSON
    if len(parsed.links) > 60:
        return False
    if getattr(parsed, "js_hint", 0):
        return True
    if len(getattr(parsed, "json_blob", "")) >= 200:
        return True
    return False



def url_prior(url: str, base_netloc: str, subsite_penalty: int) -> int:
    u = url.lower()
    s = 0

    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 200
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 35
    if "bulletin" in u:
        s -= 20

    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if COMPLIANCE_URL_PENALTY.search(u):
        s -= 90
    if AWARDS_URL_PENALTY.search(u):
        s -= 160
    if GRAD_URL_PENALTY.search(u):
        s -= 180

    # hard subsite blocks should be avoided aggressively
    if is_hard_subsite_block(url, base_netloc=base_netloc):
        s -= 10_000
    if is_subsite_like(url, base_netloc=base_netloc):
        s -= int(subsite_penalty)

    if is_canonical_hub_url(url):
        s += 160

    return s


def prioritize_links(links: List[Tuple[str, str]], base_netloc: str, subsite_penalty: int) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000

        s = url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty)

        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 45
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 90
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140

        return -s

    return sorted(links, key=rank)


# ============================================================
# Candidate structure
# ============================================================
@dataclass
class CandidatePage:
    url: str
    score: int
    sig: PageSignals
    parsed: ParsedPage


def get_parsed_page(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)


def fetch_and_score(url: str, session: requests.Session, base_netloc: str, subsite_penalty: int) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page(url, session=session)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        sc = score_inventory(sig, subsite_penalty=subsite_penalty)
        return CandidatePage(url=url, score=sc, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)


# ============================================================
# Parallel batch fetch+score helper
# ============================================================
def fetch_and_score_many(
    urls: List[str],
    base_netloc: str,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    """Fetch+score a batch of URLs concurrently (one Session per thread)."""
    # de-dupe while preserving order
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            uniq.append(u)

    if workers <= 1:
        out: List[CandidatePage] = []
        s = requests.Session()
        for u in uniq:
            cand = fetch_and_score(u, session=s, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
            if cand is not None:
                out.append(cand)
        return out

    out: List[CandidatePage] = []

    def _task(u: str) -> Optional[CandidatePage]:
        # IMPORTANT: call get_thread_session() inside the worker thread
        return fetch_and_score(
            u,
            session=get_thread_session(),
            base_netloc=base_netloc,
            subsite_penalty=subsite_penalty,
        )

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_task, u) for u in uniq]
        for fut in as_completed(futs):
            cand = fut.result()
            if cand is not None:
                out.append(cand)
    return out


def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.score, len(topk), cand))
        return
    if cand.score > topk[0][0]:
        heapq.heapreplace(topk, (cand.score, topk[0][1], cand))


def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    """Return candidates sorted by score (desc), with an optional canonical-hub tie-break."""
    cands = [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]

    # C: Prefer canonical hubs more aggressively when tie-ish.
    if cands:
        best_score = cands[0].score

        canonical_best: Optional[CandidatePage] = None
        for c in cands:
            if c.sig.canonical_hub and (best_score - c.score) <= CANONICAL_TIE_THRESHOLD:
                if canonical_best is None or c.score > canonical_best.score:
                    canonical_best = c

        if canonical_best is not None and canonical_best is not cands[0]:
            cands = [canonical_best] + [x for x in cands if x is not canonical_best]

    return cands


# ============================================================
# Candidate generation
# ============================================================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))

    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================================================
# Program title extraction
# ============================================================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_title_key_loose(s: str) -> str:
    """Looser normalization for concordance (handles '&' vs 'and', generic suffix words, etc.)."""
    s = normalize_unicode_text(s).lower()
    s = s.replace("&", " and ")
    # Drop very common generic words that frequently cause false non-matches
    s = re.sub(r"\b(program|department|major|minor|concentration|certificate|track|option)\b", " ", s)
    s = re.sub(r"\b(the|of|in|for|and|to|a|an)\b", " ", s)
    s = re.sub(r"[^\w\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _content_tokens(s: str) -> Set[str]:
    """Tokenize a title into content-bearing tokens for partial matching."""
    s = norm_title_key_loose(s)
    if not s:
        return set()
    toks = [t for t in re.split(r"[\s_-]+", s) if t]
    toks = [t for t in toks if len(t) >= 3]  # remove tiny junk tokens
    return set(toks)


def _overlap_coeff(a: Set[str], b: Set[str]) -> float:
    """|A∩B| / min(|A|,|B|)"""
    if not a or not b:
        return 0.0
    inter = len(a.intersection(b))
    denom = float(min(len(a), len(b)))
    return inter / denom if denom > 0 else 0.0


def best_partial_title_match(a_titles: List[str], b_titles: List[str]) -> Tuple[float, str, str]:
    """Return best partial match score and the (a_title, b_title) pair.

    Score is max(token overlap coefficient, SequenceMatcher ratio) on loose-normalized strings.
    """
    best_score = 0.0
    best_a = ""
    best_b = ""

    for a in (a_titles or []):
        a = normalize_unicode_text(a)
        if not a:
            continue
        a_loose = norm_title_key_loose(a)
        a_tok = _content_tokens(a)

        for b in (b_titles or []):
            b = normalize_unicode_text(b)
            if not b:
                continue
            b_loose = norm_title_key_loose(b)
            b_tok = _content_tokens(b)

            tok_score = _overlap_coeff(a_tok, b_tok)
            seq_score = difflib.SequenceMatcher(None, a_loose, b_loose).ratio() if (a_loose and b_loose) else 0.0
            score = max(tok_score, seq_score)

            if score > best_score:
                best_score = score
                best_a = a
                best_b = b

    return best_score, best_a, best_b


def partial_concordance(a_titles: List[str], b_titles: List[str], threshold: float = 0.80) -> Tuple[int, str]:
    """Binary + detail string for partial concordance between two title lists."""
    score, a_best, b_best = best_partial_title_match(a_titles, b_titles)
    ok = int(score >= float(threshold))
    detail = ""
    if ok and a_best and b_best:
        detail = f'inst="{a_best}" ~ other="{b_best}" score={score:.2f}'
    return ok, detail

def _split_program_name_field(s: str) -> List[str]:
    """Split a program-name field into candidate names (handles pipes/newlines/semicolons)."""
    s = normalize_unicode_text(s or "")
    if not s:
        return []
    parts: List[str] = [s]
    for sep in ["|", ";", "\n", "\r"]:
        tmp: List[str] = []
        for p in parts:
            if sep in p:
                tmp.extend([x.strip() for x in p.split(sep) if x.strip()])
            else:
                tmp.append(p)
        parts = tmp

    # Only split on commas when it looks like a list (avoid breaking names like
    # "African, Black & Caribbean Studies")
    tmp2: List[str] = []
    for p in parts:
        if p.count(",") >= 2:
            tmp2.extend([x.strip() for x in p.split(",") if x.strip()])
        else:
            tmp2.append(p)
    return [normalize_unicode_text(x) for x in tmp2 if x]


def _title_key_set(titles: List[str]) -> Set[str]:
    """Normalized key set for exact-match concordance checks."""
    return {norm_title_key(t) for t in (titles or []) if norm_title_key(t)}


def _field_key_set(field_val: str) -> Set[str]:
    """Normalized key set from a scalar program-name field (e.g., 2013_program_name)."""
    return {norm_title_key(x) for x in _split_program_name_field(field_val) if norm_title_key(x)}


def _any_exact_concordance(a: Set[str], b: Set[str]) -> int:
    """1 if any exact normalized title matches exist between sets; else 0."""
    if not a or not b:
        return 0
    return int(bool(a.intersection(b)))

def looks_like_program_title(s: str, progtitle_strictness: int, context: str = "hub") -> bool:
    """
    Strictness guidance:
      1-2: permissive (still requires TARGET_ANY)
      3: current default behavior
      4-5: stricter (prefer explicit program-ish keywords; tighter filters)
    context: "hub" (default) or "follow" (detail)
    """
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False

    s_low = s_norm.lower()

    # D1: obvious junk rejects
    if URL_LIKE_TEXT.search(s_norm):
        return False
    if "toggle" in s_low:
        return False

    # always require the target token family (keeps extraction scoped)
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False

    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm):
        return False

    # D1: sentence-like / prose-ish rejects
    if ". " in s_norm and len(s_norm) >= 40:
        return False
    if ": " in s_norm and len(s_norm) >= 55:
        # allow short degree-style labels, reject long descriptive clauses
        if PROGRAM_KEYWORDS.search(s_norm) is None and not re.search(r"\b(B\.?A\.?|B\.?S\.?|BA|BS)\b", s_norm, re.I):
            return False
    if PROSE_VERBS.search(s_norm) and len(s_norm) >= 45:
        # follow/detail pages are stricter; hubs can still reject obvious prose
        if context == "follow" or progtitle_strictness >= 3:
            return False

    # D1: long multi-clause strings are usually prose or navigation
    if len(s_norm) > 90:
        comma_ct = s_norm.count(",")
        if comma_ct >= 2 and PROGRAM_KEYWORDS.search(s_norm) is None:
            return False

    # NOTE: negative-context words (news/events/etc.) are handled primarily in
    # clean_program_titles() where we can condition on program intent.
    # Keep only a very strict early reject.
    if TITLE_NEGATIVE_CONTEXT.search(s_norm) and progtitle_strictness >= 5:
        return False

    # D2: follow/detail pages must show stronger "program" intent
    if context == "follow":
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    if PERSON_NAME_ONLY.match(s_norm):
        if progtitle_strictness >= 3 and not PROGRAM_KEYWORDS.search(s_norm):
            return False
        if progtitle_strictness >= 5:
            return False

    if progtitle_strictness >= 4:
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    return True
def is_program_detailish_url(url: str) -> bool:
    """Tight allowlist for one-hop follow: only follow likely program/major detail pages."""
    path = (urlparse(url).path or "")
    if any(p.search(path) for p in LISTING_LINK_PATTERNS):
        return True
    # also allow common majors-programs hubs/details
    if re.search(r"/majors-programs/[^/]+", path, re.I):
        return True
    return False


def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out


# ============================================================
# PDF fallback (best-effort)
# ============================================================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""


def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================================================
# Aggregate extraction from inventory + alts (+ limited 1-hop)
# ============================================================
def aggregate_outputs(
    pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
    progtitle_strictness: int,
    workers: int,
) -> Tuple[
    int, Set[str], Dict[str, int],
    Dict[str, List[str]],
    List[str],
    List[str],
]:
    base_netloc = urlparse(base_url).netloc

    struct_hits_union: Set[str] = set()
    control_hit_cols = {k: 0 for k in CONTROL_TERMS}
    any_control_found = 0

    tokens_agg: Dict[str, Set[str]] = {k: set() for k in TARGET_TOKEN_REGEX.keys()}
    titles: Set[str] = set()

    pdf_hits: List[str] = []

    # from chosen pages
    for cand in pages:
        any_control_found = max(any_control_found, 1 if cand.sig.control_unique_structured > 0 else 0)
        struct_hits_union.update(cand.sig.struct_hits)

        for ck, pat in CONTROL_PATS.items():
            if pat.search(cand.parsed.corpus_structured):
                control_hit_cols[ck] = 1

        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k, vals in tm.items():
            tokens_agg[k].update(vals)

        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
                titles.add(txt)

    # 1-hop follow
    follow_urls: List[str] = []
    for cand in pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            # D3: follow only if anchor text looks like a title, OR URL looks like a program detail page
            if looks_like_program_title(a_text or "", progtitle_strictness=progtitle_strictness, context="hub") or is_program_detailish_url(u):
                follow_urls.append(u)

    # sitemap assist if best looks thin
    if pages and page_is_thin(pages[0].parsed):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    # Fetch a limited number of follow URLs (optionally in parallel)
    follow_batch = uniq_follow[:ONE_HOP_MAX_FETCHES]

    def _safe_fetch_parse(u: str) -> Optional[ParsedPage]:
        try:
            # Use per-thread sessions when parallel
            s = get_thread_session() if workers and workers > 1 else session
            return get_parsed_page(u, session=s)
        except Exception:
            return None
        finally:
            # Keep some politeness throttling even when parallel
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    if workers and workers > 1 and len(follow_batch) > 1:
        max_w = min(int(workers), 8, len(follow_batch))
        with ThreadPoolExecutor(max_workers=max_w) as ex:
            futs = [ex.submit(_safe_fetch_parse, u) for u in follow_batch]
            for fut in as_completed(futs):
                parsed = fut.result()
                if not parsed:
                    continue

                tm = token_matches_from_text(parsed.corpus_any)
                for k, vals in tm.items():
                    tokens_agg[k].update(vals)

                for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                    if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                        titles.add(txt)
    else:
        for u in follow_batch:
            parsed = _safe_fetch_parse(u)
            if not parsed:
                continue

            tm = token_matches_from_text(parsed.corpus_any)
            for k, vals in tm.items():
                tokens_agg[k].update(vals)

            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                    titles.add(txt)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in pages:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k, vals in tm.items():
                            tokens_agg[k].update(vals)
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Postprocess titles: split breadcrumbs, salvage boilerplate, drop course/news/social noise,
    # and select the best variant per normalized key.
    titles_out = clean_program_titles(list(titles), progtitle_strictness=progtitle_strictness)
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}

    return any_control_found, struct_hits_union, control_hit_cols, tokens_out, titles_out, pdf_hits


#
# ============================================================
# Discovery: find candidate pages
# ============================================================
def find_candidates_for_institution(
    base_url: str,
    session: requests.Session,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_cache
        if sitemap_cache is not None:
            return sitemap_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_cache = sitemap_candidate_urls(all_urls)
        return sitemap_cache

    def consider(u: str) -> Optional[CandidatePage]:
        if not u or u in tried:
            return None
        tried.add(u)
        cand = fetch_and_score(u, session=session, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
        if cand is None:
            return None
        push_topk(topk, cand, k=TOP_K_ALTS)
        return cand

    # Phase 0
    consider(base_url)

    # ------------------------------------------------------------
    # Phase 1 (parallelizable): explicit paths + year/catalog + home links
    # ------------------------------------------------------------
    batch_urls: List[str] = []

    for p in INVENTORY_PATHS[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    batch_urls.extend(build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES])

    for p in CATALOG_PATHS[:PHASE1_MAX_CATALOG_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    # homepage links require parsing homepage once (sequential), but the fetched URLs can be scored in batch
    try:
        home = get_parsed_page(base_url, session=session)
        
        # C1: If the homepage/programs page is a JS hub (thin app shell), expand candidates via sitemap immediately
        if looks_like_js_hub(home, base_url):
            try:
                batch_urls.extend(load_sitemap_candidates()[:80])
            except Exception:
                pass
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc, subsite_penalty=subsite_penalty)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            batch_urls.append(u)
    except Exception:
        pass

    # score the batch in parallel (if workers>1)
    batch_cands = fetch_and_score_many(
        batch_urls,
        base_netloc=base_netloc,
        subsite_penalty=subsite_penalty,
        workers=workers,
    )
    for cand in batch_cands:
        tried.add(cand.url)
        push_topk(topk, cand, k=TOP_K_ALTS)

    # C2: If ANY early candidate looks like a JS-rendered hub, expand via sitemap NOW
    # (even if the thin hub didn't win the scoring yet).
    try:
        if sitemap_cache is None:
            js_hub_seen = any(looks_like_js_hub(c.parsed, c.url) for c in batch_cands)
            if js_hub_seen:
                sm_urls = load_sitemap_candidates()[:120]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
    except Exception:
        pass

    # Optional: if best so far looks weak, pull some sitemap candidates (also batchable)
    if topk:
        best_so_far = max(topk, key=lambda x: x[0])[2]
        if (best_so_far.sig.thin or best_so_far.sig.hub_sig == 0) and is_inventoryish_url(best_so_far.url):
            try:
                sm_urls = load_sitemap_candidates()[:50]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
            except Exception:
                pass

    # Best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty), depth, u))

    pq_push(base_url, 0)
    fetched = 0
    while pq and fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u)
        fetched += 1
        if depth >= PHASE2_MAX_DEPTH:
            continue
        try:
            parsed = cand.parsed if cand is not None else get_parsed_page(u, session=session)
            links = [(t, lu) for (t, lu) in parsed.links if same_domain(lu, base_netloc)]
            links = prioritize_links(links, base_netloc, subsite_penalty=subsite_penalty)
            for _, lu in links[:70]:
                pq_push(lu, depth + 1)
        except Exception:
            pass

    # Subdomain roots
    for root in build_subdomain_roots(base_url):
        c_root = consider(root)
        if not c_root:
            continue
        root_is_good = (c_root.sig.hub_sig >= 1) or (len(c_root.sig.struct_hits) >= 2) or (not c_root.sig.thin)
        if root_is_good and not c_root.sig.soft404:
            continue
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p))

    return topk_sorted(topk)


# ============================================================
# Reason string
# ============================================================
def inventory_reason(c: CandidatePage) -> str:
    s = c.sig
    flags = []
    if s.grad_penalty:
        flags.append("grad")
    if s.profile_penalty:
        flags.append("profile")
    if s.irrelevant_penalty:
        flags.append("irrelevant")
    if s.admissions_penalty:
        flags.append("admissions")
    if s.archive_penalty:
        flags.append("archive")
    if s.awards_penalty:
        flags.append("awards")
    if s.compliance_penalty:
        flags.append("compliance")
    if s.year_hit:
        flags.append("yearHit")
    if s.thin:
        flags.append("thin")
    if s.soft404:
        flags.append("soft404")
    if s.subsite_like:
        flags.append("subsiteLike")
    if getattr(s, "hard_subsite_block", 0):
        flags.append("hardSubsiteBlock")
    if getattr(s, "too_specific", 0):
        flags.append("tooSpecific")

    return ";".join([
        f"score={c.score}",
        f"tier={s.tier}",
        f"hubSig={s.hub_sig}",
        f"listingLinks={s.hub_counts.get('listing_links', 0)}",
        f"slugDiv={s.hub_counts.get('slug_diversity', 0)}",
        f"structHits={len(s.struct_hits)}",
        f"ctrlStructured={s.control_unique_structured}",
        f"canonicalHub={s.canonical_hub}",
        ("flags=" + ",".join(flags)) if flags else "flags=",
    ])


# ============================================================
# CLI
# ============================================================
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--input", type=str, default=str(DEFAULT_INPUT_PATH))
    ap.add_argument("--output", type=str, default=str(DEFAULT_OUTPUT_PATH))

    # If omitted, default to TEST_HEAD_N (notebook-friendly). Use --head 0 for full run.
    ap.add_argument("--head", type=int, default=None, help="If >0, limit rows for testing; if omitted defaults to TEST_HEAD_N")

    ap.add_argument("--out-suffix", type=str, default="", help="Append to output filename stem")
    ap.add_argument("--subsite-penalty", type=int, default=80)
    ap.add_argument("--progtitle-strictness", type=int, default=3)

    # Parallelism (keep small to be polite)
    ap.add_argument(
        "--workers",
        type=int,
        default=DEFAULT_WORKERS,
        help="Parallel fetch workers for candidate scoring (1 = off). Try 4.",
    )

    ap.add_argument(
        "--debug-jsonlen",
        action="store_true",
        help="Debug: print json_blob length (chars) for the best candidate per school",
    )

    # IMPORTANT: ignore Jupyter/ipykernel injected args like --f=...
    args, _unknown = ap.parse_known_args(argv)

    if args.head is None:
        args.head = TEST_HEAD_N

    # sanity clamp
    if args.workers is None or args.workers < 1:
        args.workers = 1
    if args.workers > MAX_WORKERS:
        args.workers = MAX_WORKERS

    return args


def apply_out_suffix(out_path: Path, suffix: str) -> Path:
    if not suffix:
        return out_path
    return out_path.with_name(out_path.stem + suffix + out_path.suffix)


# ============================================================
# Main
# ============================================================
def main() -> None:
    args = parse_args()
    input_path = Path(args.input)
    output_path = apply_out_suffix(Path(args.output), args.out_suffix)

    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {input_path}: {sorted(missing)}")

    if args.head and args.head > 0:
        df = df.head(args.head).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")
        # Reset per-institution status tally for fetch-status tagging (403-aware)
        reset_status_tally()

        best_inventory_url = "N/A"
        best_inventory_reason = ""
        alt_candidate_urls = ""
        url_tag_value = ""

        any_control_found = 0
        struct_hits_union: Set[str] = set()
        any_struct_found = 0

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        token_matches: Dict[str, List[str]] = {k: [] for k in OUTPUT_TOKEN_BUCKETS}

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0
        pdf_hits: List[str] = []
        # CollegeVine (3rd-party truth source)
        college_vine_site = 0
        college_vine_control_hits = 0
        college_vine_url = ""
        college_vine_program_title_count = 0
        college_vine_program_titles: List[str] = []

        try:
            if base_url:
                candidates = find_candidates_for_institution(
                    base_url,
                    session=session,
                    subsite_penalty=args.subsite_penalty,
                    workers=args.workers,
                )

                if candidates:
                    best = candidates[0]
                    best_inventory_url = best.url
                    best_inventory_reason = inventory_reason(best)
                    alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K_ALTS])
                    if getattr(args, "debug_jsonlen", False):
                        jb_len = len(getattr(best.parsed, "json_blob", "") or "")
                        print(f"  [debug-jsonlen] best_url={best.url} | json_blob_chars={jb_len}")

                    any_control_found, struct_hits_union, control_hits, tokens, titles, pdf_hits = aggregate_outputs(
                        pages=candidates[:TOP_K_ALTS],
                        base_url=base_url,
                        session=session,
                        progtitle_strictness=args.progtitle_strictness,
                        workers=args.workers,
                    )

                    any_struct_found = 1 if struct_hits_union else 0

                    for k, v in control_hits.items():
                        control_hit_cols[f"found_{k}"] = int(v)

                    for k in OUTPUT_TOKEN_BUCKETS:
                        token_matches[k] = tokens.get(k, [])

                    program_titles = titles
                    program_title_count = len(program_titles)
                    any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0
        except Exception:
            # keep row, but leave defaults
            pass

        # Freeze institution-site fetch-status tag BEFORE any CollegeVine calls (so CV doesn't pollute tally)
        url_tag_value = fetch_status_tag(best_inventory_url)
        # Reset status tally so any CV fetches won't affect institution url_tag
        reset_status_tally()

        # CollegeVine (separate fetch so it doesn't affect the university-site fetch-status tally)
        try:
            (
                college_vine_url,
                college_vine_site,
                college_vine_control_hits,
                college_vine_program_title_count,
                college_vine_program_titles,
            ) = fetch_collegevine_majors_page(
                school_name=name,
                session=session,
                progtitle_strictness=args.progtitle_strictness,
            )
        except Exception:
            pass

        rec: Dict[str, str] = dict(row)

        rec["best_guess_inventory_url"] = best_inventory_url
        rec["best_guess_inventory_reason"] = best_inventory_reason
        rec["url_tag"] = url_tag_value
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        for k in OUTPUT_TOKEN_BUCKETS:
            vals = token_matches.get(k, [])
            rec[f"{k}_matches"] = "|".join(vals) if vals else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        # CollegeVine truth-source columns
        rec["college_vine_site"] = str(int(college_vine_site))
        rec["college_vine_url"] = (college_vine_url or "")
        rec[f"college_vine_controls_ge_{int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB)}"] = str(
            int(college_vine_control_hits >= int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB))
        )

        # Ensure CollegeVine program titles are cleaned consistently before storage + concordance
        college_vine_program_titles_clean = clean_program_titles(
            list(college_vine_program_titles) if college_vine_program_titles else [],
            progtitle_strictness=args.progtitle_strictness,
        )
        rec["college_vine_program_title_count"] = str(len(college_vine_program_titles_clean))
        rec["college_vine_program_titles_found"] = (
            "|".join(college_vine_program_titles_clean) if college_vine_program_titles_clean else ""
        )

        # Concordance checks (exact match on normalized title keys)
        inst_keys = _title_key_set(program_titles)
        cv_keys = _title_key_set(college_vine_program_titles_clean)
        p2013_keys = _field_key_set(str(row.get("2013_program_name", "")))

        rec["concord_institution_vs_collegevine"] = str(_any_exact_concordance(inst_keys, cv_keys))
        rec["concord_institution_vs_2013_program_name"] = str(_any_exact_concordance(inst_keys, p2013_keys))
        rec["concord_collegevine_vs_2013_program_name"] = str(_any_exact_concordance(cv_keys, p2013_keys))

        # Partial concordance checks (loose normalization + token overlap / string similarity)
        # These help when names differ by punctuation, '&' vs 'and', generic suffix words (Program/Department), etc.
        inst_titles_for_match = list(program_titles or [])
        cv_titles_for_match = list(college_vine_program_titles_clean or [])
        p2013_titles_for_match = _split_program_name_field(str(row.get("2013_program_name", "")))

        inst_vs_cv_partial, inst_vs_cv_partial_detail = partial_concordance(
            inst_titles_for_match,
            cv_titles_for_match,
            threshold=0.80,
        )
        inst_vs_2013_partial, inst_vs_2013_partial_detail = partial_concordance(
            inst_titles_for_match,
            p2013_titles_for_match,
            threshold=0.80,
        )
        cv_vs_2013_partial, cv_vs_2013_partial_detail = partial_concordance(
            cv_titles_for_match,
            p2013_titles_for_match,
            threshold=0.80,
        )

        rec["partial_concord_institution_vs_collegevine"] = str(inst_vs_cv_partial)
        rec["partial_concord_institution_vs_2013_program_name"] = str(inst_vs_2013_partial)
        rec["partial_concord_collegevine_vs_2013_program_name"] = str(cv_vs_2013_partial)

        rec["partial_match_detail_institution_vs_collegevine"] = inst_vs_cv_partial_detail
        rec["partial_match_detail_institution_vs_2013_program_name"] = inst_vs_2013_partial_detail
        rec["partial_match_detail_collegevine_vs_2013_program_name"] = cv_vs_2013_partial_detail

        # (concordance columns handled above)

        # total controls found (proxy for "how many majors we saw")
        total_controls_found = int(sum(int(v) for v in control_hit_cols.values()))
        rec["total_controls_found"] = str(total_controls_found)

        # sufficiency tag based on MIN_CONTROL_HITS_FOR_CONFIDENT_HUB
        if total_controls_found < int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB):
            rec["controls_sufficiency"] = f"warning_{int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB)}_MajorsFound"
        else:
            rec["controls_sufficiency"] = "sufficient majors"

        rec.update({k: str(v) for k, v in control_hit_cols.items()})
        out_rows.append(rec)

        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "url_tag",
        "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        *[f"{k}_matches" for k in OUTPUT_TOKEN_BUCKETS],
        "any_potential_AfrAmr_found",
        "pdf_hits",
        "total_controls_found",
        "controls_sufficiency",
    ]

    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]

    # CollegeVine columns should appear near the end (before concordance), per spec.
    cv_cols_all = [
        "college_vine_site",
        "college_vine_url",
        f"college_vine_controls_ge_{int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB)}",
        "college_vine_program_title_count",
        "college_vine_program_titles_found",
    ]
    cv_cols = [c for c in cv_cols_all if c in out_df.columns]

    # Partial concordance columns should appear near the end (after CollegeVine, before exact concordance).
    partial_cols_all = [
        "partial_concord_institution_vs_collegevine",
        "partial_concord_institution_vs_2013_program_name",
        "partial_concord_collegevine_vs_2013_program_name",
        "partial_match_detail_institution_vs_collegevine",
        "partial_match_detail_institution_vs_2013_program_name",
        "partial_match_detail_collegevine_vs_2013_program_name",
    ]
    partial_cols = [c for c in partial_cols_all if c in out_df.columns]

    # Concordance columns must be the final 3 columns.
    concord_cols_all = [
        "concord_institution_vs_collegevine",
        "concord_institution_vs_2013_program_name",
        "concord_collegevine_vs_2013_program_name",
    ]
    concord_cols = [c for c in concord_cols_all if c in out_df.columns]

    remaining_cols = [
        c for c in out_df.columns
        if c not in set(front_cols + control_cols + cv_cols + partial_cols + concord_cols)
    ]

    # Final column order: exact concordance last, per spec.
    out_df = out_df[front_cols + control_cols + remaining_cols + cv_cols + partial_cols + concord_cols]

    out_df.to_csv(output_path, index=False)
    print(f"\nWrote: {output_path} | rows,cols = {out_df.shape}")
    print(out_df.head())


if __name__ == "__main__":
    main()

[1/25] unitid=188429 | Adelphi University
[2/25] unitid=138600 | Agnes Scott College
[3/25] unitid=168546 | Albion College
[4/25] unitid=164465 | Amherst College
[5/25] unitid=143084 | Augustana College
[6/25] unitid=189097 | Barnard College
[7/25] unitid=132471 | Barry University
[8/25] unitid=160977 | Bates College
[9/25] unitid=156295 | Berea College
[10/25] unitid=142115 | Boise State University
[11/25] unitid=164924 | Boston College
[12/25] unitid=164988 | Boston University
[13/25] unitid=161004 | Bowdoin College
[14/25] unitid=201441 | Bowling Green State University-Main Campus
[15/25] unitid=143358 | Bradley University
[16/25] unitid=165015 | Brandeis University
[17/25] unitid=217156 | Brown University
[18/25] unitid=211273 | Bryn Mawr College
[19/25] unitid=211291 | Bucknell University
[20/25] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/25] unitid=110529 | California State Polytechnic University-Pomona
[22/25] unitid=110538 | California State Uni

In [6]:
# trouble shooting collegevine search:
cv_url = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"

reset_status_tally()
s = requests.Session()
html = fetch_text_cached(cv_url, session=s)

print("HTML chars:", len(html))
parsed = parse_html_to_parsedpage(html, page_url=cv_url)

print("visible_text chars:", len(parsed.visible_text))
print("main_text chars:", len(parsed.main_text))
print("json_blob chars:", len(parsed.json_blob))
print("links:", len(parsed.links))
print("js_hint:", parsed.js_hint)

needles = [
    "African Studies",
    "Classics and Classical Languages, Literatures, and Linguistics, General",
]
for n in needles:
    hit_any = (n in html) or (n in parsed.visible_text) or (n in parsed.main_text) or (n in parsed.json_blob)
    print(f"{n!r} -> present_anywhere={hit_any} | in_json={n in parsed.json_blob} | in_visible={n in parsed.visible_text}")

HTML chars: 57113
visible_text chars: 623
main_text chars: 0
json_blob chars: 0
links: 21
js_hint: 1
'African Studies' -> present_anywhere=False | in_json=False | in_visible=False
'Classics and Classical Languages, Literatures, and Linguistics, General' -> present_anywhere=False | in_json=False | in_visible=False


In [7]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
title = soup.title.get_text(" ", strip=True) if soup.title else ""
print("TITLE:", title)
print(html[:4000])

TITLE: CollegeVine
<!DOCTYPE html>
<html class='h-100' lang='en'>
<head>


<meta content='text/html; charset=UTF-8' http-equiv='Content-Type'>
<meta content='width=device-width, initial-scale=1' name='viewport'>
<meta content='CollegeVine' name='apple-mobile-web-app-title'>
<link href='https://d28hdetl1q8yl2.cloudfront.net/img/favicon-32x32.png' rel='icon' sizes='32x32'>
<link href='https://d28hdetl1q8yl2.cloudfront.net/img/favicon-16x16.png' rel='icon' sizes='16x16'>
<link href='https://d28hdetl1q8yl2.cloudfront.net/img/favicon-64x64.png' rel='icon' sizes='64x64'>
<link href='https://d28hdetl1q8yl2.cloudfront.net/img/favicon-128x128.png' rel='icon' sizes='128x128'>
<link href='https://d28hdetl1q8yl2.cloudfront.net/img/favicon-180x180.png' rel='icon' sizes='180x180'>
<link href='https://d28hdetl1q8yl2.cloudfront.net/img/apple-touch-icon-120x120.png' rel='apple-touch-icon' sizes='120x120'>
<link href='https://d28hdetl1q8yl2.cloudfront.net/img/apple-touch-icon-152x152.png' rel='apple-tou

In [8]:
script_srcs = []
for sc in soup.find_all("script"):
    src = sc.get("src")
    if src:
        script_srcs.append(src)

print("script src count:", len(script_srcs))
for u in script_srcs[:30]:
    print(u)

script src count: 24
https://js.sentry-cdn.com/4d42713cd70d4bb0973933789bbaede8.min.js
https://cmp.osano.com/Azyzp2TvnpgtM3kbz/02dc74f6-748a-4d06-aae6-1dda988cd583/osano.js
https://www.googletagmanager.com/gtag/js?id=G-KT9ZYXLWZ5
https://js.sentry-cdn.com/4d42713cd70d4bb0973933789bbaede8.min.js
https://d28hdetl1q8yl2.cloudfront.net/assets/tracking-fa9713c983199b4698597ef97cb4bc9b49f5a33564ac751a6aa1bc959067ded0.js
https://d28hdetl1q8yl2.cloudfront.net/theme/libs/jquery/dist/jquery.min.js
https://d28hdetl1q8yl2.cloudfront.net/assets/select2/select2.min-308692a9122971f04b0f0652762845a3e6fcf6b3b2be36f6986bb6e0c6e4d696.js
https://d28hdetl1q8yl2.cloudfront.net/theme/libs/bootstrap/dist/js/bootstrap.bundle.min.js
https://d28hdetl1q8yl2.cloudfront.net/theme/libs/autosize/dist/autosize.min.js
https://d28hdetl1q8yl2.cloudfront.net/theme/libs/list.js/dist/list.min.js
https://d28hdetl1q8yl2.cloudfront.net/theme/js/theme.min.js
https://d28hdetl1q8yl2.cloudfront.net/assets/auth0/auth0.min-5ce8da819

In [9]:
import re

def fetch_js(url):
    # make absolute
    absu = urljoin(cv_url, url)
    return fetch_text_cached(absu, session=s)

needles = [
    "graphql", "/graphql", "/api", "majors", "Major", "cip", "Classification of Instructional Programs",
    "school", "hub/all", "agnes-scott-college",
]

for src in script_srcs[:10]:  # start small
    js = fetch_js(src)
    print("\n===", src, "chars", len(js))
    for nd in needles:
        if nd.lower() in js.lower():
            print("  HIT:", nd)
    # also extract any obvious URLs
    urls = re.findall(r"https?://[^\s\"']+", js)
    if urls:
        print("  sample URLs:", urls[:5])


=== https://js.sentry-cdn.com/4d42713cd70d4bb0973933789bbaede8.min.js chars 2839
  sample URLs: ['https://browser.sentry-cdn.com/7.120.4/bundle.es5.min.js', 'https://4d42713cd70d4bb0973933789bbaede8@o433797.ingest.us.sentry.io/5389863']

=== https://cmp.osano.com/Azyzp2TvnpgtM3kbz/02dc74f6-748a-4d06-aae6-1dda988cd583/osano.js chars 289443
  HIT: /api
  HIT: cip
  HIT: school
  sample URLs: ['https://redux.js.org/Errors?code=', 'http://www.w3.org/2000/svg', 'http://www.w3.org/2000/svg', 'https://www.osano.com/?utm_campaign=cmp&utm_source=cmp-dialog&utm_medium=drawer', 'http://www.w3.org/2000/svg']

=== https://www.googletagmanager.com/gtag/js?id=G-KT9ZYXLWZ5 chars 529006
  sample URLs: ['http://.', 'https://adservice.google.com/pagead/regclk', 'https://cct.google/taggy/agent.js', 'https://www.google.com', 'https://www.googleadservices.com']

=== https://js.sentry-cdn.com/4d42713cd70d4bb0973933789bbaede8.min.js chars 2839
  sample URLs: ['https://browser.sentry-cdn.com/7.120.4/bundle.es

In [10]:
hub_bundle = "https://d28hdetl1q8yl2.cloudfront.net/assets/src/EntryPoints/Hub/AllSchools-575a5af082dc79b1d8e092e69e7110d4dd6771eea8414bfd506aee23e4f913ec.purs"
js = fetch_text_cached(hub_bundle, session=s)

needles = [
    "graphql", "/graphql", "/api", "api.collegevine", "collegevine.com/api",
    "majors", "Major", "cip", "schools/hub", "agnes-scott-college",
    "degree", "program", "taxonomy",
]

for nd in needles:
    if nd.lower() in js.lower():
        print("HIT:", nd)

HIT: /api
HIT: majors
HIT: Major
HIT: cip
HIT: schools/hub
HIT: degree
HIT: program


In [11]:
import re
urls = sorted(set(re.findall(r"https?://[^\s\"']+", js)))
for u in urls[:200]:
    print(u)
print("total urls found:", len(urls))

http://fb.me/use-check-prop-types
http://js.pusher.com
http://www.w3.org/1998/Math/MathML
http://www.w3.org/1999/xhtml
http://www.w3.org/1999/xlink
http://www.w3.org/2000/svg
http://www.w3.org/XML/1998/namespace
https://api.mapbox.com/styles/v1/collegevine-ivy/ck8z1m02b06f41io4o6s4z5md/static/pin-s+3c8df5(
https://blog.collegevine.com/10-tips-to-improve-your-act-score/
https://blog.collegevine.com/10-tips-to-improve-your-sat-score/
https://blog.collegevine.com/category/academics/choosing-classes/
https://blog.collegevine.com/how-federal-work-study-works/
https://blog.collegevine.com/improve-your-high-school-gpa-with-these-5-strategies/
https://blog.collegevine.com/test-optional-coronavirus-policies
https://github.com/highlightjs/highlight.js/issues/2277`),gt=J,ot=Ge),Ae===void
https://github.com/highlightjs/highlight.js/wiki/security
https://github.com/pusher/pusher-js/tree/cc491015371a4bde5743d1c87a0fbac0feb53195#encrypted-channel-support
https://github.com/remarkjs/react-markdown/blo

In [12]:
paths = sorted(set(re.findall(r"\"(/[^\"\s]{3,120})\"", js)))
for p in paths:
    if "/api" in p or "graphql" in p or "major" in p or "cip" in p:
        print(p)

In [13]:
import re

api_paths = sorted(set(re.findall(r"/api/[A-Za-z0-9._~:/?#\[\]@!$&'()*+,;=%-]{1,200}", js)))
print("api_paths:", len(api_paths))
for p in api_paths[:200]:
    print(p)

api_paths: 0


In [14]:
qs = re.findall(r'"([^"\n]{1,200})"', js)  # quoted strings up to 200 chars
interesting = sorted(set(
    q for q in qs
    if any(tok in q.lower() for tok in ["api", "major", "majors", "cip", "degree", "program", "graphql"])
))
print("interesting quoted strings:", len(interesting))
for q in interesting[:300]:
    print(q)

interesting quoted strings: 51
)(j9t(a)(e.props.featuredPrograms))]),W5e(
+(GHt(Rn(e.majors))+
+[e.academicOfferings.creditForApExams.constructor.name,e.academicOfferings.creditForIbExams.constructor.name])}(),ut(e.academicOfferings.offersGradDegree)(t(
,items:e.props.programs,renderItem:function(o){return yL(
/docs/client_api_guide/client_events#trigger-events
Degree options
Enrichment Program
EnrichmentProgram
Expected encrypted event ciphertext length to be 
Failed pattern match at Mobile.Push.API.PushPermissionFailedReason (line 12, column 13 - line 18, column 16): 
Failed pattern match at Utils.API (line 103, column 44 - line 112, column 22): 
Failed pattern match at Utils.API (line 263, column 32 - line 265, column 96): 
I led an initiative that attracted 10+ participants
I led an initiative that attracted more than 1,000 participants
I led an initiative that attracted more than 100 participants
I led an initiative that attracted more than 3,500 participants
I led an initiative t

In [15]:
for tok in ["operationName", "graphql", "query:", "mutation:", "__typename"]:
    print(tok, tok in js)

operationName False
graphql False
query: False
mutation: False
__typename False


In [17]:
# Scan ALL loaded JS bundles for key API/endpoint tokens (in order)
# Assumes you already have:
#   - page_url (the CollegeVine majors page URL)
#   - html (raw HTML text fetched for page_url)
# If not, set page_url and html accordingly before running.


import re
from urllib.parse import urljoin, urldefrag, urlparse

import requests

# ---- Configure ----
TOKENS_IN_ORDER = [
    "CV.apiEndpoint",
    "apiEndpoint(",
    "Utils.API",
    "ContentModule.API",
    "graphql",
    "operationName",
    "__typename",
    "/api/",
]

# Optional: cap fetched JS size to avoid huge downloads (bytes)
MAX_JS_BYTES = 3_000_000


HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

page_url = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors" # or other school site
html = fetch_text_cached(page_url, session=requests.Session())  # or your existing fetch

def _extract_script_srcs(html_text: str, page_url: str):
    """Return de-duped absolute script src URLs found in HTML."""
    srcs = []
    for m in re.finditer(r"<script[^>]+src=['\"]([^'\"]+)['\"]", html_text, flags=re.I):
        src = m.group(1).strip()
        if not src:
            continue
        abs_url = urljoin(page_url, src)
        abs_url, _ = urldefrag(abs_url)
        srcs.append(abs_url)

    # de-dupe preserving order
    seen = set()
    out = []
    for u in srcs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def _fetch_text_limited(url: str, session: requests.Session, max_bytes: int = MAX_JS_BYTES) -> str:
    r = session.get(url, headers=HEADERS, timeout=30, allow_redirects=True, stream=True)
    r.raise_for_status()
    chunks = []
    n = 0
    for chunk in r.iter_content(chunk_size=65536):
        if not chunk:
            continue
        chunks.append(chunk)
        n += len(chunk)
        if n >= max_bytes:
            break
    data = b"".join(chunks)
    try:
        return data.decode("utf-8", errors="replace")
    except Exception:
        return data.decode("latin-1", errors="replace")

def _first_pos(s: str, needle: str) -> int:
    try:
        return s.find(needle)
    except Exception:
        return -1

def scan_scripts_for_tokens(html_text: str, page_url: str):
    """Fetch each script and scan for tokens. Print a ranked report."""
    script_urls = _extract_script_srcs(html_text, page_url)
    print(f"scripts discovered: {len(script_urls)}")
    if not script_urls:
        return []

    session = requests.Session()

    results = []
    for idx, js_url in enumerate(script_urls, 1):
        try:
            js = _fetch_text_limited(js_url, session=session)
        except Exception as e:
            results.append({
                "script_url": js_url,
                "status": "fetch_error",
                "error": repr(e),
                "size_chars": 0,
                "best_token": None,
                "best_token_pos": None,
                "hits": {},
            })
            continue

        hits = {}
        best = None  # (token_index, pos, token)
        for t_i, tok in enumerate(TOKENS_IN_ORDER):
            pos = _first_pos(js, tok)
            if pos >= 0:
                hits[tok] = pos
                if best is None or (t_i < best[0]) or (t_i == best[0] and pos < best[1]):
                    best = (t_i, pos, tok)

        results.append({
            "script_url": js_url,
            "status": "ok",
            "error": "",
            "size_chars": len(js),
            "best_token": best[2] if best else None,
            "best_token_pos": best[1] if best else None,
            "hits": hits,
        })

    # Sort: scripts that hit earlier-priority tokens first; then by token position; then by size
    def _rank(r):
        if r["status"] != "ok":
            return (999, 999999999, 999999999)
        if r["best_token"] is None:
            return (999, 999999999, r["size_chars"])
        token_rank = TOKENS_IN_ORDER.index(r["best_token"])
        return (token_rank, r["best_token_pos"] or 0, r["size_chars"])

    results_sorted = sorted(results, key=_rank)

    # Pretty print report
    print("\n=== Token scan report (sorted by earliest token in priority order) ===")
    for r in results_sorted:
        short = r["script_url"]
        if len(short) > 110:
            short = short[:107] + "..."
        if r["status"] != "ok":
            print(f"[ERR] {short}\n      {r['error']}")
            continue
        if not r["hits"]:
            print(f"[--]  {short} (chars={r['size_chars']})")
            continue

        # Print hits in token-priority order
        hit_str = ", ".join([f"{tok}@{r['hits'][tok]}" for tok in TOKENS_IN_ORDER if tok in r["hits"]])
        print(f"[OK]  {short} (chars={r['size_chars']})")
        print(f"      hits: {hit_str}")

    return results_sorted

# ---- Run ----
# Make sure `page_url` and `html` are defined in your notebook cell before running.
# Example:
# page_url = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"
# html = fetch_text_cached(page_url, session=requests.Session())  # or your existing fetch

results = scan_scripts_for_tokens(html, page_url)

# After this runs, look for scripts where the earliest hit is:
#   - CV.apiEndpoint
#   - apiEndpoint(
#   - Utils.API
# Those are your best next targets to extract actual endpoint construction.

scripts discovered: 23

=== Token scan report (sorted by earliest token in priority order) ===
[OK]  https://d28hdetl1q8yl2.cloudfront.net/assets/src/EntryPoints/Livestream/MiniPromo-6da511c6db7301687a702e534... (chars=501549)
      hits: CV.apiEndpoint@449947, Utils.API@462305
[OK]  https://d28hdetl1q8yl2.cloudfront.net/assets/src/EntryPoints/Hub/AllSchools-575a5af082dc79b1d8e092e69e7110d... (chars=3014652)
      hits: CV.apiEndpoint@749537, Utils.API@761746, ContentModule.API@1168247, graphql@2652957
[OK]  https://cmp.osano.com/Azyzp2TvnpgtM3kbz/02dc74f6-748a-4d06-aae6-1dda988cd583/osano.js (chars=289443)
      hits: /api/@270642
[OK]  https://api.mapbox.com/mapbox-gl-js/v1.8.1/mapbox-gl.js (chars=750597)
      hits: /api/@708920
[--]  https://js.hs-scripts.com/24165363.js (chars=1506)
[--]  https://d28hdetl1q8yl2.cloudfront.net/assets/performance_analytics-83186d88a07a12c063bf191d9586b54b535288c4... (chars=2540)
[--]  https://js.sentry-cdn.com/4d42713cd70d4bb0973933789bbaede8.min.js

In [18]:
import re
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept": "*/*",
}

TOKENS = [
    "CV.apiEndpoint",
    "Utils.API",
    "ContentModule.API",
    "graphql",
    "operationName",
]

BUNDLES = {
    "MiniPromo": "https://d28hdetl1q8yl2.cloudfront.net/assets/src/EntryPoints/Livestream/MiniPromo-6da511c6db7301687a702e534d8526363ff8209bb7d7d49e9969c9af9f0483dc.purs",
    "AllSchools": "https://d28hdetl1q8yl2.cloudfront.net/assets/src/EntryPoints/Hub/AllSchools-575a5af082dc79b1d8e092e69e7110d4dd6771eea8414bfd506aee23e4f913ec.purs",
}

def fetch_text(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=45)
    r.raise_for_status()
    return r.text

def find_all_positions(s: str, sub: str):
    out = []
    start = 0
    while True:
        i = s.find(sub, start)
        if i < 0:
            break
        out.append(i)
        start = i + 1
    return out

def snippet(s: str, pos: int, radius: int = 400) -> str:
    lo = max(0, pos - radius)
    hi = min(len(s), pos + radius)
    return s[lo:hi]

def extract_interesting_strings(sn: str):
    # capture quoted strings (single/double/backtick)
    qs = re.findall(r"""(['"`])((?:\\.|(?!\1).){3,500})\1""", sn)
    strs = [q[1] for q in qs]
    # filter for likely endpoint-ish or domain-ish strings
    keep = []
    for x in strs:
        xl = x.lower()
        if any(k in xl for k in ["/api", "graphql", "major", "majors", "cip", "degree", "program", "schools/hub", "contentmodule"]):
            keep.append(x)
        elif re.search(r"https?://", x):
            keep.append(x)
    # de-dupe preserve order
    seen = set()
    out = []
    for x in keep:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out[:60]

def scan_bundle(name: str, url: str):
    js = fetch_text(url)
    print(f"\n==== {name} ({len(js)} chars) ====")
    for tok in TOKENS:
        poses = find_all_positions(js, tok)
        if not poses:
            continue
        print(f"\n-- token '{tok}' hits: {len(poses)} (showing up to 3) --")
        for p in poses[:3]:
            sn = snippet(js, p, radius=450)
            print(f"\n[{tok}] @ {p}")
            print(sn.replace("\n", " ")[:1200])  # keep it readable
            interesting = extract_interesting_strings(sn)
            if interesting:
                print("  interesting strings:")
                for z in interesting[:25]:
                    print("   -", z)

for nm, u in BUNDLES.items():
    scan_bundle(nm, u)


==== MiniPromo (501552 chars) ====

-- token 'CV.apiEndpoint' hits: 3 (showing up to 3) --

[CV.apiEndpoint] @ 449950
tartTime:bT(mi)(Pe(e.streamingStartTime)),polls:PA(e.polls)(function(n){return{poll:Uh(n.poll),status:Op(n.status)}}),presenter:OA(pT)(Pe(e.presenter)),partnerType:Pe(e.partnerType),guestPresenter:UA(Pe(e.guestPresenter))(iT),zoomJoinUrl:Pe(e.zoomJoinUrl),zoomStartUrl:Pe(e.zoomStartUrl)}};var HA=typeof CV!="undefined"&&CV.pathInfo||typeof global!="undefined"&&global.CV&&global.CV.pathInfo||(e=>null);function CT(e,n){let r=typeof CV!="undefined"&&CV.apiEndpoint||typeof global!="undefined"&&global.CV&&global.CV.apiEndpoint;return r||console.error("Missing `CV.apiEndpoint` function"),r&&r(e,n)}function wT(e,n,r,i,l){return function(p,T){var d=e.newXHR(),h=e.fixupUrl(l.url,d);if(d.open(l.method||"GET",h,!0,l.username,l.password),l.headers)try{for(var P=0,y;(y=l.headers[P])!=null;P++)d.setRequestHeader(y.field,y.value)}catch(M){p(M)}var E=function(M){return function(){p(new

In [19]:
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept": "text/html,*/*",
}

def fetch(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=45)
    r.raise_for_status()
    return r.text

def show_hits(label: str, text: str, patterns, radius=350, max_hits_each=3):
    print(f"\n==== {label} ====")
    for pat in patterns:
        ms = list(re.finditer(pat, text))
        if not ms:
            continue
        print(f"\n-- pattern: {pat} | hits={len(ms)} (showing up to {max_hits_each}) --")
        for m in ms[:max_hits_each]:
            i = m.start()
            lo = max(0, i - radius)
            hi = min(len(text), i + radius)
            sn = text[lo:hi].replace("\n", " ")
            print(f"[{i}] {sn[:1200]}")

html = fetch(PAGE_URL)
soup = BeautifulSoup(html, "html.parser")

# 1) Scan inline scripts in HTML (often where CV.pathInfo is injected)
inline = "\n".join(
    (sc.string or sc.get_text(" ", strip=True) or "")
    for sc in soup.find_all("script")
    if not sc.get("src")
)

html_patterns = [
    r"\bCV\.pathInfo\b",
    r"\bpathInfo\b\s*[:=]",
    r"\bCV\.apiEndpoint\b",
    r"\bapiEndpoint\b\s*[:=]",
    r"\bapiHost\b|\bapiBase\b|\bapiUrl\b",
    r"schools_content_modules_(presence|get)_path",
]
show_hits("INLINE SCRIPT SEARCH (HTML)", inline, html_patterns)

# 2) Gather external scripts and scan each for actual definitions
script_srcs = []
for sc in soup.find_all("script"):
    src = sc.get("src")
    if src:
        script_srcs.append(urljoin(PAGE_URL, src))

print(f"\nexternal scripts discovered: {len(script_srcs)}")

js_patterns = [
    # look for real definitions
    r"CV\.apiEndpoint\s*=\s*function",
    r"apiEndpoint\s*:\s*function",
    r"apiEndpoint\s*=\s*\(",
    r"CV\.pathInfo\s*=",
    r"pathInfo\s*:\s*\{",
    r"schools_content_modules_(presence|get)_path",
    r"schools/hub",
    r"/api/",
]

for u in script_srcs:
    try:
        js = fetch(u)
    except Exception as e:
        print(f"[skip] {u} | {type(e).__name__}: {e}")
        continue

    # Only print if it contains something relevant
    if not any(re.search(p, js) for p in js_patterns):
        continue

    print(f"\n==== SCRIPT: {u} | chars={len(js)} ====")
    show_hits("SCRIPT HIT CONTEXT", js, js_patterns, radius=300, max_hits_each=2)


==== INLINE SCRIPT SEARCH (HTML) ====

-- pattern: \bCV\.pathInfo\b | hits=6 (showing up to 3) --
[7233] nrfE3LkUSGnjwjOOc2hsfLtaC');   }}();     if (CV.tracking) {     CV.tracking.pageView()   }     if (CV.currentUser) {          if (CV.tracking) {       CV.tracking.identify(CV.currentUser.cvid)     }   }     performance.mark('js:start')   performance.mark('js:lib:start')     performance.mark('js:lib:end')     window.CV = window.CV || {}      window.CV.pathInfo = function(name) {     const path = window.CV.pathInfo.endpoints[name]     if (!path) {       window.CV.pathInfo.missingPath(name)       return     }        return { path, token: 'RmaMvoTnM1iuxcWsLcttapudJROD2i0gLIDe29Op_wNrYgMw7Z071QLgp3y4nkAv-SM7pm_00UjZ72i1HDfP1A' }   }      window.CV.pathInfo.endpoints = {"autopilot_pre_join_path":"
[7288] g) {     CV.tracking.pageView()   }     if (CV.currentUser) {          if (CV.tracking) {       CV.tracking.identify(CV.currentUser.cvid)     }   }     performance.mark('js:start')   per

In [20]:
import re, json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def fetch_text(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=45)
    r.raise_for_status()
    return r.text

def extract_cv_bootstrap(html: str):
    soup = BeautifulSoup(html, "html.parser")
    inline = "\n".join(
        (sc.string or sc.get_text(" ", strip=True) or "")
        for sc in soup.find_all("script")
        if not sc.get("src")
    )

    # endpoints object: window.CV.pathInfo.endpoints = {...}
    m_end = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*(?:;|\n)", inline, flags=re.S)
    if not m_end:
        raise RuntimeError("Could not find window.CV.pathInfo.endpoints JSON in inline scripts.")
    endpoints_json = m_end.group(1)

    # token inside: token: '....'
    m_tok = re.search(r"token:\s*'([^']+)'", inline)
    token = m_tok.group(1) if m_tok else ""

    endpoints = json.loads(endpoints_json)
    return endpoints, token, inline

def find_school_ids_or_slugs(html: str):
    # best-effort heuristics: often there is a school id embedded somewhere
    # We'll return any obvious integer ids or slug hints we can find.
    hits = {}
    # common patterns
    for pat in [
        r'"school_id"\s*:\s*(\d+)',
        r'"schoolId"\s*:\s*(\d+)',
        r'\bschool_id\s*=\s*(\d+)',
        r'\bschoolId\s*=\s*(\d+)',
    ]:
        ms = re.findall(pat, html)
        if ms:
            hits[pat] = sorted(set(ms))[:10]
    return hits

html = fetch_text(PAGE_URL)
endpoints, token, inline = extract_cv_bootstrap(html)

print("token present:", bool(token), "token_len:", len(token))
print("presence path:", endpoints.get("schools_content_modules_presence_path"))
print("get path:", endpoints.get("schools_content_modules_get_path"))

id_hints = find_school_ids_or_slugs(html)
print("school id hints:", id_hints)

BASE = "https://www.collegevine.com"
presence_url = urljoin(BASE, endpoints["schools_content_modules_presence_path"])
get_url = urljoin(BASE, endpoints["schools_content_modules_get_path"])

sess = requests.Session()
sess.headers.update({
    **HEADERS,
    "Accept": "application/json,*/*",
    "Referer": PAGE_URL,
})

# We don't know the exact param schema yet.
# We'll try a few common conventions for presence:
presence_param_sets = [
    {"school_slug": "agnes-scott-college"},
    {"slug": "agnes-scott-college"},
    {"school": "agnes-scott-college"},
    {"d": "agnes-scott-college"},  # path includes /d/<slug>/
]

# If we found a numeric school_id, try that too
for pats, vals in id_hints.items():
    for v in vals:
        presence_param_sets.append({"school_id": v})
        presence_param_sets.append({"schoolId": v})

def try_get_json(url: str, params: dict):
    r = sess.get(url, params=params, timeout=45)
    # show status + a tiny snippet
    print("GET", r.url, "=>", r.status_code, "len", len(r.text))
    if r.status_code != 200:
        return None, r
    try:
        return r.json(), r
    except Exception:
        return None, r

presence_json = None
presence_resp = None
for params in presence_param_sets:
    j, r = try_get_json(presence_url, params=params)
    if j is not None and isinstance(j, dict) and len(j) > 0:
        presence_json, presence_resp = j, r
        print("✅ presence succeeded with params:", params)
        break

if presence_json is None:
    print("❌ Could not get JSON from presence with tried param sets.")
else:
    # Search presence response for likely module ids / names
    presence_txt = json.dumps(presence_json)
    print("presence keys:", list(presence_json.keys())[:30])

    # crude extraction of candidate module ids
    module_ids = sorted(set(re.findall(r'"moduleId"\s*:\s*(\d+)', presence_txt)))
    if not module_ids:
        module_ids = sorted(set(re.findall(r'"module_id"\s*:\s*(\d+)', presence_txt)))
    print("module_ids found:", module_ids[:30], "count", len(module_ids))

    # Also look for “DegreeOptions” / “majors” signals
    for needle in ["DegreeOptions", "degree", "majors", "programs", "cipCode"]:
        print(needle, "=>", needle.lower() in presence_txt.lower())

    # Try fetching a small number of module ids (if any)
    targets = [
        "African Studies",
        "Classics and Classical Languages, Literatures, and Linguistics, General",
    ]

    def contains_targets(obj) -> dict:
        txt = json.dumps(obj)
        return {t: (t in txt) for t in targets}

    got_any = False
    for mid in module_ids[:25]:
        # try common schemas for /get
        for params in [
            {"module_id": mid},
            {"moduleId": mid},
            {"id": mid},
        ]:
            j, r = try_get_json(get_url, params=params)
            if j is None:
                continue
            flags = contains_targets(j)
            print("module", mid, "targets:", flags)
            if any(flags.values()):
                got_any = True
                print("🎯 HIT module", mid, "with params", params)
                break
        if got_any:
            break

    if not got_any:
        print("No direct target hits found in first batch of module_ids.")

token present: True token_len: 86
presence path: /schools/content-modules/presence
get path: /schools/content-modules/get
school id hints: {}
GET https://www.collegevine.com/schools/content-modules/presence?school_slug=agnes-scott-college => 404 len 7010
GET https://www.collegevine.com/schools/content-modules/presence?slug=agnes-scott-college => 404 len 7009
GET https://www.collegevine.com/schools/content-modules/presence?school=agnes-scott-college => 404 len 7014
GET https://www.collegevine.com/schools/content-modules/presence?d=agnes-scott-college => 404 len 7015
❌ Could not get JSON from presence with tried param sets.


In [21]:
import re, json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"

UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
BASE = "https://www.collegevine.com"

def fetch_html(session: requests.Session, url: str) -> str:
    r = session.get(url, headers={"User-Agent": UA, "Accept": "text/html,*/*"}, timeout=45)
    r.raise_for_status()
    return r.text

def extract_bootstrap(html: str):
    soup = BeautifulSoup(html, "html.parser")

    csrf = ""
    m = soup.find("meta", attrs={"name": "csrf-token"})
    if m and m.get("content"):
        csrf = m["content"].strip()

    inline = "\n".join(
        (sc.string or sc.get_text(" ", strip=True) or "")
        for sc in soup.find_all("script")
        if not sc.get("src")
    )

    m_end = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*(?:;|\n)", inline, flags=re.S)
    if not m_end:
        raise RuntimeError("Could not find window.CV.pathInfo.endpoints JSON")

    endpoints = json.loads(m_end.group(1))

    m_tok = re.search(r"token:\s*'([^']+)'", inline)
    pathinfo_token = m_tok.group(1) if m_tok else ""

    return endpoints, csrf, pathinfo_token, inline

def sniff(resp: requests.Response, label: str):
    txt = resp.text or ""
    snippet = txt[:300].replace("\n", " ")
    print(f"{label}: {resp.status_code} len={len(txt)} ct={resp.headers.get('Content-Type','')}")
    print("  snippet:", snippet[:300])

def probe(session: requests.Session, url: str, method: str, params=None, json_body=None, headers=None):
    headers = headers or {}
    if method == "GET":
        r = session.get(url, params=params, headers=headers, timeout=45)
    else:
        r = session.post(url, params=params, json=json_body, headers=headers, timeout=45)
    return r

# 1) Establish cookies by loading the hub page
sess = requests.Session()
html = fetch_html(sess, PAGE_URL)

endpoints, csrf_token, pathinfo_token, inline = extract_bootstrap(html)
print("csrf_token present:", bool(csrf_token), "len:", len(csrf_token))
print("pathInfo token present:", bool(pathinfo_token), "len:", len(pathinfo_token))
print("presence path:", endpoints.get("schools_content_modules_presence_path"))
print("get path:", endpoints.get("schools_content_modules_get_path"))

presence_path = endpoints["schools_content_modules_presence_path"]
get_path = endpoints["schools_content_modules_get_path"]

# 2) Candidate URL variants
def variants(path: str):
    out = []
    for p in [path, path + ".json"]:
        out.append(urljoin(BASE, p))
        out.append(urljoin(BASE, "/api" + p))
        out.append(urljoin(BASE, "/api/v1" + p))
    # de-dupe preserving order
    seen = set()
    uniq = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq

presence_urls = variants(presence_path)
get_urls = variants(get_path)

# 3) Headers: behave like XHR
xhr_headers = {
    "User-Agent": UA,
    "Accept": "application/json,*/*",
    "Referer": PAGE_URL,
    "X-Requested-With": "XMLHttpRequest",
}
# include CSRF if present
if csrf_token:
    xhr_headers["X-CSRF-Token"] = csrf_token

# 4) Probe presence with GET and POST using a few bodies
presence_payloads = [
    {"school_slug": "agnes-scott-college"},
    {"slug": "agnes-scott-college"},
    {"school": "agnes-scott-college"},
    {"d": "agnes-scott-college"},
    # likely real param (seen in bundle strings)
    {"first_school_id": 0},  # intentionally wrong to see if we get 422 instead of 404
]

print("\n=== PROBE presence variants ===")
for u in presence_urls:
    # GET (params)
    r = probe(sess, u, "GET", params={"school_slug": "agnes-scott-college"}, headers=xhr_headers)
    sniff(r, f"GET {u}")
    # POST (JSON bodies)
    for body in presence_payloads[:3]:
        r2 = probe(sess, u, "POST", json_body=body, headers={**xhr_headers, "Content-Type": "application/json"})
        sniff(r2, f"POST {u} body={list(body.keys())}")
    print()

print("\n=== If we get a non-404 (422/401/403/200), stop and inspect that response ===")

csrf_token present: True len: 86
pathInfo token present: True len: 86
presence path: /schools/content-modules/presence
get path: /schools/content-modules/get

=== PROBE presence variants ===
GET https://www.collegevine.com/schools/content-modules/presence: 404 len=6095 ct=text/html; charset=utf-8
  snippet: <html lang='en'> <head> <meta content='text/html; charset=UTF-8' http-equiv='Content-Type'> <meta content='width=device-width, initial-scale=1, shrink-to-fit=no' name='viewport'> <meta content='noindex,follow' name='robots'> <title>404 Page Not Found | CollegeVine</title> <link rel="stylesheet" href
POST https://www.collegevine.com/schools/content-modules/presence body=['school_slug']: 200 len=976 ct=application/json; charset=utf-8
  snippet: {"presence":["cbec9f47-7998-4a25-9d9c-bc729ce4c367","cbab31ed-d6cb-4e74-8bb2-1f0e75e1a0a5","dc1ed90d-7438-42db-93bb-827c1db3e7fa","6bb355f1-7d26-44d3-8246-56677fb4e20b","56d6e132-2d76-48dd-9f2a-4873bf7e8d01","1b33c3cf-3b3c-48cd-b425-7739a716e73

In [22]:
import json, re
import requests
from bs4 import BeautifulSoup

PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
BASE = "https://www.collegevine.com"
SLUG = "agnes-scott-college"

def bootstrap(session):
    html = session.get(PAGE_URL, headers={"User-Agent": UA, "Accept":"text/html,*/*"}, timeout=45).text
    soup = BeautifulSoup(html, "html.parser")

    csrf = soup.find("meta", attrs={"name":"csrf-token"})["content"].strip()

    inline = "\n".join(
        (sc.string or sc.get_text(" ", strip=True) or "")
        for sc in soup.find_all("script")
        if not sc.get("src")
    )
    m_end = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*(?:;|\n)", inline, flags=re.S)
    endpoints = json.loads(m_end.group(1))
    return endpoints, csrf

def post_json(session, path, csrf, body):
    url = BASE + path
    headers = {
        "User-Agent": UA,
        "Accept": "application/json,*/*",
        "Referer": PAGE_URL,
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-Token": csrf,
        "Content-Type": "application/json",
    }
    r = session.post(url, headers=headers, json=body, timeout=45)
    return r

sess = requests.Session()
endpoints, csrf = bootstrap(sess)

presence_path = endpoints["schools_content_modules_presence_path"]
get_path      = endpoints["schools_content_modules_get_path"]

# 1) presence
r = post_json(sess, presence_path, csrf, {"school_slug": SLUG})
print("presence:", r.status_code, r.headers.get("Content-Type",""), "len", len(r.text))
presence = r.json()["presence"]
print("presence modules:", len(presence))
print("first few:", presence[:5])

# 2) try fetching a few modules via /get
# We'll sample modules and search for keywords that indicate majors/programs.
KEYWORDS = ["major", "majors", "program", "programs", "cip", "degree", "areaOfStudy", "African Studies", "Classics"]

def summarize_module(obj):
    s = json.dumps(obj, ensure_ascii=False)
    hits = [k for k in KEYWORDS if k.lower() in s.lower()]
    # try to extract a likely "type" field if present
    mod_type = None
    if isinstance(obj, dict):
        for k in ["type", "moduleType", "name", "key", "slug"]:
            if k in obj and isinstance(obj[k], str):
                mod_type = obj[k]
                break
    return mod_type, hits, len(s)

def fetch_one(module_id):
    r = post_json(sess, get_path, csrf, {"module_id": module_id})
    return r

print("\n--- probing modules (first 25) ---")
candidates = []
for mid in presence[:25]:
    rr = fetch_one(mid)
    if rr.status_code != 200:
        print("get", mid, "->", rr.status_code)
        continue
    obj = rr.json()
    mod_type, hits, sz = summarize_module(obj)
    if hits:
        candidates.append((mid, mod_type, hits, sz))
        print("HIT", mid, "type=", mod_type, "hits=", hits, "json_chars=", sz)

print("\nCandidates found:", len(candidates))
print("Top candidates:")
for row in candidates[:10]:
    print(row)

# 3) If we have candidates, pull the best one and try to extract majors list
def find_strings(obj, min_len=4):
    out = []
    if isinstance(obj, dict):
        for v in obj.values():
            out.extend(find_strings(v, min_len))
    elif isinstance(obj, list):
        for v in obj:
            out.extend(find_strings(v, min_len))
    elif isinstance(obj, str) and len(obj) >= min_len:
        out.append(obj)
    return out

if candidates:
    best = candidates[0][0]
    rr = fetch_one(best)
    data = rr.json()
    strings = find_strings(data, min_len=4)

    # keep only "program-ish" strings (heuristic)
    keep = []
    for s in strings:
        if any(x in s.lower() for x in ["studies", "languages", "literatures", "linguistics", "major", "program", "classics", "african"]):
            keep.append(s)

    keep_unique = []
    seen = set()
    for s in keep:
        if s not in seen:
            seen.add(s)
            keep_unique.append(s)

    print("\n--- extracted candidate strings (unique, first 80) ---")
    for s in keep_unique[:80]:
        print(s)
else:
    print("\nNo keyword-bearing modules in first 25; increase the probe window or change heuristics.")

presence: 200 application/json; charset=utf-8 len 976
presence modules: 24
first few: ['cbec9f47-7998-4a25-9d9c-bc729ce4c367', 'cbab31ed-d6cb-4e74-8bb2-1f0e75e1a0a5', 'dc1ed90d-7438-42db-93bb-827c1db3e7fa', '6bb355f1-7d26-44d3-8246-56677fb4e20b', '56d6e132-2d76-48dd-9f2a-4873bf7e8d01']

--- probing modules (first 25) ---

Candidates found: 0
Top candidates:

No keyword-bearing modules in first 25; increase the probe window or change heuristics.


In [23]:
import json, re, math
import requests
from bs4 import BeautifulSoup
from collections import Counter

PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
BASE = "https://www.collegevine.com"
SLUG = "agnes-scott-college"

def bootstrap(session):
    html = session.get(PAGE_URL, headers={"User-Agent": UA, "Accept":"text/html,*/*"}, timeout=45).text
    soup = BeautifulSoup(html, "html.parser")
    csrf = soup.find("meta", attrs={"name":"csrf-token"})["content"].strip()

    inline = "\n".join(
        (sc.string or sc.get_text(" ", strip=True) or "")
        for sc in soup.find_all("script")
        if not sc.get("src")
    )
    m_end = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*(?:;|\n)", inline, flags=re.S)
    endpoints = json.loads(m_end.group(1))
    return endpoints, csrf

def post_json(session, path, csrf, body):
    url = BASE + path
    headers = {
        "User-Agent": UA,
        "Accept": "application/json,*/*",
        "Referer": PAGE_URL,
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-Token": csrf,
        "Content-Type": "application/json",
    }
    r = session.post(url, headers=headers, json=body, timeout=45)
    return r

def iter_strings(obj):
    """Yield all strings found anywhere in a nested JSON-like object."""
    if isinstance(obj, dict):
        for v in obj.values():
            yield from iter_strings(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_strings(v)
    elif isinstance(obj, str):
        yield obj

def is_humanish(s: str) -> bool:
    s = s.strip()
    if len(s) < 4: 
        return False
    # reject UUIDs, hashes, purely numeric-ish
    if re.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", s, re.I):
        return False
    if re.fullmatch(r"[0-9a-f]{32,}", s, re.I):
        return False
    if re.fullmatch(r"\d+(\.\d+)?", s):
        return False
    # require at least some letters
    return any(c.isalpha() for c in s)

def summarize_payload(obj):
    s_all = list(iter_strings(obj))
    s_h = [x for x in s_all if is_humanish(x)]
    # keep “major-looking” strings separately
    s_majorish = [x for x in s_h if re.search(r"(studies|science|sciences|engineering|literature|languages|linguistics|classics|history|philosophy|math|biology|chemistry|physics|economics|business|psychology|sociology|art|music|theatre|communications|education)", x, re.I)]
    return {
        "json_chars": len(json.dumps(obj, ensure_ascii=False)),
        "strings_all": len(s_all),
        "strings_human": len(s_h),
        "strings_human_uniq": len(set(s_h)),
        "strings_majorish": len(s_majorish),
        "sample_human": list(dict.fromkeys(s_h))[:25],  # preserve order, first 25 unique
    }

sess = requests.Session()
endpoints, csrf = bootstrap(sess)

presence_path = endpoints["schools_content_modules_presence_path"]
get_path      = endpoints["schools_content_modules_get_path"]

# 1) presence
rp = post_json(sess, presence_path, csrf, {"school_slug": SLUG})
presence_json = rp.json()
presence = presence_json["presence"]
print("presence keys:", list(presence_json.keys()))
print("presence modules:", len(presence))

# 2) try multiple /get bodies per module_id, and keep the “best looking” response
GET_BODIES = [
    lambda mid: {"module_id": mid},
    lambda mid: {"module_id": mid, "school_slug": SLUG},
    lambda mid: {"module_id": mid, "slug": SLUG},
    lambda mid: {"module_id": mid, "school": SLUG},
]

results = []
for mid in presence:
    best = None
    best_sig = None
    best_body = None
    best_status = None

    for make_body in GET_BODIES:
        body = make_body(mid)
        r = post_json(sess, get_path, csrf, body)
        status = r.status_code

        # sometimes endpoints return HTML error pages with 200; guard
        ct = (r.headers.get("Content-Type") or "").lower()
        if "json" not in ct:
            continue

        try:
            obj = r.json()
        except Exception:
            continue

        sig = summarize_payload(obj)

        # pick the response with the most “human strings”
        score = (sig["strings_human_uniq"], sig["strings_human"], sig["strings_majorish"], sig["json_chars"])
        if best_sig is None or score > (best_sig["strings_human_uniq"], best_sig["strings_human"], best_sig["strings_majorish"], best_sig["json_chars"]):
            best = obj
            best_sig = sig
            best_body = body
            best_status = status

    if best_sig is None:
        results.append((mid, None, None, None))
    else:
        results.append((mid, best_status, best_body, best_sig))

# 3) rank modules by “human string richness”
ranked = [r for r in results if r[3] is not None]
ranked.sort(key=lambda x: (x[3]["strings_human_uniq"], x[3]["strings_majorish"], x[3]["json_chars"]), reverse=True)

print("\n--- top modules by human-readable content ---")
for mid, status, body, sig in ranked[:10]:
    print(f"\nMODULE {mid} | status={status} | body={body}")
    print(f"  json_chars={sig['json_chars']}  strings_all={sig['strings_all']}  human={sig['strings_human']}  human_uniq={sig['strings_human_uniq']}  majorish={sig['strings_majorish']}")
    if sig["sample_human"]:
        print("  sample_human:", sig["sample_human"][:12])

# 4) targeted search across the best modules’ extracted strings
TARGETS = [
    "African Studies",
    "Classics and Classical Languages, Literatures, and Linguistics, General"
]

print("\n--- targeted search in top-ranked modules ---")
for mid, status, body, sig in ranked[:8]:
    joined = "\n".join(sig["sample_human"])
    hits = {t: (t.lower() in joined.lower()) for t in TARGETS}
    print(mid, hits)

presence keys: ['presence', 'firstSchoolModules']
presence modules: 24

--- top modules by human-readable content ---

MODULE cbec9f47-7998-4a25-9d9c-bc729ce4c367 | status=200 | body={'module_id': 'cbec9f47-7998-4a25-9d9c-bc729ce4c367'}
  json_chars=4  strings_all=0  human=0  human_uniq=0  majorish=0

MODULE cbab31ed-d6cb-4e74-8bb2-1f0e75e1a0a5 | status=200 | body={'module_id': 'cbab31ed-d6cb-4e74-8bb2-1f0e75e1a0a5'}
  json_chars=4  strings_all=0  human=0  human_uniq=0  majorish=0

MODULE dc1ed90d-7438-42db-93bb-827c1db3e7fa | status=200 | body={'module_id': 'dc1ed90d-7438-42db-93bb-827c1db3e7fa'}
  json_chars=4  strings_all=0  human=0  human_uniq=0  majorish=0

MODULE 6bb355f1-7d26-44d3-8246-56677fb4e20b | status=200 | body={'module_id': '6bb355f1-7d26-44d3-8246-56677fb4e20b'}
  json_chars=4  strings_all=0  human=0  human_uniq=0  majorish=0

MODULE 56d6e132-2d76-48dd-9f2a-4873bf7e8d01 | status=200 | body={'module_id': '56d6e132-2d76-48dd-9f2a-4873bf7e8d01'}
  json_chars=4  strings_all

In [24]:
import json, re
import requests
from bs4 import BeautifulSoup
from collections import Counter

PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
BASE = "https://www.collegevine.com"
SLUG = "agnes-scott-college"

def bootstrap(session):
    html = session.get(PAGE_URL, headers={"User-Agent": UA, "Accept":"text/html,*/*"}, timeout=45).text
    soup = BeautifulSoup(html, "html.parser")
    csrf = soup.find("meta", attrs={"name":"csrf-token"})["content"].strip()

    inline = "\n".join(
        (sc.string or sc.get_text(" ", strip=True) or "")
        for sc in soup.find_all("script")
        if not sc.get("src")
    )
    m_end = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*(?:;|\n)", inline, flags=re.S)
    endpoints = json.loads(m_end.group(1))
    return endpoints, csrf, html

def post_json(session, path, csrf, body):
    url = BASE + path
    headers = {
        "User-Agent": UA,
        "Accept": "application/json,*/*",
        "Referer": PAGE_URL,
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-Token": csrf,
        "Content-Type": "application/json",
    }
    r = session.post(url, headers=headers, json=body, timeout=45)
    return r

def iter_strings(obj):
    if isinstance(obj, dict):
        for v in obj.values():
            yield from iter_strings(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_strings(v)
    elif isinstance(obj, str):
        yield obj

def is_humanish(s: str) -> bool:
    s = s.strip()
    if len(s) < 4:
        return False
    if re.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", s, re.I):
        return False
    if re.fullmatch(r"[0-9a-f]{32,}", s, re.I):
        return False
    if re.fullmatch(r"\d+(\.\d+)?", s):
        return False
    return any(c.isalpha() for c in s)

def summarize_payload(obj):
    s_all = list(iter_strings(obj))
    s_h = [x for x in s_all if is_humanish(x)]
    s_majorish = [x for x in s_h if re.search(r"(studies|science|sciences|engineering|literature|languages|linguistics|classics|history|philosophy|math|biology|chemistry|physics|economics|business|psychology|sociology|art|music|theatre|communications|education)", x, re.I)]
    return {
        "json_chars": len(json.dumps(obj, ensure_ascii=False)),
        "strings_all": len(s_all),
        "strings_human": len(s_h),
        "strings_human_uniq": len(set(s_h)),
        "strings_majorish": len(s_majorish),
        "sample_human": list(dict.fromkeys(s_h))[:25],
    }

sess = requests.Session()
endpoints, csrf, html = bootstrap(sess)

presence_path = endpoints["schools_content_modules_presence_path"]
get_path      = endpoints["schools_content_modules_get_path"]

# presence
rp = post_json(sess, presence_path, csrf, {"school_slug": SLUG})
pres = rp.json()
presence_ids = pres.get("presence", [])
first_school_modules = pres.get("firstSchoolModules", None)

print("presence modules:", len(presence_ids))
print("has firstSchoolModules:", first_school_modules is not None)

# 1) Inspect firstSchoolModules structure (this matters!)
print("\n=== firstSchoolModules (shape preview) ===")
if isinstance(first_school_modules, dict):
    print("dict keys:", list(first_school_modules.keys())[:30])
elif isinstance(first_school_modules, list):
    print("list len:", len(first_school_modules))
    if len(first_school_modules) > 0:
        print("first item keys:", list(first_school_modules[0].keys()) if isinstance(first_school_modules[0], dict) else type(first_school_modules[0]))
else:
    print("type:", type(first_school_modules))

# 2) Normalize candidate module records out of firstSchoolModules.
# We don't know the exact schema, so we extract any dicts that look like modules.
candidates = []

def harvest_modules(x):
    if isinstance(x, dict):
        # heuristic: treat dict as a module record if it has module-ish keys
        keys = set(x.keys())
        if {"module_id","moduleId"} & keys or {"type","typ"} & keys:
            candidates.append(x)
        for v in x.values():
            harvest_modules(v)
    elif isinstance(x, list):
        for v in x:
            harvest_modules(v)

harvest_modules(first_school_modules)

print("\n=== harvested candidate module records:", len(candidates), "===")
for i, rec in enumerate(candidates[:8]):
    print(f"[{i}] keys:", list(rec.keys())[:20])

# 3) Try /get with module_id PLUS type (and any school id if present)
def pick(*names):
    for n in names:
        if n in rec and rec[n] is not None:
            return rec[n]
    return None

results = []

for rec in candidates:
    mid  = pick("module_id","moduleId","id")
    typ  = pick("type","typ","module_type","moduleType")
    sid  = pick("school_id","schoolId","first_school_id","firstSchoolId")

    if not mid:
        continue

    bodies = []
    # base attempts
    bodies.append({"module_id": mid})
    if typ: bodies.append({"module_id": mid, "type": typ})
    if sid: bodies.append({"module_id": mid, "school_id": sid})
    if typ and sid: bodies.append({"module_id": mid, "type": typ, "school_id": sid})

    # also try slug variants (sometimes required)
    if typ: bodies.append({"module_id": mid, "type": typ, "school_slug": SLUG})
    if typ: bodies.append({"module_id": mid, "type": typ, "slug": SLUG})

    best = None
    best_sig = None
    best_body = None
    best_status = None
    best_ct = None

    for body in bodies:
        r = post_json(sess, get_path, csrf, body)
        ct = (r.headers.get("Content-Type") or "").lower()
        if "json" not in ct:
            continue
        try:
            obj = r.json()
        except Exception:
            continue

        sig = summarize_payload(obj)
        score = (sig["strings_human_uniq"], sig["strings_majorish"], sig["strings_human"], sig["json_chars"])
        if best_sig is None or score > (best_sig["strings_human_uniq"], best_sig["strings_majorish"], best_sig["strings_human"], best_sig["json_chars"]):
            best, best_sig, best_body, best_status, best_ct = obj, sig, body, r.status_code, ct

    results.append((mid, typ, sid, best_status, best_body, best_sig))

# 4) Rank by content
ranked = [r for r in results if r[5] is not None]
ranked.sort(key=lambda x: (x[5]["strings_human_uniq"], x[5]["strings_majorish"], x[5]["json_chars"]), reverse=True)

print("\n=== TOP /get results by content ===")
for mid, typ, sid, status, body, sig in ranked[:10]:
    print(f"\nmodule_id={mid} type={typ} school_id={sid} status={status}")
    print("  body:", body)
    print(f"  json_chars={sig['json_chars']} human_uniq={sig['strings_human_uniq']} majorish={sig['strings_majorish']}")
    if sig["sample_human"]:
        print("  sample_human:", sig["sample_human"][:12])

presence modules: 24
has firstSchoolModules: False

=== firstSchoolModules (shape preview) ===
type: <class 'NoneType'>

=== harvested candidate module records: 0 ===

=== TOP /get results by content ===


In [25]:
import json, re, requests
from bs4 import BeautifulSoup

PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"
UA = "Mozilla/5.0"
BASE = "https://www.collegevine.com"
SLUG = "agnes-scott-college"

def bootstrap(session):
    r = session.get(PAGE_URL, headers={"User-Agent": UA}, timeout=45)
    html = r.text
    soup = BeautifulSoup(html, "html.parser")
    csrf = soup.find("meta", attrs={"name":"csrf-token"})["content"].strip()

    inline = "\n".join(
        (sc.string or sc.get_text(" ", strip=True) or "")
        for sc in soup.find_all("script")
        if not sc.get("src")
    )

    # endpoints dict
    m_end = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*(?:;|\n)", inline, flags=re.S)
    endpoints = json.loads(m_end.group(1))

    # pathInfo token (the one returned inside CV.pathInfo)
    m_tok = re.search(r"return\s*\{\s*path\s*,\s*token\s*:\s*'([^']+)'\s*\}", inline, flags=re.S)
    pathinfo_token = m_tok.group(1) if m_tok else None

    return endpoints, csrf, pathinfo_token

def post_presence(session, path, csrf, data=None, json_body=None):
    url = BASE + path
    headers = {
        "User-Agent": UA,
        "Accept": "application/json,*/*",
        "Referer": PAGE_URL,
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-Token": csrf,
    }
    if json_body is not None:
        headers["Content-Type"] = "application/json"
        r = session.post(url, headers=headers, json=json_body, timeout=45)
    else:
        # form encoding
        r = session.post(url, headers=headers, data=data, timeout=45)
    return r

sess = requests.Session()
endpoints, csrf, pitok = bootstrap(sess)

presence_path = endpoints["schools_content_modules_presence_path"]
print("presence_path:", presence_path)
print("csrf len:", len(csrf), "pathInfo token present:", pitok is not None, "len:", len(pitok or ""))

variants = [
    ("JSON basic",      None, {"school_slug": SLUG}),
    ("FORM basic",      {"school_slug": SLUG}, None),
    ("FORM slug",       {"slug": SLUG}, None),
    ("FORM school",     {"school": SLUG}, None),
]

# Try adding pathInfo token as a param too (in case it's required)
if pitok:
    variants += [
        ("FORM +token", {"school_slug": SLUG, "token": pitok}, None),
        ("JSON +token", None, {"school_slug": SLUG, "token": pitok}),
    ]

for name, form_data, json_body in variants:
    r = post_presence(sess, presence_path, csrf, data=form_data, json_body=json_body)
    ct = (r.headers.get("Content-Type") or "").lower()
    print(f"\n== {name} == status={r.status_code} ct={ct} len={len(r.text)}")
    if "json" in ct:
        obj = r.json()
        print("keys:", list(obj.keys()))
        print("presence:", len(obj.get("presence", [])))
        print("firstSchoolModules type:", type(obj.get("firstSchoolModules", None)))
    else:
        print("snippet:", r.text[:200])

presence_path: /schools/content-modules/presence
csrf len: 86 pathInfo token present: True len: 86

== JSON basic == status=200 ct=application/json; charset=utf-8 len=976
keys: ['presence', 'firstSchoolModules']
presence: 24
firstSchoolModules type: <class 'NoneType'>

== FORM basic == status=200 ct=application/json; charset=utf-8 len=976
keys: ['presence', 'firstSchoolModules']
presence: 24
firstSchoolModules type: <class 'NoneType'>

== FORM slug == status=200 ct=application/json; charset=utf-8 len=976
keys: ['presence', 'firstSchoolModules']
presence: 24
firstSchoolModules type: <class 'NoneType'>

== FORM school == status=200 ct=application/json; charset=utf-8 len=976
keys: ['presence', 'firstSchoolModules']
presence: 24
firstSchoolModules type: <class 'NoneType'>

== FORM +token == status=200 ct=application/json; charset=utf-8 len=976
keys: ['presence', 'firstSchoolModules']
presence: 24
firstSchoolModules type: <class 'NoneType'>

== JSON +token == status=200 ct=application/json;

In [26]:
import re, json, requests
from bs4 import BeautifulSoup

UA = "Mozilla/5.0"
CANDIDATE_PAGES = [
    "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors",
    "https://www.collegevine.com/schools/agnes-scott-college",
]

def extract_school_id_from_page(url):
    html = requests.get(url, headers={"User-Agent": UA}, timeout=45).text
    soup = BeautifulSoup(html, "html.parser")

    # 1) direct attributes (rare but easy)
    for attr in ["data-school-id", "data_school_id", "school-id", "school_id"]:
        hit = soup.find(attrs={attr: True})
        if hit:
            try:
                return int(hit[attr])
            except:
                pass

    # 2) scan inline scripts for common patterns
    inline = "\n".join(
        (sc.string or sc.get_text(" ", strip=True) or "")
        for sc in soup.find_all("script")
        if not sc.get("src")
    )

    patterns = [
        r'"school_id"\s*:\s*(\d+)',
        r'"schoolId"\s*:\s*(\d+)',
        r'\bschool_id\s*=\s*(\d+)',
        r'\bschoolId\s*=\s*(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, inline)
        if m:
            return int(m.group(1))

    # 3) slug-neighborhood heuristic (often works in serialized blobs)
    slug = "agnes-scott-college"
    idx = inline.find(slug)
    if idx != -1:
        window = inline[max(0, idx-2000): idx+2000]
        m = re.search(r'"id"\s*:\s*(\d+)\s*,\s*"slug"\s*:\s*"agnes-scott-college"', window)
        if m:
            return int(m.group(1))

    return None

for u in CANDIDATE_PAGES:
    sid = extract_school_id_from_page(u)
    print(u, "=> school_id:", sid)

https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors => school_id: None
https://www.collegevine.com/schools/agnes-scott-college => school_id: None


In [27]:
import requests, json

BASE = "https://www.collegevine.com"
PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"
UA = "Mozilla/5.0"

def probe_presence_with_unitid(sess, presence_path, csrf_token, unitid, slug="agnes-scott-college"):
    url = BASE + presence_path
    headers = {
        "User-Agent": UA,
        "Accept": "application/json,*/*",
        "Referer": PAGE_URL,
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-Token": csrf_token,
    }

    key_variants = [
        # what we *think* it wants (internal school id)
        "first_school_id", "school_id",

        # plausible alternates if they support IPEDS/UnitID
        "unitid", "unit_id", "ipeds", "ipeds_id", "IPEDS", "IPEDS_ID",
        "college_unitid", "college_unit_id", "college_ipeds_id",

        # camelCase variants
        "firstSchoolId", "schoolId", "ipedsId", "unitId",
    ]

    bodies = []
    for k in key_variants:
        bodies.append({k: unitid})
        # sometimes they require slug too
        bodies.append({k: unitid, "slug": slug})
        bodies.append({k: unitid, "school_slug": slug})

    # also try the "known-working" slug-only form for baseline comparison
    bodies.insert(0, {"slug": slug})

    best = None
    for b in bodies:
        r = sess.post(url, headers=headers, data=b, timeout=45)
        ct = (r.headers.get("Content-Type") or "").lower()
        ok_json = "json" in ct
        try:
            obj = r.json() if ok_json else None
        except:
            obj = None

        presence_n = len(obj.get("presence", [])) if isinstance(obj, dict) else None
        fsm = obj.get("firstSchoolModules") if isinstance(obj, dict) else None
        fsm_type = type(fsm).__name__
        fsm_len = len(fsm) if isinstance(fsm, list) else None

        print(f"body={list(b.keys())} status={r.status_code} ct={ct.split(';')[0]} "
              f"presence={presence_n} firstSchoolModules={fsm_type}({fsm_len})")

        # “best” = anything that makes firstSchoolModules not null OR changes presence size materially
        score = 0
        if isinstance(fsm, list): score += 1000 + len(fsm)
        if isinstance(presence_n, int): score += presence_n
        if best is None or score > best[0]:
            best = (score, b, r.status_code, obj)

    return best  # (score, body, status, json_obj)

# Usage:
# sess = requests.Session()
# best = probe_presence_with_unitid(sess, endpoints["schools_content_modules_presence_path"], csrf, 138600)
# print("\nBEST:", best[1], "score:", best[0])

In [28]:
def probe_get_with_unitid(sess, get_path, csrf_token, module_id, unitid, slug="agnes-scott-college"):
    url = BASE + get_path
    headers = {
        "User-Agent": UA,
        "Accept": "application/json,*/*",
        "Referer": PAGE_URL,
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-Token": csrf_token,
    }

    id_keys = [
        "school_id","schoolId","first_school_id","firstSchoolId",
        "unitid","unit_id","ipeds_id","ipedsId","unitId","ipeds",
    ]

    bodies = [
        {"module_id": module_id},
        {"module_id": module_id, "slug": slug},
    ]
    for k in id_keys:
        bodies.append({"module_id": module_id, k: unitid})
        bodies.append({"module_id": module_id, k: unitid, "slug": slug})

    best = None
    for b in bodies:
        r = sess.post(url, headers=headers, data=b, timeout=45)
        ct = (r.headers.get("Content-Type") or "").lower()
        txt = r.text
        try:
            obj = r.json() if "json" in ct else None
        except:
            obj = None

        # “content score”: how non-trivial is the payload?
        score = 0
        if isinstance(obj, dict):
            score += 10 * len(obj.keys())
            score += len(json.dumps(obj))
        else:
            score += len(txt)

        # print a quick summary
        if isinstance(obj, dict):
            keys = list(obj.keys())[:25]
            print(f"body={list(b.keys())} status={r.status_code} keys={keys} json_chars={len(json.dumps(obj))}")
        else:
            print(f"body={list(b.keys())} status={r.status_code} ct={ct.split(';')[0]} len={len(txt)}")

        if best is None or score > best[0]:
            best = (score, b, r.status_code, obj, txt[:300])

    return best

# Usage:
# module_id = presence_json["presence"][0]
# best_get = probe_get_with_unitid(sess, endpoints["schools_content_modules_get_path"], csrf, module_id, 138600)
# print("\nBEST GET BODY:", best_get[1])
# print("BEST GET snippet:", best_get[4])

In [34]:
import re, json, requests
from pprint import pprint

UA = "Mozilla/5.0"
PAGE_URL = "https://www.collegevine.com/schools/hub/all/d/agnes-scott-college/majors"

def fetch_html(url):
    r = requests.get(url, headers={"User-Agent": UA}, timeout=45)
    r.raise_for_status()
    return r.text

def extract_js_object_by_bracematch(text, anchor_regex):
    """
    Finds anchor_regex, then finds the first '{' after it and returns the
    substring corresponding to the balanced {...} object (brace-matched),
    ignoring braces inside quoted strings.
    """
    m = re.search(anchor_regex, text)
    if not m:
        return None, f"Anchor not found: {anchor_regex}"

    i = m.end()
    # find first '{' after anchor
    start = text.find("{", i)
    if start == -1:
        return None, "Could not find '{' after anchor."

    depth = 0
    in_str = False
    esc = False
    quote = None

    for j in range(start, len(text)):
        ch = text[j]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
                quote = None
            continue

        # not in string
        if ch in ("'", '"'):
            in_str = True
            quote = ch
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                # inclusive end
                return text[start:j+1], None

    return None, "Reached end of text without closing brace-match."

def extract_cv_endpoints(html):
    # Anchor on the assignment, then brace-match the object
    anchor = r"window\.CV\.pathInfo\.endpoints\s*=\s*"
    blob, err = extract_js_object_by_bracematch(html, anchor)
    if err:
        return None, err

    # The endpoints object on this site is usually valid JSON (double quotes).
    # If it’s JS object literal instead, we’ll detect and give a clearer error.
    try:
        return json.loads(blob), None
    except Exception as e:
        return None, f"Brace-matched object found but json.loads failed: {e}\nFirst 200 chars:\n{blob[:200]}"

html = fetch_html(PAGE_URL)
print("HTML chars:", len(html))

endpoints, err = extract_cv_endpoints(html)
if err:
    print("ERROR:", err)
else:
    print("endpoints keys:", len(endpoints))
    # Quick sanity sample
    print("\nSample (first 10):")
    pprint(list(endpoints.items())[:10])

    # Helpful groups
    hub_keys = sorted([k for k in endpoints if k.startswith("schools_hub")])
    print("\nschools_hub* keys:", len(hub_keys))
    print("\n".join(hub_keys[:80]))

HTML chars: 58546
endpoints keys: 68

Sample (first 10):
[('autopilot_pre_join_path', '/applications/pre-join'),
 ('flag_set_path', '/flag/set'),
 ('schools_content_modules_get_path', '/schools/content-modules/get'),
 ('schools_content_modules_presence_path', '/schools/content-modules/presence'),
 ('schools_hub_admissions_path', '/schools/hub/admissions'),
 ('schools_hub_data_chances_and_financials_path',
  '/schools/hub/data/chances-and-financials'),
 ('schools_hub_data_recommended_schools_path',
  '/schools/hub/data/recommended-schools'),
 ('schools_hub_notify_filter_selection_path',
  '/schools/hub/notify-filter-selection'),
 ('schools_hub_save_view_config_path', '/schools/hub/save-view-config'),
 ('schools_hub_whatif_chancing_path', '/schools/hub/whatif-chancing')]

schools_hub* keys: 6
schools_hub_admissions_path
schools_hub_data_chances_and_financials_path
schools_hub_data_recommended_schools_path
schools_hub_notify_filter_selection_path
schools_hub_save_view_config_path
schools_

In [35]:
import re

def find_endpoint_keys(endpoints, *needles):
    needles = [n.lower() for n in needles]
    hits = []
    for k, v in endpoints.items():
        hay = (k + " " + v).lower()
        if any(n in hay for n in needles):
            hits.append((k, v))
    return sorted(hits, key=lambda x: x[0])

# likely school-id / lookup / search / slug mapping candidates
cands = find_endpoint_keys(
    endpoints,
    "school", "schools", "search", "lookup", "find", "show", "autocomplete", "slug", "id"
)

print("candidate endpoint keys:", len(cands))
for k, v in cands:
    print(f"{k:45s} -> {v}")

candidate endpoint keys: 25
enrollment_survey_remove_school_from_list_path -> /enrollment-survey/remove-school
experts_search_advisors_path                  -> nope
high_schools_invitations_invite_sem_school_path -> /high-schools/invitations/sem-school/invite
livestreams_polls_report_shown_path           -> /livestreams/polls/shown
livestreams_video_pipeline_status_path        -> /livestreams/conferences/pipeline-status
login_path                                    -> /users/auth/auth0?origin=%2Fschools%2Fhub%2Fall%2Fd%2Fagnes-scott-college%2Fmajors
network_connections_invite_school_path        -> /network/connections/invite
profile_search_extracurricular_activities_path -> /profile/search-extracurriculars
profile_validate_username_path                -> /profile/validate-username
schools_content_modules_get_path              -> /schools/content-modules/get
schools_content_modules_presence_path         -> /schools/content-modules/presence
schools_hub_admissions_path                   -

In [37]:
import requests, re, json
from urllib.parse import urljoin

BASE = "https://www.collegevine.com"
UA = "Mozilla/5.0"

def get_csrf_and_session(majors_url):
    s = requests.Session()
    html = s.get(majors_url, headers={"User-Agent": UA}, timeout=45).text
    m = re.search(r'name="csrf-token"\s+content="([^"]+)"', html)
    csrf = m.group(1) if m else None
    return s, csrf

def try_search_endpoint(slug="agnes-scott-college"):
    majors_url = f"{BASE}/schools/hub/all/d/{slug}/majors"
    s, csrf = get_csrf_and_session(majors_url)

    url = urljoin(BASE, "/schools/search")
    headers = {"User-Agent": UA, "X-Requested-With":"XMLHttpRequest"}
    if csrf:
        headers["X-CSRF-Token"] = csrf

    queries = [slug, slug.replace("-", " "), "Agnes Scott"]
    param_sets = [
        ("GET",  {"q": None}),
        ("GET",  {"query": None}),
        ("GET",  {"term": None}),
        ("GET",  {"search": None}),
        ("POST", {"q": None}),
        ("POST", {"query": None}),
        ("POST", {"term": None}),
        ("POST", {"search": None}),
    ]

    for q in queries:
        for method, base_params in param_sets:
            params = dict(base_params)
            for k in params:
                params[k] = q

            try:
                if method == "GET":
                    r = s.get(url, params=params, headers=headers, timeout=45)
                else:
                    r = s.post(url, data=params, headers=headers, timeout=45)

                ct = r.headers.get("content-type","")
                if "application/json" in ct:
                    obj = r.json()
                    print("\n✅ /schools/search JSON", method, params)
                    print("status:", r.status_code, "len:", len(r.text))
                    print(json.dumps(obj)[:1200])
                    return obj
                else:
                    # sometimes JSON is returned as text/plain
                    if r.text.strip().startswith("{") or r.text.strip().startswith("["):
                        try:
                            obj = json.loads(r.text)
                            print("\n✅ /schools/search JSON-ish", method, params)
                            print("status:", r.status_code, "len:", len(r.text))
                            print(json.dumps(obj)[:1200])
                            return obj
                        except:
                            pass

            except Exception:
                pass

    print("❌ no JSON from /schools/search")
    return None

obj = try_search_endpoint()


✅ /schools/search JSON POST {'q': 'agnes-scott-college'}
status: 200 len: 11942
[{"id": "24442334-2624-4005-a67f-2959cc8bb65a", "name": "AI Miami International University of Art and Design", "slug": "ai-miami-international-university-of-art-and-design", "imageUrl": "https://collegevine.imgix.net/24442334-2624-4005-a67f-2959cc8bb65a.jpg?"}, {"id": "dc1ed90d-7438-42db-93bb-827c1db3e7fa", "name": "AMDA College of the Performing Arts", "slug": "american-musical-and-dramatic-academy", "imageUrl": "https://collegevine.imgix.net/dc1ed90d-7438-42db-93bb-827c1db3e7fa.jpg?"}, {"id": "f47eda01-3489-4191-ba48-d2e345901ccb", "name": "ASA College", "slug": "asa-college", "imageUrl": "https://collegevine.imgix.net/f47eda01-3489-4191-ba48-d2e345901ccb.jpg?"}, {"id": "90a6fbbd-1141-434e-872a-f96329efc97a", "name": "Aaniiih Nakoda College", "slug": "aaniiih-nakoda-college", "imageUrl": "https://collegevine.imgix.net/90a6fbbd-1141-434e-872a-f96329efc97a.jpg?"}, {"id": "19bf133b-dc4e-4a89-afbd-69da3c53bf

In [39]:
import json
import requests

BASE = "https://www.collegevine.com"

def post_form_json(session, url, data, csrf=None):
    headers = {"X-Requested-With": "XMLHttpRequest"}
    if csrf:
        headers["X-CSRF-Token"] = csrf
    r = session.post(url, data=data, headers=headers, timeout=45)
    ct = r.headers.get("content-type", "")
    if "json" in ct:
        return r.status_code, ct, r.text, r.json()
    return r.status_code, ct, r.text, None

def find_school_id(search_results, target_slug="agnes-scott-college"):
    exact = [r for r in search_results if (r.get("slug") == target_slug)]
    if exact:
        return exact[0]["id"], exact[0]
    # fallback: slug contains
    partial = [r for r in search_results if target_slug in (r.get("slug") or "")]
    if partial:
        return partial[0]["id"], partial[0]
    return None, None

# Use your existing session/csrf if you already have them; otherwise create a session:
try:
    sess
except NameError:
    sess = requests.Session()

# If you already scraped csrf_token earlier, keep using it; otherwise leave None
try:
    csrf_token
except NameError:
    csrf_token = None

# 1) Run /schools/search
q = "Agnes Scott College"   # you can change this; "Agnes Scott" also works well
status, ct, raw, results_json = post_form_json(
    sess,
    f"{BASE}/schools/search",
    {"q": q},
    csrf=csrf_token
)

print("status:", status, "ct:", ct, "len:", len(raw))
print("result type:", type(results_json), "n:", (len(results_json) if isinstance(results_json, list) else None))

# 2) Save to disk so we can reference later even if we lose variables
out_path = "cv_schools_search_agnes_scott.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results_json, f, ensure_ascii=False, indent=2)

print("saved:", out_path)

# 3) Extract the exact CV school id by slug
school_id, rec = find_school_id(results_json or [], "agnes-scott-college")
print("school_id:", school_id)
print("record:", rec)

status: 200 ct: application/json; charset=utf-8 len: 11942
result type: <class 'list'> n: 59
saved: cv_schools_search_agnes_scott.json
school_id: aed6fe96-65d4-4252-aea4-b66281033b5c
record: {'id': 'aed6fe96-65d4-4252-aea4-b66281033b5c', 'name': 'Agnes Scott College', 'slug': 'agnes-scott-college', 'imageUrl': 'https://collegevine.imgix.net/aed6fe96-65d4-4252-aea4-b66281033b5c.jpg?'}


In [40]:
def find_school_id(search_results, target_slug="agnes-scott-college"):
    exact = [r for r in search_results if r.get("slug") == target_slug]
    if exact:
        return exact[0]["id"], exact[0]

    # fallback: slug contains
    partial = [r for r in search_results if target_slug in (r.get("slug") or "")]
    if partial:
        return partial[0]["id"], partial[0]

    return None, None

school_id, rec = find_school_id(results_json, "agnes-scott-college")
print("school_id:", school_id)
print("record:", rec)

school_id: aed6fe96-65d4-4252-aea4-b66281033b5c
record: {'id': 'aed6fe96-65d4-4252-aea4-b66281033b5c', 'name': 'Agnes Scott College', 'slug': 'agnes-scott-college', 'imageUrl': 'https://collegevine.imgix.net/aed6fe96-65d4-4252-aea4-b66281033b5c.jpg?'}


In [41]:
import requests

BASE = "https://www.collegevine.com"

def post_json(session, url, data, csrf=None):
    headers = {"X-Requested-With": "XMLHttpRequest"}
    if csrf:
        headers["X-CSRF-Token"] = csrf
    r = session.post(url, data=data, headers=headers, timeout=45)
    return r.status_code, r.headers.get("content-type",""), r.text, (r.json() if "json" in r.headers.get("content-type","") else None)

presence_url = f"{BASE}/schools/content-modules/presence"

status, ct, text, obj = post_json(sess, presence_url, {"first_school_id": school_id}, csrf=csrf_token)
print("presence status:", status, "ct:", ct, "len:", len(text))
print("keys:", list(obj.keys()) if isinstance(obj, dict) else type(obj))
print("presence n:", len(obj.get("presence", [])) if isinstance(obj, dict) else None)
print("firstSchoolModules type:", type(obj.get("firstSchoolModules")) if isinstance(obj, dict) else None)

presence status: 422 ct: text/html; charset=utf-8 len: 5894
keys: <class 'NoneType'>
presence n: None
firstSchoolModules type: None


In [42]:
import re
import json
import requests
from bs4 import BeautifulSoup

BASE = "https://www.collegevine.com"
MAJORS_URL = f"{BASE}/schools/hub/all/d/agnes-scott-college/majors"

def extract_csrf_and_endpoints(html):
    soup = BeautifulSoup(html, "html.parser")
    csrf = None
    meta = soup.select_one("meta[name='csrf-token']")
    if meta:
        csrf = meta.get("content")

    # endpoints blob lives in inline script: window.CV.pathInfo.endpoints = {...}
    m = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*[\r\n;]", html, re.DOTALL)
    endpoints = {}
    if m:
        blob = m.group(1)
        # blob is valid JSON-ish already in your prior run; try json loads directly
        endpoints = json.loads(blob)
    return csrf, endpoints

def post_presence(session, presence_url, data, csrf, referer):
    headers = {
        "X-Requested-With": "XMLHttpRequest",
        "X-CSRF-Token": csrf,
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Origin": BASE,
        "Referer": referer,
    }
    r = session.post(presence_url, data=data, headers=headers, timeout=45)
    ct = r.headers.get("content-type", "")
    parsed = None
    if "json" in ct:
        try:
            parsed = r.json()
        except Exception:
            parsed = None
    return r.status_code, ct, r.text, parsed

# --- Bootstrap session by GETting the page first ---
sess = requests.Session()
r0 = sess.get(MAJORS_URL, timeout=45)
print("GET majors:", r0.status_code, "len:", len(r0.text), "ct:", r0.headers.get("content-type"))

csrf_token, endpoints = extract_csrf_and_endpoints(r0.text)
print("csrf present:", bool(csrf_token), "len:", (len(csrf_token) if csrf_token else None))
print("endpoints keys:", len(endpoints))

presence_path = endpoints.get("schools_content_modules_presence_path")
get_path      = endpoints.get("schools_content_modules_get_path")
print("presence_path:", presence_path)
print("get_path:", get_path)

presence_url = BASE + presence_path

school_id = "aed6fe96-65d4-4252-aea4-b66281033b5c"  # Agnes Scott College UUID

# --- Try likely parameter names (this is the key) ---
param_variants = [
    {"first_school_id": school_id},   # from compiled JS snippet (first_school_id)
    {"firstSchool": school_id},
    {"first_school": school_id},
    {"school_id": school_id},
    {"id": school_id},
    {"slug": "agnes-scott-college"},          # you already saw this returns 200 JSON
    {"school_slug": "agnes-scott-college"},
    {"school": "agnes-scott-college"},
]

for data in param_variants:
    status, ct, text, parsed = post_presence(sess, presence_url, data, csrf_token, MAJORS_URL)
    print("\nPOST presence data=", list(data.keys()), "=>", status, ct, "len:", len(text))
    if parsed is not None:
        print("  JSON keys:", list(parsed.keys()))
        print("  presence n:", len(parsed.get("presence", []) or []))
        print("  firstSchoolModules type:", type(parsed.get("firstSchoolModules")).__name__)
    else:
        # show why we got 422 (usually "Invalid authenticity token" etc.)
        print("  snippet:", text[:400].replace("\n", " ") )

GET majors: 200 len: 57114 ct: text/html; charset=utf-8
csrf present: True len: 86
endpoints keys: 68
presence_path: /schools/content-modules/presence
get_path: /schools/content-modules/get

POST presence data= ['first_school_id'] => 200 application/json; charset=utf-8 len: 976
  JSON keys: ['presence', 'firstSchoolModules']
  presence n: 24
  firstSchoolModules type: NoneType

POST presence data= ['firstSchool'] => 200 application/json; charset=utf-8 len: 976
  JSON keys: ['presence', 'firstSchoolModules']
  presence n: 24
  firstSchoolModules type: NoneType

POST presence data= ['first_school'] => 200 application/json; charset=utf-8 len: 976
  JSON keys: ['presence', 'firstSchoolModules']
  presence n: 24
  firstSchoolModules type: NoneType

POST presence data= ['school_id'] => 200 application/json; charset=utf-8 len: 976
  JSON keys: ['presence', 'firstSchoolModules']
  presence n: 24
  firstSchoolModules type: NoneType

POST presence data= ['id'] => 200 application/json; charset=ut

In [43]:
import re, json, requests

BASE = "https://www.collegevine.com"
MAJORS_URL = f"{BASE}/schools/hub/all/d/agnes-scott-college/majors"
school_id = "aed6fe96-65d4-4252-aea4-b66281033b5c"

targets = [
    "African Studies",
    "Classics and Classical Languages, Literatures, and Linguistics, General",
    "cip", "cipCode", "major", "majors", "program"
]

# reuse the session bootstrap pattern you already have
sess = requests.Session()
html = sess.get(MAJORS_URL, timeout=45).text

# extract endpoints blob
m = re.search(r"window\.CV\.pathInfo\.endpoints\s*=\s*(\{.*?\})\s*[\r\n;]", html, re.DOTALL)
endpoints = json.loads(m.group(1))
static_url = endpoints.get("schools_static_data_url")

print("static_url:", static_url)

r = sess.get(static_url, timeout=45)
print("GET static:", r.status_code, "len:", len(r.text), "ct:", r.headers.get("content-type"))

# quick presence scan
for t in targets:
    print(f"{t!r} present:", (t.lower() in r.text.lower()))

# try to parse JSON (may be json; sometimes it's gzipped json but requests handles)
data = None
try:
    data = r.json()
    print("parsed JSON type:", type(data))
except Exception as e:
    print("JSON parse failed:", repr(e))

# If it parsed, try to locate Agnes Scott by school_id anywhere in the object
def find_paths(obj, needle, max_hits=20):
    hits = []
    def rec(x, path):
        if len(hits) >= max_hits:
            return
        if isinstance(x, dict):
            for k, v in x.items():
                rec(v, path + [k])
        elif isinstance(x, list):
            for i, v in enumerate(x):
                rec(v, path + [i])
        else:
            if isinstance(x, str) and needle in x:
                hits.append(path)
    rec(obj, [])
    return hits

if data is not None:
    id_hits = find_paths(data, school_id, max_hits=50)
    print("school_id path hits:", len(id_hits))
    for p in id_hits[:10]:
        print("  path:", p)

    # also search for target strings
    for t in targets[:2]:
        thits = find_paths(data, t, max_hits=20)
        print(f"\nTarget {t!r} path hits:", len(thits))
        for p in thits[:10]:
            print("  path:", p)

static_url: https://d28hdetl1q8yl2.cloudfront.net/schools/hub/data/static/db4f9e9ec10877fd275f07ab07f0d62032a5d88749de6b80f7930eff8ecbe1a3
GET static: 200 len: 9293104 ct: application/json; charset=utf-8
'African Studies' present: True
'Classics and Classical Languages, Literatures, and Linguistics, General' present: True
'cip' present: True
'cipCode' present: True
'major' present: True
'majors' present: True
'program' present: True
parsed JSON type: <class 'dict'>
school_id path hits: 2
  path: ['staticSchools', 78, 'id']
  path: ['staticSchools', 78, 'imgixPath']

Target 'African Studies' path hits: 1
  path: ['majorsMap', 95, 'name']

Target 'Classics and Classical Languages, Literatures, and Linguistics, General' path hits: 1
  path: ['majorsMap', 97, 'name']


In [44]:
import re, json

school_id = "aed6fe96-65d4-4252-aea4-b66281033b5c"
school_slug = "agnes-scott-college"

# data = r.json() from your static fetch
# (assuming you've kept it in `data` already)

# 1) pull the school record
school_rec = next(s for s in data["staticSchools"] if s.get("id") == school_id)
print("School:", school_rec.get("name"), "| idx:", data["staticSchools"].index(school_rec))

# 2) build lookup for majorsMap
majors = data["majorsMap"]

# majorsMap may be list of dicts with ids OR dict keyed by id; handle both
major_by_id = {}
if isinstance(majors, list):
    # infer id field name
    # common: "id" or "cipCode" or "cip_code"
    id_keys = ["id", "cipCode", "cip_code", "cip", "code"]
    for m in majors:
        if not isinstance(m, dict): 
            continue
        mid = None
        for k in id_keys:
            if k in m:
                mid = m[k]
                break
        if mid is not None:
            major_by_id[mid] = m
else:
    # already dict keyed by id
    major_by_id = majors

print("majorsMap size:", len(major_by_id))

# 3) helper: recursively find "major-like" id lists inside a dict
def find_id_lists(obj, path=()):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            hits += find_id_lists(v, path + (k,))
    elif isinstance(obj, list):
        # candidate if list of ints/strings and not huge nested objects
        if obj and all(isinstance(x, (int, str)) for x in obj) and len(obj) < 5000:
            hits.append((path, obj))
        else:
            for i, v in enumerate(obj[:200]):  # avoid huge blowups
                hits += find_id_lists(v, path + (i,))
    return hits

cands = find_id_lists(school_rec)
print("candidate id-lists inside school record:", len(cands))

# 4) score candidates by how many elements match majorsMap ids
def score_candidate(lst):
    # try exact match vs stringified match
    keys = set(major_by_id.keys())
    s_keys = set(str(k) for k in keys)
    hit_exact = sum(1 for x in lst if x in keys)
    hit_str = sum(1 for x in lst if str(x) in s_keys)
    return max(hit_exact, hit_str), hit_exact, hit_str

scored = []
for path, lst in cands:
    sc, he, hs = score_candidate(lst)
    if sc > 0:
        scored.append((sc, he, hs, path, lst))

scored.sort(reverse=True, key=lambda x: x[0])

print("\nTop candidate lists that look like major IDs:")
for sc, he, hs, path, lst in scored[:10]:
    print(f" score={sc} (exact={he}, str={hs}) path={path} len={len(lst)} sample={lst[:8]}")

# 5) pick the best candidate (highest score) as the majors list
best = scored[0] if scored else None
if not best:
    print("\n❌ No obvious majors list found in school record. We should look for a global map keyed by school id.")
else:
    sc, he, hs, path, major_ids = best
    # normalize ids to match key types
    if he >= hs:
        norm_ids = major_ids
        lookup = major_by_id
        get_major = lambda mid: lookup.get(mid)
    else:
        norm_ids = [str(x) for x in major_ids]
        lookup = {str(k): v for k, v in major_by_id.items()}
        get_major = lambda mid: lookup.get(mid)

    names = [get_major(mid)["name"] for mid in norm_ids if get_major(mid) and "name" in get_major(mid)]
    print("\n✅ Using majors list at path:", path)
    print("Majors found:", len(names))
    print("First 25 majors:", names[:25])

    # 6) explicit checks
    target1 = "African Studies"
    target2 = "Classics and Classical Languages, Literatures, and Linguistics, General"
    print("\nTarget presence:")
    print(target1, "->", target1 in names)
    print(target2, "->", target2 in names)

School: Agnes Scott College | idx: 78
majorsMap size: 1046
candidate id-lists inside school record: 1

Top candidate lists that look like major IDs:
 score=34 (exact=34, str=34) path=('majors',) len=34 sample=['05.0101', '05.0207', '16.0501', '16.0901', '16.0905', '16.1200', '23.0101', '23.1302']

✅ Using majors list at path: ('majors',)
Majors found: 34
First 25 majors: ['African Studies', "Women's Studies", 'German Language and Literature', 'French Language and Literature', 'Spanish Language and Literature', 'Classics and Classical Languages, Literatures, and Linguistics, General', 'English Language and Literature, General', 'Creative Writing', 'Biology/Biological Sciences, General', 'Biochemistry', 'Neuroscience', 'Mathematics, General', 'Mathematics, Other', 'Multi-/Interdisciplinary Studies, Other', 'Philosophy', 'Religion/Religious Studies', 'Religion/Religious Studies, Other', 'Astrophysics', 'Chemistry, General', 'Physics, General', 'Psychology, General', 'Economics, General', 

test with new collegevine api scrape

In [51]:
#!/usr/bin/env python3
"""
v14: program-inventory discovery + scoring + tagging (clean)

v14 additions / changes (relative to v13):
- Robust fetch-status tagging via a per-institution status tally:
    * Deterministic row-level `url_tag` derived from all attempted institution-site fetches
    * Explicit buckets for 403, 429, any 5xx, request exceptions (timeouts/connection errors),
      and a catch-all for other 4xx (excluding 403/429)
    * Status tally is reset between the institution crawl and CollegeVine to keep `url_tag`
      institution-only

- Program-title extraction accuracy improvements (cleaning/postprocessing):
    * Stronger course detection (e.g., "AAAS 115a", "AAAS/WGS 125a")
    * Negative-context / announcement filtering (news/events/job postings/platform links)
    * Salvage of common boilerplate prefixes ("Welcome to", "About the program in", etc.)
    * Breadcrumb splitting, degree-suffix normalization, scoring, and deduping

- Expanded target token family for matching and outputs:
    * Africa/African/Africana, Pan-African, African-American, Afro-American, Africology,
      plus Ethnic/Ethnicity, Black, Race/Racial
    * Fixed output token buckets: afri / ethnic / black / race

- Hub confidence proxy from control majors:
    * `total_controls_found` = sum(found_<control>) across the CONTROL_TERMS list
    * `controls_sufficiency` = "warning_<X>_MajorsFound" if total_controls_found < X else "sufficient majors"
    * X is controlled by `MIN_CONTROL_HITS_FOR_CONFIDENT_HUB`

- Canonical-hub preference + JS-hub handling:
    * Canonical hubs can win tie-ish scores (CANONICAL_TIE_THRESHOLD)
    * JS-rendered thin hubs trigger earlier sitemap expansion (looks_like_js_hub)

- CollegeVine (3rd-party truth source) integration:
    * Builds a small slug candidate set for each school name and probes
      `https://www.collegevine.com/schools/hub/all/d/<slug>/majors`
    * Outputs `college_vine_url` (traceability), `college_vine_site` (1/0),
      `college_vine_control_hits`, and CollegeVine-derived program titles
      (extracted with the same heuristics + cleaning as institution pages)
    * Optional control threshold flag: `college_vine_controls_ge_<X>`

- Concordance checks (exact normalized matches; may be expanded later):
    * institution programs vs CollegeVine programs
    * institution programs vs `2013_program_name`
    * CollegeVine programs vs `2013_program_name`

Key features (overall):
- Candidate discovery: path templates + homepage link prioritization + best-first crawl + sitemap assist
- Page scoring: hubness signatures + structured term hits + control-term proxy + penalties
- Row tagging for best_guess_inventory_url: Correct hub / Correct-ish but too specific / Wrong subsite / Wrong non-academic
- CLI knobs for sweep runs from ipynb:
    --subsite-penalty (int): penalty applied to subsite-like pages
    --progtitle-strictness (int): controls how strict program title extraction is
    --head (int): limit rows for quick testing
    --out-suffix (str): appended to output filename stem
    --input / --output: file paths

Notebook-friendly:
- TEST_HEAD_N provides default subset size when --head is not provided. Use --head 0 for full run.
- DEFAULT_WORKERS provides a top-level default for --workers.

INPUT:
- Carnegie_Carla_2013_subset_with_web.csv (unitid, name, Web_address)

OUTPUT:
- Carnegie_Carla_2013_subset_majors_scan_v14.csv (default)
"""

from __future__ import annotations

import argparse
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import heapq
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
from urllib.parse import urljoin, urlparse, urldefrag

import pandas as pd
import requests
from bs4 import BeautifulSoup

import difflib

# ============================================================
# Thread-local requests sessions (for safe parallel fetching)
# ============================================================
_thread_local = threading.local()


def get_thread_session() -> requests.Session:
    """Return a per-thread requests.Session (requests.Session is not thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        _thread_local.session = s
    return s

# ============================================================
# Per-institution fetch-status tally (403-aware)
# ============================================================
# We process institutions sequentially, but fetch pages in parallel per institution.
# This shared tally lets us tag rows when any attempted fetch hit a 403 (or similar).
_STATUS_LOCK = threading.Lock()
_STATUS_COUNTS: Dict[int, int] = {}


def reset_status_tally() -> None:
    with _STATUS_LOCK:
        _STATUS_COUNTS.clear()


def record_http_status(code: int) -> None:
    if not code:
        return
    with _STATUS_LOCK:
        _STATUS_COUNTS[code] = _STATUS_COUNTS.get(code, 0) + 1


# New: Record request exceptions for row-level tagging
def record_request_exception() -> None:
    """Record that a request raised an exception (timeout, connection error, etc.)."""
    with _STATUS_LOCK:
        _STATUS_COUNTS[-1] = _STATUS_COUNTS.get(-1, 0) + 1


def fetch_status_tag(best_url: str) -> str:
    """Row tag describing fetch status / blocks encountered during candidate discovery."""
    with _STATUS_LOCK:
        counts = dict(_STATUS_COUNTS)

    # Prioritize explicit blocks
    if counts.get(403, 0) > 0:
        return "403_forbidden_seen"
    if counts.get(429, 0) > 0:
        return "429_rate_limited_seen"

    # Any server errors encountered
    if any(isinstance(c, int) and 500 <= c <= 599 for c in counts.keys()):
        return "5xx_seen"

    # Request exceptions (timeouts, connection errors, etc.)
    if counts.get(-1, 0) > 0:
        return "exceptions_seen"

    # Any other 4xx (excluding 403/429)
    if any(isinstance(c, int) and 400 <= c <= 499 and c not in (403, 429) for c in counts.keys()):
        return "4xx_seen"

    # If nothing worked
    if not best_url or best_url == "N/A":
        return "no_candidate"

    return "ok"

# ============================================================
# CollegeVine fetch-status tally (separate from institution crawl)
# ============================================================
# CollegeVine can rate-limit (429) or block (403) separately from institution sites.
# Track these separately so `url_tag` remains institution-only.
_CV_STATUS_LOCK = threading.Lock()
_CV_STATUS_COUNTS: Dict[int, int] = {}


def reset_cv_status_tally() -> None:
    with _CV_STATUS_LOCK:
        _CV_STATUS_COUNTS.clear()


def record_cv_http_status(code: int) -> None:
    if not code:
        return
    with _CV_STATUS_LOCK:
        _CV_STATUS_COUNTS[code] = _CV_STATUS_COUNTS.get(code, 0) + 1


def record_cv_request_exception() -> None:
    """Record that a CollegeVine request raised an exception."""
    with _CV_STATUS_LOCK:
        _CV_STATUS_COUNTS[-1] = _CV_STATUS_COUNTS.get(-1, 0) + 1


def cv_status_snapshot() -> Dict[int, int]:
    with _CV_STATUS_LOCK:
        return dict(_CV_STATUS_COUNTS)


def cv_block_or_ratelimit_seen() -> bool:
    """True if we saw a clear CV-side block/rate-limit signal."""
    snap = cv_status_snapshot()
    return (snap.get(429, 0) > 0) or (snap.get(403, 0) > 0)

# ============================================================
# Notebook-friendly default test mode
# ============================================================
# If you run from an ipynb as:  !python v14_program_inventory.py
# it will default to processing TEST_HEAD_N rows unless you pass --head.
# Use --head 0 for full run.
TEST_HEAD_N = 100

# ============================================================
# Top-level parallelism defaults
# ============================================================
# You can override via CLI: --workers
# Keep MAX_WORKERS small to stay polite to university sites.

DEFAULT_WORKERS = 6
MAX_WORKERS = 8

# ============================================================
# Control-term coverage threshold (majors proxy)
# ============================================================
# We use the number of CONTROL_TERMS matched as a rough proxy for whether we landed on a
# real programs/majors hub page. If total controls found is below this threshold, we flag.
# (Do NOT reintroduce found_bio; this counts only the current CONTROL_TERMS list.)
MIN_CONTROL_HITS_FOR_CONFIDENT_HUB = 10

# ============================================================
# CollegeVine (3rd-party truth source): static majors list + control hits
# ============================================================
# We previously attempted to scrape majors from the rendered majors page or via
# content-modules APIs. The robust route is CollegeVine's static data blob
# referenced in CV.pathInfo.endpoints as `schools_static_data_url`.
#
# Flow:
#   1) Bootstrap a CV session by fetching a public hub page and extracting:
#        - CSRF token (meta tag)
#        - CV.pathInfo.endpoints (JSON-like object embedded in HTML)
#   2) Resolve the school id via POST /schools/search
#   3) Fetch the static JSON blob (schools_static_data_url)
#   4) Map the school's `majors` CIP-code list to human names via majorsMap
#   5) Derive:
#        - control hits proxy (distinct CONTROL_TERMS present among offered majors)
#        - target program titles (subset of majors matching TARGET_ANY_REGEX)
#
# Majors pages follow:
#   https://www.collegevine.com/schools/hub/all/d/<school-slug>/majors
# We still output that URL for traceability, but we do not rely on its HTML.

COLLEGEVINE_BASE = "https://www.collegevine.com/schools/hub/all/d"
COLLEGEVINE_BOOTSTRAP_URL = "https://www.collegevine.com/schools/hub/all"
COLLEGEVINE_TIMEOUT_SEC = 20

# In-memory cache for the very large static JSON blob (per run)
_COLLEGEVINE_STATIC_CACHE: Optional[dict] = None
_COLLEGEVINE_STATIC_CACHE_URL: str = ""


def slugify_collegevine_school_name(name: str) -> str:
    """Convert a school name to CollegeVine's expected hyphenated slug."""
    s = normalize_unicode_text(name or "").lower()

    # Replace '&' with 'and' to match common slug conventions
    s = s.replace("&", " and ")

    # Drop apostrophes/quotes; keep alnum + spaces + hyphens
    s = re.sub(r"[\"']", "", s)

    # Collapse non-alphanumeric into spaces, then hyphenate
    s = re.sub(r"[^a-z0-9\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ", "-")

    # Collapse duplicate hyphens
    s = re.sub(r"-+", "-", s).strip("-")
    return s


def collegevine_slug_candidates(name: str) -> List[str]:
    """Generate a small set of plausible CollegeVine slugs for a school name."""
    base = normalize_unicode_text(name or "")
    if not base:
        return []

    cands: List[str] = []

    def _add(nm: str) -> None:
        s = slugify_collegevine_school_name(nm)
        if s and s not in cands:
            cands.append(s)

    # 1) direct
    _add(base)

    # 2) drop common campus qualifiers / parentheticals
    nm2 = re.sub(r"\s*\([^)]*\)\s*", " ", base)
    nm2 = re.sub(r"\b(main\s+campus|downtown\s+campus|online|global\s+campus)\b", " ", nm2, flags=re.I)
    nm2 = normalize_unicode_text(nm2)
    _add(nm2)

    # 3) remove common suffix tokens (but keep order)
    nm3 = re.sub(
        r"\b(university|college|institute|school|state\s+university|community\s+college|polytechnic|campus)\b",
        " ",
        base,
        flags=re.I,
    )
    nm3 = normalize_unicode_text(nm3)
    _add(nm3)

    # 4) remove leading 'the'
    nm4 = re.sub(r"^the\s+", "", base, flags=re.I)
    nm4 = normalize_unicode_text(nm4)
    _add(nm4)

    return cands[:4]


def count_control_hits_in_text(text: str) -> int:
    """Count how many distinct CONTROL_TERMS match in the provided text."""
    blob = normalize_unicode_text(text or "")
    return int(sum(1 for pat in CONTROL_PATS.values() if pat.search(blob)))


def _extract_balanced_braces_object(html: str, marker: str) -> str:
    """Extract a JSON-like object literal that follows `marker` using brace balancing."""
    if not html or not marker:
        return ""
    i = html.find(marker)
    if i < 0:
        return ""

    # Find first '{' after marker
    j = html.find("{", i)
    if j < 0:
        return ""

    depth = 0
    in_str = False
    esc = False
    for k in range(j, len(html)):
        ch = html[k]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue

        if ch == '"':
            in_str = True
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return html[j : k + 1]

    return ""


def _bootstrap_collegevine_session(session: requests.Session, bootstrap_urls: Optional[List[str]] = None) -> Tuple[str, Dict[str, str]]:
    """Fetch a public CollegeVine page and extract (csrf_token, endpoints_map).

    IMPORTANT: Not all CV pages embed `window.CV.pathInfo.endpoints = {...}`.
    In practice, the per-school majors page (or school page) is often the most reliable.

    We therefore try a small list of candidate bootstrap URLs (in order) until we can
    extract an endpoints map.
    """
    urls = [COLLEGEVINE_BOOTSTRAP_URL]
    for u in (bootstrap_urls or []):
        if u and u not in urls:
            urls.append(u)

    for url in urls:
        try:
            r = session.get(url, headers=HEADERS, timeout=COLLEGEVINE_TIMEOUT_SEC, allow_redirects=True)
        except Exception:
            try:
                record_cv_request_exception()
            except Exception:
                pass
            continue

        try:
            record_cv_http_status(getattr(r, "status_code", 0) or 0)
        except Exception:
            pass

        # React to CV rate limiting / blocking: don't keep probing bootstrap URLs.
        if getattr(r, "status_code", 0) in (403, 429):
            return "", {}

        if getattr(r, "status_code", 0) != 200:
            continue

        try:
            html = r.text or ""
        except Exception:
            try:
                html = (r.content or b"").decode("utf-8", errors="replace")
            except Exception:
                html = ""

        if not html:
            continue

        soup = BeautifulSoup(html, "html.parser")
        csrf = ""
        m = soup.find("meta", attrs={"name": "csrf-token"})
        if m and m.get("content"):
            csrf = normalize_unicode_text(m.get("content"))

        # endpoints object: window.CV.pathInfo.endpoints = {...}
        # Use a more specific marker so we don't accidentally start brace-balancing
        # inside an unrelated JS function body.
        marker_candidates = [
            "window.CV.pathInfo.endpoints =",
            "window.CV.pathInfo.endpoints=",
        ]

        obj_txt = ""
        for marker in marker_candidates:
            obj_txt = _extract_balanced_braces_object(html, marker)
            if obj_txt:
                break

        if not obj_txt:
            # try next bootstrap URL
            continue

        try:
            endpoints = json.loads(obj_txt)
        except Exception:
            # Sometimes the extracted blob can include trailing semicolons; try to sanitize.
            obj2 = obj_txt.strip().rstrip(";")
            try:
                endpoints = json.loads(obj2)
            except Exception:
                continue

        if not isinstance(endpoints, dict):
            continue

        # Ensure string -> string
        out: Dict[str, str] = {}
        for k, v in endpoints.items():
            if isinstance(k, str) and isinstance(v, str):
                out[k] = v

        if out:
            return csrf, out

    return "", {}


def _collegevine_request_headers(csrf_token: str) -> Dict[str, str]:
    """Headers for CV XHR-ish POST endpoints."""
    h = dict(HEADERS)
    h.update(
        {
            "Accept": "application/json, text/plain, */*",
            "Content-Type": "application/json;charset=UTF-8",
            "X-Requested-With": "XMLHttpRequest",
        }
    )
    if csrf_token:
        h["X-CSRF-Token"] = csrf_token
    return h


def _resolve_collegevine_school(session: requests.Session, csrf: str, endpoints: Dict[str, str], school_name: str) -> Tuple[str, str, str]:
    """Return (school_id, school_slug, school_display_name) from /schools/search.

    Why this can fail:
      - /schools/search may return a large, noisy list for short/slug-like queries.
      - Some school slugs are easily confused (e.g., "augustana" vs "asa").

    Strategy:
      1) Try multiple queries (slug candidates AND the raw school name).
      2) Prefer exact slug matches.
      3) For fuzzy picks, require overlap on at least one "key token" from the school name
         (e.g., "augustana", "albion", "amherst") to avoid false positives.
    """

    search_path = endpoints.get("schools_search_path") or "/schools/search"

    # --- Key tokens: remove generic stopwords and institutional suffixes ---
    name_norm = normalize_unicode_text(school_name or "").lower()
    name_norm = re.sub(r"[^a-z0-9\s-]", " ", name_norm)
    name_norm = re.sub(r"\s+", " ", name_norm).strip()

    stop = {
        "the", "of", "in", "for", "and", "to", "a", "an", "at",
        "university", "college", "institute", "school", "campus",
        "state", "community", "polytechnic",
    }
    key_tokens = {t for t in name_norm.replace("-", " ").split() if t and t not in stop and len(t) >= 4}

    target_slug = slugify_collegevine_school_name(school_name)

    # Query strategy (expanded):
    #   1) Raw name first (best precision)
    #   2) Key-token-only fallbacks (e.g., "albion")
    #   3) Slug candidates (slug + spaced slug)
    #   4) Remaining normalized variants
    queries: List[str] = []

    def _add_q(q: str) -> None:
        qn = normalize_unicode_text(q)
        if qn and qn not in queries:
            queries.append(qn)

    # 1) Raw name first
    _add_q(school_name)
    _add_q(name_norm)

    # 2) Key-token-only fallbacks (longer/more distinctive first)
    if key_tokens:
        for tok in sorted(key_tokens, key=lambda x: (-len(x), x)):
            _add_q(tok)

    # 3) Slug candidates
    for c in collegevine_slug_candidates(school_name):
        if not c:
            continue
        _add_q(c)
        _add_q(c.replace("-", " "))

    # 4) Lightly de-suffixed variant (sometimes CV names omit "College"/"University")
    nm_simple = re.sub(
        r"\b(university|college|institute|school|state\s+university|community\s+college|polytechnic|campus)\b",
        " ",
        school_name or "",
        flags=re.I,
    )
    nm_simple = normalize_unicode_text(nm_simple)
    if nm_simple and nm_simple.lower() != (school_name or "").lower():
        _add_q(nm_simple)

    # Cap attempts higher: some schools need a few different query shapes.
    for q in queries[:12]:
        try:
            r = session.post(
                urljoin("https://www.collegevine.com", search_path),
                headers=_collegevine_request_headers(csrf),
                json={"q": q},
                timeout=COLLEGEVINE_TIMEOUT_SEC,
            )
        except Exception:
            try:
                record_cv_request_exception()
            except Exception:
                pass
            continue

        try:
            record_cv_http_status(getattr(r, "status_code", 0) or 0)
        except Exception:
            pass

        # React to CV rate limiting / blocking: stop trying further queries.
        if getattr(r, "status_code", 0) in (403, 429):
            break

        if getattr(r, "status_code", 0) != 200:
            continue

        try:
            arr = r.json()
        except Exception:
            continue

        if not isinstance(arr, list) or not arr:
            continue

        # 1) Exact slug match wins immediately.
        for rec in arr:
            if not isinstance(rec, dict):
                continue
            if str(rec.get("slug") or "").strip().lower() == target_slug:
                sid = rec.get("id") or ""
                return str(sid), str(rec.get("slug") or ""), str(rec.get("name") or "")

        # 2) Fuzzy: require key token overlap to avoid false positives like ASA vs Augustana.
        scored: List[Tuple[float, dict]] = []
        for rec in arr:
            if not isinstance(rec, dict):
                continue

            nm = normalize_unicode_text(str(rec.get("name") or ""))
            sl = normalize_unicode_text(str(rec.get("slug") or ""))
            if not nm and not sl:
                continue

            blob = (nm + " " + sl).lower()
            if key_tokens:
                if not any(tok in blob for tok in key_tokens):
                    continue

            score = 0.0
            if sl:
                score = max(score, difflib.SequenceMatcher(None, target_slug, sl.lower()).ratio())
            if nm:
                score = max(score, difflib.SequenceMatcher(None, name_norm, nm.lower()).ratio())

            scored.append((score, rec))

        if not scored:
            continue

        scored.sort(key=lambda x: x[0], reverse=True)
        best_score, best = scored[0]

        # With token-overlap gating, we can accept slightly lower thresholds.
        if best_score >= 0.74:
            sid = str(best.get("id") or "")
            sl = str(best.get("slug") or "")
            nm = str(best.get("name") or "")
            return sid, sl, nm

    return "", "", ""


def _load_collegevine_static_data(session: requests.Session, endpoints: Dict[str, str]) -> Optional[dict]:
    """Fetch and cache the large CollegeVine static JSON blob."""
    global _COLLEGEVINE_STATIC_CACHE, _COLLEGEVINE_STATIC_CACHE_URL

    static_url = endpoints.get("schools_static_data_url") or ""
    if not static_url:
        return None

    if _COLLEGEVINE_STATIC_CACHE is not None and static_url == _COLLEGEVINE_STATIC_CACHE_URL:
        return _COLLEGEVINE_STATIC_CACHE

    try:
        r = session.get(static_url, headers=HEADERS, timeout=COLLEGEVINE_TIMEOUT_SEC, allow_redirects=True)
    except Exception:
        return None

    if getattr(r, "status_code", 0) != 200:
        return None

    try:
        data = r.json()
    except Exception:
        # fallback: parse text
        try:
            data = json.loads(r.text)
        except Exception:
            return None

    if not isinstance(data, dict):
        return None

    _COLLEGEVINE_STATIC_CACHE = data
    _COLLEGEVINE_STATIC_CACHE_URL = static_url
    return data


def _build_majors_lookup(static_data: dict) -> Dict[str, str]:
    """Return mapping CIP-code -> major name from CollegeVine static majorsMap."""
    majors_map = static_data.get("majorsMap")
    out: Dict[str, str] = {}

    if isinstance(majors_map, dict):
        # already keyed by code
        for k, v in majors_map.items():
            if isinstance(v, dict) and v.get("name"):
                out[str(k)] = str(v.get("name"))
        return out

    if isinstance(majors_map, list):
        # entries like {"cipCode": "05.0101", "name": "African Studies", ...}
        for rec in majors_map:
            if not isinstance(rec, dict):
                continue
            nm = rec.get("name")
            if not nm:
                continue

            code = rec.get("cipCode")
            if code is None:
                code = rec.get("cip")
            if code is None:
                code = rec.get("code")
            if code is None:
                code = rec.get("id")
            if code is None:
                continue

            out[str(code)] = str(nm)

    return out


def fetch_collegevine_majors_page(
    school_name: str,
    session: requests.Session,
    progtitle_strictness: int,
) -> Tuple[str, int, int, int, List[str]]:
    """CollegeVine majors integration via static data.

    Return: (collegevine_url, college_vine_site, college_vine_control_hits,
             college_vine_program_title_count, college_vine_program_titles)

    - college_vine_site is 1 if we can resolve the school and map majors from the static blob.
    - control hits are computed against offered major *names*.
    - program titles are the subset of offered major names that match TARGET_ANY_REGEX,
      optionally passed through clean_program_titles for consistency.

    NOTE: This fetch is intentionally isolated from the per-institution _STATUS_COUNTS
    used for tagging university-site 403/429s.
    """
    # Reset CV-only status tally for this institution's CollegeVine attempt.
    try:
        reset_cv_status_tally()
    except Exception:
        pass
    # 1) Bootstrap session for endpoints + csrf
    # Try to bootstrap from pages that reliably embed CV.pathInfo.endpoints.
    slugs_for_bootstrap = collegevine_slug_candidates(school_name)
    bootstrap_urls: List[str] = []
    for slug in slugs_for_bootstrap:
        bootstrap_urls.append(f"{COLLEGEVINE_BASE}/{slug}/majors")
        bootstrap_urls.append(f"https://www.collegevine.com/schools/{slug}")

    csrf, endpoints = _bootstrap_collegevine_session(session, bootstrap_urls=bootstrap_urls)
    if not endpoints:
        # If CV is rate-limiting or blocking us, avoid further probing.
        if cv_block_or_ratelimit_seen():
            slugs = slugs_for_bootstrap
            if slugs:
                return f"{COLLEGEVINE_BASE}/{slugs[0]}/majors", 0, 0, 0, []
            return "", 0, 0, 0, []
        # fall back to old behavior: attempt HTML majors page fetch (best-effort)
        slugs = slugs_for_bootstrap
        if not slugs:
            return "", 0, 0, 0, []
        last_url = f"{COLLEGEVINE_BASE}/{slugs[0]}/majors"
        for slug in slugs:
            cv_url = f"{COLLEGEVINE_BASE}/{slug}/majors"
            last_url = cv_url
            try:
                r = session.get(cv_url, headers=HEADERS, timeout=COLLEGEVINE_TIMEOUT_SEC, allow_redirects=True)
            except Exception:
                try:
                    record_cv_request_exception()
                except Exception:
                    pass
                continue

            try:
                record_cv_http_status(getattr(r, "status_code", 0) or 0)
            except Exception:
                pass

            # React to CV rate limiting / blocking.
            if getattr(r, "status_code", 0) in (403, 429):
                break

            if getattr(r, "status_code", 0) != 200:
                continue
            try:
                html = r.text or ""
            except Exception:
                html = ""
            if not html:
                continue
            parsed = parse_html_to_parsedpage(html, page_url=cv_url)
            if is_soft_404(parsed):
                continue
            ctrl_hits = count_control_hits_in_text(parsed.corpus_any)
            raw_titles: Set[str] = set()
            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
                    raw_titles.add(txt)
            for txt in [getattr(parsed, "h1", ""), getattr(parsed, "html_title", "")]:
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
                    raw_titles.add(txt)
            cleaned_titles = clean_program_titles(list(raw_titles), progtitle_strictness=progtitle_strictness)
            return cv_url, 1, ctrl_hits, int(len(cleaned_titles)), cleaned_titles
        return last_url, 0, 0, 0, []

    # 2) Resolve school id + slug
    school_id, school_slug, _school_display_name = _resolve_collegevine_school(session, csrf, endpoints, school_name)
    if not school_id:
        # Fallback: try to resolve directly from the static blob by slug/name (avoids /schools/search failures).
        static_data = _load_collegevine_static_data(session, endpoints)
        if isinstance(static_data, dict):
            slug_guess = slugify_collegevine_school_name(school_name)
            name_norm = normalize_unicode_text(school_name or "").lower()
            name_norm = re.sub(r"[^a-z0-9\s-]", " ", name_norm)
            name_norm = re.sub(r"\s+", " ", name_norm).strip()

            # Build key tokens (same logic as resolver) to prevent wrong picks.
            stop = {
                "the", "of", "in", "for", "and", "to", "a", "an", "at",
                "university", "college", "institute", "school", "campus",
                "state", "community", "polytechnic",
            }
            key_tokens = {t for t in name_norm.replace("-", " ").split() if t and t not in stop and len(t) >= 4}

            best_rec = None
            best_score = 0.0

            for rec in (static_data.get("staticSchools") or []):
                if not isinstance(rec, dict):
                    continue

                # Some versions of the blob include `slug`; others may not.
                sl = normalize_unicode_text(str(rec.get("slug") or rec.get("schoolSlug") or ""))
                nm = normalize_unicode_text(str(rec.get("name") or rec.get("displayName") or rec.get("schoolName") or ""))

                # Exact slug match if available.
                if sl and sl.lower() == slug_guess:
                    best_rec = rec
                    best_score = 1.0
                    break

                blob = (nm + " " + sl).lower()
                if key_tokens:
                    if not any(tok in blob for tok in key_tokens):
                        continue

                sc = 0.0
                if sl:
                    sc = max(sc, difflib.SequenceMatcher(None, slug_guess, sl.lower()).ratio())
                if nm:
                    sc = max(sc, difflib.SequenceMatcher(None, name_norm, nm.lower()).ratio())

                if sc > best_score:
                    best_score = sc
                    best_rec = rec

            if best_rec is not None and best_score >= 0.78:
                school_id = str(best_rec.get("id") or "")
                if not school_slug:
                    # Prefer the blob slug if present; otherwise use our guess.
                    school_slug = str(best_rec.get("slug") or best_rec.get("schoolSlug") or slug_guess)

        # If still unresolved, output a traceable majors URL based on our best slug guess.
        if not school_id:
            slugs = slugs_for_bootstrap
            if slugs:
                return f"{COLLEGEVINE_BASE}/{slugs[0]}/majors", 0, 0, 0, []
            return "", 0, 0, 0, []

    # 3) Load static blob
    static_data = _load_collegevine_static_data(session, endpoints)
    if not static_data:
        return f"{COLLEGEVINE_BASE}/{school_slug}/majors", 0, 0, 0, []

    majors_lookup = _build_majors_lookup(static_data)
    if not majors_lookup:
        return f"{COLLEGEVINE_BASE}/{school_slug}/majors", 0, 0, 0, []

    # 4) Find school record in staticSchools and map majors
    school_rec = None
    for rec in (static_data.get("staticSchools") or []):
        if isinstance(rec, dict) and str(rec.get("id") or "") == str(school_id):
            school_rec = rec
            break

    if not school_rec or not isinstance(school_rec, dict):
        return f"{COLLEGEVINE_BASE}/{school_slug}/majors", 0, 0, 0, []

    major_codes = school_rec.get("majors") or []
    if not isinstance(major_codes, list) or not major_codes:
        return f"{COLLEGEVINE_BASE}/{school_slug}/majors", 0, 0, 0, []

    offered_names: List[str] = []
    for code in major_codes:
        nm = majors_lookup.get(str(code))
        if nm:
            offered_names.append(normalize_unicode_text(nm))

    if not offered_names:
        return f"{COLLEGEVINE_BASE}/{school_slug}/majors", 0, 0, 0, []

    # 5) Compute control hits and target program titles
    ctrl_hits = count_control_hits_in_text(" ".join(offered_names))

    raw_titles = [t for t in offered_names if TARGET_ANY_REGEX.search(t)]
    # These are already clean major names, but keep the cleaner for consistency
    cleaned_titles = clean_program_titles(list(set(raw_titles)), progtitle_strictness=progtitle_strictness)

    cv_url = f"{COLLEGEVINE_BASE}/{school_slug}/majors"
    return cv_url, 1, int(ctrl_hits), int(len(cleaned_titles)), cleaned_titles

# ============================================================
# Defaults (override via CLI)
# ============================================================
DEFAULT_INPUT_PATH = Path("Carnegie_Carla_2013_subset_with_web.csv")
DEFAULT_OUTPUT_PATH = Path("Carnegie_Carla_2013_subset_majors_scan_v14.csv")

UNITID_COL = "unitid"
NAME_COL = "name"
WEB_COL = "Web_address"

REQUEST_TIMEOUT_SEC = 25
SLEEP_BETWEEN_REQUESTS_SEC = 0.25
SLEEP_BETWEEN_INSTITUTIONS_SEC = 0.20

MAX_BYTES_PER_PAGE = 2_000_000
MAX_JSON_BLOB_CHARS = 200_000

CACHE_DIR = Path(".cache_v14_html")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TTL_SECONDS = 14 * 24 * 3600

# Per-institution budgets
PHASE1_MAX_INVENTORY_CANDIDATES = 18
PHASE1_MAX_CATALOG_CANDIDATES = 10
PHASE1_MAX_HOMEPAGE_LINKS = 22
PHASE2_BESTFIRST_MAX_PAGES = 55
PHASE2_MAX_DEPTH = 3

TOP_K_ALTS = 6
ONE_HOP_MAX_LINKS_TOTAL = 40
ONE_HOP_MAX_FETCHES = 18

SITEMAP_MAX_FETCHES = 5
SITEMAP_MAX_URLS = 2500
SITEMAP_CANDIDATE_CAP = 240

MIN_VISIBLE_TEXT_LEN = 900
MIN_MAIN_TEXT_LEN = 650

ENABLE_PDF_FALLBACK = True
PDF_MAX_FETCHES_PER_INSTITUTION = 2

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; majors-scan/14.0; +https://nces.ed.gov/)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


# ============================================================
# URL templates
# ============================================================
INVENTORY_PATHS = [
    "/academics/programs", "/academics/programs/",
    "/programs", "/programs/",
    "/academics/majors-minors", "/academics/majors-minors/",
    "/majors-minors", "/majors-minors/",
    "/academics/majors", "/academics/majors/",
    "/majors", "/majors/",
    "/undergraduate/majors", "/undergraduate/majors/",
    "/departments-and-programs", "/departments-and-programs/",
    "/academics/departments-and-programs", "/academics/departments-and-programs/",
    "/departments", "/departments/",
    "/academics/departments", "/academics/departments/",
    "/areas-of-study", "/areas-of-study/",
    "/fields-of-study", "/fields-of-study/",
    "/academic-programs", "/academic-programs/",
    "/academics/academic-programs", "/academics/academic-programs/",
    "/find-your-program", "/en/find-your-program", "/en/find-your-program/",
]

CATALOG_PATHS = [
    "/college-catalog", "/college-catalog/",
    "/course-catalog", "/course-catalog/",
    "/catalog", "/catalog/",
    "/catalogue", "/catalogue/",
    "/bulletin", "/bulletin/",
    "/academics/catalog", "/academics/catalog/",
]

YEAR_TOKENS = [
    "2526", "20252026", "2025-2026", "2025–2026", "2025—2026", "2025_2026", "202526",
    "25-26", "25–26", "25_26",
]
YEAR_PATH_TEMPLATES = [
    "/catalog/{y}",
    "/catalogue/{y}",
    "/college-catalog/{y}",
    "/course-catalog/{y}",
    "/academic-catalog/{y}",
    "/bulletin/{y}",
    "/academics/catalog/{y}",
    "/academiclife/college-catalog/{y}",
]


# ============================================================
# Patterns / scoring signals
# ============================================================
YEAR_REGEX = re.compile(
    r"(2025\s*[-–—_/]?\s*2026)|(\b2526\b)|(\b202526\b)|(\b20252026\b)|(\b25\s*[-–—_/]\s*26\b)",
    flags=re.IGNORECASE,
)

GRAD_URL_PENALTY = re.compile(
    r"\b(graduate|grad-studies|gradstudies|master|masters|doctoral|phd|mba|mfa|ms|ma)\b",
    flags=re.IGNORECASE,
)
PROFILE_URL_PENALTY = re.compile(
    r"(/people/|/person/|/faculty/|/faculty-directory/|/staff/|/directory/|/profiles?/|/bio/|/biography/)",
    flags=re.IGNORECASE,
)
IRRELEVANT_URL_PENALTY = re.compile(
    r"(/news/|/events/|/event/|/calendar/|/press/|/alumni/|/giving/|/donate/|/commencement|/story/|/blog/|/announcement/)",
    flags=re.IGNORECASE,
)
ADMISSIONS_URL_PENALTY = re.compile(
    r"(/admissions/|/apply/|/visit/|/financial-aid/|/tuition/|/request-info/)",
    flags=re.IGNORECASE,
)
ARCHIVE_URL_PENALTY = re.compile(r"\barchive\b|/past-|/past/", flags=re.IGNORECASE)
AWARDS_URL_PENALTY = re.compile(r"\b(awards?|honors?)\b", flags=re.IGNORECASE)
COMPLIANCE_URL_PENALTY = re.compile(
    r"(consumer-information|disclosures?|cip\b|gainful-employment|institutional-research)",
    flags=re.IGNORECASE,
)

SOFT_404_TEXT = re.compile(
    r"\b(page\s+not\s+found|404|not\s+found|we\s+can'?t\s+find|does\s+not\s+exist)\b",
    flags=re.IGNORECASE,
)

UNDERGRAD_TITLE_BOOST = re.compile(
    r"\b(undergraduate|majors\s+and\s+minors|majors-minors|bachelor|B\.?A\.?|B\.?S\.?)\b",
    flags=re.IGNORECASE,
)

PDF_LINK_REGEX = re.compile(r"\.pdf(\?|$)", flags=re.IGNORECASE)

CONTROL_TERMS: Dict[str, str] = {
    "anthropology": r"\banthropolog[a-z]*\b",
    "math": r"\bmath[a-z]*\b",
    # "bio": r"\bbio[a-z]*\b", # not a good control due to "biography" and other simple matches
    "linguistics": r"\blinguist[a-z]*\b",
    "chem": r"\bchem(?:istry|ical)?\b",
    "architect": r"\barchitect[a-z]*\b",
    "economics": r"\beconomics?\b",
    "psychology": r"\bpsycholog[a-z]*\b",
    "sociology": r"\bsociolog[a-z]*\b",
    "history": r"\bhistory\b",
    "english": r"\benglish\b",
    "political_science": r"\bpolitical\s+science\b|\bpolisci\b",
    "philosophy": r"\bphilosoph[a-z]*\b",
    "computer_science": r"\bcomputer\s+science\b|\bcomp\s+sci\b|\bcs\b(?![a-z])",
    "engineering": r"\bengineering\b",
    "physics": r"\bphysics\b",
    "geology": r"\bgeolog[a-z]*\b",
    "statistics": r"\bstatistic[a-z]*\b",
    "neuroscience": r"\bneuroscien[a-z]*\b",
}

STRUCT_TERMS: Dict[str, str] = {
    "majors": r"\bmajors?\b",
    "minors": r"\bminors?\b",
    "degrees": r"\bdegrees?\b",
    "programs": r"\bprograms?\b",
    "bachelor": r"\bbachelor(?:'s)?\b",
    "BA": r"\bB\.?A\.?\b",
    "BS": r"\bB\.?S\.?\b",
}

#
# Expanded target token family (v12): includes Africa/African/Africana, Pan-African,
# African American, Afro-American, Africology, Race, plus existing afri/ethnic/black.
#
# NOTE: Keep "TARGET_ANY_REGEX" broad (union) so extraction triggers properly,
# but keep per-bucket regexes interpretable for output columns.
TARGET_TOKEN_REGEX = {
    # "afri" bucket (kept for output column stability) WITHOUT broad afri* wildcard.
    "afri": re.compile(
        r"\b(africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology)\b",
        flags=re.IGNORECASE,
    ),
    # Include both "ethnic*" and "ethnicity" so the bucket reflects real page phrasing.
    "ethnic": re.compile(r"\b(ethnic[a-z\-]*|ethnicity)\b", flags=re.IGNORECASE),
    "black": re.compile(r"\bblack[a-z\-]*\b", flags=re.IGNORECASE),
    # New bucket: race / racial
    "race": re.compile(r"\b(race|racial)\b", flags=re.IGNORECASE),
}

# Trigger regex used to decide whether text is in-scope for extraction.
# Keep this broad (union of all target families) so we don't miss relevant programs.
TARGET_ANY_REGEX = re.compile(
    r"\b(africa(?:n|na)?|pan[-\s]?african|african[-\s]?american|afro[-\s]?american|africology|"
    r"ethnic[a-z\-]*|ethnicity|black[a-z\-]*|race|racial)\b",
    flags=re.IGNORECASE,
)

# Output column stability: keep these buckets as fixed output columns.
# If TARGET_TOKEN_REGEX changes, these columns will still be present (empty if missing).
OUTPUT_TOKEN_BUCKETS = ["afri", "ethnic", "black", "race"]

TITLE_NEGATIVE_CONTEXT = re.compile(
    r"\b(student\s+union|black\s+student|history\s+month|cultural\s+center|lives\s+matter|alumni|news|event|calendar|office)\b",
    flags=re.IGNORECASE,
)

COURSE_CODE = re.compile(r"^\s*[A-Z]{2,6}(?:/[A-Z]{2,6})*\s*[-]?\s*\d{1,3}[A-Za-z]?\b")
COURSE_WORDS = re.compile(r"\b(units?|credits?|semester|fall|spring|course(s)?|syllabus)\b", flags=re.IGNORECASE)
PERSON_NAME_ONLY = re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+){1,2}$")
PROGRAM_KEYWORDS = re.compile(
    r"\b(studies|program|department|major|minor|concentration|track|certificate|center)\b",
    flags=re.IGNORECASE,
)

# --- Title junk / prose detection ---
URL_LIKE_TEXT = re.compile(r"(https?://|www\.|\b[a-z0-9\-]+\.(edu|com|org|net)(/|\b))", flags=re.IGNORECASE)
PROSE_VERBS = re.compile(
    r"\b(define|apply|enrich|explore|learn|prepare|prepares|focus(es)?|offers|provides|designed|helps|aims)\b",
    flags=re.IGNORECASE,
)

# --- Program-title cleaning / postprocessing (v14) ---
TITLE_PLATFORM_KEYWORDS = re.compile(
    r"\b(facebook|instagram|twitter|linkedin|youtube|blackboard|canvas|email|login)\b",
    flags=re.IGNORECASE,
)
TITLE_SLUG_LIKE = re.compile(r"^[a-z0-9\-]{10,}$")
TITLE_LEADING_SLASH = re.compile(r"^/\s*")

TITLE_STRIP_PREFIXES = [
    # salvage patterns (capture group is the likely program name)
    re.compile(r"^\s*welcome\s+to\s+(.+)$", re.I),
    re.compile(r"^\s*about\s+(?:the\s+)?(?:program|department)\s+(?:in|of)\s+(.+)$", re.I),
    re.compile(r"^\s*view\s+(.+)$", re.I),
    re.compile(r"^\s*connect\s+with\s+(.+)$", re.I),
    re.compile(r"^\s*prospective\s+students\s*:\s*(.+)$", re.I),
    re.compile(r"^\s*learn\s+more\s+about\s+(.+)$", re.I),
    re.compile(r"^\s*explore\s+(.+)$", re.I),
]

TITLE_HARD_DROPS = re.compile(
    r"\b(lecturer\s+pool|job|position|apply\b|hiring\b|employment\b)\b",
    flags=re.IGNORECASE,
)

TITLE_CONTENTY_WORDS = re.compile(
    r"\b(discusses|announces|becomes|celebrate|event|workshop|lecture|seminar|news|story|profile|alumni|upcoming)\b",
    flags=re.IGNORECASE,
)

TITLE_YEARISH = re.compile(r"\b(20\d{2}|\d{2}\s*[-–—/]\s*\d{2})\b")


def _has_program_intent(s: str) -> bool:
    s_low = (s or "").lower()
    # Strong program intent: explicit keywords OR common framing
    if PROGRAM_KEYWORDS.search(s) is not None:
        return True
    if s_low.endswith(" studies") or s_low.endswith(" studies program"):
        return True
    if s_low.startswith("department of ") or s_low.startswith("center for ") or s_low.startswith("centre for "):
        return True
    return False


def _split_title_candidates(s: str) -> List[str]:
    """Split breadcrumb-ish or concatenated titles into candidate chunks."""
    s = normalize_unicode_text(s)
    if not s:
        return []

    parts: List[str] = [s]
    for sep in ["|", "•", "·", "–", "—"]:
        if sep in s:
            tmp: List[str] = []
            for p in parts:
                tmp.extend([x.strip() for x in p.split(sep) if x.strip()])
            parts = tmp

    # Light split on " / " only (avoid breaking AAAS/WGS course codes)
    tmp2: List[str] = []
    for p in parts:
        if " / " in p:
            tmp2.extend([x.strip() for x in p.split(" / ") if x.strip()])
        else:
            tmp2.append(p)
    parts = tmp2

    return [normalize_unicode_text(p) for p in parts if p]


def _salvage_prefix(s: str) -> List[str]:
    s = normalize_unicode_text(s)
    if not s:
        return []
    out = [s]
    for rx in TITLE_STRIP_PREFIXES:
        m = rx.match(s)
        if m:
            g = normalize_unicode_text(m.group(1))
            if g and g != s:
                out.append(g)
    return out


def _normalize_degree_suffix(s: str) -> str:
    # Normalize common degree-label patterns that appear after colons or at end.
    s = normalize_unicode_text(s)
    s = re.sub(r"\s*:\s*(B\.?A\.?|B\.?S\.?|BA|BS)\b\s*", " ", s, flags=re.I)
    s = re.sub(r"\s+\b(B\.?A\.?|B\.?S\.?|BA|BS)\b\s*$", "", s, flags=re.I)
    return normalize_unicode_text(s)


# ============================================================
# Loose-normalized concordance helpers
# ============================================================

def _title_key_set_loose(titles: List[str]) -> Set[str]:
    """Loose normalized key set for concordance checks.

    Mirrors `_title_key_set` but uses `norm_title_key_loose()` so common formatting
    differences (e.g., '&' vs 'and', generic suffix words) don't block matches.

    IMPORTANT: Apply `apply_synonym_map()` before loose-normalization so known
    spelling/synonym fixes used in partial matching (e.g., carribbean->caribbean)
    also influence loose concordance.
    """
    out: Set[str] = set()
    for t in (titles or []):
        t_norm = normalize_unicode_text(t)
        if not t_norm:
            continue
        t_pre = apply_synonym_map(t_norm)
        k = norm_title_key_loose(t_pre)
        if k:
            out.add(k)
    return out


def _field_key_set_loose(field_val: str) -> Set[str]:
    """Loose normalized key set from a scalar program-name field.

    Mirrors `_field_key_set` but uses `norm_title_key_loose()`.

    IMPORTANT: Apply `apply_synonym_map()` before loose-normalization so known
    spelling/synonym fixes used in partial matching are reflected here too.
    """
    out: Set[str] = set()
    for x in _split_program_name_field(field_val):
        x_norm = normalize_unicode_text(x)
        if not x_norm:
            continue
        x_pre = apply_synonym_map(x_norm)
        k = norm_title_key_loose(x_pre)
        if k:
            out.add(k)
    return out


def _score_title_variant(s: str) -> int:
    """Higher is better."""
    s_norm = normalize_unicode_text(s)
    s_low = s_norm.lower()
    score = 0

    if _has_program_intent(s_norm):
        score += 40
    if re.search(r"\b(major|minor|concentration|certificate)\b", s_norm, re.I):
        score += 15
    if s_low.startswith("department of "):
        score += 8
    if s_low.endswith(" studies"):
        score += 10

    # Penalize content-y / announcement-y strings
    if TITLE_HARD_DROPS.search(s_norm):
        score -= 200
    if TITLE_YEARISH.search(s_norm):
        score -= 80
    if TITLE_CONTENTY_WORDS.search(s_norm) and not _has_program_intent(s_norm):
        score -= 120

    # Mild penalty for boilerplate verbs/prefixes that survived
    if re.match(r"^(about|welcome|view|connect|prospective\s+students)\b", s_low):
        score -= 40

    # Prefer reasonably short, name-like strings
    if 8 <= len(s_norm) <= 80:
        score += 5
    if len(s_norm) > 110:
        score -= 25

    return score


def clean_program_titles(raw_titles: List[str], progtitle_strictness: int) -> List[str]:
    """Clean, salvage, split, score, and dedupe extracted program-title candidates."""
    expanded: List[str] = []
    for t in raw_titles:
        for s in _salvage_prefix(t):
            expanded.extend(_split_title_candidates(s))

    keep: List[str] = []
    for t in expanded:
        t = _normalize_degree_suffix(t)
        if not t:
            continue

        # Hard drops
        if TITLE_PLATFORM_KEYWORDS.search(t):
            continue
        if TITLE_LEADING_SLASH.search(t):
            continue
        if TITLE_SLUG_LIKE.match(t):
            continue
        if URL_LIKE_TEXT.search(t):
            continue
        if TITLE_HARD_DROPS.search(t):
            continue

        # Course detection (must drop)
        if COURSE_CODE.match(t):
            continue
        if COURSE_WORDS.search(t):
            continue

        # Always require target token family (keeps extraction scoped)
        if TARGET_ANY_REGEX.search(t) is None:
            continue

        # Contenty strings are only acceptable if they ALSO show program intent
        if (TITLE_CONTENTY_WORDS.search(t) or PROSE_VERBS.search(t)) and not _has_program_intent(t):
            if progtitle_strictness >= 3:
                continue

        # Drop sentence-like cases
        if ". " in t and len(t) >= 40:
            continue
        if ": " in t and len(t) >= 60 and not _has_program_intent(t):
            continue

        keep.append(t)

    # Score + dedupe by normalized key
    best_by_key: Dict[str, Tuple[int, str]] = {}
    for t in keep:
        k = norm_title_key(t)
        if not k:
            continue
        sc = _score_title_variant(t)
        cur = best_by_key.get(k)
        if cur is None or sc > cur[0] or (sc == cur[0] and len(t) < len(cur[1])):
            best_by_key[k] = (sc, t)

    out = [v[1] for v in best_by_key.values()]
    return sorted(out)

INVENTORY_URL_HINTS = (
    "/programs", "/academics/programs",
    "/majors-minors", "/majors",
    "/departments-and-programs", "/departments",
    "/areas-of-study", "/fields-of-study",
    "/academic-programs", "find-your-program",
)
CATALOG_URL_HINTS = ("catalog", "catalogue", "college-catalog", "course-catalog", "bulletin")

# "Listing-like" URL patterns for anchors we count toward hubness
LISTING_LINK_PATTERNS = [
    re.compile(r"/programs/[^/]+/?$", re.I),
    re.compile(r"/majors-minors/[^/]+/?$", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", re.I),
    re.compile(r"/departments/[^/]+/?$", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
]

# Broader detail-page detection (so single-major pages don’t beat hubs)
DETAIL_URL_PATTERNS = [
    re.compile(r"/programs/[^/]+/.+", re.I),
    re.compile(r"/majors-minors/[^/]+/.+", re.I),
    re.compile(r"/departments-and-programs/[^/]+/.+", re.I),
    re.compile(r"/departments/[^/]+/.+", re.I),
    re.compile(r"/majors/[^/]+(\.html|\.shtml)?$", re.I),
    re.compile(r"/majors-programs/[^/]+", re.I),
    re.compile(r"/academiclife/departments/[^/]+/?$", re.I),
    re.compile(r"/content/.*/departments/[^/]+\.html$", re.I),
]

SITEMAP_PROGRAM_URL_PATTERNS = [
    re.compile(r"/academics/majors-minors/[^/]+/index\.html?$", flags=re.IGNORECASE),
    re.compile(r"/academics/majors-minors/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/academics/departments-and-programs/[^/]+/?$", flags=re.IGNORECASE),
    re.compile(r"/departments/[^/]+/?$", flags=re.IGNORECASE),
]


# ============================================================
# Row tagging heuristics
# ============================================================
HUB_PATH_HINTS = [
    "majors", "minors", "programs", "degrees", "department", "departments",
    "departments-and-programs", "academics", "academic", "catalog", "bulletin",
    "undergraduate", "undergraduate-programs", "courses", "areas-of-study"
]

NON_ACADEMIC_HINTS = [
    "news", "event", "calendar", "giving", "alumni", "advancement",
    "athletics", "sports", "tickets", "shop", "bookstore",
    "it", "its", "help", "support", "login", "privacy", "accessibility",
    "admissions", "apply", "visit", "tuition", "financial-aid",
    "student-life", "housing", "dining", "library",
    "about", "contact", "directory", "people", "faculty", "staff",
    "press", "media", "magazine", "blog", "policy"
]

SUBSITE_HINTS = [
    "center", "centre", "institute", "office", "community", "belonging",
    "extension", "continuing-ed", "continuing-education",
    "professional", "outreach",
    # dataset-specific common traps
    "nwo", "cge",
]

# High-precision traps: if these appear in host+path, treat page as an automatic loss.
HARD_SUBSITE_BLOCK_TOKENS = [
    "nwo",  # dataset-specific trap
    "cge",  # dataset-specific trap
]

# Separate from the broad detail penalty: this targets department/program/detail pages that can look
# academically relevant but are still not the intended "programs hub" landing page.
# Starting value calibrated from sweep snapshots: “Correct-ish but too specific” was rare (~2/25 rows, ~8%),
# and usually didn’t have an obvious canonical hub alternative in the top-K alts; use a moderate penalty
# (similar scale to other URL penalties) to nudge toward hubs without overcorrecting.
TOO_SPECIFIC_PENALTY = 240

# If a canonical hub is within this many points of best, prefer it.
CANONICAL_TIE_THRESHOLD = 60

TOO_SPECIFIC_PATTERNS = [
    r"/departments?/[a-z0-9_-]+",
    r"/departments-and-programs/[a-z0-9_-]+",
    r"/programs?/[a-z0-9_-]+",
    r"/academics/.+/(major|minors|programs)\b",
]


# ============================================================
# Unicode normalization
# ============================================================

# Helper: best-effort mojibake repair for common UTF-8/latin-1 issues
def _maybe_fix_mojibake(s: str) -> str:
    """Best-effort repair for common mojibake (UTF-8 bytes decoded as latin-1/cp1252).

    This mainly targets artifacts like "\u201a\u00c4\u00ae" / "\u00c3" / "\u00e2" sequences seen in some CSVs.
    We only apply the repair when it measurably reduces these suspicious markers.
    """
    if not s:
        return s

    # Heuristic trigger: only attempt if we see common mojibake markers.
    if re.search(r"(\u201a\u00c4|\u00c3|\u00e2|\u00c2)", s) is None:
        return s

    try:
        fixed = s.encode("latin-1", errors="ignore").decode("utf-8", errors="ignore")
    except Exception:
        return s

    if not fixed:
        return s

    def _badness(x: str) -> int:
        return (
            x.count("\u201a\u00c4")
            + x.count("\u00c3")
            + x.count("\u00e2")
            + x.count("\u00c2")
        )

    return fixed if _badness(fixed) < _badness(s) else s

def normalize_unicode_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = _maybe_fix_mojibake(s)
    s = unicodedata.normalize("NFKC", s)
    for ch in ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2212", "\u00AD"]:
        s = s.replace(ch, "-")
    s = (s.replace("\u2018", "'")
           .replace("\u2019", "'")
           .replace("\u201C", '"')
           .replace("\u201D", '"'))
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compile_patterns(term_map: Dict[str, str]) -> Dict[str, re.Pattern]:
    return {k: re.compile(v, flags=re.IGNORECASE) for k, v in term_map.items()}


CONTROL_PATS = compile_patterns(CONTROL_TERMS)
STRUCT_PATS = compile_patterns(STRUCT_TERMS)


# ============================================================
# Cache + fetch
# ============================================================
def _cache_key(url: str) -> str:
    return hashlib.sha256(url.encode("utf-8")).hexdigest()


def _cache_paths(url: str) -> Tuple[Path, Path]:
    key = _cache_key(url)
    return CACHE_DIR / f"{key}.bin", CACHE_DIR / f"{key}.meta"


def fetch_bytes_cached(url: str, session: requests.Session) -> bytes:
    bin_path, meta_path = _cache_paths(url)
    if bin_path.exists() and meta_path.exists():
        try:
            ts = int(meta_path.read_text().strip())
            if time.time() - ts < CACHE_TTL_SECONDS:
                return bin_path.read_bytes()
        except Exception:
            pass

    try:
        r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC, allow_redirects=True, stream=True)
    except Exception:
        # Record request exceptions for row-level tagging (timeouts, connection errors, etc.)
        try:
            record_request_exception()
        except Exception:
            pass
        raise
    # Record fetch status for row-level tagging (403-aware)
    try:
        record_http_status(getattr(r, "status_code", 0) or 0)
    except Exception:
        pass
    r.raise_for_status()
    content = r.content
    if len(content) > MAX_BYTES_PER_PAGE:
        content = content[:MAX_BYTES_PER_PAGE]

    try:
        bin_path.write_bytes(content)
        meta_path.write_text(str(int(time.time())))
    except Exception:
        pass
    return content


def fetch_text_cached(url: str, session: requests.Session) -> str:
    b = fetch_bytes_cached(url, session=session)
    try:
        return b.decode("utf-8", errors="replace")
    except Exception:
        return b.decode("latin-1", errors="replace")


# ============================================================
# URL helpers
# ============================================================
def ensure_scheme(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return ""
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def same_domain(url: str, base_netloc: str) -> bool:
    try:
        return urlparse(url).netloc.lower().endswith(base_netloc.lower())
    except Exception:
        return False


def root_domain(netloc: str) -> str:
    n = (netloc or "").lower().strip()
    for p in ("www.", "m.", "web."):
        if n.startswith(p):
            n = n[len(p):]
    return n


def rootish_domain(netloc: str) -> str:
    """
    Lightweight "registrable-ish" domain: last two labels if possible.
    This is imperfect for .edu subdomains but works well enough for drift detection.
    """
    n = root_domain(netloc)
    parts = [p for p in n.split(".") if p]
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return n


ALLOWED_ACADEMIC_SUBDOMAINS = {"www", "majors", "programs", "catalog", "bulletin", "courses"}


def is_subsite_like(url: str, base_netloc: str) -> bool:
    """
    Heuristic: "subsite-like" means the URL is likely an office/center/outreach subsite,
    or lives on a non-standard subdomain that frequently hosts non-inventory pages.

    Rules:
    - Different rootish domain => subsite-like
    - Subdomain first-label not in allowlist => subsite-like
      (unless it's literally the base host itself)
    - Path contains subsite hint words => subsite-like
    """
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Offsite-ish drift
    if rootish_domain(host) != rootish_domain(base):
        return True

    # Subdomain gating: majors.<root>, catalog.<root>, etc allowed; others penalized
    if host != base:
        # get first label (e.g., "catalog" in "catalog.bu.edu")
        parts = [p for p in host.split(".") if p]
        sub = parts[0] if parts else ""
        if sub and sub not in ALLOWED_ACADEMIC_SUBDOMAINS:
            return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in SUBSITE_HINTS):
        return True

    return False

def is_hard_subsite_block(url: str, base_netloc: str) -> bool:
    """True if URL is a known high-precision subsite trap (auto-loss)."""
    u = ensure_scheme(url)
    if not u:
        return True

    host = (urlparse(u).netloc or "").lower()
    base = (base_netloc or "").lower()

    # Drift to different registrable-ish domain => hard-block
    if rootish_domain(host) != rootish_domain(base):
        return True

    path = (urlparse(u).path or "").lower()
    full = host + path

    return any(tok in full for tok in HARD_SUBSITE_BLOCK_TOKENS)

def is_too_specific_url(url: str) -> bool:
    """True if URL path matches TOO_SPECIFIC_PATTERNS."""
    path = (urlparse(url).path or "").lower()
    for pat in TOO_SPECIFIC_PATTERNS:
        if re.search(pat, path):
            return True
    return False

def is_inventoryish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in INVENTORY_URL_HINTS)


def is_catalogish_url(url: str) -> bool:
    ul = url.lower()
    return any(h in ul for h in CATALOG_URL_HINTS)


def is_canonical_hub_url(url: str) -> bool:
    p = (urlparse(url).path or "/").rstrip("/")
    p_low = p.lower()
    canonical = {
        "/programs",
        "/academics/programs",
        "/majors-minors",
        "/academics/majors-minors",
        "/majors",
        "/academics/majors",
        "/departments-and-programs",
        "/academics/departments-and-programs",
        "/departments",
        "/academics/departments",
        "/areas-of-study",
        "/fields-of-study",
        "/academic-programs",
        "/academics/academic-programs",
    }
    return p_low in canonical


def looks_like_detail_url(url: str) -> bool:
    path = (urlparse(url).path or "").lower()
    return any(p.search(path) for p in DETAIL_URL_PATTERNS)


# ============================================================
# Row tagging
# ============================================================
def _norm_host_from_web_address(web_address: str) -> str:
    u = ensure_scheme(web_address)
    if not u:
        return ""
    return (urlparse(u).netloc or "").lower()


def tag_row(web_address: str, best_url: str) -> str:
    """
    Returns one of:
      - 'Correct hub'
      - 'Correct-ish but too specific'
      - 'Wrong subsite'
      - 'Wrong non-academic'
    """
    base_host = _norm_host_from_web_address(web_address)
    u = ensure_scheme(best_url)
    if not u or best_url == "N/A":
        return "Wrong non-academic"

    host = (urlparse(u).netloc or "").lower()
    path = (urlparse(u).path or "").lower()
    full = host + path

    if any(x in full for x in NON_ACADEMIC_HINTS):
        return "Wrong non-academic"

    # off-root-domain or obvious subsite hints
    if base_host and rootish_domain(host) != rootish_domain(base_host):
        return "Wrong subsite"
    if any(x in full for x in SUBSITE_HINTS):
        return "Wrong subsite"

    # hub-ish path
    if any(x in path for x in HUB_PATH_HINTS):
        for pat in TOO_SPECIFIC_PATTERNS:
            if re.search(pat, path):
                return "Correct-ish but too specific"
        return "Correct hub"

    # weaker academic fallback
    if "academ" in path or "catalog" in path or "bulletin" in path:
        return "Correct-ish but too specific"

    return "Wrong subsite"


# ============================================================
# HTML parse + main-content extraction
# ============================================================
@dataclass
class ParsedPage:
    url: str
    visible_text: str
    main_text: str
    anchor_texts: List[str]
    anchor_attr_texts: List[str]
    links: List[Tuple[str, str]]  # (anchor_text, abs_url)
    title_like: List[str]
    html_title: str
    h1: str
    corpus_any: str
    corpus_structured: str
    json_blob: str
    js_hint: int


def parse_html_to_parsedpage(html: str, page_url: str) -> ParsedPage:
    soup = BeautifulSoup(html, "html.parser")

    ttag = soup.find("title")
    html_title = normalize_unicode_text(ttag.get_text(" ", strip=True)) if ttag else ""

    h1_tag = soup.find("h1")
    h1 = normalize_unicode_text(h1_tag.get_text(" ", strip=True)) if h1_tag else ""

    # --- Embedded JSON support (v12 B): keep machine-readable program lists ---
    # Many modern hubs embed program lists as JSON (ld+json, app/json, Next/Nuxt payloads).
    # We extract likely-JSON script contents into `json_blob` and then remove scripts so visible text stays clean.
    json_chunks: List[str] = []
    for sc in soup.find_all("script"):
        typ = (sc.get("type") or "").strip().lower()
        sid = (sc.get("id") or "").strip().lower()

        is_json_type = ("json" in typ)  # includes application/ld+json, application/json, etc.
        is_framework_payload = sid in ("__next_data__", "__nuxt__")

        if not (is_json_type or is_framework_payload):
            continue

        raw = sc.string if sc.string is not None else sc.get_text(" ", strip=True)
        raw = raw.strip() if raw else ""
        if not raw:
            continue

        raw_norm = normalize_unicode_text(raw)
        if not raw_norm:
            continue

        # Keep only content that looks plausibly JSON-like to avoid pulling in inline JS.
        lstripped = raw_norm.lstrip()
        if not (
            lstripped.startswith("{")
            or lstripped.startswith("[")
            or '"@context"' in raw_norm
            or '"@type"' in raw_norm
            or '"@graph"' in raw_norm
            or '"itemListElement"' in raw_norm
        ):
            continue

        json_chunks.append(raw_norm)

    json_blob = normalize_unicode_text(" ".join(json_chunks))
    if len(json_blob) > MAX_JSON_BLOB_CHARS:
        json_blob = json_blob[:MAX_JSON_BLOB_CHARS]

    # Heuristic: if page looks like a JS-rendered app shell, mark it so scoring/candidate expansion can adapt
    raw_lower = (html or "").lower()
    js_hint = 1 if (
        "__next_data__" in raw_lower
        or "window.__nuxt__" in raw_lower
        or "data-reactroot" in raw_lower
        or "id=\"root\"" in raw_lower
        or "id=\"app\"" in raw_lower
        or "ng-version" in raw_lower
    ) else 0

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    links: List[Tuple[str, str]] = []
    anchor_texts: List[str] = []
    anchor_attr_texts: List[str] = []

    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(page_url, href)
        abs_url, _ = urldefrag(abs_url)

        a_text = normalize_unicode_text(a.get_text(" ", strip=True))
        aria = normalize_unicode_text(a.get("aria-label") or "")
        title_attr = normalize_unicode_text(a.get("title") or "")

        if a_text:
            anchor_texts.append(a_text)
        if aria:
            anchor_attr_texts.append(aria)
        if title_attr:
            anchor_attr_texts.append(title_attr)

        links.append((a_text, abs_url))

    visible_text = normalize_unicode_text(soup.get_text(separator=" ", strip=True))

    main_node = soup.find("main") or soup.find(id=re.compile(r"content|main", re.I)) or soup.find("article")
    if main_node:
        main_text = normalize_unicode_text(main_node.get_text(separator=" ", strip=True))
    else:
        soup2 = BeautifulSoup(str(soup), "html.parser")
        for tag in soup2(["header", "footer", "nav"]):
            tag.decompose()
        main_text = normalize_unicode_text(soup2.get_text(separator=" ", strip=True))

    # Title-like candidates should be pulled from "main-ish" content when possible.
    # This reduces nav/news/social noise before later postprocessing.
    title_like: List[str] = []
    title_scope = main_node
    if title_scope is None:
        soup_scope = BeautifulSoup(str(soup), "html.parser")
        for tag in soup_scope(["header", "footer", "nav", "aside"]):
            tag.decompose()
        title_scope = soup_scope

    for tag in title_scope.find_all(["h1", "h2", "h3", "li"]):
        t = normalize_unicode_text(tag.get_text(" ", strip=True))
        if t:
            title_like.append(t)

    corpus_any = normalize_unicode_text(
        " ".join(
            x
            for x in [visible_text, json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts)]
            if x
        )
    )
    corpus_structured = normalize_unicode_text(
        " ".join(
            x
            for x in [json_blob, " ".join(anchor_texts), " ".join(anchor_attr_texts), " ".join(title_like)]
            if x
        )
    )

    return ParsedPage(
        url=page_url,
        visible_text=visible_text,
        main_text=main_text,
        anchor_texts=anchor_texts,
        anchor_attr_texts=anchor_attr_texts,
        links=links,
        title_like=title_like,
        html_title=html_title,
        h1=h1,
        corpus_any=corpus_any,
        corpus_structured=corpus_structured,
        json_blob=json_blob,
        js_hint=js_hint,
    )


def is_soft_404(parsed: ParsedPage) -> bool:
    blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1] + parsed.title_like[:5]))
    if SOFT_404_TEXT.search(blob):
        return True
    if len(parsed.main_text) < 600 and SOFT_404_TEXT.search(parsed.visible_text):
        return True
    if len(parsed.main_text) < 250 and len(parsed.anchor_texts) < 5:
        return True
    return False


def page_is_thin(parsed: ParsedPage) -> bool:
    return (len(parsed.visible_text) < MIN_VISIBLE_TEXT_LEN) and (len(parsed.main_text) < MIN_MAIN_TEXT_LEN)


# ============================================================
# Sitemap parsing
# ============================================================
def _xml_text(el: Optional[ET.Element]) -> str:
    if el is None or el.text is None:
        return ""
    return el.text.strip()


def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def fetch_sitemap_urls(base_url: str, session: requests.Session, base_netloc: str) -> List[str]:
    start = urljoin(base_url, "/sitemap.xml")
    to_fetch = [start]
    fetched = 0
    urls: List[str] = []
    seen_fetch: Set[str] = set()
    seen_urls: Set[str] = set()

    while to_fetch and fetched < SITEMAP_MAX_FETCHES and len(urls) < SITEMAP_MAX_URLS:
        sm_url = to_fetch.pop(0)
        if sm_url in seen_fetch:
            continue
        seen_fetch.add(sm_url)
        fetched += 1

        try:
            xml_txt = fetch_text_cached(sm_url, session=session)
            root = ET.fromstring(xml_txt.encode("utf-8", errors="ignore"))
        except Exception:
            continue

        root_tag = _strip_ns(root.tag).lower()

        if root_tag == "sitemapindex":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if loc and same_domain(loc, base_netloc):
                        to_fetch.append(loc)
            continue

        if root_tag == "urlset":
            for el in root.findall(".//"):
                if _strip_ns(el.tag).lower() == "loc":
                    loc = _xml_text(el)
                    if not loc:
                        continue
                    if not same_domain(loc, base_netloc):
                        continue
                    loc, _ = urldefrag(loc)
                    if loc not in seen_urls:
                        seen_urls.add(loc)
                        urls.append(loc)
                        if len(urls) >= SITEMAP_MAX_URLS:
                            break

    return urls


def sitemap_candidate_urls(all_urls: List[str]) -> List[str]:
    cands: List[str] = []
    for u in all_urls:
        ul = u.lower()

        # hard skips
        if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
            continue
        if ARCHIVE_URL_PENALTY.search(ul):
            continue
        if "graduate" in ul or "grad" in ul:
            continue

        if any(p.search(ul) for p in SITEMAP_PROGRAM_URL_PATTERNS):
            cands.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in cands:
        if u not in seen:
            seen.add(u)
            out.append(u)
        if len(out) >= SITEMAP_CANDIDATE_CAP:
            break

    return out


# ============================================================
# Hubness: program-listing-only link counts
# ============================================================
def is_listing_link(url: str) -> bool:
    path = (urlparse(url).path or "")
    return any(p.search(path) for p in LISTING_LINK_PATTERNS)


def hubness_signature(parsed: ParsedPage, base_netloc: str) -> Tuple[int, Dict[str, int]]:
    listing_links = 0
    slug_set: Set[str] = set()
    distinct_anchor: Set[str] = set()

    for a_text, u in parsed.links:
        if not same_domain(u, base_netloc):
            continue
        if not is_listing_link(u):
            continue

        listing_links += 1

        t = (a_text or "").strip()
        if len(t) >= 6:
            distinct_anchor.add(t.lower())

        path = (urlparse(u).path or "").strip("/").lower()
        segs = [s for s in path.split("/") if s]
        if segs:
            slug_set.add(segs[-1])

    # weak secondary signal
    li_like = sum(1 for t in parsed.title_like if len(t) >= 6)

    sig = 0
    if listing_links >= 12:
        sig += 1
    if listing_links >= 25:
        sig += 1
    if len(slug_set) >= 8:
        sig += 1
    if len(slug_set) >= 16:
        sig += 1
    if len(distinct_anchor) >= 18:
        sig += 1
    if li_like >= 40 and (listing_links >= 10):
        sig += 1

    counts = {
        "listing_links": listing_links,
        "slug_diversity": len(slug_set),
        "distinct_anchor": len(distinct_anchor),
        "li_like": li_like,
    }
    return sig, counts


# ============================================================
# Scoring
# ============================================================
@dataclass
class PageSignals:
    url: str
    tier: str
    soft404: int
    thin: int

    canonical_hub: int
    hub_sig: int
    hub_counts: Dict[str, int]

    struct_hits: List[str]
    control_unique_structured: int

    year_hit: int
    undergrad_title_boost: int

    grad_penalty: int
    profile_penalty: int
    irrelevant_penalty: int
    admissions_penalty: int
    archive_penalty: int
    awards_penalty: int
    compliance_penalty: int

    subsite_like: int  # configurable penalty via knob
    hard_subsite_block: int  # auto-loss for known traps
    too_specific: int  # explicit penalty separate from generic detail penalty
    js_hub_hint: int


def page_tier(url: str) -> str:
    u = url.lower()
    path = urlparse(url).path or "/"
    if path == "/" or path == "":
        return "homepage"
    if is_inventoryish_url(url):
        return "inventory"
    if "bulletin" in u:
        return "bulletin"
    if any(h in u for h in ("catalog", "catalogue", "college-catalog", "course-catalog")):
        return "catalog"
    return "other"


def compute_signals(url: str, parsed: ParsedPage, base_netloc: str) -> PageSignals:
    soft404 = 1 if is_soft_404(parsed) else 0
    thin = 1 if page_is_thin(parsed) else 0

    control_unique_structured = sum(1 for pat in CONTROL_PATS.values() if pat.search(parsed.corpus_structured))
    struct_hits = [k for k, pat in STRUCT_PATS.items() if pat.search(parsed.corpus_any)]

    title_blob = normalize_unicode_text(" ".join([parsed.html_title, parsed.h1]))
    year_hit = 1 if (YEAR_REGEX.search(url) or YEAR_REGEX.search(title_blob)) else 0
    undergrad_title_boost = 1 if UNDERGRAD_TITLE_BOOST.search(title_blob) else 0

    u_low = url.lower()
    grad_penalty = 1 if (GRAD_URL_PENALTY.search(u_low) or re.search(r"\bgraduate\b", title_blob, re.I)) else 0
    profile_penalty = 1 if (PROFILE_URL_PENALTY.search(u_low) or re.search(r"\bfaculty directory\b", title_blob, re.I)) else 0
    irrelevant_penalty = 1 if IRRELEVANT_URL_PENALTY.search(u_low) else 0
    admissions_penalty = 1 if ADMISSIONS_URL_PENALTY.search(u_low) else 0
    archive_penalty = 1 if ARCHIVE_URL_PENALTY.search(u_low) else 0
    awards_penalty = 1 if AWARDS_URL_PENALTY.search(u_low) else 0
    compliance_penalty = 1 if COMPLIANCE_URL_PENALTY.search(u_low) else 0

    hub_sig, hub_counts = hubness_signature(parsed, base_netloc=base_netloc)
    canonical_hub = 1 if is_canonical_hub_url(url) else 0

    subsite_like = 1 if is_subsite_like(url, base_netloc=base_netloc) else 0
    hard_subsite_block = 1 if is_hard_subsite_block(url, base_netloc=base_netloc) else 0
    too_specific = 1 if (is_too_specific_url(url) and not is_canonical_hub_url(url)) else 0

    js_hub_hint = 1 if (getattr(parsed, "js_hint", 0) and (len(getattr(parsed, "json_blob", "")) >= 200)) else 0
    return PageSignals(
        url=url,
        tier=page_tier(url),
        soft404=soft404,
        thin=thin,
        canonical_hub=canonical_hub,
        hub_sig=hub_sig,
        hub_counts=hub_counts,
        struct_hits=struct_hits,
        control_unique_structured=control_unique_structured,
        year_hit=year_hit,
        undergrad_title_boost=undergrad_title_boost,
        grad_penalty=grad_penalty,
        profile_penalty=profile_penalty,
        irrelevant_penalty=irrelevant_penalty,
        admissions_penalty=admissions_penalty,
        archive_penalty=archive_penalty,
        awards_penalty=awards_penalty,
        compliance_penalty=compliance_penalty,
        subsite_like=subsite_like,
        hard_subsite_block=hard_subsite_block,
        too_specific=too_specific,
        js_hub_hint=js_hub_hint,
    )


def canonical_bonus_allowed(sig: PageSignals) -> bool:
    if sig.soft404:
        return False
    if sig.thin and sig.hub_sig == 0 and len(sig.struct_hits) < 2:
        return False
    return True


def score_inventory(sig: PageSignals, subsite_penalty: int) -> int:
    if sig.soft404:
        return -10_000
    
    # hard subsite blocks are automatic losses
    if getattr(sig, "hard_subsite_block", 0):
        return -10_000    

    score = 0

    # tier base
    if sig.tier == "inventory":
        score += 240
    elif sig.tier == "catalog":
        score += 130
    elif sig.tier == "bulletin":
        score += 85
    else:
        score += 40

    # hubness + structured hits
    score += 85 * sig.hub_sig
    score += 10 * len(sig.struct_hits)
    score += 4 * sig.control_unique_structured

    # canonical hub bonus (gated)
    if sig.canonical_hub and canonical_bonus_allowed(sig):
        score += 160

    # undergrad title boost
    if sig.undergrad_title_boost:
        score += 35

    # year boost only for catalog/bulletin pages
    if sig.year_hit and sig.tier in ("catalog", "bulletin") and is_catalogish_url(sig.url):
        score += 70

    # penalties (URL/title/H1)
    if sig.profile_penalty:
        score -= 260
    if sig.grad_penalty:
        score -= 170
    if sig.irrelevant_penalty:
        score -= 260
    if sig.admissions_penalty:
        score -= 160
    if sig.archive_penalty:
        score -= 220
    if sig.awards_penalty:
        score -= 200
    if sig.compliance_penalty:
        score -= 150

    # configurable subsite penalty
    if sig.subsite_like:
        score -= int(subsite_penalty)

    # explicit penalty for "too specific" department/program pages
    if getattr(sig, "too_specific", 0):
        score -= int(TOO_SPECIFIC_PENALTY)    

    # broad detail penalty so hubs win
    if looks_like_detail_url(sig.url):
        if sig.hub_sig <= 1:
            score -= 220
        elif sig.hub_sig == 2:
            score -= 90
        else:
            score -= 25

    # thin penalty (C2: JS hubs can be thin app shells; barely penalize)
    if sig.thin and sig.hub_sig == 0:
        if getattr(sig, "js_hub_hint", 0) and is_inventoryish_url(sig.url):
            score -= 20
        else:
            score -= 180

    return score
# ============================================================
# Candidate generation
# ============================================================

def looks_like_js_hub(parsed: ParsedPage, url: str) -> bool:
    """Detect JS-rendered program hubs early so we can expand via sitemap sooner."""
    if not page_is_thin(parsed):
        return False

    u = (url or "").lower()
    host = (urlparse(url).netloc or "").lower()
    inventoryish = is_inventoryish_url(url) or (host.startswith("majors.") or host.startswith("programs."))
    if not inventoryish and "find-your-program" not in u:
        return False

    # Typical app shells have few usable anchors and often embed data in JSON
    if len(parsed.links) > 60:
        return False
    if getattr(parsed, "js_hint", 0):
        return True
    if len(getattr(parsed, "json_blob", "")) >= 200:
        return True
    return False



def url_prior(url: str, base_netloc: str, subsite_penalty: int) -> int:
    u = url.lower()
    s = 0

    if any(h in u for h in INVENTORY_URL_HINTS):
        s += 200
    if any(k in u for k in ("academics", "academic", "program", "major", "minor", "depart", "areas-of-study", "fields-of-study")):
        s += 90
    if any(k in u for k in ("catalog", "catalogue")):
        s += 35
    if "bulletin" in u:
        s -= 20

    if IRRELEVANT_URL_PENALTY.search(u):
        s -= 260
    if ADMISSIONS_URL_PENALTY.search(u):
        s -= 160
    if PROFILE_URL_PENALTY.search(u):
        s -= 320
    if ARCHIVE_URL_PENALTY.search(u):
        s -= 140
    if COMPLIANCE_URL_PENALTY.search(u):
        s -= 90
    if AWARDS_URL_PENALTY.search(u):
        s -= 160
    if GRAD_URL_PENALTY.search(u):
        s -= 180

    # hard subsite blocks should be avoided aggressively
    if is_hard_subsite_block(url, base_netloc=base_netloc):
        s -= 10_000
    if is_subsite_like(url, base_netloc=base_netloc):
        s -= int(subsite_penalty)

    if is_canonical_hub_url(url):
        s += 160

    return s


def prioritize_links(links: List[Tuple[str, str]], base_netloc: str, subsite_penalty: int) -> List[Tuple[str, str]]:
    def rank(item: Tuple[str, str]) -> int:
        a_text, u = item
        if not same_domain(u, base_netloc):
            return 10_000

        s = url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty)

        at = (a_text or "").lower()
        if re.search(r"\b(majors|minors|programs|departments|areas|fields|undergraduate)\b", at):
            s += 45
        if re.search(r"\b(admissions|apply|visit)\b", at):
            s -= 90
        if re.search(r"\b(faculty|staff|directory)\b", at):
            s -= 140

        return -s

    return sorted(links, key=rank)


# ============================================================
# Candidate structure
# ============================================================
@dataclass
class CandidatePage:
    url: str
    score: int
    sig: PageSignals
    parsed: ParsedPage


def get_parsed_page(url: str, session: requests.Session) -> ParsedPage:
    html = fetch_text_cached(url, session=session)
    return parse_html_to_parsedpage(html, page_url=url)


def fetch_and_score(url: str, session: requests.Session, base_netloc: str, subsite_penalty: int) -> Optional[CandidatePage]:
    try:
        parsed = get_parsed_page(url, session=session)
        sig = compute_signals(url, parsed, base_netloc=base_netloc)
        sc = score_inventory(sig, subsite_penalty=subsite_penalty)
        return CandidatePage(url=url, score=sc, sig=sig, parsed=parsed)
    except Exception:
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)


# ============================================================
# Parallel batch fetch+score helper
# ============================================================
def fetch_and_score_many(
    urls: List[str],
    base_netloc: str,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    """Fetch+score a batch of URLs concurrently (one Session per thread)."""
    # de-dupe while preserving order
    seen: Set[str] = set()
    uniq: List[str] = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            uniq.append(u)

    if workers <= 1:
        out: List[CandidatePage] = []
        s = requests.Session()
        for u in uniq:
            cand = fetch_and_score(u, session=s, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
            if cand is not None:
                out.append(cand)
        return out

    out: List[CandidatePage] = []

    def _task(u: str) -> Optional[CandidatePage]:
        # IMPORTANT: call get_thread_session() inside the worker thread
        return fetch_and_score(
            u,
            session=get_thread_session(),
            base_netloc=base_netloc,
            subsite_penalty=subsite_penalty,
        )

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_task, u) for u in uniq]
        for fut in as_completed(futs):
            cand = fut.result()
            if cand is not None:
                out.append(cand)
    return out


def push_topk(topk: List[Tuple[int, int, CandidatePage]], cand: CandidatePage, k: int) -> None:
    if len(topk) < k:
        heapq.heappush(topk, (cand.score, len(topk), cand))
        return
    if cand.score > topk[0][0]:
        heapq.heapreplace(topk, (cand.score, topk[0][1], cand))


def topk_sorted(topk: List[Tuple[int, int, CandidatePage]]) -> List[CandidatePage]:
    """Return candidates sorted by score (desc), with an optional canonical-hub tie-break."""
    cands = [t[2] for t in sorted(topk, key=lambda x: x[0], reverse=True)]

    # C: Prefer canonical hubs more aggressively when tie-ish.
    if cands:
        best_score = cands[0].score

        canonical_best: Optional[CandidatePage] = None
        for c in cands:
            if c.sig.canonical_hub and (best_score - c.score) <= CANONICAL_TIE_THRESHOLD:
                if canonical_best is None or c.score > canonical_best.score:
                    canonical_best = c

        if canonical_best is not None and canonical_best is not cands[0]:
            cands = [canonical_best] + [x for x in cands if x is not canonical_best]

    return cands


# ============================================================
# Candidate generation
# ============================================================
def build_year_candidates(base_url: str) -> List[str]:
    out: List[str] = []
    for y in YEAR_TOKENS:
        for tpl in YEAR_PATH_TEMPLATES:
            out.append(urljoin(base_url, tpl.format(y=y)))

    seen: Set[str] = set()
    uniq: List[str] = []
    for u in out:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq


def build_subdomain_roots(base_url: str) -> List[str]:
    netloc = urlparse(base_url).netloc
    rd = root_domain(netloc)
    if "." not in rd:
        return []
    return [
        f"https://majors.{rd}/",
        f"https://programs.{rd}/",
        f"https://catalog.{rd}/",
        f"https://bulletin.{rd}/",
    ]


# ============================================================
# Program title extraction
# ============================================================
def norm_title_key(s: str) -> str:
    s = normalize_unicode_text(s).lower()
    s = re.sub(r"[^\w\s&-]", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_title_key_loose(s: str) -> str:
    """Looser normalization for concordance (handles '&' vs 'and', generic suffix words, etc.)."""
    s = normalize_unicode_text(s).lower()
    s = s.replace("&", " and ")
    # Drop very common generic words that frequently cause false non-matches
    s = re.sub(r"\b(program|department|major|minor|concentration|certificate|track|option)\b", " ", s)
    s = re.sub(r"\b(the|of|in|for|and|to|a|an)\b", " ", s)
    s = re.sub(r"[^\w\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ============================================================
# Synonym mapping: used ONLY for partial concordance + change tracking
# ============================================================

# Keep this intentionally conservative: we want to merge obvious equivalences without
# over-normalizing distinct programs.
# NOTE: applied ONLY in partial concordance scoring / alignment.
SYNONYM_REPLACEMENTS: List[Tuple[re.Pattern, str]] = [
    # ampersand handled earlier too, but keep for safety
    (re.compile(r"&", re.I), " and "),

    # common short forms
    (re.compile(r"\bdept\.?\b", re.I), " department "),
    (re.compile(r"\bprog\.?\b", re.I), " program "),

    # african-american variants
    (re.compile(r"\bafrican\s*[-–—]\s*american\b", re.I), " african american "),
    (re.compile(r"\bafro\s*[-–—]?\s*american\b", re.I), " african american "),

    # black studies / africana common equivalences (very conservative)
    (re.compile(r"\bblack\s+studies\b", re.I), " africana studies "),

    # gender/women's apostrophe variants
    (re.compile(r"\bwomen['’]s\b", re.I), " women "),
    (re.compile(r"\bgender\s+and\s+sexuality\b", re.I), " gender sexuality "),

    # common misspelling seen in some sources
    (re.compile(r"\bcarribbean\b", re.I), " caribbean "),
]


def apply_synonym_map(s: str) -> str:
    """Apply conservative synonym replacements.

    This is ONLY used for partial concordance scoring / alignment and does not affect
    strict or loose exact concordance columns.
    """
    s = normalize_unicode_text(s)
    if not s:
        return ""
    out = s
    for rx, repl in SYNONYM_REPLACEMENTS:
        out = rx.sub(repl, out)
    out = normalize_unicode_text(out)
    return out


def _content_tokens(s: str) -> Set[str]:
    """Tokenize a title into content-bearing tokens for partial matching.

    IMPORTANT: synonym mapping is applied by the caller (partial-only). This function
    assumes it is receiving the already-preprocessed string.
    """
    s = norm_title_key_loose(s)
    if not s:
        return set()
    toks = [t for t in re.split(r"[\s_-]+", s) if t]
    toks = [t for t in toks if len(t) >= 3]  # remove tiny junk tokens
    return set(toks)


def _overlap_coeff(a: Set[str], b: Set[str]) -> float:
    """|A∩B| / min(|A|,|B|)"""
    if not a or not b:
        return 0.0
    inter = len(a.intersection(b))
    denom = float(min(len(a), len(b)))
    return inter / denom if denom > 0 else 0.0


def best_partial_title_match(a_titles: List[str], b_titles: List[str], use_synonyms: bool = False) -> Tuple[float, str, str]:
    """Return best partial match score and the (a_title, b_title) pair.

    Score is max(token overlap coefficient, SequenceMatcher ratio) on loose-normalized strings.

    If `use_synonyms` is True, apply `apply_synonym_map()` BEFORE loose normalization.
    This is used ONLY for partial concordance + change tracking.
    """
    best_score = 0.0
    best_a = ""
    best_b = ""

    for a in (a_titles or []):
        a_raw = normalize_unicode_text(a)
        if not a_raw:
            continue
        a_pre = apply_synonym_map(a_raw) if use_synonyms else a_raw
        a_loose = norm_title_key_loose(a_pre)
        a_tok = _content_tokens(a_pre)

        for b in (b_titles or []):
            b_raw = normalize_unicode_text(b)
            if not b_raw:
                continue
            b_pre = apply_synonym_map(b_raw) if use_synonyms else b_raw
            b_loose = norm_title_key_loose(b_pre)
            b_tok = _content_tokens(b_pre)

            tok_score = _overlap_coeff(a_tok, b_tok)
            seq_score = difflib.SequenceMatcher(None, a_loose, b_loose).ratio() if (a_loose and b_loose) else 0.0
            score = max(tok_score, seq_score)

            if score > best_score:
                best_score = score
                best_a = a_raw
                best_b = b_raw

    return best_score, best_a, best_b


def partial_concordance(
    a_titles: List[str],
    b_titles: List[str],
    threshold: float = 0.80,
    use_synonyms: bool = False,
) -> Tuple[int, str]:
    """Binary + detail string for partial concordance between two title lists.

    If `use_synonyms` is True, apply synonym mapping ONLY during partial scoring.
    """
    score, a_best, b_best = best_partial_title_match(a_titles, b_titles, use_synonyms=use_synonyms)
    ok = int(score >= float(threshold))
    detail = ""
    if ok and a_best and b_best:
        detail = f'inst="{a_best}" ~ other="{b_best}" score={score:.2f}'
    return ok, detail


# ============================================================
# Change tracking (E): align program lists + report adds/losses
# ============================================================

def _pair_score(a: str, b: str, use_synonyms: bool = True) -> float:
    """Compute a pairwise similarity score for alignment."""
    a_raw = normalize_unicode_text(a)
    b_raw = normalize_unicode_text(b)
    if not a_raw or not b_raw:
        return 0.0

    a_pre = apply_synonym_map(a_raw) if use_synonyms else a_raw
    b_pre = apply_synonym_map(b_raw) if use_synonyms else b_raw

    a_loose = norm_title_key_loose(a_pre)
    b_loose = norm_title_key_loose(b_pre)

    a_tok = _content_tokens(a_pre)
    b_tok = _content_tokens(b_pre)

    tok_score = _overlap_coeff(a_tok, b_tok)
    seq_score = difflib.SequenceMatcher(None, a_loose, b_loose).ratio() if (a_loose and b_loose) else 0.0
    return float(max(tok_score, seq_score))


def align_title_lists(
    a_titles: List[str],
    b_titles: List[str],
    threshold: float = 0.80,
    use_synonyms: bool = True,
) -> Tuple[List[Tuple[str, str, float]], List[str], List[str]]:
    """Greedy 1:1 alignment between two title lists.

    Returns:
      matches: list of (a_title, b_title, score) with score>=threshold
      a_only: titles only in A (unmatched)
      b_only: titles only in B (unmatched)

    IMPORTANT: synonym mapping is applied ONLY if `use_synonyms` is True.
    """
    a_list = [normalize_unicode_text(x) for x in (a_titles or []) if normalize_unicode_text(x)]
    b_list = [normalize_unicode_text(x) for x in (b_titles or []) if normalize_unicode_text(x)]

    if not a_list and not b_list:
        return [], [], []

    # score all pairs
    pairs: List[Tuple[float, int, int]] = []
    for i, a in enumerate(a_list):
        for j, b in enumerate(b_list):
            sc = _pair_score(a, b, use_synonyms=use_synonyms)
            if sc >= float(threshold):
                pairs.append((sc, i, j))

    pairs.sort(key=lambda x: x[0], reverse=True)

    used_a: Set[int] = set()
    used_b: Set[int] = set()
    matches: List[Tuple[str, str, float]] = []

    for sc, i, j in pairs:
        if i in used_a or j in used_b:
            continue
        used_a.add(i)
        used_b.add(j)
        matches.append((a_list[i], b_list[j], float(sc)))

    a_only = [a for i, a in enumerate(a_list) if i not in used_a]
    b_only = [b for j, b in enumerate(b_list) if j not in used_b]

    return matches, a_only, b_only


def format_alignment_pairs(matches: List[Tuple[str, str, float]]) -> str:
    if not matches:
        return ""
    parts = []
    for a, b, sc in matches:
        parts.append(f'a="{a}" ~ b="{b}" score={sc:.2f}')
    return "|".join(parts)

def _split_program_name_field(s: str) -> List[str]:
    """Split a program-name field into candidate names (handles pipes/newlines/semicolons)."""
    s = normalize_unicode_text(s or "")
    if not s:
        return []
    parts: List[str] = [s]
    for sep in ["|", ";", "\n", "\r"]:
        tmp: List[str] = []
        for p in parts:
            if sep in p:
                tmp.extend([x.strip() for x in p.split(sep) if x.strip()])
            else:
                tmp.append(p)
        parts = tmp

    # Only split on commas when it looks like a list (avoid breaking names like
    # "African, Black & Caribbean Studies")
    tmp2: List[str] = []
    for p in parts:
        if p.count(",") >= 2:
            tmp2.extend([x.strip() for x in p.split(",") if x.strip()])
        else:
            tmp2.append(p)
    return [normalize_unicode_text(x) for x in tmp2 if x]


def _title_key_set(titles: List[str]) -> Set[str]:
    """Normalized key set for exact-match concordance checks."""
    return {norm_title_key(t) for t in (titles or []) if norm_title_key(t)}


def _field_key_set(field_val: str) -> Set[str]:
    """Normalized key set from a scalar program-name field (e.g., 2013_program_name)."""
    return {norm_title_key(x) for x in _split_program_name_field(field_val) if norm_title_key(x)}


def _any_exact_concordance(a: Set[str], b: Set[str]) -> int:
    """1 if any exact normalized title matches exist between sets; else 0."""
    if not a or not b:
        return 0
    return int(bool(a.intersection(b)))

def looks_like_program_title(s: str, progtitle_strictness: int, context: str = "hub") -> bool:
    """
    Strictness guidance:
      1-2: permissive (still requires TARGET_ANY)
      3: current default behavior
      4-5: stricter (prefer explicit program-ish keywords; tighter filters)
    context: "hub" (default) or "follow" (detail)
    """
    s_norm = normalize_unicode_text(s)
    if len(s_norm) < 6 or len(s_norm) > 140:
        return False

    s_low = s_norm.lower()

    # D1: obvious junk rejects
    if URL_LIKE_TEXT.search(s_norm):
        return False
    if "toggle" in s_low:
        return False

    # always require the target token family (keeps extraction scoped)
    if TARGET_ANY_REGEX.search(s_norm) is None:
        return False

    if COURSE_CODE.match(s_norm):
        return False
    if COURSE_WORDS.search(s_norm):
        return False

    # D1: sentence-like / prose-ish rejects
    if ". " in s_norm and len(s_norm) >= 40:
        return False
    if ": " in s_norm and len(s_norm) >= 55:
        # allow short degree-style labels, reject long descriptive clauses
        if PROGRAM_KEYWORDS.search(s_norm) is None and not re.search(r"\b(B\.?A\.?|B\.?S\.?|BA|BS)\b", s_norm, re.I):
            return False
    if PROSE_VERBS.search(s_norm) and len(s_norm) >= 45:
        # follow/detail pages are stricter; hubs can still reject obvious prose
        if context == "follow" or progtitle_strictness >= 3:
            return False

    # D1: long multi-clause strings are usually prose or navigation
    if len(s_norm) > 90:
        comma_ct = s_norm.count(",")
        if comma_ct >= 2 and PROGRAM_KEYWORDS.search(s_norm) is None:
            return False

    # NOTE: negative-context words (news/events/etc.) are handled primarily in
    # clean_program_titles() where we can condition on program intent.
    # Keep only a very strict early reject.
    if TITLE_NEGATIVE_CONTEXT.search(s_norm) and progtitle_strictness >= 5:
        return False

    # D2: follow/detail pages must show stronger "program" intent
    if context == "follow":
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    if PERSON_NAME_ONLY.match(s_norm):
        if progtitle_strictness >= 3 and not PROGRAM_KEYWORDS.search(s_norm):
            return False
        if progtitle_strictness >= 5:
            return False

    if progtitle_strictness >= 4:
        if PROGRAM_KEYWORDS.search(s_norm) is None and "studies" not in s_low:
            return False

    return True
def is_program_detailish_url(url: str) -> bool:
    """Tight allowlist for one-hop follow: only follow likely program/major detail pages."""
    path = (urlparse(url).path or "")
    if any(p.search(path) for p in LISTING_LINK_PATTERNS):
        return True
    # also allow common majors-programs hubs/details
    if re.search(r"/majors-programs/[^/]+", path, re.I):
        return True
    return False


def token_matches_from_text(txt: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for key, pat in TARGET_TOKEN_REGEX.items():
        out[key] = sorted(set(m.group(0) for m in pat.finditer(txt)))
    return out


# ============================================================
# PDF fallback (best-effort)
# ============================================================
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    try:
        from io import BytesIO
        from pdfminer.high_level import extract_text  # type: ignore
        return normalize_unicode_text(extract_text(BytesIO(pdf_bytes)))
    except Exception:
        return ""


def find_yearish_major_pdf_links(parsed: ParsedPage) -> List[str]:
    pdfs: List[str] = []
    for a_text, u in parsed.links:
        ul = (u or "").lower()
        at = (a_text or "").lower()
        if not ul:
            continue
        if not PDF_LINK_REGEX.search(ul):
            continue
        if YEAR_REGEX.search(ul) or YEAR_REGEX.search(at) or re.search(r"\b(major|minor|program|undergrad)\b", at):
            pdfs.append(u)

    out: List[str] = []
    seen: Set[str] = set()
    for u in pdfs:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


# ============================================================
# Aggregate extraction from inventory + alts (+ limited 1-hop)
# ============================================================
def aggregate_outputs(
    pages: List[CandidatePage],
    base_url: str,
    session: requests.Session,
    progtitle_strictness: int,
    workers: int,
) -> Tuple[
    int, Set[str], Dict[str, int],
    Dict[str, List[str]],
    List[str],
    List[str],
]:
    base_netloc = urlparse(base_url).netloc

    struct_hits_union: Set[str] = set()
    control_hit_cols = {k: 0 for k in CONTROL_TERMS}
    any_control_found = 0

    tokens_agg: Dict[str, Set[str]] = {k: set() for k in TARGET_TOKEN_REGEX.keys()}
    titles: Set[str] = set()

    pdf_hits: List[str] = []

    # from chosen pages
    for cand in pages:
        any_control_found = max(any_control_found, 1 if cand.sig.control_unique_structured > 0 else 0)
        struct_hits_union.update(cand.sig.struct_hits)

        for ck, pat in CONTROL_PATS.items():
            if pat.search(cand.parsed.corpus_structured):
                control_hit_cols[ck] = 1

        tm = token_matches_from_text(cand.parsed.corpus_any)
        for k, vals in tm.items():
            tokens_agg[k].update(vals)

        for txt in (cand.parsed.anchor_texts + cand.parsed.anchor_attr_texts + cand.parsed.title_like):
            if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="hub"):
                titles.add(txt)

    # 1-hop follow
    follow_urls: List[str] = []
    for cand in pages:
        for a_text, u in cand.parsed.links:
            if not u or not same_domain(u, base_netloc):
                continue
            ul = u.lower()
            if IRRELEVANT_URL_PENALTY.search(ul) or ADMISSIONS_URL_PENALTY.search(ul) or PROFILE_URL_PENALTY.search(ul):
                continue
            if "graduate" in ul or "grad" in ul:
                continue
            # D3: follow only if anchor text looks like a title, OR URL looks like a program detail page
            if looks_like_program_title(a_text or "", progtitle_strictness=progtitle_strictness, context="hub") or is_program_detailish_url(u):
                follow_urls.append(u)

    # sitemap assist if best looks thin
    if pages and page_is_thin(pages[0].parsed):
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        follow_urls.extend(sitemap_candidate_urls(all_urls))

    seen_u: Set[str] = set()
    uniq_follow: List[str] = []
    for u in follow_urls:
        if u not in seen_u:
            seen_u.add(u)
            uniq_follow.append(u)
        if len(uniq_follow) >= ONE_HOP_MAX_LINKS_TOTAL:
            break

    # Fetch a limited number of follow URLs (optionally in parallel)
    follow_batch = uniq_follow[:ONE_HOP_MAX_FETCHES]

    def _safe_fetch_parse(u: str) -> Optional[ParsedPage]:
        try:
            # Use per-thread sessions when parallel
            s = get_thread_session() if workers and workers > 1 else session
            return get_parsed_page(u, session=s)
        except Exception:
            return None
        finally:
            # Keep some politeness throttling even when parallel
            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    if workers and workers > 1 and len(follow_batch) > 1:
        max_w = min(int(workers), 8, len(follow_batch))
        with ThreadPoolExecutor(max_workers=max_w) as ex:
            futs = [ex.submit(_safe_fetch_parse, u) for u in follow_batch]
            for fut in as_completed(futs):
                parsed = fut.result()
                if not parsed:
                    continue

                tm = token_matches_from_text(parsed.corpus_any)
                for k, vals in tm.items():
                    tokens_agg[k].update(vals)

                for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                    if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                        titles.add(txt)
    else:
        for u in follow_batch:
            parsed = _safe_fetch_parse(u)
            if not parsed:
                continue

            tm = token_matches_from_text(parsed.corpus_any)
            for k, vals in tm.items():
                tokens_agg[k].update(vals)

            for txt in (parsed.anchor_texts + parsed.anchor_attr_texts + parsed.title_like):
                if txt and looks_like_program_title(txt, progtitle_strictness=progtitle_strictness, context="follow"):
                    titles.add(txt)

    # PDF fallback (rare)
    if ENABLE_PDF_FALLBACK:
        pdf_fetches = 0
        for cand in pages:
            if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                break
            for pdf_url in find_yearish_major_pdf_links(cand.parsed):
                if pdf_fetches >= PDF_MAX_FETCHES_PER_INSTITUTION:
                    break
                try:
                    pdf_bytes = fetch_bytes_cached(pdf_url, session=session)
                    txt = extract_text_from_pdf_bytes(pdf_bytes)
                    if not txt:
                        continue
                    if re.search(r"\b(major|minor|program|undergraduate)\b", txt, re.I):
                        tm = token_matches_from_text(txt)
                        for k, vals in tm.items():
                            tokens_agg[k].update(vals)
                        pdf_hits.append(pdf_url)
                        pdf_fetches += 1
                except Exception:
                    continue

    # Postprocess titles: split breadcrumbs, salvage boilerplate, drop course/news/social noise,
    # and select the best variant per normalized key.
    titles_out = clean_program_titles(list(titles), progtitle_strictness=progtitle_strictness)
    tokens_out = {k: sorted(v) for k, v in tokens_agg.items()}

    return any_control_found, struct_hits_union, control_hit_cols, tokens_out, titles_out, pdf_hits


#
# ============================================================
# Discovery: find candidate pages
# ============================================================
def find_candidates_for_institution(
    base_url: str,
    session: requests.Session,
    subsite_penalty: int,
    workers: int,
) -> List[CandidatePage]:
    base_netloc = urlparse(base_url).netloc
    tried: Set[str] = set()
    topk: List[Tuple[int, int, CandidatePage]] = []

    sitemap_cache: Optional[List[str]] = None

    def load_sitemap_candidates() -> List[str]:
        nonlocal sitemap_cache
        if sitemap_cache is not None:
            return sitemap_cache
        all_urls = fetch_sitemap_urls(base_url, session=session, base_netloc=base_netloc)
        sitemap_cache = sitemap_candidate_urls(all_urls)
        return sitemap_cache

    def consider(u: str) -> Optional[CandidatePage]:
        if not u or u in tried:
            return None
        tried.add(u)
        cand = fetch_and_score(u, session=session, base_netloc=base_netloc, subsite_penalty=subsite_penalty)
        if cand is None:
            return None
        push_topk(topk, cand, k=TOP_K_ALTS)
        return cand

    # Phase 0
    consider(base_url)

    # ------------------------------------------------------------
    # Phase 1 (parallelizable): explicit paths + year/catalog + home links
    # ------------------------------------------------------------
    batch_urls: List[str] = []

    for p in INVENTORY_PATHS[:PHASE1_MAX_INVENTORY_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    batch_urls.extend(build_year_candidates(base_url)[:PHASE1_MAX_CATALOG_CANDIDATES])

    for p in CATALOG_PATHS[:PHASE1_MAX_CATALOG_CANDIDATES]:
        batch_urls.append(urljoin(base_url, p))

    # homepage links require parsing homepage once (sequential), but the fetched URLs can be scored in batch
    try:
        home = get_parsed_page(base_url, session=session)
        
        # C1: If the homepage/programs page is a JS hub (thin app shell), expand candidates via sitemap immediately
        if looks_like_js_hub(home, base_url):
            try:
                batch_urls.extend(load_sitemap_candidates()[:80])
            except Exception:
                pass
        home_links = [(t, u) for (t, u) in home.links if same_domain(u, base_netloc)]
        home_links = prioritize_links(home_links, base_netloc, subsite_penalty=subsite_penalty)
        for _, u in home_links[:PHASE1_MAX_HOMEPAGE_LINKS]:
            batch_urls.append(u)
    except Exception:
        pass

    # score the batch in parallel (if workers>1)
    batch_cands = fetch_and_score_many(
        batch_urls,
        base_netloc=base_netloc,
        subsite_penalty=subsite_penalty,
        workers=workers,
    )
    for cand in batch_cands:
        tried.add(cand.url)
        push_topk(topk, cand, k=TOP_K_ALTS)

    # C2: If ANY early candidate looks like a JS-rendered hub, expand via sitemap NOW
    # (even if the thin hub didn't win the scoring yet).
    try:
        if sitemap_cache is None:
            js_hub_seen = any(looks_like_js_hub(c.parsed, c.url) for c in batch_cands)
            if js_hub_seen:
                sm_urls = load_sitemap_candidates()[:120]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
    except Exception:
        pass

    # Optional: if best so far looks weak, pull some sitemap candidates (also batchable)
    if topk:
        best_so_far = max(topk, key=lambda x: x[0])[2]
        if (best_so_far.sig.thin or best_so_far.sig.hub_sig == 0) and is_inventoryish_url(best_so_far.url):
            try:
                sm_urls = load_sitemap_candidates()[:50]
                sm_cands = fetch_and_score_many(
                    sm_urls,
                    base_netloc=base_netloc,
                    subsite_penalty=subsite_penalty,
                    workers=workers,
                )
                for cand in sm_cands:
                    tried.add(cand.url)
                    push_topk(topk, cand, k=TOP_K_ALTS)
            except Exception:
                pass

    # Best-first crawl
    pq: List[Tuple[int, int, str]] = []
    seen: Set[str] = set()

    def pq_push(u: str, depth: int) -> None:
        if not u or u in seen:
            return
        if not same_domain(u, base_netloc):
            return
        seen.add(u)
        heapq.heappush(pq, (-url_prior(u, base_netloc=base_netloc, subsite_penalty=subsite_penalty), depth, u))

    pq_push(base_url, 0)
    fetched = 0
    while pq and fetched < PHASE2_BESTFIRST_MAX_PAGES:
        _, depth, u = heapq.heappop(pq)
        cand = consider(u)
        fetched += 1
        if depth >= PHASE2_MAX_DEPTH:
            continue
        try:
            parsed = cand.parsed if cand is not None else get_parsed_page(u, session=session)
            links = [(t, lu) for (t, lu) in parsed.links if same_domain(lu, base_netloc)]
            links = prioritize_links(links, base_netloc, subsite_penalty=subsite_penalty)
            for _, lu in links[:70]:
                pq_push(lu, depth + 1)
        except Exception:
            pass

    # Subdomain roots
    for root in build_subdomain_roots(base_url):
        c_root = consider(root)
        if not c_root:
            continue
        root_is_good = (c_root.sig.hub_sig >= 1) or (len(c_root.sig.struct_hits) >= 2) or (not c_root.sig.thin)
        if root_is_good and not c_root.sig.soft404:
            continue
        for p in ("/programs", "/majors", "/majors-minors", "/academics/programs", "/departments-and-programs"):
            consider(urljoin(root, p))

    return topk_sorted(topk)


# ============================================================
# Reason string
# ============================================================
def inventory_reason(c: CandidatePage) -> str:
    s = c.sig
    flags = []
    if s.grad_penalty:
        flags.append("grad")
    if s.profile_penalty:
        flags.append("profile")
    if s.irrelevant_penalty:
        flags.append("irrelevant")
    if s.admissions_penalty:
        flags.append("admissions")
    if s.archive_penalty:
        flags.append("archive")
    if s.awards_penalty:
        flags.append("awards")
    if s.compliance_penalty:
        flags.append("compliance")
    if s.year_hit:
        flags.append("yearHit")
    if s.thin:
        flags.append("thin")
    if s.soft404:
        flags.append("soft404")
    if s.subsite_like:
        flags.append("subsiteLike")
    if getattr(s, "hard_subsite_block", 0):
        flags.append("hardSubsiteBlock")
    if getattr(s, "too_specific", 0):
        flags.append("tooSpecific")

    return ";".join([
        f"score={c.score}",
        f"tier={s.tier}",
        f"hubSig={s.hub_sig}",
        f"listingLinks={s.hub_counts.get('listing_links', 0)}",
        f"slugDiv={s.hub_counts.get('slug_diversity', 0)}",
        f"structHits={len(s.struct_hits)}",
        f"ctrlStructured={s.control_unique_structured}",
        f"canonicalHub={s.canonical_hub}",
        ("flags=" + ",".join(flags)) if flags else "flags=",
    ])


# ============================================================
# CLI
# ============================================================
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--input", type=str, default=str(DEFAULT_INPUT_PATH))
    ap.add_argument("--output", type=str, default=str(DEFAULT_OUTPUT_PATH))

    # If omitted, default to TEST_HEAD_N (notebook-friendly). Use --head 0 for full run.
    ap.add_argument("--head", type=int, default=None, help="If >0, limit rows for testing; if omitted defaults to TEST_HEAD_N")

    ap.add_argument("--out-suffix", type=str, default="", help="Append to output filename stem")
    ap.add_argument("--subsite-penalty", type=int, default=80)
    ap.add_argument("--progtitle-strictness", type=int, default=3)

    # Parallelism (keep small to be polite)
    ap.add_argument(
        "--workers",
        type=int,
        default=DEFAULT_WORKERS,
        help="Parallel fetch workers for candidate scoring (1 = off). Try 4.",
    )

    ap.add_argument(
        "--debug-jsonlen",
        action="store_true",
        help="Debug: print json_blob length (chars) for the best candidate per school",
    )

    # IMPORTANT: ignore Jupyter/ipykernel injected args like --f=...
    args, _unknown = ap.parse_known_args(argv)

    if args.head is None:
        args.head = TEST_HEAD_N

    # sanity clamp
    if args.workers is None or args.workers < 1:
        args.workers = 1
    if args.workers > MAX_WORKERS:
        args.workers = MAX_WORKERS

    return args


def apply_out_suffix(out_path: Path, suffix: str) -> Path:
    if not suffix:
        return out_path
    return out_path.with_name(out_path.stem + suffix + out_path.suffix)


# ============================================================
# Main
# ============================================================
def main() -> None:
    args = parse_args()
    input_path = Path(args.input)
    output_path = apply_out_suffix(Path(args.output), args.out_suffix)

    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    required = {UNITID_COL, NAME_COL, WEB_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {input_path}: {sorted(missing)}")

    if args.head and args.head > 0:
        df = df.head(args.head).copy()

    session = requests.Session()
    out_rows: List[Dict[str, str]] = []

    for i, row in df.iterrows():
        unitid = str(row.get(UNITID_COL, "")).strip()
        name = str(row.get(NAME_COL, "")).strip()
        base_url = ensure_scheme(str(row.get(WEB_COL, "")).strip())

        print(f"[{i+1}/{len(df)}] unitid={unitid} | {name}")
        # Reset per-institution status tally for fetch-status tagging (403-aware)
        reset_status_tally()

        best_inventory_url = "N/A"
        best_inventory_reason = ""
        alt_candidate_urls = ""
        url_tag_value = ""

        any_control_found = 0
        struct_hits_union: Set[str] = set()
        any_struct_found = 0

        control_hit_cols = {f"found_{k}": 0 for k in CONTROL_TERMS}

        token_matches: Dict[str, List[str]] = {k: [] for k in OUTPUT_TOKEN_BUCKETS}

        program_titles: List[str] = []
        program_title_count = 0
        any_potential_AfrAmr_found = 0
        pdf_hits: List[str] = []
        # CollegeVine (3rd-party truth source)
        college_vine_site = 0
        college_vine_control_hits = 0
        college_vine_url = ""
        college_vine_program_title_count = 0
        college_vine_program_titles: List[str] = []

        try:
            if base_url:
                candidates = find_candidates_for_institution(
                    base_url,
                    session=session,
                    subsite_penalty=args.subsite_penalty,
                    workers=args.workers,
                )

                if candidates:
                    best = candidates[0]
                    best_inventory_url = best.url
                    best_inventory_reason = inventory_reason(best)
                    alt_candidate_urls = "|".join(c.url for c in candidates[:TOP_K_ALTS])
                    if getattr(args, "debug_jsonlen", False):
                        jb_len = len(getattr(best.parsed, "json_blob", "") or "")
                        print(f"  [debug-jsonlen] best_url={best.url} | json_blob_chars={jb_len}")

                    any_control_found, struct_hits_union, control_hits, tokens, titles, pdf_hits = aggregate_outputs(
                        pages=candidates[:TOP_K_ALTS],
                        base_url=base_url,
                        session=session,
                        progtitle_strictness=args.progtitle_strictness,
                        workers=args.workers,
                    )

                    any_struct_found = 1 if struct_hits_union else 0

                    for k, v in control_hits.items():
                        control_hit_cols[f"found_{k}"] = int(v)

                    for k in OUTPUT_TOKEN_BUCKETS:
                        token_matches[k] = tokens.get(k, [])

                    program_titles = titles
                    program_title_count = len(program_titles)
                    any_potential_AfrAmr_found = 1 if program_title_count > 0 else 0
        except Exception:
            # keep row, but leave defaults
            pass

        # Freeze institution-site fetch-status tag BEFORE any CollegeVine calls (so CV doesn't pollute tally)
        url_tag_value = fetch_status_tag(best_inventory_url)
        # Reset status tally so any CV fetches won't affect institution url_tag
        reset_status_tally()

        # CollegeVine (separate fetch so it doesn't affect the university-site fetch-status tally)
        try:
            (
                college_vine_url,
                college_vine_site,
                college_vine_control_hits,
                college_vine_program_title_count,
                college_vine_program_titles,
            ) = fetch_collegevine_majors_page(
                school_name=name,
                session=session,
                progtitle_strictness=args.progtitle_strictness,
            )
        except Exception:
            pass

        rec: Dict[str, str] = dict(row)
        # Normalize *all* incoming text columns (fix mojibake like "‚Äê" / "Äê")
        # so baseline fields such as `2013_program_name` are clean in the output.
        try:
            for _k, _v in list(rec.items()):
                if isinstance(_v, str):
                    rec[_k] = normalize_unicode_text(_v)
                elif _v is None:
                    rec[_k] = ""
                else:
                    rec[_k] = normalize_unicode_text(str(_v))
        except Exception:
            pass

        rec["best_guess_inventory_url"] = best_inventory_url
        rec["best_guess_inventory_reason"] = best_inventory_reason
        rec["url_tag"] = url_tag_value
        rec["alt_candidate_urls"] = alt_candidate_urls

        rec["any_control_found"] = str(any_control_found)
        rec["struct_hits_union"] = "|".join(sorted(struct_hits_union)) if struct_hits_union else ""
        rec[">0_struct_hits_found"] = str(any_struct_found)

        for k in OUTPUT_TOKEN_BUCKETS:
            vals = token_matches.get(k, [])
            rec[f"{k}_matches"] = "|".join(vals) if vals else ""

        rec["program_title_count"] = str(program_title_count)
        rec["program_titles_found"] = "|".join(program_titles) if program_titles else ""
        rec["any_potential_AfrAmr_found"] = str(any_potential_AfrAmr_found)

        rec["pdf_hits"] = "|".join(pdf_hits) if pdf_hits else ""

        # CollegeVine truth-source columns
        rec["college_vine_site"] = str(int(college_vine_site))
        rec["college_vine_url"] = (college_vine_url or "")
        # Status label for CV control-threshold outcome (keeps `college_vine_site` as “CV resolved”).
        if int(college_vine_site):
            rec["college_vine_ctrl_status"] = (
                "passed_ctrl_thresh" if int(college_vine_control_hits) >= int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB)
                else "failed_ctrl_thresh"
            )
        else:
            rec["college_vine_ctrl_status"] = "cv_unresolved"
        rec[f"college_vine_controls_ge_{int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB)}"] = str(
            int(college_vine_control_hits >= int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB))
        )

        # Ensure CollegeVine program titles are cleaned consistently before storage + concordance
        college_vine_program_titles_clean = clean_program_titles(
            list(college_vine_program_titles) if college_vine_program_titles else [],
            progtitle_strictness=args.progtitle_strictness,
        )
        rec["college_vine_program_title_count"] = str(len(college_vine_program_titles_clean))
        rec["college_vine_program_titles_found"] = (
            "|".join(college_vine_program_titles_clean) if college_vine_program_titles_clean else ""
        )

        # Combined indicator: any target-program matches from institution crawl OR CollegeVine
        # (Use cleaned CV titles and extracted institution titles, both of which are scoped to TARGET_ANY.)
        rec["any_program_matches_inst_or_collegevine"] = str(
            int(bool(program_titles) or bool(college_vine_program_titles_clean))
        )

        # Concordance checks (strict exact match on normalized title keys)
        inst_keys = _title_key_set(program_titles)
        cv_keys = _title_key_set(college_vine_program_titles_clean)
        p2013_keys = _field_key_set(str(rec.get("2013_program_name", "")))

        rec["concord_institution_vs_collegevine"] = str(_any_exact_concordance(inst_keys, cv_keys))
        rec["concord_institution_vs_2013_program_name"] = str(_any_exact_concordance(inst_keys, p2013_keys))
        rec["concord_collegevine_vs_2013_program_name"] = str(_any_exact_concordance(cv_keys, p2013_keys))

        # Concordance checks (loose exact match on loose-normalized title keys)
        # Uses `norm_title_key_loose()` via `_title_key_set_loose()` / `_field_key_set_loose()` so
        # common formatting differences (e.g., '&' vs 'and', generic suffix words) don't block matches.
        inst_keys_loose = _title_key_set_loose(program_titles)
        cv_keys_loose = _title_key_set_loose(college_vine_program_titles_clean)
        p2013_keys_loose = _field_key_set_loose(str(rec.get("2013_program_name", "")))

        rec["concord_loose_institution_vs_collegevine"] = str(_any_exact_concordance(inst_keys_loose, cv_keys_loose))
        rec["concord_loose_institution_vs_2013_program_name"] = str(_any_exact_concordance(inst_keys_loose, p2013_keys_loose))
        rec["concord_loose_collegevine_vs_2013_program_name"] = str(_any_exact_concordance(cv_keys_loose, p2013_keys_loose))

        # Partial concordance checks (loose normalization + token overlap / string similarity)
        # These help when names differ by punctuation, '&' vs 'and', generic suffix words (Program/Department), etc.
        inst_titles_for_match = list(program_titles or [])
        cv_titles_for_match = list(college_vine_program_titles_clean or [])
        p2013_titles_for_match = _split_program_name_field(str(rec.get("2013_program_name", "")))

        inst_vs_cv_partial, inst_vs_cv_partial_detail = partial_concordance(
            inst_titles_for_match,
            cv_titles_for_match,
            threshold=0.80,
            use_synonyms=True,
        )
        inst_vs_2013_partial, inst_vs_2013_partial_detail = partial_concordance(
            inst_titles_for_match,
            p2013_titles_for_match,
            threshold=0.80,
            use_synonyms=True,
        )
        cv_vs_2013_partial, cv_vs_2013_partial_detail = partial_concordance(
            cv_titles_for_match,
            p2013_titles_for_match,
            threshold=0.80,
            use_synonyms=True,
        )

        rec["partial_concord_institution_vs_collegevine"] = str(inst_vs_cv_partial)
        rec["partial_concord_institution_vs_2013_program_name"] = str(inst_vs_2013_partial)
        rec["partial_concord_collegevine_vs_2013_program_name"] = str(cv_vs_2013_partial)

        rec["partial_match_detail_institution_vs_collegevine"] = inst_vs_cv_partial_detail
        rec["partial_match_detail_institution_vs_2013_program_name"] = inst_vs_2013_partial_detail
        rec["partial_match_detail_collegevine_vs_2013_program_name"] = cv_vs_2013_partial_detail

        # ============================================================
        # Change tracking columns (adds/losses/renames) using synonym-aware partial alignment
        # NOTE: synonym mapping is applied ONLY here (partial-only), not in strict/loose exact columns.
        # ============================================================

        p2013_present = int(bool(normalize_unicode_text(str(rec.get("2013_program_name", "")))))
        inst_present = int(bool(program_titles))
        cv_present = int(bool(college_vine_program_titles_clean))

        # Align 2013 -> Institution extracted titles
        m_i13, only_2013_i, only_inst = align_title_lists(
            a_titles=p2013_titles_for_match,
            b_titles=inst_titles_for_match,
            threshold=0.80,
            use_synonyms=True,
        )

        # Align 2013 -> CollegeVine titles
        m_cv13, only_2013_cv, only_cv = align_title_lists(
            a_titles=p2013_titles_for_match,
            b_titles=cv_titles_for_match,
            threshold=0.80,
            use_synonyms=True,
        )

        # Row-level cases
        if not p2013_present and inst_present:
            inst_vs_2013_case = "2013_missing__new_present"
        elif p2013_present and not inst_present:
            inst_vs_2013_case = "2013_present__new_missing"
        elif not p2013_present and not inst_present:
            inst_vs_2013_case = "2013_missing__new_missing"
        else:
            inst_vs_2013_case = "2013_present__new_present"

        if not p2013_present and cv_present:
            cv_vs_2013_case = "2013_missing__new_present"
        elif p2013_present and not cv_present:
            cv_vs_2013_case = "2013_present__new_missing"
        elif not p2013_present and not cv_present:
            cv_vs_2013_case = "2013_missing__new_missing"
        else:
            cv_vs_2013_case = "2013_present__new_present"

        # Lightweight context: if 2013 is present but new is missing, flag likely extraction blockage
        likely_blocked_or_bad_hub = int(
            (p2013_present == 1 and inst_present == 0)
            and (
                str(url_tag_value or "").strip() != ""
                or str(rec.get("controls_sufficiency", "")).startswith("warning_")
                or int(any_control_found) == 0
            )
        )

        rec["2013_program_present"] = str(p2013_present)
        rec["inst_programs_present"] = str(inst_present)
        rec["collegevine_programs_present"] = str(cv_present)

        rec["change_case_institution_vs_2013"] = inst_vs_2013_case
        rec["change_case_collegevine_vs_2013"] = cv_vs_2013_case

        # Program-level changes (note each)
        rec["inst_added_programs_vs_2013"] = "|".join(only_inst) if only_inst else ""
        rec["inst_missing_2013_programs_vs_inst"] = "|".join(only_2013_i) if only_2013_i else ""
        rec["inst_match_pairs_vs_2013"] = format_alignment_pairs(m_i13)

        rec["collegevine_added_programs_vs_2013"] = "|".join(only_cv) if only_cv else ""
        rec["collegevine_missing_2013_programs_vs_collegevine"] = "|".join(only_2013_cv) if only_2013_cv else ""
        rec["collegevine_match_pairs_vs_2013"] = format_alignment_pairs(m_cv13)

        # Helpful flag when 2013 exists but new is missing (loss vs extraction issue)
        rec["likely_blocked_or_bad_hub_when_2013_present_new_missing"] = str(likely_blocked_or_bad_hub)

        # (concordance columns handled above)

        # total controls found (proxy for "how many majors we saw")
        total_controls_found = int(sum(int(v) for v in control_hit_cols.values()))
        rec["total_controls_found"] = str(total_controls_found)

        # sufficiency tag based on MIN_CONTROL_HITS_FOR_CONFIDENT_HUB
        if total_controls_found < int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB):
            rec["controls_sufficiency"] = f"warning_{int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB)}_MajorsFound"
        else:
            rec["controls_sufficiency"] = "sufficient majors"

        rec.update({k: str(v) for k, v in control_hit_cols.items()})
        out_rows.append(rec)

        time.sleep(SLEEP_BETWEEN_INSTITUTIONS_SEC)

    out_df = pd.DataFrame(out_rows)

    ordered_front = [
        UNITID_COL, NAME_COL, WEB_COL,
        "best_guess_inventory_url", "best_guess_inventory_reason",
        "url_tag",
        "alt_candidate_urls",
        "any_control_found", "struct_hits_union", ">0_struct_hits_found",
        "program_title_count", "program_titles_found",
        *[f"{k}_matches" for k in OUTPUT_TOKEN_BUCKETS],
        "any_potential_AfrAmr_found",
        # "any_program_matches_inst_or_collegevine",  # removed from here to put at end
        "pdf_hits",
        "total_controls_found",
        "controls_sufficiency",
    ]

    front_cols = [c for c in ordered_front if c in out_df.columns]
    control_cols = [f"found_{k}" for k in CONTROL_TERMS if f"found_{k}" in out_df.columns]

    # CollegeVine columns should appear near the end (before concordance), per spec.
    cv_cols_all = [
        "college_vine_site",
        "college_vine_url",
        "college_vine_ctrl_status",
        f"college_vine_controls_ge_{int(MIN_CONTROL_HITS_FOR_CONFIDENT_HUB)}",
        "college_vine_program_title_count",
        "college_vine_program_titles_found",
    ]
    cv_cols = [c for c in cv_cols_all if c in out_df.columns]

    # Partial concordance columns should appear near the end (after CollegeVine, before loose and strict exact concordance).
    partial_cols_all = [
        "partial_concord_institution_vs_collegevine",
        "partial_concord_institution_vs_2013_program_name",
        "partial_concord_collegevine_vs_2013_program_name",
        "partial_match_detail_institution_vs_collegevine",
        "partial_match_detail_institution_vs_2013_program_name",
        "partial_match_detail_collegevine_vs_2013_program_name",
        # E-tracking columns (appended immediately after the partial_match_detail_* columns)
        "2013_program_present",
        "inst_programs_present",
        "collegevine_programs_present",
        "change_case_institution_vs_2013",
        "change_case_collegevine_vs_2013",
        "inst_added_programs_vs_2013",
        "inst_missing_2013_programs_vs_inst",
        "inst_match_pairs_vs_2013",
        "collegevine_added_programs_vs_2013",
        "collegevine_missing_2013_programs_vs_collegevine",
        "collegevine_match_pairs_vs_2013",
        "likely_blocked_or_bad_hub_when_2013_present_new_missing",
    ]
    partial_cols = [c for c in partial_cols_all if c in out_df.columns]

    # Loose exact concordance columns should appear near the end (after partial, before strict exact concordance).
    loose_cols_all = [
        "concord_loose_institution_vs_collegevine",
        "concord_loose_institution_vs_2013_program_name",
        "concord_loose_collegevine_vs_2013_program_name",
    ]
    loose_cols = [c for c in loose_cols_all if c in out_df.columns]

    # Concordance columns must be the final 3 columns.
    concord_cols_all = [
        "concord_institution_vs_collegevine",
        "concord_institution_vs_2013_program_name",
        "concord_collegevine_vs_2013_program_name",
    ]
    concord_cols = [c for c in concord_cols_all if c in out_df.columns]

    remaining_cols = [
        c for c in out_df.columns
        if c not in set(front_cols + control_cols + cv_cols + partial_cols + loose_cols + concord_cols)
    ]

    # Move this combined indicator to the very end for easier scanning.
    tail_cols_all = ["any_program_matches_inst_or_collegevine"]
    tail_cols = [c for c in tail_cols_all if c in out_df.columns]

    # Final column order: loose exact concordance just before strict concordance, per spec.
    base_order = front_cols + control_cols + remaining_cols + cv_cols + partial_cols + loose_cols + concord_cols
    # Ensure tail columns are last (and not duplicated).
    base_order = [c for c in base_order if c not in set(tail_cols)]
    out_df = out_df[base_order + tail_cols]

    out_df.to_csv(output_path, index=False)
    print(f"\nWrote: {output_path} | rows,cols = {out_df.shape}")
    print(out_df.head())


if __name__ == "__main__":
    main()

[1/100] unitid=188429 | Adelphi University
[2/100] unitid=138600 | Agnes Scott College
[3/100] unitid=168546 | Albion College
[4/100] unitid=164465 | Amherst College
[5/100] unitid=143084 | Augustana College
[6/100] unitid=189097 | Barnard College
[7/100] unitid=132471 | Barry University
[8/100] unitid=160977 | Bates College
[9/100] unitid=156295 | Berea College
[10/100] unitid=142115 | Boise State University
[11/100] unitid=164924 | Boston College
[12/100] unitid=164988 | Boston University
[13/100] unitid=161004 | Bowdoin College
[14/100] unitid=201441 | Bowling Green State University-Main Campus
[15/100] unitid=143358 | Bradley University
[16/100] unitid=165015 | Brandeis University
[17/100] unitid=217156 | Brown University
[18/100] unitid=211273 | Bryn Mawr College
[19/100] unitid=211291 | Bucknell University
[20/100] unitid=110422 | California Polytechnic State University-San Luis Obispo
[21/100] unitid=110529 | California State Polytechnic University-Pomona
[22/100] unitid=110538 

/Users/thoma/Documents/Professional_files/Consulting/AfrAmr_studies_Harvard/venv/lib/python3.9/site-packages/bs4/__init__.py:473: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self._feed()
/var/folders/0_/vt5npj3x5wq9q_4bzlj28tqw0000gn/T/ipykernel_7104/3118025207.py:1868: XMLParsedAsHTMLWarning: It looks like you're usi

[32/100] unitid=110495 | California State University-Stanislaus
[33/100] unitid=173258 | Carleton College
[34/100] unitid=201645 | Case Western Reserve University
[35/100] unitid=128771 | Central Connecticut State University
[36/100] unitid=234827 | Central Washington University
[37/100] unitid=156408 | Centre College
[38/100] unitid=144005 | Chicago State University
[39/100] unitid=217864 | Citadel Military College of South Carolina
[40/100] unitid=138947 | Clark Atlanta University
[41/100] unitid=217882 | Clemson University
[42/100] unitid=202134 | Cleveland State University
[43/100] unitid=153144 | Coe College
[44/100] unitid=161086 | Colby College
[45/100] unitid=190099 | Colgate University
[46/100] unitid=126678 | Colorado College
[47/100] unitid=153162 | Cornell College
[48/100] unitid=190415 | Cornell University
[49/100] unitid=181002 | Creighton University
[50/100] unitid=190512 | CUNY Bernard M Baruch College
[51/100] unitid=190549 | CUNY Brooklyn College
[52/100] unitid=19056